# Reconstrucción experimental sin el modelo final — AGE (`tweet-age-2026-puj`)

**Competencia:** `tweet-age-2026-puj` (Kaggle, `NLP_Course_PUJ`)  
**Programa:** Maestría en Inteligencia Artificial — PLN, Pontificia Universidad Javeriana  
**Equipo:** Grupo 2 - PLN (`carlosgarnica`)  
**Métrica oficial:** macro-F1 sobre 6 rangos de edad ordenados

---

## Alcance de este cuaderno

Este cuaderno reconstruye **desde el corpus** la evidencia experimental del proyecto: auditoría y EDA,
la fuga por *near-duplicados* y los folds canónicos, las representaciones, los 14 miembros, el
meta-modelo V12, V19/V20/V29/V30, la matriz de test y la segunda partición. Documenta además las
falsaciones V44–V52 y la frontera 2026.

**No contiene el modelo final**: los componentes neuronales, los consensos V48/V56, su entrenamiento y
la *submission* están en `modelo_final_entrenamiento.ipynb`; la entrega consolidada, en
`99_entrega_consolidado.ipynb`.


## Resumen ejecutivo

Este cuaderno reconstruye **desde el corpus** la evidencia experimental del proyecto: la auditoría y el
análisis exploratorio, la fuga por *near-duplicados* y los folds agrupados, las representaciones y el
meta-modelo de catorce miembros, y el linaje V19/V20/V29/V30 hasta la matriz de test. Documenta además
las falsaciones V44–V52 y la frontera 2026.

El **modelo final** (V48, *leaderboard* 0.63724; y V56, CV 0.6189929) y su *submission* se construyen en
`modelo_final_entrenamiento.ipynb`; la entrega consolidada con las cifras verificadas está en
`99_entrega_consolidado.ipynb`.


## Guía de lectura

El cuaderno sigue la metodología **CRISP-ML** hasta la evaluación de la evidencia:

- **Fase 1 — Comprensión del problema:** la tarea, la métrica y las restricciones.
- **Fase 2 — Comprensión de los datos:** auditoría, EDA, *near-duplicados* y folds.
- **Fase 3 — Preparación y representación:** representaciones, features, miembros y meta-modelo V12.
- **Fase 4 — Modelado:** V18–V29 documentado; V30, la regla *plug-in*, los fracasos seleccionados y las
  falsaciones V44–V52; frontera 2026 documentada (sin re-ejecutar sus componentes).
- **Fase 5 — Evaluación:** protocolo de validación y análisis matemáticos y de calibración.

El modelo final y su *submission* se ejecutan en el cuaderno aparte.


## Protocolo de reproducibilidad

Este notebook es **independiente y determinista**:

1. **Corpus autónomo.** La primera celda localiza los archivos de la competencia (datos adjuntos de Kaggle,
   copias locales o descarga directa vía API oficial). No se clona ningún repositorio.
2. **Cadena embebida.** Todo el código de reconstrucción (representaciones, 14 miembros, features,
   meta-modelo, V20/V30, regla plug-in y redes neuronales) viaja dentro del notebook; cada módulo es un
   porte fiel del script original del proyecto, con su referencia en el docstring.
3. **Protocolo reproducible.** La partición canónica de validación (5 folds agrupados por clústeres de
   near-duplicados) se genera en la propia ejecución con la regla documentada —`StratifiedGroupKFold`,
   `shuffle=True`, semilla 42— y se verifica por propiedades; los pesos de las reglas de decisión son
   constantes legibles del protocolo. Ejecutar dos veces produce exactamente los mismos valores.
4. **Trazabilidad.** Cada experimento reporta su resultado con la fuente documental del proyecto
   (registro de corridas, actas de decisión e informes), y el notebook verifica que lo reconstruido
   coincide con lo documentado.

**Tiempo de ejecución completo:** del orden de 4 horas (entre 3 y 5) en CPU de 8 hilos + GPU (los
componentes neuronales usan GPU si está disponible; sin ella corren en CPU con diferencias del orden
de la precisión de máquina, documentadas en la sección 6.2).

## 0.1 Entorno de ejecución

In [ ]:
# Versiones exactas del protocolo experimental: la particion canonica y los reajustes de
# arboles se generaron con scikit-learn 1.8.0 y LightGBM 4.6.0. Se fijan al arranque para
# que la ejecucion en Kaggle reproduzca las cifras documentadas (no las del entorno local).
import subprocess as _subprocess
import sys as _sys

_subprocess.run([_sys.executable, "-m", "pip", "install", "-q",
                 "scikit-learn==1.8.0", "lightgbm==4.6.0"], check=True)

# Entorno: semillas fijas, estilo de figuras y utilidades basicas del notebook.
import hashlib
import json
import os
import random
import sys
import time
import warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

SEMILLA = 42
os.environ["PYTHONHASHSEED"] = str(SEMILLA)
random.seed(SEMILLA)
np.random.seed(SEMILLA)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

CLASSES = ["13-17", "18-24", "25-34", "35-49", "50-64", "65-xx"]

import sklearn  # noqa: E402  (se importa aqui solo para reportar la version)

print("Entorno listo | Python:", sys.version.split()[0],
      "| numpy:", np.__version__, "| pandas:", pd.__version__,
      "| scikit-learn:", sklearn.__version__)
try:
    import torch
    print("torch:", torch.__version__, "| GPU disponible:", torch.cuda.is_available())
except ImportError:
    print("torch no disponible: los componentes neuronales correran en CPU numpy equivalente")

## 0.2 Resolución autónoma del corpus

Esta sección localiza los tres archivos oficiales (`age_train.csv`, `age_test.csv` y la plantilla de envío).
El orden de búsqueda es: datos adjuntos de la competencia en Kaggle, copias locales habituales y, si no
aparecen, descarga directa con la API oficial (requiere credenciales de Kaggle en el entorno). Todo ocurre
dentro del notebook, sin dependencias externas.

In [ ]:
# Resolucion autonoma del corpus: Kaggle adjunto -> copias locales -> API oficial.
def _buscar_en(base, nombres):
    """Devuelve la primera ruta existente de una lista de nombres dentro de una base."""
    for nombre in nombres:
        ruta = base / nombre
        if ruta.is_file():
            return ruta
    return None


def _descarga_api(destino):
    """Descarga y descomprime el corpus oficial via la API de Kaggle (solo stdlib)."""
    import urllib.request
    import zipfile

    token = os.environ.get("KAGGLE_API_TOKEN", "").strip()
    if not token:
        credencial = Path.home() / ".kaggle" / "kaggle.json"
        if credencial.is_file():
            datos = json.loads(credencial.read_text(encoding="utf-8"))
            token = str(datos.get("token", "")).strip()
    if not token:
        raise RuntimeError(
            "No hay corpus local ni credenciales de Kaggle (KAGGLE_API_TOKEN o ~/.kaggle/kaggle.json)")
    url = "https://www.kaggle.com/api/v1/competitions/data/download-all/tweet-age-2026-puj"
    peticion = urllib.request.Request(url, headers={"Authorization": "Bearer " + token})
    destino.mkdir(parents=True, exist_ok=True)
    archivo = destino / "tweet-age-2026-puj.zip"
    with urllib.request.urlopen(peticion, timeout=300) as respuesta:
        archivo.write_bytes(respuesta.read())
    with zipfile.ZipFile(archivo) as comprimido:
        comprimido.extractall(destino)
    return destino


def resolver_corpus():
    """Devuelve (train, test, submission) localizando o descargando el corpus oficial."""
    bases = []
    if Path("/kaggle/input").exists():
        bases.extend(sorted(p for p in Path("/kaggle/input").iterdir() if p.is_dir()))
    # Kaggle monta las competiciones en /kaggle/input/competitions/<slug>/
    if Path("/kaggle/input/competitions").exists():
        bases.extend(sorted(p for p in Path("/kaggle/input/competitions").iterdir()
                             if p.is_dir()))
    # Kaggle monta las competiciones en /kaggle/input/competitions/<slug>/
    if Path("/kaggle/input/competitions").exists():
        bases.extend(sorted(p for p in Path("/kaggle/input/competitions").iterdir()
                             if p.is_dir()))
    bases.extend([Path("data/raw"), Path("../data/raw"), Path.cwd()])
    for base in bases:
        train = _buscar_en(base, ("age_train.csv", "Age_train.csv"))
        test = _buscar_en(base, ("age_test.csv", "Age_test.csv"))
        envio = _buscar_en(base, ("age_submission.csv", "sample_submission.csv"))
        if train and test:
            return train, test, envio
    print("Corpus no encontrado localmente: descargando via API de Kaggle...")
    base = _descarga_api(Path("data/raw"))
    train = _buscar_en(base, ("age_train.csv",))
    test = _buscar_en(base, ("age_test.csv",))
    envio = _buscar_en(base, ("age_submission.csv",))
    if not (train and test):
        raise FileNotFoundError("La descarga no produjo age_train.csv / age_test.csv")
    return train, test, envio


RUTA_TRAIN, RUTA_TEST, RUTA_ENVIO = resolver_corpus()
print("train :", RUTA_TRAIN)
print("test  :", RUTA_TEST)
print("envio :", RUTA_ENVIO if RUTA_ENVIO else "(se generara desde los ids de test)")

In [ ]:
# Carga del corpus con la convencion exacta del proyecto (separador ';', utf-8-sig, texto limpio).
train = pd.read_csv(RUTA_TRAIN, sep=";", encoding="utf-8-sig")
test = pd.read_csv(RUTA_TEST, sep=";", encoding="utf-8-sig")
train["text"] = train["text"].fillna("").astype(str).str.strip()
test["text"] = test["text"].fillna("").astype(str).str.strip()

assert train.shape == (169336, 3), f"train inesperado: {train.shape}"
assert test.shape == (112892, 2), f"test inesperado: {test.shape}"
assert list(train.columns) == ["id", "text", "age_range"], train.columns

y = train["age_range"].astype(str).to_numpy()
clase = np.array([CLASSES.index(valor) for valor in y])

print(f"train: {len(train):,} filas | test: {len(test):,} filas")
print("distribucion de clases (train):")
for etiqueta, cuenta in train["age_range"].value_counts().reindex(CLASSES).items():
    print(f"  {etiqueta}: {cuenta:,} ({cuenta / len(train) * 100:.2f}%)")

## 0.3 La cadena de reconstrucción (código embebido)

Todo el código de reconstrucción viaja dentro de este notebook: cada módulo es un porte
fiel del script original del proyecto (la referencia exacta aparece en su docstring).
No hay dependencias externas: con el corpus descargado, la cadena se reconstruye entera.
Los módulos se presentan en orden de dependencias.

### Módulo `comun.py`

In [ ]:
"""Utilidades s de la cadena AGE (porte fiel del proyecto KAGGLE_AGE_2026_PUJ).

Regla de oro: cada funcion es un PORTE LITERAL del original citado; no se mejora ni se
reinterpreta nada. ASCII puro. Sin efectos en import.
"""

from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd

CLASSES = ["13-17", "18-24", "25-34", "35-49", "50-64", "65-xx"]
SEED = 42

CHAR_300K = {"analyzer": "char_wb", "ngram_range": (3, 6), "min_df": 5,
             "sublinear_tf": True, "max_features": 300_000, "lowercase": True}
CHAR_700K = {"analyzer": "char_wb", "ngram_range": (3, 6), "min_df": 3,
             "sublinear_tf": True, "max_features": 700_000, "lowercase": True}
MENTION = {"analyzer": "word", "ngram_range": (1, 1), "min_df": 2,
           "sublinear_tf": True, "lowercase": True}
HASHTAG = {"analyzer": "word", "ngram_range": (1, 1), "min_df": 2,
           "sublinear_tf": True, "lowercase": True}
WORD = {"analyzer": "word", "ngram_range": (1, 2), "min_df": 3,
        "sublinear_tf": True, "max_features": 300_000, "lowercase": True}

CLASS_MIDPOINTS = {"13-17": 15.0, "18-24": 21.0, "25-34": 29.5, "35-49": 42.0,
                   "50-64": 57.0, "65-xx": 70.0}

ADJACENT_PAIRS = [("13-17", "18-24"), ("18-24", "25-34"), ("25-34", "35-49"),
                  ("35-49", "50-64"), ("50-64", "65-xx")]

SPACES_RE = re.compile(r"\s+")


def collapse_spaces(series: pd.Series) -> pd.Series:
    """Colapsa espacios repetidos y recorta extremos (src/preprocessing.py:98-100)."""
    return series.fillna("").astype(str).str.replace(SPACES_RE, " ", regex=True).str.strip()


def tokens_of(series: pd.Series, pattern: str) -> pd.Series:
    """Extrae tokens por patron y los une en minusculas (src/pipeline.py:75-78)."""
    return collapse_spaces(series.str.findall(pattern).apply(lambda values: " ".join(values).lower()))


def cargar_train(raiz: Path) -> pd.DataFrame:
    """Lee age_train.csv como data.load_train(): sep ';', utf-8-sig, texto sin padding."""
    marco = pd.read_csv(raiz / "data" / "raw" / "age_train.csv", sep=";", encoding="utf-8-sig")
    marco["text"] = marco["text"].fillna("").astype(str).str.strip()
    return marco


def cargar_test(raiz: Path) -> pd.DataFrame:
    """Lee age_test.csv con la misma convencion."""
    marco = pd.read_csv(raiz / "data" / "raw" / "age_test.csv", sep=";", encoding="utf-8-sig")
    marco["text"] = marco["text"].fillna("").astype(str).str.strip()
    return marco


def cargar_folds(raiz: Path) -> np.ndarray:
    """fold_idx canonico (protocolo congelado): 169.336 valores 0-4.

    En los spikes se lee del carril de artefactos; el notebook embebe este vector.
    """
    return np.load(raiz / "artifacts" / "p33_scores" / "todos" / "fold_idx.npy")


def to_scores(raw: np.ndarray, classes, probabilistic: bool) -> np.ndarray:
    """Reordena columnas al orden canonico y aplica log a las probabilidades (src/pipeline.py:89-94)."""
    orden = [list(classes).index(etiqueta) for etiqueta in CLASSES]
    scores = raw[:, orden]
    return np.log(np.clip(scores, 1e-12, None)) if probabilistic else scores


def softmax_ordinal(continuous: np.ndarray) -> np.ndarray:
    """Convierte la prediccion ordinal continua en distribucion suave (src/pipeline.py:81-86)."""
    midpoints = np.array([CLASS_MIDPOINTS[etiqueta] for etiqueta in CLASSES])
    distances = np.abs(continuous[:, None] - midpoints)
    soft = np.exp(-distances / 5.0)
    return soft / soft.sum(axis=1, keepdims=True)

### Módulo `clusters.py`

In [ ]:
"""Clusters de near-duplicados (porte fiel de src/cv.py:33-229).

Construye los clusters de duplicado exacto o near-duplicado sobre el conjunto completo de textos
(train + test, sin usar etiquetas) con *blocking* por tokens raros y union-find. Es el soporte del
protocolo congelado de folds (D-003). ASCII puro. Sin efectos en import.
"""

from __future__ import annotations

import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

#: Longitud minima del texto normalizado para participar en el agrupamiento.
MIN_LEN = 20
#: Umbral de Jaccard sobre conjuntos de tokens para considerar near-duplicado.
JACCARD_THRESHOLD = 0.8
#: Numero de tokens raros usados como clave de bloqueo.
BLOCK_TOKENS = 3
#: Frecuencia documental maxima para que un token sirva como clave de bloqueo.
MAX_DF_FOR_KEY = 200
#: Tope de documentos comparados dentro de un mismo bloque (evita costo cuadratico).
MAX_BLOCK_SIZE = 400

_URL_RE = re.compile(r"https?://\S+|www\.\S+")
_MENTION_RE = re.compile(r"@\w+")
_RT_RE = re.compile(r"^\s*rt\s*:?\s*", flags=re.IGNORECASE)
_NOISE_RE = re.compile("[^0-9a-z\\u00e1\\u00e9\\u00ed\\u00f3\\u00fa\\u00fc\\u00f1 ]+")
_SPACES_RE = re.compile(r"\s+")


def normalize_for_dedup(series: pd.Series) -> pd.Series:
    """Normaliza el texto para detectar duplicados y near-duplicados (src/cv.py:51-73)."""
    text = series.fillna("").astype(str).str.lower()
    text = text.str.replace(_URL_RE, " ", regex=True)
    text = text.str.replace(_MENTION_RE, " ", regex=True)
    text = text.str.replace(_RT_RE, "", regex=True)
    text = text.str.replace(_NOISE_RE, " ", regex=True)
    return text.str.replace(_SPACES_RE, " ", regex=True).str.strip()


def dedup_signature(normalized: pd.Series, chars: int = 60) -> pd.Series:
    """Firma informativa de prefijo (solo metadato, no agrupa; src/cv.py:76-78)."""
    return normalized.str.slice(0, chars)


def jaccard(left: set, right: set) -> float:
    """Similitud de Jaccard entre dos conjuntos de tokens (src/cv.py:81-93)."""
    if not left or not right:
        return 0.0
    if len(left) > len(right):
        left, right = right, left
    if len(left) / len(right) < JACCARD_THRESHOLD and len(left) < JACCARD_THRESHOLD * len(right):
        return 0.0
    intersection = len(left & right)
    if intersection == 0:
        return 0.0
    return intersection / (len(left) + len(right) - intersection)


class _UnionFind:
    """Union-find con compresion de caminos (src/cv.py:96-115)."""

    def __init__(self, size: int) -> None:
        self.parent = list(range(size))

    def find(self, item: int) -> int:
        """Raiz del conjunto de ``item``."""
        root = item
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[item] != root:
            self.parent[item], item = root, self.parent[item]
        return root

    def union(self, left: int, right: int) -> None:
        """Une los conjuntos de ``left`` y ``right``."""
        root_left, root_right = self.find(left), self.find(right)
        if root_left != root_right:
            self.parent[root_right] = root_left


def construir_clusters(textos: pd.Series, min_len: int = MIN_LEN,
                       threshold: float = JACCARD_THRESHOLD,
                       block_tokens: int = BLOCK_TOKENS,
                       max_df_for_key: int = MAX_DF_FOR_KEY,
                       max_block_size: int = MAX_BLOCK_SIZE,
                       pair_budget: int = 20_000_000,
                       stats: dict | None = None) -> pd.DataFrame:
    """Agrupa textos en clusters de duplicado exacto o near-duplicado (src/cv.py:118-229).

    Dos filas caen en el mismo cluster si su texto normalizado es identico o si la similitud de
    Jaccard entre sus conjuntos de tokens es >= ``threshold``. El blocking por los ``block_tokens``
    tokens mas raros evita el costo cuadratico.
    """
    normalized = normalize_for_dedup(textos)
    signature = dedup_signature(normalized)
    normalized_array = normalized.to_numpy()
    valid = normalized.str.len().to_numpy() >= min_len
    union = _UnionFind(len(normalized))

    exact_groups: dict = defaultdict(list)
    for position, key in enumerate(normalized_array):
        if valid[position]:
            exact_groups[key].append(position)
    for positions in exact_groups.values():
        anchor = positions[0]
        for position in positions[1:]:
            union.union(anchor, position)

    token_sets = [frozenset(tokens.split()) for tokens in normalized_array]
    document_frequency: Counter = Counter()
    for tokens in token_sets:
        document_frequency.update(tokens)

    blocks: dict = defaultdict(list)
    for position, tokens in enumerate(token_sets):
        if not valid[position]:
            continue
        rare = sorted((document_frequency[token], token) for token in tokens
                      if document_frequency[token] <= max_df_for_key)
        for _, token in rare[:block_tokens]:
            blocks[token].append(position)

    comparisons = 0
    budget_exhausted = False
    for documents in sorted(blocks.values(), key=len):
        if len(documents) < 2:
            continue
        if len(documents) > max_block_size:
            documents = documents[:max_block_size]
        for left_index in range(len(documents)):
            left = documents[left_index]
            left_tokens = token_sets[left]
            if len(left_tokens) < 2:
                continue
            for right_index in range(left_index + 1, len(documents)):
                right = documents[right_index]
                if union.find(left) == union.find(right):
                    continue
                comparisons += 1
                if comparisons > pair_budget:
                    budget_exhausted = True
                    break
                if jaccard(left_tokens, token_sets[right]) >= threshold:
                    union.union(left, right)
            if budget_exhausted:
                break
        if budget_exhausted:
            break

    roots = np.array([union.find(index) for index in range(len(normalized))])
    _, cluster = np.unique(roots, return_inverse=True)

    if stats is not None:
        stats.update({"blocks": len(blocks), "comparisons": comparisons,
                      "pair_budget_exhausted": budget_exhausted, "threshold": threshold,
                      "max_df_for_key": max_df_for_key, "block_tokens": block_tokens})
    return pd.DataFrame({"norm": normalized_array, "sig": signature.to_numpy(),
                         "cluster": cluster})

### Módulo `dense.py`

In [ ]:
"""Textometria y features estilometricas a nivel de documento (porte literal).

Porte de ``src/textometry.py`` (lineas 1-173) a ``repro/cadena/dense.py``. La funcion
publica ``extraer_textometria`` reproduce exactamente el original: 38 columnas en el
orden ``TEXTOMETRY_FEATURES + STYLE_FEATURES``. Las regex ``_EMOJI_RE`` y ``_URL_RE``
se copian de ``src/preprocessing.py:18,25-32`` como constantes locales del modulo.

Todas las features son deterministas y locales al documento: no aprenden parametros del
corpus, por lo que pueden calcularse sobre train y test sin riesgo de leakage.

Sin dependencias del proyecto (solo stdlib + numpy/pandas), ASCII puro y sin efectos en
import.
"""

from __future__ import annotations

import re

import numpy as np
import pandas as pd

# Emoticonos textuales clasicos (con fronteras de palabra para evitar falsos positivos).
# src/textometry.py:24-26
EMOTICON_RE = re.compile(
    r"(?<![\w:/])(?:[:;=8][-o*']?[\)\]\(\[dDpP/:}{@|]|x[dDpP](?!\w)|XD(?!\w)|<3)(?![\w])")
# Palabras con elongacion (3 o mas repeticiones de un caracter). src/textometry.py:27-28
ELONGATED_RE = re.compile(r"\b\w*(\w)\1{2,}\w*\b", flags=re.UNICODE)
# Secuencias de puntuacion repetida (!!, ??, ..., !?, etc.). src/textometry.py:29-30
MULTI_PUNCT_RE = re.compile(r"[!?.,;:]{2,}")
# Signos de apertura del espanol. src/textometry.py:31-32
OPEN_PUNCT_RE = re.compile(r"[\u00a1\u00bf]")
# Palabras completamente en mayusculas (2+ letras). src/textometry.py:33-34
ALL_CAPS_RE = re.compile(r"\b[A-Z\u00c1\u00c9\u00cd\u00d3\u00da\u00dc\u00d1]{2,}\b")
# Separador de frases. src/textometry.py:35-36
SENTENCE_SPLIT_RE = re.compile(r"[.!?\u2026]+|\n+")

# Lista fija de stopwords en espanol (recurso lexico cerrado, sin parametros aprendidos).
# src/textometry.py:38-48
SPANISH_STOPWORDS = frozenset(["a", "al", "algo", "algunas", "alguno", "algunos", "ambos", "ante", "antes", "aquel", "aquella", "aquellas", "aquellos", "aqui", "asi", "aun", "aunque", "bien", "cada", "casi", "como", "con", "contra", "cual", "cuales", "cuando", "cuanto", "cuenta", "da", "de", "del", "desde", "donde", "dos", "el", "ella", "ellas", "ello", "ellos", "en", "entonces", "entre", "era", "eran", "es", "esa", "esas", "ese", "eso", "esos", "esta", "estaba", "estado", "estan", "estar", "estas", "este", "esto", "estos", "fue", "fueron", "ha", "habia", "han", "hasta", "hay", "la", "las", "le", "les", "lo", "los", "mas", "me", "mi", "mientras", "mis", "mucho", "muchos", "muy", "nada", "ni", "no", "nos", "nosotros", "nuestra", "nuestro", "nuestros", "o", "os", "otra", "otras", "otro", "otros", "para", "pero", "poco", "por", "porque", "que", "quien", "quienes", "se", "sea", "segun", "ser", "si", "sin", "sobre", "solo", "son", "su", "sus", "tal", "tambien", "tanto", "te", "tener", "tiene", "tienen", "todo", "todos", "tu", "tus", "un", "una", "uno", "unos", "usted", "ustedes", "va", "vamos", "van", "varias", "varios", "ver", "y", "ya", "yo"])

# Familias de features (orden exacto del original). src/textometry.py:50-62
TEXTOMETRY_FEATURES = [
    "n_chars", "n_words", "n_unique_words", "ttr", "cttr", "hapax_ratio", "lexical_entropy",
    "avg_word_len", "max_word_len", "std_word_len", "n_sentences", "avg_sentence_len_words",
    "stopword_ratio", "content_ratio", "long_word_ratio",
]
STYLE_FEATURES = [
    "upper_ratio", "digit_ratio", "punct_ratio", "n_excl", "n_quest", "n_open_punct",
    "multi_punct_count", "elongated_count", "all_caps_count", "has_all_caps",
    "n_emoji", "emoji_diversity", "has_emoji", "n_emoticon", "has_emoticon",
    "n_hashtag", "n_mention", "n_url", "is_rt", "starts_mention", "n_repeated_char_runs",
    "punct_per_word", "unique_char_ratio",
]

# Patrones compartidos copiados de src/preprocessing.py (_URL_RE en la linea 18;
# _EMOJI_RE en las lineas 25-32).
_URL_RE = re.compile(r"https?://\S+|www\.\S+")
_EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001F5FF\U0001F600-\U0001F64F\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F\U0001F780-\U0001F7FF\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF\U0001FA00-\U0001FAFF\U0001F1E6-\U0001F1FF"
    "\U00002600-\U000027BF\U00002B00-\U00002BFF\U0000FE0F"
    "]"
)


def _entropy(counts: np.ndarray) -> float:
    """Entropia de Shannon (base 2) de un vector de conteos (src/textometry.py:65-71)."""
    total = counts.sum()
    if total <= 0:
        return 0.0
    probabilities = counts[counts > 0] / total
    return float(-(probabilities * np.log2(probabilities)).sum())


def extraer_textometria(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula las features textometricas y estilometricas de cada documento.

    Porte literal de ``src/textometry.py:83-173`` (helper ``_entropy`` en 65-71). El
    original acepta ``text_col: str = "text"``; en este porte la columna es fija
    ``"text"``, segun la interfaz de ``repro/cadena``. La entrada debe traer el texto
    ya limpio (strip) y la salida conserva el indice de ``df``, con 38 columnas en el
    orden ``TEXTOMETRY_FEATURES + STYLE_FEATURES``.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame con la columna ``text`` (ya sin padding).

    Returns
    -------
    pandas.DataFrame
        Una fila por documento, con las columnas de ``TEXTOMETRY_FEATURES`` y
        ``STYLE_FEATURES``.
    """
    text = df["text"].fillna("").astype(str)
    lowered = text.str.lower()
    words = lowered.str.split()
    # Los emoticonos se cuentan sobre el texto sin URLs: "http://" contiene ":/" y
    # contaminaba el conteo (detectado en el EDA de la Fase 2).
    text_without_urls = text.str.replace(_URL_RE, " ", regex=True)

    n_chars = text.str.len()
    n_words = words.str.len()
    n_unique = words.apply(lambda tokens: len(set(tokens)) if tokens else 0)
    n_hapax = words.apply(
        lambda tokens: int(sum(1 for count in pd.Series(tokens).value_counts() if count == 1))
        if tokens else 0)
    entropy = words.apply(lambda tokens: _entropy(np.unique(tokens, return_counts=True)[1])
                          if tokens else 0.0)
    word_len_mean = words.apply(lambda tokens: float(np.mean([len(t) for t in tokens]))
                                if tokens else 0.0)
    word_len_max = words.apply(lambda tokens: int(max((len(t) for t in tokens), default=0)))
    word_len_std = words.apply(lambda tokens: float(np.std([len(t) for t in tokens]))
                               if tokens else 0.0)

    letters = text.str.count(r"[A-Za-z\u00c1\u00c9\u00cd\u00d3\u00da\u00dc\u00d1\u00e1\u00e9\u00ed\u00f3\u00fa\u00fc\u00f1]")
    upper = text.str.count(r"[A-Z\u00c1\u00c9\u00cd\u00d3\u00da\u00dc\u00d1]")
    digits = text.str.count(r"\d")
    punct = text.str.count(r"[^\w\s]")
    sentences = text.apply(lambda value: len([s for s in SENTENCE_SPLIT_RE.split(value) if s.strip()]))
    stopword_hits = words.apply(lambda tokens: sum(1 for token in tokens
                                                   if token.strip(".,;:!?\u00a1\u00bf\"'()") in SPANISH_STOPWORDS))
    long_words = words.apply(lambda tokens: sum(1 for token in tokens if len(token) >= 8))

    features = pd.DataFrame({
        # --- textometria ---
        "n_chars": n_chars,
        "n_words": n_words,
        "n_unique_words": n_unique,
        "ttr": (n_unique / n_words.replace(0, np.nan)).fillna(0.0),
        "cttr": (n_unique / np.sqrt(2 * n_words.replace(0, np.nan))).fillna(0.0),
        "hapax_ratio": (n_hapax / n_words.replace(0, np.nan)).fillna(0.0),
        "lexical_entropy": entropy,
        "avg_word_len": word_len_mean,
        "max_word_len": word_len_max,
        "std_word_len": word_len_std,
        "n_sentences": sentences,
        "avg_sentence_len_words": (n_words / sentences.replace(0, np.nan)).fillna(0.0),
        "stopword_ratio": (stopword_hits / n_words.replace(0, np.nan)).fillna(0.0),
        "content_ratio": (1 - (stopword_hits / n_words.replace(0, np.nan))).fillna(0.0),
        "long_word_ratio": (long_words / n_words.replace(0, np.nan)).fillna(0.0),
        # --- estilo ---
        "upper_ratio": (upper / letters.replace(0, np.nan)).fillna(0.0),
        "digit_ratio": (digits / n_chars.replace(0, np.nan)).fillna(0.0),
        "punct_ratio": (punct / n_chars.replace(0, np.nan)).fillna(0.0),
        "n_excl": text.str.count("!"),
        "n_quest": text.str.count(r"\?"),
        "n_open_punct": text.str.count(OPEN_PUNCT_RE.pattern),
        "multi_punct_count": text.str.count(MULTI_PUNCT_RE.pattern),
        "elongated_count": text.str.count(ELONGATED_RE.pattern),
        "all_caps_count": text.str.count(ALL_CAPS_RE.pattern),
        "has_all_caps": text.str.contains(ALL_CAPS_RE.pattern).astype(int),
        "n_emoji": text.str.count(_EMOJI_RE.pattern),
        "emoji_diversity": text.apply(
            lambda value: len(set(_EMOJI_RE.findall(value))) if value else 0),
        "has_emoji": text.str.contains(_EMOJI_RE.pattern).astype(int),
        "n_emoticon": text_without_urls.str.count(EMOTICON_RE.pattern),
        "has_emoticon": text_without_urls.str.contains(EMOTICON_RE.pattern).astype(int),
        "n_hashtag": text.str.count("#"),
        "n_mention": text.str.count("@"),
        "n_url": text.str.count(r"https?://|www\."),
        "is_rt": text.str.match(r"^\s*RT\b", case=False).astype(int),
        "starts_mention": text.str.match(r"^\s*@").astype(int),
        "n_repeated_char_runs": text.str.count(r"(\w)\1{1,}"),
        "punct_per_word": (punct / n_words.replace(0, np.nan)).fillna(0.0),
        "unique_char_ratio": (text.apply(lambda value: len(set(value)))
                              / n_chars.replace(0, np.nan)).fillna(0.0),
    })
    return features

### Módulo `structured.py`

In [ ]:
"""Features estructuradas de segunda generacion (87 columnas) - porte literal.

Porte fiel de las seis familias locales de ``src/advanced_features.py``, en el orden de
ensamblado de ``scripts/advanced_02_features.py:38-45,94-104`` (``LOCAL_BUILDERS`` /
``build_local_features``) y de ``scripts/phase3b_recorregido.py:54-69`` (``build_local``):

    1. ``distribution_features``       (src/advanced_features.py:75-111)  -> 21 cols wlen_*/prop_*
    2. ``char_distribution_features``  (src/advanced_features.py:117-153) -> 10 cols char_*
    3. ``lexical_richness_features``   (src/advanced_features.py:159-189) -> 9 cols de riqueza lexica
    4. ``word_shape_features``         (src/advanced_features.py:195-226) -> 10 cols shape_*/repeat_*
    5. ``position_features``           (src/advanced_features.py:232-267) -> 20 cols pos_*/first_*/last_*
    6. ``structure_entropy_features``  (src/advanced_features.py:273-306) -> 17 cols punct_*/entropia

mas la correccion vigente de ``scripts/phase3b_recorregido.py:42-51`` (``ALNUM_RE`` y recalculo
de ``shape_alphanumeric_ratio``), que es la version que produjo el artefacto
``features/age_structured_features.parquet``.

Orden y conteo verificados contra ese parquet: 21 + 10 + 9 + 10 + 20 + 17 = 87 columnas
(73 float64 + 14 int64). Nota: el plan cita 19/18 para las familias 5/6, pero el codigo literal
y el artefacto confirman 20/17 con el mismo total de 87.

Entrada: DataFrame con columna ``text`` limpia (se procesa en orden de fila). Salida: DataFrame
de 87 columnas con el mismo indice que la entrada. Determinista, sin RNG. Solo
numpy/pandas/scipy/stdlib; sin efectos en import. ASCII puro: los caracteres acentuados de las
regex se escriben con escapes unicode (``\\uXXXX``) para mantener el fuente en ASCII sin
cambiar el patron compilado.
"""

from __future__ import annotations

import re
import zlib
from collections import Counter

import numpy as np
import pandas as pd
from scipy import stats

# --- Patrones (src/advanced_features.py:28-37) ------------------------------------------
MENTION_RE = re.compile(r"@(\w+)")
HASHTAG_RE = re.compile(r"#(\w+)")
URL_RE = re.compile(r"https?://\S+|www\.\S+")
#: Alternativa de palabra con vocales acentuadas (mismo patron que el original; el fuente usa
#: escapes unicode para quedar en ASCII).
WORD_RE = re.compile(r"[A-Za-z"
                     "\u00c1\u00c9\u00cd\u00d3\u00da\u00dc\u00d1"
                     "\u00e1\u00e9\u00ed\u00f3\u00fa\u00fc\u00f1"
                     r"0-9]+")
#: Tokenizador propio de tweets. La alternativa de palabra **debe** aceptar mayusculas: si no,
#: las palabras capitalizadas o en mayusculas se fragmentan y las features de forma quedan en
#: cero (bug detectado en la auditoria de alineacion de la Fase 3).
TOKEN_RE = re.compile(r"@\w+|#\w+|https?://\S+|[A-Za-z"
                      "\u00c1\u00c9\u00cd\u00d3\u00da\u00dc\u00d1"
                      "\u00e1\u00e9\u00ed\u00f3\u00fa\u00fc\u00f1"
                      r"]+|\d+|[^\s\w]", flags=re.UNICODE)
#: Tokens que contienen letras **y** digitos (p. ej. "abc123"), que el tokenizador separa
#: (``scripts/phase3b_recorregido.py:42``).
ALNUM_RE = re.compile(r"\b(?=\w*\d)(?=\w*[A-Za-z])\w+\b")


def _safe_entropy(counts: np.ndarray) -> float:
    """Entropia de Shannon (base 2) segura para vectores vacios (src/advanced_features.py:56-62)."""
    total = counts.sum()
    if total <= 0:
        return 0.0
    probabilities = counts[counts > 0] / total
    return float(-(probabilities * np.log2(probabilities)).sum())


def _percentiles(values: np.ndarray, points: tuple[int, ...] = (10, 25, 50, 75, 90)) -> dict:
    """Percentiles de un vector (0 si esta vacio) (src/advanced_features.py:65-69)."""
    if values.size == 0:
        return {f"p{p}": 0.0 for p in points}
    return {f"p{p}": float(np.percentile(values, p)) for p in points}


# --------------------------------------------------------------------------------------
# Familias A, C - distribucion de la longitud de palabra (src/advanced_features.py:75-111)
# --------------------------------------------------------------------------------------
def distribution_features(texts: pd.Series) -> pd.DataFrame:
    """Familia A/C: distribucion interna de la longitud de palabra y de los tokens."""
    tokens = texts.fillna("").str.lower().str.findall(WORD_RE.pattern)
    rows: list[dict] = []
    for token_list in tokens:
        lengths = np.array([len(token) for token in token_list], dtype=float)
        if lengths.size == 0:
            rows.append({"wlen_mean": 0.0, "wlen_median": 0.0, "wlen_std": 0.0, "wlen_cv": 0.0,
                         "wlen_skew": 0.0, "wlen_kurtosis": 0.0, "wlen_iqr": 0.0, "wlen_mad": 0.0,
                         "wlen_range": 0.0, "wlen_p10": 0.0, "wlen_p25": 0.0, "wlen_p50": 0.0,
                         "wlen_p75": 0.0, "wlen_p90": 0.0, "wlen_tail_ratio": 0.0,
                         "wlen_head_ratio": 0.0, "prop_len1": 0.0, "prop_len2": 0.0,
                         "prop_len3": 0.0, "prop_len_ge8": 0.0, "prop_len_ge10": 0.0})
            continue
        percentiles = {f"wlen_{key}": value for key, value in _percentiles(lengths).items()}
        mean = float(lengths.mean())
        std = float(lengths.std())
        has_variance = lengths.std() > 0
        rows.append({
            "wlen_mean": mean, "wlen_median": float(np.median(lengths)), "wlen_std": std,
            "wlen_cv": std / mean if mean > 0 else 0.0,
            # Asimetria y curtosis requieren varianza no nula (si todos los tokens miden igual,
            # la forma no esta definida y se reporta 0 en lugar de NaN).
            "wlen_skew": float(stats.skew(lengths)) if lengths.size > 2 and has_variance else 0.0,
            "wlen_kurtosis": (float(stats.kurtosis(lengths))
                              if lengths.size > 3 and has_variance else 0.0),
            "wlen_iqr": float(np.percentile(lengths, 75) - np.percentile(lengths, 25)),
            "wlen_mad": float(np.mean(np.abs(lengths - mean))),
            "wlen_range": float(lengths.max() - lengths.min()),
            **percentiles,
            "wlen_tail_ratio": float((lengths >= np.percentile(lengths, 90)).mean()),
            "wlen_head_ratio": float((lengths <= np.percentile(lengths, 10)).mean()),
            "prop_len1": float((lengths == 1).mean()), "prop_len2": float((lengths == 2).mean()),
            "prop_len3": float((lengths == 3).mean()), "prop_len_ge8": float((lengths >= 8).mean()),
            "prop_len_ge10": float((lengths >= 10).mean()),
        })
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------------------
# Familia B, R - distribucion de caracteres y entropias (src/advanced_features.py:117-153)
# --------------------------------------------------------------------------------------
def char_distribution_features(texts: pd.Series) -> pd.DataFrame:
    """Familia B/R: composicion y concentracion del alfabeto usado en el tweet."""
    clean = texts.fillna("").astype(str)
    lowered = clean.str.lower()
    total = clean.str.len().replace(0, np.nan)
    vowels = lowered.str.count(r"[aeiou" "\u00e1\u00e9\u00ed\u00f3\u00fa\u00fc" r"]")
    consonants = lowered.str.count(r"[bcdfghjklmnpqrstvwxyz" "\u00f1" r"]")
    letters = vowels + consonants
    rows = pd.DataFrame({
        "char_vowel_ratio": (vowels / letters.replace(0, np.nan)).fillna(0.0),
        "char_consonant_ratio": (consonants / letters.replace(0, np.nan)).fillna(0.0),
        "char_alpha_ratio": (letters / total).fillna(0.0),
        "char_space_ratio": (clean.str.count(r"\s") / total).fillna(0.0),
        "char_symbol_ratio": (clean.str.count(r"[^\w\s]") / total).fillna(0.0),
    })
    character_entropy, simpson, hhi, dominant, unique_ratio = [], [], [], [], []
    for value in clean:
        if not value:
            character_entropy.append(0.0)
            simpson.append(0.0)
            hhi.append(0.0)
            dominant.append(0.0)
            unique_ratio.append(0.0)
            continue
        counts = np.array(list(Counter(value).values()), dtype=float)
        probabilities = counts / counts.sum()
        character_entropy.append(_safe_entropy(counts))
        simpson.append(float(1 - (probabilities ** 2).sum()))
        hhi.append(float((probabilities ** 2).sum()))
        dominant.append(float(probabilities.max()))
        unique_ratio.append(len(set(value)) / len(value))
    rows["char_entropy"] = character_entropy
    rows["char_simpson"] = simpson
    rows["char_hhi"] = hhi
    rows["char_dominant_ratio"] = dominant
    rows["char_unique_ratio"] = unique_ratio
    return rows


# --------------------------------------------------------------------------------------
# Familia D - riqueza lexica avanzada (src/advanced_features.py:159-189)
# --------------------------------------------------------------------------------------
def lexical_richness_features(texts: pd.Series) -> pd.DataFrame:
    """Familia D: diversidad lexica estable para textos cortos (MATTR, Yule's K, Simpson)."""
    tokens = texts.fillna("").str.lower().str.findall(WORD_RE.pattern)
    rows: list[dict] = []
    for token_list in tokens:
        n = len(token_list)
        if n == 0:
            rows.append({"mattr_25": 0.0, "root_ttr": 0.0, "log_ttr": 0.0, "maas": 0.0,
                         "yule_k": 0.0, "simpson_d": 0.0, "dis_legomena_ratio": 0.0,
                         "repeated_token_ratio": 0.0, "top_token_concentration": 0.0})
            continue
        counts = Counter(token_list)
        window = min(25, n)
        windows = [token_list[start:start + window] for start in range(0, max(1, n - window + 1))]
        mattr = float(np.mean([len(set(w)) / len(w) for w in windows if w]))
        types = len(counts)
        frequencies = np.array(list(counts.values()), dtype=float)
        yule_k = float(1e4 * ((frequencies ** 2).sum() - n) / (n ** 2)) if n > 1 else 0.0
        rows.append({
            "mattr_25": mattr,
            "root_ttr": types / np.sqrt(n),
            "log_ttr": np.log(types) / np.log(n) if n > 1 and types > 1 else 0.0,
            "maas": (np.log(n) - np.log(types)) / (np.log(n) ** 2) if n > 1 and types > 1 else 0.0,
            "yule_k": yule_k,
            "simpson_d": float((frequencies * (frequencies - 1)).sum() / (n * (n - 1)))
            if n > 1 else 0.0,
            "dis_legomena_ratio": float((frequencies == 2).sum() / types) if types else 0.0,
            "repeated_token_ratio": float((n - types) / n),
            "top_token_concentration": float(frequencies.max() / n),
        })
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------------------
# Familia H, T - forma de palabra y repeticion (src/advanced_features.py:195-226)
# --------------------------------------------------------------------------------------
def word_shape_features(texts: pd.Series) -> pd.DataFrame:
    """Familia H/T: patrones de escritura de las palabras y repeticion interna."""
    tokens = texts.fillna("").str.findall(TOKEN_RE.pattern)
    rows: list[dict] = []
    for token_list in tokens:
        # ``WORD_RE`` acepta mayusculas: si no, las features de forma de palabra quedarian en
        # cero (bug detectado en la auditoria de alineacion de la Fase 3).
        words = [token for token in token_list if WORD_RE.fullmatch(token)]
        n = len(words) or 1
        all_caps = sum(1 for token in words if token.isupper() and len(token) > 1)
        capitalized = sum(1 for token in words if token[:1].isupper() and token[1:].islower())
        numeric = sum(1 for token in words if token.isdigit())
        alphanumeric = sum(1 for token in words if any(c.isdigit() for c in token)
                           and any(c.isalpha() for c in token))
        repeated = sum(1 for token in words if re.search(r"(\w)\1{1,}", token))
        bigrams = list(zip(token_list, token_list[1:])) if len(token_list) > 1 else []
        trigrams = (list(zip(token_list, token_list[1:], token_list[2:]))
                    if len(token_list) > 2 else [])
        rows.append({
            "shape_allcaps_ratio": all_caps / n,
            "shape_capitalized_ratio": capitalized / n,
            "shape_lower_ratio": sum(1 for token in words if token.islower()) / n,
            "shape_numeric_ratio": numeric / n,
            "shape_alphanumeric_ratio": alphanumeric / n,
            "shape_repeated_letter_token_ratio": repeated / n,
            "repeat_token_ratio": float(1 - len(set(token_list)) / len(token_list))
            if token_list else 0.0,
            "repeat_bigram_ratio": float(1 - len(set(bigrams)) / len(bigrams)) if bigrams else 0.0,
            "repeat_trigram_ratio": float(1 - len(set(trigrams)) / len(trigrams)) if trigrams else 0.0,
            "adjacent_repeat_ratio": float(np.mean([a == b for a, b in bigrams])) if bigrams else 0.0,
        })
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------------------
# Familia I, M, U - posicion de elementos, URLs y posicion lexica
# (src/advanced_features.py:232-267)
# --------------------------------------------------------------------------------------
def position_features(texts: pd.Series) -> pd.DataFrame:
    """Familias I/M/U: posicion normalizada de entidades y de la primera/ultima palabra."""
    clean = texts.fillna("").astype(str)
    rows: list[dict] = []
    for value in clean:
        length = max(len(value), 1)
        tokens = TOKEN_RE.findall(value)

        def positions(pattern: re.Pattern) -> tuple[float, float, float]:
            matches = [m.start() / length for m in pattern.finditer(value)]
            if not matches:
                return 0.0, 0.0, 0.0
            return matches[0], matches[-1], float(np.mean(matches))

        mention_first, mention_last, mention_mean = positions(MENTION_RE)
        hashtag_first, hashtag_last, hashtag_mean = positions(HASHTAG_RE)
        url_first, url_last, url_mean = positions(URL_RE)
        first_word = tokens[0] if tokens else ""
        last_word = tokens[-1] if tokens else ""
        rows.append({
            "pos_mention_first": mention_first, "pos_mention_last": mention_last,
            "pos_mention_mean": mention_mean, "pos_hashtag_first": hashtag_first,
            "pos_hashtag_last": hashtag_last, "pos_hashtag_mean": hashtag_mean,
            "pos_url_first": url_first, "pos_url_last": url_last, "pos_url_mean": url_mean,
            "first_token_is_mention": int(bool(tokens) and tokens[0].startswith("@")),
            "last_token_is_mention": int(bool(tokens) and tokens[-1].startswith("@")),
            "first_token_is_hashtag": int(bool(tokens) and tokens[0].startswith("#")),
            "last_token_is_hashtag": int(bool(tokens) and tokens[-1].startswith("#")),
            "first_token_is_url": int(bool(tokens) and tokens[0].startswith("http")),
            "last_token_is_url": int(bool(tokens) and tokens[-1].startswith("http")),
            "first_word_len": len(first_word), "last_word_len": len(last_word),
            "first_word_is_word": int(bool(WORD_RE.fullmatch(first_word))),
            "last_word_is_word": int(bool(WORD_RE.fullmatch(last_word))),
            "first_word_is_upper": int(first_word[:1].isupper() and len(first_word) > 1),
        })
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------------------
# Familia N, R, S - estructura, entropia y compresibilidad (src/advanced_features.py:273-306)
# --------------------------------------------------------------------------------------
def structure_entropy_features(texts: pd.Series) -> pd.DataFrame:
    """Familias N/R/S: puntuacion, entropias por tipo y ratio de compresion."""
    clean = texts.fillna("").astype(str)
    total = clean.str.len().replace(0, np.nan)
    punctuation = {name: clean.str.count(pattern) for name, pattern in
                   (("comma", ","), ("period", r"\."), ("colon", ":"), ("semicolon", ";"),
                    ("parenthesis", r"[()]"), ("quote", r"[\"']"), ("dash", "[-\u2013\u2014]"),
                    ("excl", "!"), ("quest", r"\?"))}
    rows = pd.DataFrame({
        f"punct_{name}_density": (count / total).fillna(0.0)
        for name, count in punctuation.items()
    })
    rows["punct_type_diversity"] = clean.apply(
        lambda value: len(set(re.findall(r"[^\w\s]", value))))
    rows["punct_transitions"] = clean.apply(
        lambda value: len(re.findall(r"[^\w\s]{2,}", value)))
    rows["token_entropy"] = clean.apply(
        lambda value: _safe_entropy(np.array(list(Counter(TOKEN_RE.findall(value)).values()),
                                             dtype=float)) if value else 0.0)
    rows["punct_entropy"] = clean.apply(
        lambda value: _safe_entropy(np.array(list(Counter(re.findall(r"[^\w\s]", value)).values()),
                                             dtype=float)) if value else 0.0)
    rows["entity_count"] = clean.apply(lambda value: len(MENTION_RE.findall(value))
                                       + len(HASHTAG_RE.findall(value)))
    rows["entity_entropy"] = clean.apply(
        lambda value: _safe_entropy(np.array(list(Counter(
            [m.lower() for m in MENTION_RE.findall(value)]
            + [h.lower() for h in HASHTAG_RE.findall(value)]).values()), dtype=float))
        if value else 0.0)
    rows["compression_ratio"] = clean.apply(
        lambda value: len(zlib.compress(value.encode("utf-8"), 6)) / max(len(value), 1))
    rows["compression_gain"] = clean.apply(
        lambda value: 1 - len(zlib.compress(value.encode("utf-8"), 6)) / max(len(value), 1))
    return rows


# --------------------------------------------------------------------------------------
# Correccion vigente (scripts/phase3b_recorregido.py:42-51)
# --------------------------------------------------------------------------------------
def add_alphanumeric_ratio(frame: pd.DataFrame, texts: pd.Series) -> pd.DataFrame:
    """Recalcula ``shape_alphanumeric_ratio`` sobre el texto crudo (phase3b:45-51).

    El tokenizador separa letras y digitos, asi que el conteo interno de ``word_shape_features``
    es siempre cero; aqui se cuentan los tokens mixtos con ``ALNUM_RE`` y se dividen por el
    numero de tokens de ``TOKEN_RE`` (denominador con piso 1).
    """
    counts = texts.fillna("").astype(str).apply(lambda value: len(ALNUM_RE.findall(value)))
    words = texts.fillna("").astype(str).str.findall(TOKEN_RE.pattern).apply(len).replace(0, 1)
    frame = frame.copy()
    frame["shape_alphanumeric_ratio"] = (counts / words).to_numpy()
    return frame


# --------------------------------------------------------------------------------------
# Ensamblado (scripts/advanced_02_features.py:94-104 / scripts/phase3b_recorregido.py:54-69)
# --------------------------------------------------------------------------------------
def extraer_estructuradas(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula las 87 features estructuradas de segunda generacion.

    Ensambla las seis familias en el orden original (``scripts/advanced_02_features.py:38-45,
    94-104``; ``scripts/phase3b_recorregido.py:54-69``) y aplica la correccion vigente de
    ``scripts/phase3b_recorregido.py:42-51`` (recalculo de ``shape_alphanumeric_ratio`` con
    ``ALNUM_RE`` sobre el texto crudo).

    Parameters
    ----------
    df : pd.DataFrame
        Marco con columna ``text`` (texto limpio, sin padding).

    Returns
    -------
    pd.DataFrame
        87 columnas (73 float64 + 14 int64) con el mismo indice que ``df``.
    """
    # Los textos se procesan en orden de fila con indice neutro (los builders construyen sus
    # marcos de forma posicional); al final se restituye el indice de la entrada.
    textos = df["text"].reset_index(drop=True)
    builders = (("A_distribucion", distribution_features),
                ("B_char", char_distribution_features),
                ("D_riqueza", lexical_richness_features),
                ("HT_forma", word_shape_features),
                ("IMU_posicion", position_features),
                ("NRS_estructura", structure_entropy_features))
    frames = [builder(textos) for _name, builder in builders]
    combined = pd.concat(frames, axis=1)
    combined = add_alphanumeric_ratio(combined, textos)
    combined.index = df.index
    return combined

### Módulo `supervised.py`

In [ ]:
"""Features supervisadas in-fold de la cadena AGE (porte fiel del proyecto original).

Reproduce los 5 bloques supervisados con el orden EXACTO de columnas (73):
mention 17 + hashtag 17 + rt_source 17 + rarity 12 + lexicon 10.

- Perfil de clase por entidad: P(clase|entidad) con suavizado ``smoothing=20.0`` sobre el
  prior del train recibido; sin entidad conocida ``[prior, prior, 0, 0, 0, 0, 0]``
  (src/advanced_features.py:326-393, linea 377).
- Rareza lexica: frecuencias solo del train recibido; OOV con log10 centinela ``-6.0``
  (src/advanced_features.py:396-434, linea 419).
- Lexico por clase: log-odds de Monroe et al. 2008 con ``alpha0=100.0, min_count=5,
  top_k=3000`` (src/advanced_features.py:437-490).
- Regexes ``MENTION_RE``/``HASHTAG_RE``/``RT_SOURCE_RE``/``WORD_RE`` de
  src/advanced_features.py:28-32.

Lado OOF (LOFO estricto): scripts/advanced_02_features.py:205-234; cada fold ajusta los
5 bloques con el train de los otros 4 y transforma su validacion, reensamblando por
indice. Lado test: train completo como fuente
(scripts/phase25_submission_fresca.py:69-75).

ASCII puro. Sin efectos en import. Solo numpy/pandas/stdlib (+ cadena..
"""

from __future__ import annotations

import re
from collections import Counter

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------
# Patrones de entidades (src/advanced_features.py:28-32)
# --------------------------------------------------------------------------------------
#: En WORD_RE los caracteres acentuados van como escapes \uXXXX para mantener el fichero
#: en ASCII puro; el patron compilado es identico al literal del original.
MENTION_RE = re.compile(r"@(\w+)")
HASHTAG_RE = re.compile(r"#(\w+)")
RT_SOURCE_RE = re.compile(r"^\s*RT\s+@(\w+)", flags=re.IGNORECASE)
WORD_RE = re.compile("[A-Za-z\u00c1\u00c9\u00cd\u00d3\u00da\u00dc\u00d1\u00e1\u00e9\u00ed\u00f3\u00fa\u00fc\u00f10-9]+")


def _safe_entropy(counts: np.ndarray) -> float:
    """Entropia de Shannon (base 2) segura para vectores vacios (src/advanced_features.py:56-62)."""
    total = counts.sum()
    if total <= 0:
        return 0.0
    probabilities = counts[counts > 0] / total
    return float(-(probabilities * np.log2(probabilities)).sum())


def _entity_documents(texts: pd.Series, kind: str) -> list[list[str]]:
    """Lista de entidades por documento segun el tipo pedido (src/advanced_features.py:312-323)."""
    if kind == "mention":
        return [[m.lower() for m in MENTION_RE.findall(value)]
                for value in texts.fillna("").astype(str)]
    if kind == "hashtag":
        return [[h.lower() for h in HASHTAG_RE.findall(value)]
                for value in texts.fillna("").astype(str)]
    if kind == "rt_source":
        return [[m.group(1).lower()] if (m := RT_SOURCE_RE.match(value)) else []
                for value in texts.fillna("").astype(str)]
    raise ValueError(f"tipo de entidad desconocido: {kind}")


def supervised_entity_features(train_texts: pd.Series, y_train: np.ndarray,
                               target_texts: pd.Series, kind: str = "mention",
                               smoothing: float = 20.0) -> pd.DataFrame:
    """Perfil de clase supervisado de las entidades (src/advanced_features.py:326-393).

    Aprende P(clase|entidad) con suavizado empirico-bayesiano solo con el train recibido
    (por tanto es seguro dentro de un fold) y resume cada documento con el perfil medio,
    el perfil de su entidad mas informativa, el margen, la entropia y la cobertura. Sin
    entidad conocida devuelve ``[prior, prior, 0, 0, 0, 0, 0]`` (17 columnas).

    Parameters
    ----------
    train_texts : pandas.Series
        Textos de train del fold.
    y_train : numpy.ndarray
        Etiquetas de train del fold.
    target_texts : pandas.Series
        Textos a transformar (validacion o test).
    kind : {"mention", "hashtag", "rt_source"}
        Tipo de entidad.
    smoothing : float
        Peso del prior de clase (cuanto mayor, mas se acerca a la distribucion global).

    Returns
    -------
    pandas.DataFrame
        17 columnas: ``{kind}_mean_c0..c5``, ``{kind}_best_c0..c5``, ``{kind}_margin``,
        ``{kind}_entropy``, ``{kind}_known_count``, ``{kind}_unknown_ratio`` y
        ``{kind}_dominance``.
    """
    classes = CLASSES
    class_index = {label: index for index, label in enumerate(classes)}
    train_entities = _entity_documents(train_texts, kind)
    target_entities = _entity_documents(target_texts, kind)

    prior = np.bincount([class_index[label] for label in y_train],
                        minlength=len(classes)).astype(float)
    prior /= prior.sum()

    counts: dict[str, np.ndarray] = {}
    for entities, label in zip(train_entities, y_train, strict=True):
        for entity in set(entities):
            counts.setdefault(entity, np.zeros(len(classes)))[class_index[label]] += 1

    profiles: dict[str, np.ndarray] = {}
    for entity, vector in counts.items():
        total = vector.sum()
        profiles[entity] = (vector + smoothing * prior) / (total + smoothing)

    rows: list[np.ndarray] = []
    for entities in target_entities:
        known = [profiles[entity] for entity in entities if entity in profiles]
        if not known:
            rows.append(np.concatenate([prior, prior, [0.0, 0.0, 0.0, 0.0, 0.0]]))
            continue
        stack = np.vstack(known)
        mean_profile = stack.mean(axis=0)
        best = stack[np.argmax(stack.max(axis=1))]
        ordered = np.sort(best)[::-1]
        margin = float(ordered[0] - ordered[1])
        entropy = _safe_entropy(best)
        rows.append(np.concatenate([mean_profile, best,
                                    [margin, entropy, float(len(stack)),
                                     float(1 - len(stack) / max(len(entities), 1)),
                                     float((np.array(entities) == Counter(entities).most_common(1)[0][0]).mean())]]))
    columns = ([f"{kind}_mean_c{i}" for i in range(len(classes))]
               + [f"{kind}_best_c{i}" for i in range(len(classes))]
               + [f"{kind}_margin", f"{kind}_entropy", f"{kind}_known_count",
                  f"{kind}_unknown_ratio", f"{kind}_dominance"])
    return pd.DataFrame(np.vstack(rows), columns=columns)


def lexical_rarity_features(train_texts: pd.Series, target_texts: pd.Series) -> pd.DataFrame:
    """Rareza y novedad lexica respecto del vocabulario del train recibido
    (src/advanced_features.py:396-434).

    Nunca usa estadisticas del conjunto objetivo (regla 74 del brief): las frecuencias se
    aprenden solo de ``train_texts``. Los OOV reciben el log10 centinela ``-6.0``
    (linea 419). Devuelve 12 columnas: ``rarity_mean_logfreq``, ``rarity_median_logfreq``,
    ``rarity_min_logfreq``, ``rarity_max_logfreq``, ``rarity_mean_freq``,
    ``rare_token_ratio``, ``very_rare_token_ratio``, ``oov_ratio``, ``hapax_token_ratio``,
    ``top_frequency_ratio``, ``lexical_novelty`` y ``lexical_familiarity``.
    """
    frequency: Counter = Counter()
    for tokens in train_texts.fillna("").str.lower().str.findall(WORD_RE.pattern):
        frequency.update(tokens)
    total = sum(frequency.values()) or 1
    log_frequencies = {token: np.log10(count / total) for token, count in frequency.items()}

    rows: list[dict] = []
    for tokens in target_texts.fillna("").str.lower().str.findall(WORD_RE.pattern):
        if not tokens:
            rows.append({"rarity_mean_logfreq": 0.0, "rarity_median_logfreq": 0.0,
                         "rarity_min_logfreq": 0.0, "rarity_max_logfreq": 0.0,
                         "rarity_mean_freq": 0.0, "rare_token_ratio": 0.0,
                         "very_rare_token_ratio": 0.0, "oov_ratio": 0.0,
                         "hapax_token_ratio": 0.0, "top_frequency_ratio": 0.0,
                         "lexical_novelty": 0.0, "lexical_familiarity": 0.0})
            continue
        frequencies = np.array([frequency.get(token, 0) for token in tokens], dtype=float)
        log_freqs = np.array([log_frequencies.get(token, -6.0) for token in tokens])
        rows.append({
            "rarity_mean_logfreq": float(log_freqs.mean()),
            "rarity_median_logfreq": float(np.median(log_freqs)),
            "rarity_min_logfreq": float(log_freqs.min()),
            "rarity_max_logfreq": float(log_freqs.max()),
            "rarity_mean_freq": float(frequencies.mean()),
            "rare_token_ratio": float((frequencies <= 5).mean()),
            "very_rare_token_ratio": float((frequencies <= 2).mean()),
            "oov_ratio": float((frequencies == 0).mean()),
            "hapax_token_ratio": float((frequencies == 1).mean()),
            "top_frequency_ratio": float(np.mean(log_freqs >= np.percentile(log_freqs, 90))),
            "lexical_novelty": float((frequencies <= 2).mean()),
            "lexical_familiarity": float(np.mean(log_freqs)),
        })
    return pd.DataFrame(rows)


def class_lexicon_scores(train_texts: pd.Series, y_train: np.ndarray,
                         target_texts: pd.Series, alpha0: float = 100.0,
                         min_count: int = 5, top_k: int = 3000) -> pd.DataFrame:
    """Distancia al lexico de cada clase mediante log-odds (Monroe et al. 2008)
    (src/advanced_features.py:437-490).

    Aprende, solo con el train recibido, los ``top_k`` terminos mas distintivos de cada
    clase (frecuencia minima ``min_count``, prior ``alpha0``) y resume cada documento con
    la puntuacion media de sus tokens frente a cada lexico, mas el margen, la entropia, la
    clase ganadora y la desviacion. Devuelve 10 columnas: ``lexicon_c0..c5``,
    ``lexicon_margin``, ``lexicon_entropy``, ``lexicon_best_class`` y ``lexicon_std``.
    """
    classes = CLASSES
    class_index = {label: index for index, label in enumerate(classes)}
    train_tokens = train_texts.fillna("").str.lower().str.findall(WORD_RE.pattern)
    counter_global: Counter = Counter()
    counters_by_class = [Counter() for _ in classes]
    for tokens, label in zip(train_tokens, y_train, strict=True):
        position = class_index[label]
        counter_global.update(tokens)
        counters_by_class[position].update(tokens)

    total = sum(counter_global.values()) or 1
    prior = {token: alpha0 * count / total for token, count in counter_global.items()}
    class_scores: list[dict[str, float]] = []
    totals_by_class = [sum(counter.values()) for counter in counters_by_class]
    for position in range(len(classes)):
        scores: dict[str, float] = {}
        other_total = total - totals_by_class[position]
        for token, count in counter_global.items():
            if count < min_count:
                continue
            in_class = counters_by_class[position].get(token, 0)
            out_class = count - in_class
            if in_class == 0:
                continue
            log_odds = (np.log((in_class + prior[token])
                               / (totals_by_class[position] + alpha0 - in_class - prior[token]))
                        - np.log((out_class + prior[token])
                                 / (other_total + alpha0 - out_class - prior[token])))
            scores[token] = float(log_odds)
        top_items = sorted(scores.items(), key=lambda item: -item[1])[:top_k]
        class_scores.append({token: value for token, value in top_items})

    rows: list[dict] = []
    for tokens in target_texts.fillna("").str.lower().str.findall(WORD_RE.pattern):
        vector = np.zeros(len(classes))
        for position, scores in enumerate(class_scores):
            if tokens:
                vector[position] = float(np.mean([scores.get(token, 0.0) for token in tokens]))
        ordered = np.sort(vector)[::-1]
        rows.append({**{f"lexicon_c{i}": vector[i] for i in range(len(classes))},
                     "lexicon_margin": float(ordered[0] - ordered[1]),
                     "lexicon_entropy": _safe_entropy(vector - vector.min() + 1e-9),
                     "lexicon_best_class": float(np.argmax(vector)),
                     "lexicon_std": float(vector.std())})
    return pd.DataFrame(rows)


def supervisadas_infold(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    """Features supervisadas LOFO estricto, (n, 73) float64.

    Porte de scripts/advanced_02_features.py:205-234. Para cada fold se ajustan los 5
    bloques con el train de los otros 4 folds y se transforma la validacion; el resultado
    se reensambla por indice (posicional) en el orden original. Orden de columnas:
    mention 17 + hashtag 17 + rt_source 17 + rarity 12 + lexicon 10.

    Parameters
    ----------
    textos : pandas.Series
        Textos del train completo (columna de texto, ya limpia).
    y : numpy.ndarray
        Etiquetas del train completo (strings de CLASSES).
    fold_idx : numpy.ndarray
        Protocolo de folds congelado: valor 0-4 por fila.

    Returns
    -------
    numpy.ndarray
        Matriz (n, 73) float64 en el orden de filas de ``textos``.
    """
    partes: list[pd.DataFrame] = []
    for fold in sorted(np.unique(fold_idx)):
        train_idx = np.flatnonzero(fold_idx != fold)
        valid_idx = np.flatnonzero(fold_idx == fold)
        train_texts = textos.iloc[train_idx]
        valid_texts = textos.iloc[valid_idx]
        y_train = y[train_idx]
        # Mismo orden que el dict `blocks` del original: mencion, hashtag, fuente_rt,
        # rareza, lexico (scripts/advanced_02_features.py:215-221).
        block = pd.concat([
            supervised_entity_features(train_texts, y_train, valid_texts, "mention"),
            supervised_entity_features(train_texts, y_train, valid_texts, "hashtag"),
            supervised_entity_features(train_texts, y_train, valid_texts, "rt_source"),
            lexical_rarity_features(train_texts, valid_texts),
            class_lexicon_scores(train_texts, y_train, valid_texts),
        ], axis=1)
        block.index = valid_idx
        partes.append(block)
    matriz = pd.concat(partes).sort_index().to_numpy(float)
    assert matriz.shape[1] == 73, matriz.shape
    return matriz


def supervisadas_test(textos_train: pd.Series, y: np.ndarray,
                      textos_test: pd.Series) -> np.ndarray:
    """Features supervisadas con el train completo como fuente, (m, 73) float64.

    Porte de scripts/phase25_submission_fresca.py:69-75: misma funcion que in-fold pero
    ajustando los 5 bloques con todo el train y transformando el test. Orden de columnas:
    mention 17 + hashtag 17 + rt_source 17 + rarity 12 + lexicon 10.

    Parameters
    ----------
    textos_train : pandas.Series
        Textos del train completo.
    y : numpy.ndarray
        Etiquetas del train completo (strings de CLASSES).
    textos_test : pandas.Series
        Textos del test a transformar.

    Returns
    -------
    numpy.ndarray
        Matriz (m, 73) float64 en el orden de filas de ``textos_test``.
    """
    matriz = pd.concat([
        supervised_entity_features(textos_train, y, textos_test, "mention"),
        supervised_entity_features(textos_train, y, textos_test, "hashtag"),
        supervised_entity_features(textos_train, y, textos_test, "rt_source"),
        lexical_rarity_features(textos_train, textos_test),
        class_lexicon_scores(textos_train, y, textos_test),
    ], axis=1).to_numpy(float)
    assert matriz.shape[1] == 73, matriz.shape
    return matriz

### Módulo `nb_svm.py`

In [ ]:
r"""NB-SVM (Wang & Manning 2012) uno-vs-rest: log-count ratio + SVM lineal por clase.

Porte literal de los originales del proyecto KAGGLE_AGE_2026_PUJ:
- scripts/advanced_05_nbsvm_seleccion.py:66-94 (ADV05: char_wb 3-6 300k + menciones).
- scripts/phase7_nbsvm_canal_meta.py:57-78 (ADV07: solo n-gramas de palabra).
- Lado test (mismo calculo reajustado con todo el train): scripts/phase3_holdout_v11.py:115-127
  y scripts/phase25_submission_fresca.py:77-89.

Reglas del paquete: ASCII puro; sin efectos en import; solo numpy/pandas/scipy/sklearn y
`cadena.. Interfaz en plan/kaggle_notebook/port_interface.md.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC



def nb_log_count_ratio(x_train: sp.csr_matrix, binary_target: np.ndarray) -> np.ndarray:
    r"""Log-count ratio de la clase positiva frente al resto (Wang & Manning 2012).

    Porte literal de scripts/advanced_05_nbsvm_seleccion.py:66-70 y
    scripts/phase7_nbsvm_canal_meta.py:57-61:
    log((pos / pos.sum()) / (neg / neg.sum())) con +1.0 de suavizado en los conteos.
    """
    positive = np.asarray(x_train[binary_target == 1].sum(axis=0)).ravel() + 1.0
    negative = np.asarray(x_train[binary_target == 0].sum(axis=0)).ravel() + 1.0
    return np.log((positive / positive.sum()) / (negative / negative.sum()))


def nb_svm_margins(textos: pd.Series, y: np.ndarray, train_idx: np.ndarray,
                   valid_idx: np.ndarray, config: dict) -> np.ndarray:
    r"""Margenes NB-SVM uno-vs-rest del fold retenido (float32, columnas en CLASSES).

    Porte de scripts/advanced_05_nbsvm_seleccion.py:73-94 (ADV05, char300 + menciones) y
    scripts/phase7_nbsvm_canal_meta.py:64-78 (ADV07, solo palabras): vectorizador(es)
    TF-IDF ajustados SOLO con textos.iloc[train_idx]; por clase (en orden CLASSES)
    log-count ratio sobre el train del fold y LinearSVC(C=0.2, max_iter=3000) entrenado
    sobre x_train.multiply(ratio).tocsr(); decision_function sobre el fold retenido
    multiplicado por el mismo ratio. Las columnas ya quedan en el orden de CLASSES
    porque el bucle entrena una clase por posicion.

    config: parametros de TfidfVectorizer del canal principal. Clave opcional
    "mentions" = {"config": {...}, "pattern": r"@\w+"} para anadir un segundo canal
    TF-IDF sobre tokens_of(textos, pattern). ADV05 la usa con CHAR_300K + MENTION y
    patron r"@\w+"; ADV07 pasa WORD sin canal de menciones (el texto tal cual).

    train_idx/valid_idx: indices enteros (p. ej. np.flatnonzero) de las filas de
    entrenamiento y validacion, como fold["train_idx"]/fold["val_idx"] del protocolo
    congelado. Salida (len(valid_idx), 6) float32.
    """
    config = dict(config)
    mentions = config.pop("mentions", None)
    y = np.asarray(y)
    vectorizer = TfidfVectorizer(**config)
    if mentions is None:
        x_train = vectorizer.fit_transform(textos.iloc[train_idx])
        x_valid = vectorizer.transform(textos.iloc[valid_idx])
    else:
        mention_vectorizer = TfidfVectorizer(**mentions["config"])
        pattern = mentions["pattern"]
        x_train = sp.hstack([
            vectorizer.fit_transform(textos.iloc[train_idx]),
            mention_vectorizer.fit_transform(tokens_of(textos.iloc[train_idx], pattern)),
        ]).tocsr()
        x_valid = sp.hstack([
            vectorizer.transform(textos.iloc[valid_idx]),
            mention_vectorizer.transform(tokens_of(textos.iloc[valid_idx], pattern)),
        ]).tocsr()
    y_train = y[train_idx]

    margins = np.zeros((len(valid_idx), len(CLASSES)), dtype=np.float32)
    for position, label in enumerate(CLASSES):
        binary_target = (y_train == label).astype(int)
        ratio = nb_log_count_ratio(x_train, binary_target)
        model = LinearSVC(C=0.2, max_iter=3000)
        model.fit(x_train.multiply(ratio).tocsr(), binary_target)
        margins[:, position] = model.decision_function(x_valid.multiply(ratio).tocsr())
    return margins

### Módulo `miembros_oof_a.py`

In [ ]:
"""Miembros OOF del bloque A de la cadena AGE (porte fiel del proyecto original).

Cada funcion reproduce EXACTAMENTE el script citado: mismos modelos, hiperparametros,
representaciones y orden de columnas. Protocolo  entrenar con ``fold_idx != fold``,
predecir ``fold_idx == fold`` y reensamblar por indice.

Fuentes:
- E-013: scripts/experimento_e013_barrido_c.py:80-92,112
- E-016: scripts/experimento_e016_e017.py:76-87,126
- E-009 / E-010 / E-011: scripts/experimento_e009_modelos_clasicos.py:55-74,147-151,171-175
- E-014 / E-015: scripts/experimento_e014_e015_hipotesis_error.py:101-112,122-132,151-153

Salidas: margenes y probabilidades (n, 6) float32 reordenados a ``CLASSES``;
E-015 ordinal (n,) float32. ASCII puro. Sin efectos en import.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge, SGDClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC


# Helpers compartidos (representacion y orden de columnas).


def _orden_canonico(classes: np.ndarray) -> list[int]:
    """Indices para reordenar columnas al orden de ``CLASSES``.

    Patron literal de los originales:
    ``[list(model.classes_).index(label) for label in CLASSES]``
    (scripts/experimento_e013_barrido_c.py:91-92).
    """
    return [list(classes).index(etiqueta) for etiqueta in CLASSES]


def _representacion_char_menciones(train_texts: pd.Series, valid_texts: pd.Series,
                                   char_config: dict) -> tuple[csr_matrix, csr_matrix]:
    """Representacion char + menciones ajustada solo con el train del fold.

    Fuente: scripts/experimento_e009_modelos_clasicos.py:90-98 (``build_representation``);
    bloque identico al de E-013/E-016/E-014/E-015
    (scripts/experimento_e013_barrido_c.py:71-76).
    """
    char = TfidfVectorizer(**char_config)
    menciones = TfidfVectorizer(**MENTION)
    x_train = hstack([char.fit_transform(train_texts),
                      menciones.fit_transform(tokens_of(train_texts, r"@\w+"))]).tocsr()
    x_valid = hstack([char.transform(valid_texts),
                      menciones.transform(tokens_of(valid_texts, r"@\w+"))]).tocsr()
    return x_train, x_valid


# Miembros OOF (uno por artefacto del bloque A).


def oof_e013(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    """Margenes OOF de E-013.

    Fuente: scripts/experimento_e013_barrido_c.py:80-92,112.
    LinearSVC(C=0.2, max_iter=3000) sobre char_wb 3-6 (min_df=5, max_features=300k) +
    menciones word 1-1 (min_df=2); vectorizadores ajustados in-fold. Devuelve (n, 6)
    float32 con ``decision_function`` reordenado a ``CLASSES``.
    """
    salida = np.zeros((len(textos), len(CLASSES)), dtype=np.float32)
    for fold in np.unique(fold_idx):
        train_idx = fold_idx != fold
        valid_idx = fold_idx == fold
        x_train, x_valid = _representacion_char_menciones(textos.iloc[train_idx],
                                                          textos.iloc[valid_idx],
                                                          CHAR_300K)
        modelo = LinearSVC(C=0.2, max_iter=3000)
        modelo.fit(x_train, y[train_idx])
        salida[valid_idx] = modelo.decision_function(x_valid)[:, _orden_canonico(modelo.classes_)]
    return salida


def oof_e016(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    """Margenes OOF de E-016.

    Fuente: scripts/experimento_e016_e017.py:76-87,126.
    LinearSVC(C=0.2, max_iter=3000) sobre char_wb 3-6 (min_df=3, max_features=700k) +
    menciones word 1-1 (min_df=2); vectorizadores ajustados in-fold. Devuelve (n, 6)
    float32 con ``decision_function`` reordenado a ``CLASSES``.
    """
    salida = np.zeros((len(textos), len(CLASSES)), dtype=np.float32)
    for fold in np.unique(fold_idx):
        train_idx = fold_idx != fold
        valid_idx = fold_idx == fold
        x_train, x_valid = _representacion_char_menciones(textos.iloc[train_idx],
                                                          textos.iloc[valid_idx],
                                                          CHAR_700K)
        modelo = LinearSVC(C=0.2, max_iter=3000)
        modelo.fit(x_train, y[train_idx])
        salida[valid_idx] = modelo.decision_function(x_valid)[:, _orden_canonico(modelo.classes_)]
    return salida


def oof_e009(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    """Probabilidades OOF de E-009.

    Fuente: scripts/experimento_e009_modelos_clasicos.py:55-61,147-151,171-175.
    OneVsRestClassifier(LogisticRegression(C=4.0, max_iter=1000, solver="liblinear",
    random_state=42)) sobre char_wb 3-6 (300k) + menciones. Devuelve (n, 6) float32 con
    ``predict_proba`` reordenada a ``CLASSES``.
    """
    salida = np.zeros((len(textos), len(CLASSES)), dtype=np.float32)
    for fold in np.unique(fold_idx):
        train_idx = fold_idx != fold
        valid_idx = fold_idx == fold
        x_train, x_valid = _representacion_char_menciones(textos.iloc[train_idx],
                                                          textos.iloc[valid_idx],
                                                          CHAR_300K)
        modelo = OneVsRestClassifier(LogisticRegression(C=4.0, max_iter=1000,
                                                        solver="liblinear",
                                                        random_state=SEED))
        modelo.fit(x_train, y[train_idx])
        salida[valid_idx] = modelo.predict_proba(x_valid)[:, _orden_canonico(modelo.classes_)]
    return salida


def oof_e011(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    """Probabilidades OOF de E-011.

    Fuente: scripts/experimento_e009_modelos_clasicos.py:68-74,147-151,171-175.
    SGDClassifier(loss="log_loss", alpha=1e-6, max_iter=30, tol=1e-4, random_state=42,
    n_jobs=32) sobre char_wb 3-6 (300k) + menciones. Devuelve (n, 6) float32 con
    ``predict_proba`` reordenada a ``CLASSES``.
    """
    salida = np.zeros((len(textos), len(CLASSES)), dtype=np.float32)
    for fold in np.unique(fold_idx):
        train_idx = fold_idx != fold
        valid_idx = fold_idx == fold
        x_train, x_valid = _representacion_char_menciones(textos.iloc[train_idx],
                                                          textos.iloc[valid_idx],
                                                          CHAR_300K)
        modelo = SGDClassifier(loss="log_loss", alpha=1e-6, max_iter=30, tol=1e-4,
                               random_state=SEED, n_jobs=32)
        modelo.fit(x_train, y[train_idx])
        salida[valid_idx] = modelo.predict_proba(x_valid)[:, _orden_canonico(modelo.classes_)]
    return salida


def oof_e010(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    """Probabilidades OOF de E-010.

    Fuente: scripts/experimento_e009_modelos_clasicos.py:62-67,147-151,171-175.
    ComplementNB(alpha=0.3) sobre char_wb 3-6 (300k) + menciones. Devuelve (n, 6) float32
    con ``predict_proba`` reordenada a ``CLASSES``.
    """
    salida = np.zeros((len(textos), len(CLASSES)), dtype=np.float32)
    for fold in np.unique(fold_idx):
        train_idx = fold_idx != fold
        valid_idx = fold_idx == fold
        x_train, x_valid = _representacion_char_menciones(textos.iloc[train_idx],
                                                          textos.iloc[valid_idx],
                                                          CHAR_300K)
        modelo = ComplementNB(alpha=0.3)
        modelo.fit(x_train, y[train_idx])
        salida[valid_idx] = modelo.predict_proba(x_valid)[:, _orden_canonico(modelo.classes_)]
    return salida


def oof_e014(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    """Margenes OOF de E-014.

    Fuente: scripts/experimento_e014_e015_hipotesis_error.py:101-112,151.
    LinearSVC(C=0.2, max_iter=3000) sobre char_wb 3-6 (300k) + menciones + hashtags
    word 1-1 (min_df=2); vectorizadores ajustados in-fold. Devuelve (n, 6) float32 con
    ``decision_function`` reordenado a ``CLASSES``.
    """
    salida = np.zeros((len(textos), len(CLASSES)), dtype=np.float32)
    for fold in np.unique(fold_idx):
        train_idx = fold_idx != fold
        valid_idx = fold_idx == fold
        x_train, x_valid = _representacion_char_menciones(textos.iloc[train_idx],
                                                          textos.iloc[valid_idx],
                                                          CHAR_300K)
        hashtags = TfidfVectorizer(**HASHTAG)
        x_train_h = hstack([x_train, hashtags.fit_transform(
            tokens_of(textos.iloc[train_idx], r"#\w+"))]).tocsr()
        x_valid_h = hstack([x_valid, hashtags.transform(
            tokens_of(textos.iloc[valid_idx], r"#\w+"))]).tocsr()
        modelo = LinearSVC(C=0.2, max_iter=3000)
        modelo.fit(x_train_h, y[train_idx])
        salida[valid_idx] = modelo.decision_function(x_valid_h)[:, _orden_canonico(modelo.classes_)]
    return salida


def oof_e015_ordinal(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    """Prediccion ordinal continua OOF de E-015.

    Fuente: scripts/experimento_e014_e015_hipotesis_error.py:122-128,152.
    Ridge(alpha=1.0, solver="sparse_cg", max_iter=2000, random_state=42) sobre
    char_wb 3-6 (300k) + menciones, con objetivo ``CLASS_MIDPOINTS`` de cada
    etiqueta (como el ``ordinal_target`` original). Devuelve (n,) float32.
    """
    objetivo = np.array([CLASS_MIDPOINTS[etiqueta] for etiqueta in y])
    salida = np.zeros(len(textos), dtype=np.float32)
    for fold in np.unique(fold_idx):
        train_idx = fold_idx != fold
        valid_idx = fold_idx == fold
        x_train, x_valid = _representacion_char_menciones(textos.iloc[train_idx],
                                                          textos.iloc[valid_idx],
                                                          CHAR_300K)
        modelo = Ridge(alpha=1.0, solver="sparse_cg", max_iter=2000, random_state=SEED)
        modelo.fit(x_train, objetivo[train_idx])
        salida[valid_idx] = modelo.predict(x_valid)
    return salida


def oof_e015_softmax(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    """Distribucion suave OOF de E-015 (softmax ordinal).

    Fuente: scripts/experimento_e014_e015_hipotesis_error.py:129-132,153.
    Aplica ``softmax_ordinal`` a la salida de ``oof_e015_ordinal``. Devuelve (n, 6)
    float32. El original calculaba el softmax sobre la prediccion continua float64 previa
    al casteo; aqui se aplica sobre la salida float32 (diferencia numerica esperada del
    orden de 1e-6).
    """
    return softmax_ordinal(oof_e015_ordinal(textos, y, fold_idx)).astype(np.float32)

### Módulo `miembros_oof_b.py`

In [ ]:
r"""Miembros OOF del paquete B de la cadena AGE: E-017, E-019b, E-021a, E-021b, E-028a y ADV05.

Firma `(textos, y, fold_idx)`: por fold del protocolo congelado (5 folds) se entrena
con `fold_idx != fold` y se predice `fold_idx == fold`, reensamblando por indice. Salidas
float32 con columnas en CLASSES, salvo E-028a (columnas = ADJACENT_PAIRS).

Fuentes (porte literal, sin mejorar ni reinterpretar):
- E-017: scripts/experimento_e016_e017.py:97-108,127.
- E-019b: scripts/experimento_e019_modelos_densos.py:65-80,98-99,113-131,157.
- E-021a: scripts/experimento_e021_ordinal_centroides.py:57-75,110-122,150.
- E-021b: scripts/experimento_e021_ordinal_centroides.py:78-91,110-127,151.
- E-028a: scripts/experimento_e028_ordinal_fino.py:92-106,147.
- ADV05: scripts/advanced_05_nbsvm_seleccion.py:73-94,138-146.

Reglas del paquete: ASCII puro; sin efectos en import; solo numpy/pandas/scipy/sklearn y
`cadena` (nb_svm). Interfaz en plan/kaggle_notebook/port_interface.md.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import LinearSVC



def _folds_estandar(fold_idx: np.ndarray):
    """Genera (fold, train_idx, valid_idx) del protocolo congelado (5 folds).

    Entrena con fold_idx != fold y predice fold_idx == fold; devuelve indices enteros,
    como fold["train_idx"]/fold["val_idx"] de src/cv.py:332-374 (make_folds agrupado).
    """
    fold_idx = np.asarray(fold_idx)
    for fold in range(5):
        yield fold, np.flatnonzero(fold_idx != fold), np.flatnonzero(fold_idx == fold)


def _tfidf_char_menciones(textos: pd.Series, train_idx: np.ndarray, valid_idx: np.ndarray):
    r"""TF-IDF in-fold char_wb 3-6 (CHAR_300K) + canal de menciones (MENTION, r"@\w+").

    Porte de scripts/experimento_e021_ordinal_centroides.py:112-117; identico en
    scripts/experimento_e019_modelos_densos.py:114-119 y
    scripts/experimento_e028_ordinal_fino.py:84-89.
    """
    char = TfidfVectorizer(**CHAR_300K)
    menciones = TfidfVectorizer(**MENTION)
    x_train = hstack([char.fit_transform(textos.iloc[train_idx]),
                      menciones.fit_transform(tokens_of(textos.iloc[train_idx],
                                                        r"@\w+"))]).tocsr()
    x_valid = hstack([char.transform(textos.iloc[valid_idx]),
                      menciones.transform(tokens_of(textos.iloc[valid_idx],
                                                    r"@\w+"))]).tocsr()
    return x_train, x_valid


def _ordinal_probabilidades(x_train, y_index: np.ndarray, x_valid) -> np.ndarray:
    r"""Distribucion de clase por modelo ordinal acumulativo (Frank & Hall 2001).

    Porte de scripts/experimento_e021_ordinal_centroides.py:57-75: K-1 clasificadores
    binarios P(y > k) con LogisticRegression(C=1.0, max_iter=1000, solver="liblinear",
    random_state=SEED); acumuladas monotonizadas con minimum.accumulate y clip [0, 1];
    P(y = k) por diferencias (-diff + ultima columna), clip 1e-12 y normalizacion.
    """
    thresholds = np.arange(1, len(CLASSES))
    cumulative = np.ones((x_valid.shape[0], len(thresholds) + 1), dtype=np.float64)
    for position, threshold in enumerate(thresholds):
        binary_target = (y_index >= threshold).astype(int)
        model = LogisticRegression(C=1.0, max_iter=1000, solver="liblinear",
                                   random_state=SEED)
        model.fit(x_train, binary_target)
        cumulative[:, position + 1] = model.predict_proba(x_valid)[:, 1]
    cumulative = np.clip(np.minimum.accumulate(cumulative, axis=1), 0.0, 1.0)
    # P(y = k) = P(y > k-1) - P(y > k); la ultima clase es P(y > K-1) - 0.
    probabilities = np.column_stack([-np.diff(cumulative, axis=1), cumulative[:, -1]])
    probabilities = np.clip(probabilities, 1e-12, None)
    return probabilities / probabilities.sum(axis=1, keepdims=True)


def _rocchio_probabilidades(x_train, y: np.ndarray, x_valid) -> np.ndarray:
    r"""Clasificador de centroides (Rocchio) con similitud coseno y softmax T=8.

    Porte de scripts/experimento_e021_ordinal_centroides.py:78-91: centroide medio por
    clase normalizado L2 (epsilon 1e-12); norma de cada fila valida (sqrt de la suma de
    cuadrados, epsilon 1e-12); similitud coseno; softmax desplazada por el maximo por
    fila con temperatura 8.0.
    """
    centroids = []
    for label in CLASSES:
        mask = y == label
        centroid = np.asarray(x_train[mask].mean(axis=0)).ravel()
        centroids.append(centroid / (np.linalg.norm(centroid) + 1e-12))
    centroid_matrix = np.vstack(centroids)

    valid_norms = np.sqrt(np.asarray(x_valid.multiply(x_valid).sum(axis=1)).ravel()) + 1e-12
    similarity = np.asarray(x_valid @ centroid_matrix.T) / valid_norms[:, None]
    shifted = similarity - similarity.max(axis=1, keepdims=True)
    exponentials = np.exp(shifted * 8.0)  # temperatura: afina la distribucion
    return exponentials / exponentials.sum(axis=1, keepdims=True)


def oof_e017(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    r"""Probabilidades OOF de E-017: LogReg OvR C=4.0 sobre word 1-2 (min_df=3, 300k).

    Porte de scripts/experimento_e016_e017.py:97-108 y 127: TfidfVectorizer(**WORD)
    ajustado in-fold; OneVsRestClassifier(LogisticRegression(C=4.0, max_iter=1000,
    solver="liblinear", random_state=SEED)); predict_proba del fold retenido reordenada
    a CLASSES con logreg.classes_. Salida (n, 6) float32.
    """
    y = np.asarray(y)
    probabilidades = np.zeros((len(textos), len(CLASSES)), dtype=np.float32)
    for _fold, train_idx, valid_idx in _folds_estandar(fold_idx):
        word = TfidfVectorizer(**WORD)
        x_train = word.fit_transform(textos.iloc[train_idx])
        x_valid = word.transform(textos.iloc[valid_idx])
        logreg = OneVsRestClassifier(LogisticRegression(C=4.0, max_iter=1000,
                                                        solver="liblinear",
                                                        random_state=SEED))
        logreg.fit(x_train, y[train_idx])
        orden = [list(logreg.classes_).index(etiqueta) for etiqueta in CLASSES]
        probabilidades[valid_idx] = logreg.predict_proba(x_valid)[:, orden]
    return probabilidades


def oof_e019b(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray,
              dense: pd.DataFrame) -> np.ndarray:
    r"""Probabilidades OOF de E-019b: MLP(256,128) sobre [35 densas + SVD(64)].

    Porte de scripts/experimento_e019_modelos_densos.py:113-131 y 157 (variante B) mas
    fit_mlp (:65-80) y el LabelEncoder con assert de orden CLASSES (:98-99): TF-IDF
    in-fold char_wb 3-6 (CHAR_300K) + menciones (MENTION, r"@\w+"); TruncatedSVD(
    n_components=64, random_state=SEED) ajustada SOLO con el train del fold; entrada
    [densas | z] escalada con StandardScaler (ajustado in-fold); MLPClassifier(
    hidden_layer_sizes=(256, 128), alpha=1e-3, batch_size=512, learning_rate_init=1e-3,
    max_iter=60, early_stopping=True, n_iter_no_change=6, random_state=SEED) con etiquetas
    codificadas como enteros (scikit-learn 1.8 falla en early stopping con etiquetas
    cadena). Las 35 densas = columnas de `dense` sin {n_emoji, emoji_diversity, has_emoji}.
    Salida (n, 6) float32.
    """
    y = np.asarray(y)
    columnas = [col for col in dense.columns
                if col not in {"n_emoji", "emoji_diversity", "has_emoji"}]
    assert len(columnas) == 35, f"se esperaban 35 densas y hay {len(columnas)}"
    densas = dense[columnas].to_numpy(float)
    encoder = LabelEncoder().fit(y)
    assert list(encoder.classes_) == list(CLASSES), "orden de clases inesperado"
    probabilidades = np.zeros((len(textos), len(CLASSES)), dtype=np.float32)
    for _fold, train_idx, valid_idx in _folds_estandar(fold_idx):
        x_train, x_valid = _tfidf_char_menciones(textos, train_idx, valid_idx)
        svd = TruncatedSVD(n_components=64, random_state=SEED)
        z_train = svd.fit_transform(x_train)
        z_valid = svd.transform(x_valid)
        combined_train = np.hstack([densas[train_idx], z_train])
        combined_valid = np.hstack([densas[valid_idx], z_valid])
        scaler = StandardScaler().fit(combined_train)
        model = MLPClassifier(hidden_layer_sizes=(256, 128), alpha=1e-3, batch_size=512,
                              learning_rate_init=1e-3, max_iter=60, early_stopping=True,
                              n_iter_no_change=6, random_state=SEED)
        model.fit(scaler.transform(combined_train), encoder.transform(y[train_idx]))
        probabilidades[valid_idx] = model.predict_proba(scaler.transform(combined_valid))
    return probabilidades


def oof_e021a(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    r"""Probabilidades OOF de E-021a: ordinal acumulativo Frank & Hall (5 binarios).

    Porte de scripts/experimento_e021_ordinal_centroides.py:110-122 y 150: por fold,
    representacion in-fold char_wb 3-6 (CHAR_300K) + menciones; y_index = posicion de la
    clase en CLASSES; modelo ordinal acumulativo :57-75. Salida (n, 6) float32.
    """
    y = np.asarray(y)
    y_index = np.array([CLASSES.index(etiqueta) for etiqueta in y])
    probabilidades = np.zeros((len(textos), len(CLASSES)), dtype=np.float32)
    for _fold, train_idx, valid_idx in _folds_estandar(fold_idx):
        x_train, x_valid = _tfidf_char_menciones(textos, train_idx, valid_idx)
        probabilidades[valid_idx] = _ordinal_probabilidades(x_train, y_index[train_idx],
                                                            x_valid)
    return probabilidades


def oof_e021b(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    r"""Probabilidades OOF de E-021b: centroides Rocchio + softmax temperatura 8.0.

    Porte de scripts/experimento_e021_ordinal_centroides.py:110-127 y 151: por fold,
    representacion in-fold char_wb 3-6 (CHAR_300K) + menciones; centroides Rocchio :78-91
    sobre el train del fold. Salida (n, 6) float32.
    """
    y = np.asarray(y)
    probabilidades = np.zeros((len(textos), len(CLASSES)), dtype=np.float32)
    for _fold, train_idx, valid_idx in _folds_estandar(fold_idx):
        x_train, x_valid = _tfidf_char_menciones(textos, train_idx, valid_idx)
        probabilidades[valid_idx] = _rocchio_probabilidades(x_train, y[train_idx], x_valid)
    return probabilidades


def oof_e028a(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    r"""Margenes OOF de E-028a: 5 SVM entrenados SOLO con cada par adyacente.

    Porte de scripts/experimento_e028_ordinal_fino.py:92-106 y 147: representacion
    in-fold char_wb 3-6 (CHAR_300K) + menciones; por par (lower, upper) de ADJACENT_PAIRS
    mascara con las dos clases, objetivo (y_train == upper) y LinearSVC(C=0.2,
    max_iter=3000); decision_function del fold retenido (margen positivo -> clase mayor).
    Columnas = ADJACENT_PAIRS. Salida (n, 5) float32.
    """
    y = np.asarray(y)
    margenes = np.zeros((len(textos), len(ADJACENT_PAIRS)), dtype=np.float32)
    for _fold, train_idx, valid_idx in _folds_estandar(fold_idx):
        x_train, x_valid = _tfidf_char_menciones(textos, train_idx, valid_idx)
        y_train = y[train_idx]
        pair_margins = np.zeros((len(valid_idx), len(ADJACENT_PAIRS)), dtype=np.float32)
        for position, (lower, upper) in enumerate(ADJACENT_PAIRS):
            mask = (y_train == lower) | (y_train == upper)
            binary_target = (y_train[mask] == upper).astype(int)
            model = LinearSVC(C=0.2, max_iter=3000)
            model.fit(x_train[mask], binary_target)
            margin = model.decision_function(x_valid)
            pair_margins[:, position] = margin
        margenes[valid_idx] = pair_margins
    return margenes


def oof_adv05(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    r"""Margenes OOF de ADV05: NB-SVM uno-vs-rest sobre char300 + menciones.

    Porte de scripts/advanced_05_nbsvm_seleccion.py:73-94 (nb_svm_fold) y 138-146 (bucle
    OOF y guardado): por fold, TfidfVectorizer(**CHAR_300K) + TfidfVectorizer(**MENTION)
    sobre tokens_of(textos, r"@\w+"), log-count ratio por clase y LinearSVC(C=0.2,
    max_iter=3000) uno-vs-rest via nb_svm_margins. Salida (n, 6) float32.
    """
    y = np.asarray(y)
    config = dict(CHAR_300K)
    config["mentions"] = {"config": dict(MENTION), "pattern": r"@\w+"}
    margenes = np.zeros((len(textos), len(CLASSES)), dtype=np.float32)
    for _fold, train_idx, valid_idx in _folds_estandar(fold_idx):
        margenes[valid_idx] = nb_svm_margins(textos, y, train_idx, valid_idx, config)
    return margenes

### Módulo `meta.py`

In [ ]:
"""Meta-modelo y matrices del meta (porte fiel de phase2/phase3/phase7/phase25 + v19).

Ordenes EXACTOS:
- MEMBER_FILES (14 miembros, 78 columnas) + densas (35) + estructuradas (87) + supervisadas (73) = 273.
- matriz 279 (train) = 273 + ADV07 (6).  ASCII puro. Sin efectos en import.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler


#: Miembros y transformacion aplicada al construir el meta (scripts/phase2_ponderacion.py:49-55).
MEMBER_FILES = [("e013", "raw"), ("e016", "raw"), ("e009", "log"), ("e011", "log"),
                ("e010", "log"), ("e014", "raw"), ("e015_ordinal", "raw"),
                ("e015_softmax", "log"), ("e017", "log"), ("e019b", "log"),
                ("e021a", "log"), ("e021b", "log"), ("e028a", "raw"), ("adv05", "raw")]

#: Columnas densas usadas en el meta (35 de 38): se excluyen las 3 de emoji.
DENSE_EXCLUIDAS = {"n_emoji", "emoji_diversity", "has_emoji"}


class EncodedMLP:
    """MLP con etiquetas codificadas a enteros (scripts/phase2_ponderacion.py:106-125)."""

    def __init__(self) -> None:
        self.encoder = LabelEncoder()
        self.scaler = StandardScaler()
        self.model = MLPClassifier(hidden_layer_sizes=(64,), alpha=1e-3, batch_size=512,
                                   learning_rate_init=1e-3, max_iter=40, early_stopping=True,
                                   n_iter_no_change=5, random_state=SEED)

    def fit(self, features: np.ndarray, labels: np.ndarray) -> EncodedMLP:
        self.encoder.fit(labels)
        self.model.fit(self.scaler.fit_transform(features), self.encoder.transform(labels))
        return self

    def predict(self, features: np.ndarray) -> np.ndarray:
        return self.encoder.inverse_transform(self.model.predict(self.scaler.transform(features)))

    def predict_proba(self, features: np.ndarray) -> np.ndarray:
        crudo = self.model.predict_proba(self.scaler.transform(features))
        completa = np.zeros((len(features), len(CLASSES)))
        for posicion, codigo in enumerate(self.model.classes_):
            completa[:, int(codigo)] = crudo[:, posicion]
        return completa


def logreg() -> Pipeline:
    """LogReg del proyecto (scripts/v13_p1b_fusion_tokens.py:79-83)."""
    return Pipeline([("scaler", StandardScaler()),
                     ("clf", LogisticRegression(C=1.0, max_iter=3000, solver="lbfgs",
                                                random_state=SEED))])


def construir_miembros_oof(textos: pd.Series, y: np.ndarray, fold_idx: np.ndarray,
                           dense: pd.DataFrame) -> dict[str, np.ndarray]:
    """Reconstruye los 14 OOF de miembros (formas y dtypes de los artefactos guardados)."""
    return {
        "e013": oof_e013(textos, y, fold_idx),
        "e016": oof_e016(textos, y, fold_idx),
        "e009": oof_e009(textos, y, fold_idx),
        "e011": oof_e011(textos, y, fold_idx),
        "e010": oof_e010(textos, y, fold_idx),
        "e014": oof_e014(textos, y, fold_idx),
        "e015_ordinal": oof_e015_ordinal(textos, y, fold_idx),
        "e015_softmax": oof_e015_softmax(textos, y, fold_idx),
        "e017": oof_e017(textos, y, fold_idx),
        "e019b": oof_e019b(textos, y, fold_idx, dense),
        "e021a": oof_e021a(textos, y, fold_idx),
        "e021b": oof_e021b(textos, y, fold_idx),
        "e028a": oof_e028a(textos, y, fold_idx),
        "adv05": oof_adv05(textos, y, fold_idx),
    }


def build_features_273(oofs: dict[str, np.ndarray], densas35: np.ndarray,
                       estructuradas: np.ndarray, supervisadas: np.ndarray) -> np.ndarray:
    """Concatenacion exacta del meta (scripts/phase2_ponderacion.py:61-81)."""
    partes = []
    for nombre, transform in MEMBER_FILES:
        matriz = oofs[nombre]
        if matriz.ndim == 1:
            matriz = matriz.reshape(-1, 1)
        partes.append(np.log(np.clip(matriz, 1e-12, None)) if transform == "log" else matriz)
    features = np.concatenate([np.concatenate(partes, axis=1),
                               densas35, estructuradas, supervisadas], axis=1)
    assert features.shape[1] == 273, features.shape
    return features


def matriz_279(meta273: np.ndarray, adv07: np.ndarray) -> np.ndarray:
    """Matriz 279 del campeon (scripts/v19_b1_boosting.py:97-108)."""
    matriz = np.concatenate([meta273.astype(np.float32), adv07.astype(np.float32)], axis=1)
    assert matriz.shape[1] == 279, matriz.shape
    return matriz


def meta_v12_oof(meta273: np.ndarray, y: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    """OOF del combinador V12: 0.5*(LogReg + EncodedMLP) sobre 273 (scripts/phase8:80-107)."""
    n = len(y)
    salida = np.zeros((n, len(CLASSES)))
    for fold in range(5):
        tr = fold_idx != fold
        va = fold_idx == fold
        lr = logreg().fit(meta273[tr], y[tr])
        red = EncodedMLP().fit(meta273[tr], y[tr])
        p_lr = np.zeros((int(va.sum()), len(CLASSES)))
        for posicion, etiqueta in enumerate(lr.classes_):
            p_lr[:, CLASSES.index(str(etiqueta))] = lr.predict_proba(meta273[va])[:, posicion]
        p_red = red.predict_proba(meta273[va])
        salida[va] = 0.5 * (p_lr + p_red)
    return salida


def densas35(dense: pd.DataFrame) -> np.ndarray:
    """Las 35 columnas densas usadas en el meta (excluye las 3 de emoji)."""
    columnas = [c for c in dense.columns if c not in DENSE_EXCLUIDAS]
    return dense[columnas].to_numpy(float)

### Módulo `v30.py`

In [ ]:
"""V19/V20 (cuatro cuartos), ensemble V30 y regla plug-in (porte fiel de v19_b1/v20_f3/v30_03/age_plugin).

Configuraciones congeladas embebidas como constantes del protocolo (no se leen de ficheros):
- Pesos del ensemble V30: media de los vectores Optuna por fold, renormalizada (v30_02_ensemble.json).
- Configs V29 (reports/v29-01_trials.csv) y V30 (v30_01_optuna_refinamiento.json), en orden de fold.
- Pesos plug-in congelados (age_umbrales_validacion.json, commit bd08aba).
ASCII puro. Sin efectos en import.
"""

from __future__ import annotations

import lightgbm as lgb
import numpy as np


#: Arbol pre-registrado del proyecto (scripts/v19_b1_boosting.py:83-86).
HIERBA = {"objective": "multiclass", "num_class": len(CLASSES), "n_estimators": 300,
          "learning_rate": 0.05, "num_leaves": 8, "max_depth": 3, "min_child_samples": 200,
          "colsample_bytree": 0.5, "subsample": 0.8, "subsample_freq": 1, "reg_lambda": 1.0,
          "n_jobs": 32, "random_state": SEED, "verbose": -1}

#: Pesos del ensemble V30 (v30_02_ensemble.json -> manifest): v20, v29, v30.
PESOS_ENSEMBLE = (0.3548, 0.3354, 0.3098)

#: Pesos plug-in congelados (age_plugin_submission.py:79).
PESOS_PLUG_IN = np.array([1.0000, 1.1749, 1.2721, 1.1630, 1.0010, 0.8128])

#: Mejor trial por fold de V29 (reports/v29-01_trials.csv), en orden de fold.
V29_CONFIGS = [
    {"num_leaves": 87, "max_depth": 5, "learning_rate": 0.04301669500288421,
     "min_child_samples": 76, "subsample": 0.6581570543882859,
     "colsample_bytree": 0.70766450002428, "reg_alpha": 0.6147385859030499,
     "reg_lambda": 9.792145504680492e-05, "n_estimators": 816},
    {"num_leaves": 55, "max_depth": 6, "learning_rate": 0.03352698655336303,
     "min_child_samples": 54, "subsample": 0.8941120537508466,
     "colsample_bytree": 0.6299160105578635, "reg_alpha": 4.026902217415037e-07,
     "reg_lambda": 6.6850377528309e-07, "n_estimators": 1372},
    {"num_leaves": 57, "max_depth": 8, "learning_rate": 0.022124342973521553,
     "min_child_samples": 94, "subsample": 0.9225205673898446,
     "colsample_bytree": 0.52058407816027, "reg_alpha": 1.8772169842126896,
     "reg_lambda": 0.018194045769254683, "n_estimators": 912},
    {"num_leaves": 43, "max_depth": 9, "learning_rate": 0.03000286836410436,
     "min_child_samples": 22, "subsample": 0.6998254389554196,
     "colsample_bytree": 0.5250455092128341, "reg_alpha": 1.0090819656249398e-05,
     "reg_lambda": 7.87982213180386e-06, "n_estimators": 863},
    {"num_leaves": 56, "max_depth": 7, "learning_rate": 0.020811077189914432,
     "min_child_samples": 29, "subsample": 0.7762118181212365,
     "colsample_bytree": 0.6242271327287489, "reg_alpha": 1.3833851148235308e-05,
     "reg_lambda": 0.0004885693686069363, "n_estimators": 1084},
]

#: Mejor trial por fold de V30 (v30_01_optuna_refinamiento.json), en orden de fold.
V30_CONFIGS = [
    {"num_leaves": 41, "max_depth": 10, "learning_rate": 0.015612823179143214,
     "min_child_samples": 79, "subsample": 0.775913545060191,
     "colsample_bytree": 0.5969526009433974, "reg_alpha": 1.0941449983677585e-05,
     "reg_lambda": 0.00036896192791107643, "n_estimators": 1011},
    {"num_leaves": 32, "max_depth": 7, "learning_rate": 0.017387326275059966,
     "min_child_samples": 23, "subsample": 0.8215355497227395,
     "colsample_bytree": 0.8291548707315466, "reg_alpha": 0.8747434883808647,
     "reg_lambda": 2.053077365511584e-05, "n_estimators": 1486},
    {"num_leaves": 63, "max_depth": 7, "learning_rate": 0.02867292541012883,
     "min_child_samples": 110, "subsample": 0.7807106452264457,
     "colsample_bytree": 0.4021908926823267, "reg_alpha": 1.1486360263326448e-05,
     "reg_lambda": 0.00031823134119350724, "n_estimators": 1023},
    {"num_leaves": 42, "max_depth": 7, "learning_rate": 0.015729268043126833,
     "min_child_samples": 41, "subsample": 0.999685646683389,
     "colsample_bytree": 0.596195315949217, "reg_alpha": 0.004978399116554409,
     "reg_lambda": 1.8436108646384413e-08, "n_estimators": 1477},
    {"num_leaves": 64, "max_depth": 6, "learning_rate": 0.03818854692388361,
     "min_child_samples": 52, "subsample": 0.8731483359860639,
     "colsample_bytree": 0.46934472778342573, "reg_alpha": 1.927060050589197e-08,
     "reg_lambda": 0.004339321331656464, "n_estimators": 1000},
]

PARAMETROS_ENTEROS = ("num_leaves", "max_depth", "min_child_samples", "subsample_freq",
                      "n_estimators")


def a_clases(probabilidad: np.ndarray, clases, filas: int) -> np.ndarray:
    """Reordena columnas al orden canonico (v30_03_submission.py:72-90)."""
    salida = np.zeros((filas, len(CLASSES)))
    existentes = list(clases)
    son_enteros = all(isinstance(v, (int, np.integer)) for v in existentes)
    for destino, etiqueta in enumerate(CLASSES):
        buscar = destino if son_enteros else etiqueta
        salida[:, destino] = probabilidad[:, existentes.index(buscar)]
    return salida


def _normaliza(params: dict) -> dict:
    """Convierte a int los parametros enteros de LightGBM (v30_03_submission.py:139-146)."""
    salida = dict(params)
    for clave in PARAMETROS_ENTEROS:
        if clave in salida and salida[clave] is not None:
            salida[clave] = int(round(float(salida[clave])))
    return salida


def _arbol(semilla: int = SEED) -> lgb.LGBMClassifier:
    """Arbol HIERBA con semilla (scripts/v19_b1_boosting.py:110-115)."""
    return lgb.LGBMClassifier(**{**HIERBA, "random_state": semilla})


def v19_oofs(matriz: np.ndarray, y: np.ndarray, clase: np.ndarray, fold_idx: np.ndarray
             ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """OOF de los tres miembros de V19 (logreg, mlp, arbol) sobre la matriz 279."""
    n = len(y)
    p_lr = np.zeros((n, 6))
    p_mlp = np.zeros((n, 6))
    p_arbol = np.zeros((n, 6))
    for fold in range(5):
        tr = fold_idx != fold
        va = fold_idx == fold
        lr = logreg().fit(matriz[tr], y[tr])
        completa = np.zeros((int(va.sum()), 6))
        for posicion, nombre in enumerate(lr.classes_):
            completa[:, CLASSES.index(str(nombre))] = lr.predict_proba(matriz[va])[:, posicion]
        p_lr[va] = completa
        red = EncodedMLP().fit(matriz[tr], y[tr])
        p_mlp[va] = red.predict_proba(matriz[va])
        modelo = _arbol(SEED).fit(matriz[tr], y[tr])
        p_arbol[va] = a_clases(modelo.predict_proba(matriz[va]), modelo.classes_, int(va.sum()))
    return p_lr, p_mlp, p_arbol


def v20_cadena_oof(matriz: np.ndarray, clase: np.ndarray, fold_idx: np.ndarray) -> np.ndarray:
    """Cadena de 5 umbrales anidados, OOF (scripts/v20_f3_umbrales.py:145-172 + 196-206)."""
    n = len(clase)
    n_clases = len(CLASSES)
    acumuladas = np.zeros((n, n_clases - 1))
    for fold in range(5):
        tr = fold_idx != fold
        va = fold_idx == fold
        for k in range(1, n_clases):
            objetivo = (clase <= k - 1).astype(int)
            modelo = lgb.LGBMClassifier(**HIERBA)
            modelo.fit(matriz[tr], objetivo[tr])
            acumuladas[va, k - 1] = modelo.predict_proba(matriz[va])[:, 1]
    for k in range(1, n_clases - 1):
        acumuladas[:, k] = np.maximum(acumuladas[:, k], acumuladas[:, k - 1])
    seis = np.zeros((n, n_clases))
    seis[:, 0] = acumuladas[:, 0]
    for k in range(1, n_clases - 1):
        seis[:, k] = acumuladas[:, k] - acumuladas[:, k - 1]
    seis[:, n_clases - 1] = 1.0 - acumuladas[:, n_clases - 2]
    seis = np.clip(seis, 0.0, None)
    return seis / np.clip(seis.sum(axis=1, keepdims=True), 1e-9, None)


def v20_oof(matriz: np.ndarray, y: np.ndarray, clase: np.ndarray, fold_idx: np.ndarray
            ) -> np.ndarray:
    """OOF del candidato V20: 0.25 de cada cuarto (v20_f3_umbrales.py:207-213)."""
    p_lr, p_mlp, p_arbol = v19_oofs(matriz, y, clase, fold_idx)
    seis = v20_cadena_oof(matriz, clase, fold_idx)
    return 0.25 * p_lr + 0.25 * p_mlp + 0.25 * p_arbol + 0.25 * seis


def v20_test(matriz: np.ndarray, matriz_test: np.ndarray, y: np.ndarray, clase: np.ndarray
             ) -> np.ndarray:
    """Receta exacta de V20 sobre el train completo (v30_03_submission.py:93-117)."""
    n_test = matriz_test.shape[0]
    modelo_lr = logreg().fit(matriz, y)
    prob_lr = a_clases(modelo_lr.predict_proba(matriz_test), modelo_lr.classes_, n_test)
    red = EncodedMLP().fit(matriz, y)
    prob_mlp = a_clases(red.model.predict_proba(red.scaler.transform(matriz_test)),
                        red.encoder.classes_, n_test)
    modelo_arbol = lgb.LGBMClassifier(**HIERBA).fit(matriz, y)
    prob_arbol = a_clases(modelo_arbol.predict_proba(matriz_test), modelo_arbol.classes_, n_test)
    acumuladas = np.zeros((n_test, len(CLASSES) - 1))
    for k in range(1, len(CLASSES)):
        objetivo = (clase <= k - 1).astype(int)
        modelo = lgb.LGBMClassifier(**HIERBA).fit(matriz, objetivo)
        acumuladas[:, k - 1] = modelo.predict_proba(matriz_test)[:, 1]
    for k in range(1, len(CLASSES) - 1):
        acumuladas[:, k] = np.maximum(acumuladas[:, k], acumuladas[:, k - 1])
    seis = np.zeros((n_test, len(CLASSES)))
    seis[:, 0] = acumuladas[:, 0]
    for k in range(1, len(CLASSES) - 1):
        seis[:, k] = acumuladas[:, k] - acumuladas[:, k - 1]
    seis[:, len(CLASSES) - 1] = 1.0 - acumuladas[:, len(CLASSES) - 2]
    seis = np.clip(seis, 0.0, None)
    seis = seis / np.clip(seis.sum(axis=1, keepdims=True), 1e-9, None)
    return 0.25 * prob_lr + 0.25 * prob_mlp + 0.25 * prob_arbol + 0.25 * seis


def lgbm_refit_oof(matriz: np.ndarray, clase: np.ndarray, fold_idx: np.ndarray,
                   configs: list[dict], base: dict | None = None) -> np.ndarray:
    """OOF de un LightGBM con la configuracion congelada por fold y early stopping del original.

    Porte de ``scripts/v29_04_optuna_individual.py:243-266`` (``_entrena``, rama lightgbm; la
    base incluye ``random_state=SEED``) y ``scripts/v30_01_optuna_refinamiento.py:164-176``
    (``_entrena``; la base NO incluye ``random_state``). El early stopping (60 rondas de
    ``multi_logloss`` sobre el fold retenido) es parte del protocolo original y el corte
    ocurrio de hecho en los reajustes documentados.

    Nota de idempotencia: se anaden ``deterministic=True`` y ``force_col_wise=True`` porque la
    construccion multihilo de histogramas de LightGBM no es estable entre procesos con la
    configuracion original (medido con un probe de dos procesos: el OOF reconstruido cambiaba);
    con ambos parametros dos corridas completas reproducen el mismo resultado bit a bit.
    """
    n = len(clase)
    oof = np.zeros((n, len(CLASSES)))
    for fold in range(5):
        tr = fold_idx != fold
        va = fold_idx == fold
        params = {"objective": "multiclass", "num_class": len(CLASSES), "verbose": -1,
                  "n_jobs": 32, "subsample_freq": 1, "deterministic": True,
                  "force_col_wise": True, **(base or {}),
                  **_normaliza(configs[fold])}
        techo = int(params.pop("n_estimators"))
        modelo = lgb.LGBMClassifier(**params, n_estimators=techo)
        modelo.fit(matriz[tr], clase[tr],
                   eval_set=[(matriz[va], clase[va])], eval_metric="multi_logloss",
                   callbacks=[lgb.early_stopping(60, verbose=False)])
        oof[va] = a_clases(modelo.predict_proba(matriz[va]), modelo.classes_, int(va.sum()))
    return oof


def lgbm_refit_test(matriz: np.ndarray, clase: np.ndarray, matriz_test: np.ndarray,
                    configs: list[dict]) -> np.ndarray:
    """Cinco reajustes (uno por configuracion congelada) sobre todo el train, promediados.

    Incluye ``deterministic=True`` y ``force_col_wise=True`` por la misma razon que
    ``lgbm_refit_oof`` (estabilidad entre procesos de la construccion de histogramas).
    """
    n_test = matriz_test.shape[0]
    predicciones = []
    for params in configs:
        p = {"objective": "multiclass", "num_class": len(CLASSES), "verbose": -1,
             "n_jobs": 28, "subsample_freq": 1, "deterministic": True,
             "force_col_wise": True, **_normaliza(params)}
        techo = int(p.pop("n_estimators"))
        modelo = lgb.LGBMClassifier(**p, n_estimators=techo)
        modelo.fit(matriz, clase)
        predicciones.append(a_clases(modelo.predict_proba(matriz_test), modelo.classes_, n_test))
    return np.mean(predicciones, axis=0)


def mezcla(p20: np.ndarray, p29: np.ndarray, p30: np.ndarray) -> np.ndarray:
    """Mezcla del ensemble V30 con los pesos congelados (v30_03_submission.py:232-240)."""
    vector = np.array(PESOS_ENSEMBLE)
    vector = vector / vector.sum()
    return np.tensordot(vector, np.stack([p20, p29, p30]), axes=(0, 0))


def regla_plug_in(prob: np.ndarray) -> np.ndarray:
    """Decision plug-in por clase: argmax(prob * PESOS_PLUG_IN) (age_plugin_submission.py:258-261)."""
    return np.argmax(prob * PESOS_PLUG_IN, axis=1)


def macro_f1(clase: np.ndarray, pred: np.ndarray) -> float:
    """Macro-F1 sobre indices 0-5 (sin dependencia de sklearn para el chequeo rapido)."""
    from sklearn.metrics import f1_score
    return float(f1_score(clase, pred, average="macro", labels=list(range(len(CLASSES)))))


def _cuartos_v20_fold(matriz: np.ndarray, clase: np.ndarray, y: np.ndarray,
                      tren: np.ndarray, val: np.ndarray) -> np.ndarray:
    """Los cuatro cuartos de V20 sobre `tren`, medidos en `val` (v30_05:146-175)."""
    prob_lr = a_clases(logreg().fit(matriz[tren], y[tren]).predict_proba(matriz[val]),
                       np.array(CLASSES), len(val))
    red = EncodedMLP().fit(matriz[tren], y[tren])
    prob_mlp = a_clases(red.model.predict_proba(red.scaler.transform(matriz[val])),
                        red.encoder.classes_, len(val))
    prob_arbol = a_clases(lgb.LGBMClassifier(**HIERBA).fit(matriz[tren], y[tren])
                          .predict_proba(matriz[val]), np.array(CLASSES), len(val))
    acumuladas = np.zeros((len(val), len(CLASSES) - 1))
    for k in range(1, len(CLASSES)):
        objetivo = (clase <= k - 1).astype(int)
        modelo = lgb.LGBMClassifier(**HIERBA).fit(matriz[tren], objetivo[tren])
        acumuladas[:, k - 1] = modelo.predict_proba(matriz[val])[:, 1]
    for k in range(1, len(CLASSES) - 1):
        acumuladas[:, k] = np.maximum(acumuladas[:, k], acumuladas[:, k - 1])
    seis = np.zeros((len(val), len(CLASSES)))
    seis[:, 0] = acumuladas[:, 0]
    for k in range(1, len(CLASSES) - 1):
        seis[:, k] = acumuladas[:, k] - acumuladas[:, k - 1]
    seis[:, len(CLASSES) - 1] = 1.0 - acumuladas[:, len(CLASSES) - 2]
    seis = np.clip(seis, 0.0, None)
    seis = seis / np.clip(seis.sum(axis=1, keepdims=True), 1e-9, None)
    return 0.25 * prob_lr + 0.25 * prob_mlp + 0.25 * prob_arbol + 0.25 * seis


def cuartos_v20_oof(matriz: np.ndarray, y: np.ndarray, clase: np.ndarray,
                    fold_idx: np.ndarray) -> np.ndarray:
    """OOF de los cuatro cuartos por fold (protocolo de la segunda particion, v30_05:146-175)."""
    n = len(clase)
    oof = np.zeros((n, len(CLASSES)))
    for fold in range(5):
        tren = np.flatnonzero(fold_idx != fold)
        val = np.flatnonzero(fold_idx == fold)
        oof[val] = _cuartos_v20_fold(matriz, clase, y, tren, val)
    return oof


def lgbm_refit_oof_sin_es(matriz: np.ndarray, clase: np.ndarray, y: np.ndarray,
                          fold_idx: np.ndarray, configs: list[dict]) -> np.ndarray:
    """Refit del techo congelado SIN early stopping (age_plugin_confirmacion.py:141-147).

    Incluye ``deterministic=True`` y ``force_col_wise=True``: sin ellos este reajuste NO era
    estable entre procesos (medido con el probe de determinismo), y es el miembro V30 de la
    segunda particion y de la evaluacion publicada del campeon.
    """
    n = len(clase)
    oof = np.zeros((n, len(CLASSES)))
    for fold in range(5):
        tr = fold_idx != fold
        va = fold_idx == fold
        params = {"objective": "multiclass", "num_class": len(CLASSES), "verbose": -1,
                  "n_jobs": 28, "subsample_freq": 1, "deterministic": True,
                  "force_col_wise": True, **_normaliza(configs[fold])}
        techo = int(params.pop("n_estimators"))
        modelo = lgb.LGBMClassifier(**params, n_estimators=techo).fit(matriz[tr], y[tr])
        oof[va] = a_clases(modelo.predict_proba(matriz[va]), modelo.classes_, int(va.sum()))
    return oof


def v30_oof_segunda(matriz: np.ndarray, y: np.ndarray, clase: np.ndarray,
                    fold_idx2: np.ndarray) -> np.ndarray:
    """V30 OOF en la particion alternativa (age_plugin_confirmacion.py:126-156).

    Misma receta y pesos congelados, con los reajustes del techo SIN early stopping (protocolo
    propio de la segunda particion) y los cuatro cuartos por fold.
    """
    p20 = cuartos_v20_oof(matriz, y, clase, fold_idx2)
    p29 = lgbm_refit_oof_sin_es(matriz, clase, y, fold_idx2, V29_CONFIGS)
    p30 = lgbm_refit_oof_sin_es(matriz, clase, y, fold_idx2, V30_CONFIGS)
    return mezcla(p20, p29, p30)

### Módulo `neuronales.py`

In [ ]:
"""Componentes neuronales V38-V53 + HGB (porte fiel de scripts/v38_01..v53_01).

Arquitecturas, perdidas, semillas, SWA y estrategias de test EXACTAS de cada script original:
- V38/V39: TabularResNet, seeds 42+fold, 14 epocas, Cosine(T_max=14), sin SWA; test = modelo full-train.
- V40: SETabularResNet, OrdinalDistanceLoss, seeds 42+fold*100, 12 epocas, sin scheduler, sin SWA; test = media/5.
- V41: SETabularResNet, OrdinalDistanceLoss, 16 epocas, sin scheduler, SWA swa_start=8; test = media/5.
- V42: SETabularResNet, SmoothOrdinalDistanceLoss, 16 epocas, Cosine(T_max=16, eta_min=1e-4), SWA swa_start=6; test = media/5.
- V43: SETabularResNet, CramerDistanceLoss, 16 epocas, Cosine(T_max=16, eta_min=1e-4), SWA swa_start=6; test = media/5.
- V47: SETabularResNet, WeightedCramerLoss (gamma por frontera), recorte Z [-6,+6], seeds 42+fold*100,
  16 epocas, Cosine(T_max=16, eta_min=1e-4), SWA swa_start=6 con update_bn en loader 512 sin barajar; test = media/5.
- V51: SETabularResNet, ConsistentSmoothedCramerLoss, 16 epocas, Cosine(T_max=16, eta_min=1e-4), SWA swa_start=6; test = media/5.
- V53: TabularDenseNet (init 192, growth 128) con SEChannelBlock, ConsistentSmoothedCramerLoss, recorte Z [-6,+6],
  16 epocas, Cosine(T_max=16, eta_min=1e-4), SWA swa_start=6; test = media/5.
- V39-HGB: HistGradientBoostingClassifier por fold; test = modelo full-train.
ASCII puro. Sin efectos en import.
"""

from __future__ import annotations

import numpy as np
import torch
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from torch import nn, optim
from torch.optim.swa_utils import AveragedModel, update_bn
from torch.utils.data import DataLoader, TensorDataset


#: Configuracion exacta por componente (pack 04; los scripts originales mandan).
CONFIGS: dict[str, dict] = {
    "v38": {"modelo": "tabular", "loss": "ce", "epocas": 14, "batch": 256, "lr": 1e-3,
            "wd": 1e-4, "coseno": True, "eta_min": 0.0, "swa_start": None,
            "seed": "42+fold", "test": "full"},
    "v39_ordinal": {"modelo": "tabular", "loss": "ordinal_ce", "epocas": 14, "batch": 256,
                    "lr": 1e-3, "wd": 1e-4, "coseno": True, "eta_min": 0.0, "swa_start": None,
                    "seed": "42+fold", "test": "full"},
    "v40": {"modelo": "se", "loss": "ordinal_dist", "epocas": 12, "batch": 256, "lr": 1e-3,
            "wd": 1e-4, "coseno": False, "eta_min": 0.0, "swa_start": None,
            "seed": "42+fold*100", "test": "media"},
    "v41": {"modelo": "se", "loss": "ordinal_dist", "epocas": 16, "batch": 256, "lr": 1e-3,
            "wd": 1e-4, "coseno": False, "eta_min": 0.0, "swa_start": 8,
            "seed": "42+fold*100", "test": "media"},
    "v42": {"modelo": "se", "loss": "smooth_ordinal", "epocas": 16, "batch": 256, "lr": 1e-3,
            "wd": 1e-4, "coseno": True, "eta_min": 1e-4, "swa_start": 6,
            "seed": "42+fold*100", "test": "media"},
    "v43": {"modelo": "se", "loss": "cramer", "epocas": 16, "batch": 256, "lr": 1e-3,
            "wd": 1e-4, "coseno": True, "eta_min": 1e-4, "swa_start": 6,
            "seed": "42+fold*100", "test": "media"},
    "v47": {"modelo": "se", "loss": "weighted_cramer", "epocas": 16, "batch": 256, "lr": 1e-3,
            "wd": 1e-4, "coseno": True, "eta_min": 1e-4, "swa_start": 6,
            "seed": "42+fold*100", "test": "media", "clip": 6.0, "bn_batch": 512},
    "v51": {"modelo": "se", "loss": "consistent_cramer", "epocas": 16, "batch": 256, "lr": 1e-3,
            "wd": 1e-4, "coseno": True, "eta_min": 1e-4, "swa_start": 6,
            "seed": "42+fold*100", "test": "media"},
    "v53": {"modelo": "dense_se", "loss": "consistent_cramer", "epocas": 16, "batch": 256,
            "lr": 1e-3, "wd": 1e-4, "coseno": True, "eta_min": 1e-4, "swa_start": 6,
            "seed": "42+fold*100", "test": "media", "clip": 6.0},
}


class TabularResNet(nn.Module):
    """Red residual tabular (scripts/v38_01_tabularnet_5folds.py:44-71)."""

    def __init__(self, in_features: int = 279, hidden: int = 256, num_classes: int = 6,
                 drop: float = 0.2):
        super().__init__()
        self.in_proj = nn.Linear(in_features, hidden)
        self.bn0 = nn.BatchNorm1d(hidden)
        self.act0 = nn.SiLU()
        self.fc1 = nn.Linear(hidden, hidden)
        self.bn1 = nn.BatchNorm1d(hidden)
        self.act1 = nn.SiLU()
        self.drop1 = nn.Dropout(drop)
        self.fc2 = nn.Linear(hidden, hidden)
        self.bn2 = nn.BatchNorm1d(hidden)
        self.act2 = nn.SiLU()
        self.drop2 = nn.Dropout(drop)
        self.head = nn.Linear(hidden, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h0 = self.act0(self.bn0(self.in_proj(x)))
        h1 = h0 + self.drop1(self.act1(self.bn1(self.fc1(h0))))
        h2 = h1 + self.drop2(self.act2(self.bn2(self.fc2(h1))))
        return self.head(h2)


class SETabularResNet(nn.Module):
    """Red residual con compuertas Squeeze-and-Excitation (scripts/v40_01..v43_01)."""

    def __init__(self, in_features: int = 279, hidden: int = 256, num_classes: int = 6,
                 drop: float = 0.2, reduction: int = 4):
        super().__init__()
        self.in_proj = nn.Linear(in_features, hidden)
        self.bn0 = nn.BatchNorm1d(hidden)
        self.act0 = nn.SiLU()
        self.fc1 = nn.Linear(hidden, hidden)
        self.bn1 = nn.BatchNorm1d(hidden)
        self.act1 = nn.SiLU()
        self.drop1 = nn.Dropout(drop)
        self.se1_fc1 = nn.Linear(hidden, hidden // reduction)
        self.se1_act = nn.SiLU()
        self.se1_fc2 = nn.Linear(hidden // reduction, hidden)
        self.se1_sig = nn.Sigmoid()
        self.fc2 = nn.Linear(hidden, hidden)
        self.bn2 = nn.BatchNorm1d(hidden)
        self.act2 = nn.SiLU()
        self.drop2 = nn.Dropout(drop)
        self.se2_fc1 = nn.Linear(hidden, hidden // reduction)
        self.se2_act = nn.SiLU()
        self.se2_fc2 = nn.Linear(hidden // reduction, hidden)
        self.se2_sig = nn.Sigmoid()
        self.head = nn.Linear(hidden, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h0 = self.act0(self.bn0(self.in_proj(x)))
        res1 = self.drop1(self.act1(self.bn1(self.fc1(h0))))
        se1 = self.se1_sig(self.se1_fc2(self.se1_act(self.se1_fc1(res1))))
        h1 = h0 + res1 * se1
        res2 = self.drop2(self.act2(self.bn2(self.fc2(h1))))
        se2 = self.se2_sig(self.se2_fc2(self.se2_act(self.se2_fc1(res2))))
        h2 = h1 + res2 * se2
        return self.head(h2)


class SEChannelBlock(nn.Module):
    """Compuerta Squeeze-and-Excitation por canal (scripts/v53_01_dense_se_tabularnet_5folds.py:65-77)."""

    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.act = nn.SiLU()
        self.fc2 = nn.Linear(channels // reduction, channels)
        self.sig = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        w = self.sig(self.fc2(self.act(self.fc1(x))))
        return x * w


class TabularDenseNet(nn.Module):
    """Red densa tabular con concatenacion multi-escala y compuertas SE (scripts/v53_01:80-142)."""

    def __init__(self, in_features: int = 279, init_dim: int = 192, growth_rate: int = 128,
                 num_classes: int = 6, drop: float = 0.20):
        super().__init__()
        self.in_proj = nn.Linear(in_features, init_dim)
        self.bn0 = nn.BatchNorm1d(init_dim)
        self.act0 = nn.SiLU()
        self.bn1 = nn.BatchNorm1d(init_dim)
        self.fc1 = nn.Linear(init_dim, growth_rate)
        self.act1 = nn.SiLU()
        self.drop1 = nn.Dropout(drop)
        self.se1 = SEChannelBlock(growth_rate, reduction=4)
        in_dim2 = init_dim + growth_rate
        self.bn2 = nn.BatchNorm1d(in_dim2)
        self.fc2 = nn.Linear(in_dim2, growth_rate)
        self.act2 = nn.SiLU()
        self.drop2 = nn.Dropout(drop)
        self.se2 = SEChannelBlock(growth_rate, reduction=4)
        in_dim3 = in_dim2 + growth_rate
        self.bn3 = nn.BatchNorm1d(in_dim3)
        self.fc3 = nn.Linear(in_dim3, growth_rate)
        self.act3 = nn.SiLU()
        self.drop3 = nn.Dropout(drop)
        self.se3 = SEChannelBlock(growth_rate, reduction=4)
        total_dense_dim = in_dim3 + growth_rate
        self.bn_trans = nn.BatchNorm1d(total_dense_dim)
        self.fc_trans = nn.Linear(total_dense_dim, 256)
        self.act_trans = nn.SiLU()
        self.drop_trans = nn.Dropout(0.10)
        self.head = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h0 = self.act0(self.bn0(self.in_proj(x)))
        z1 = self.se1(self.drop1(self.act1(self.fc1(self.bn1(h0)))))
        h1 = torch.cat([h0, z1], dim=1)
        z2 = self.se2(self.drop2(self.act2(self.fc2(self.bn2(h1)))))
        h2 = torch.cat([h1, z2], dim=1)
        z3 = self.se3(self.drop3(self.act3(self.fc3(self.bn3(h2)))))
        h3 = torch.cat([h2, z3], dim=1)
        ht = self.drop_trans(self.act_trans(self.fc_trans(self.bn_trans(h3))))
        return self.head(ht)


class OrdinalCrossEntropyLoss(nn.Module):
    """CE + distancia ordinal esperada (scripts/v39_01: OrdinalCrossEntropyLoss)."""

    def __init__(self, num_classes: int = 6, lambda_dist: float = 0.2):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self.lambda_dist = lambda_dist
        clases = torch.arange(num_classes, dtype=torch.float32)
        dist_mat = torch.abs(clases.unsqueeze(0) - clases.unsqueeze(1))
        self.register_buffer("dist_mat", dist_mat)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        loss_ce = self.ce(logits, targets)
        probs = torch.softmax(logits, dim=1)
        batch_dists = self.dist_mat[targets]
        dist_loss = torch.sum(probs * batch_dists, dim=1).mean()
        return loss_ce + self.lambda_dist * dist_loss


class OrdinalDistanceLoss(nn.Module):
    """CE + distancia ordinal esperada (scripts/v40_01/v41_01)."""

    def __init__(self, num_classes: int = 6, lambda_dist: float = 0.2):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self.lambda_dist = lambda_dist
        c = torch.arange(num_classes).float()
        self.register_buffer("dist_matrix", torch.abs(c.unsqueeze(0) - c.unsqueeze(1)))

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        loss_ce = self.ce(logits, targets)
        probs = torch.softmax(logits, dim=1)
        dists = self.dist_matrix[targets]
        loss_dist = torch.sum(probs * dists, dim=1).mean()
        return loss_ce + self.lambda_dist * loss_dist


class SmoothOrdinalDistanceLoss(nn.Module):
    """CE con suavizado 0.10 + distancia ordinal (scripts/v42_01)."""

    def __init__(self, num_classes: int = 6, lambda_dist: float = 0.2, smoothing: float = 0.10):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(label_smoothing=smoothing)
        self.lambda_dist = lambda_dist
        c = torch.arange(num_classes).float()
        self.register_buffer("dist_matrix", torch.abs(c.unsqueeze(0) - c.unsqueeze(1)))

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        loss_ce = self.ce(logits, targets)
        probs = torch.softmax(logits, dim=1)
        dists = self.dist_matrix[targets]
        loss_dist = torch.sum(probs * dists, dim=1).mean()
        return loss_ce + self.lambda_dist * loss_dist


class CramerDistanceLoss(nn.Module):
    """CE (suavizado 0.10) + Cramer-von Mises sobre la CDF (scripts/v43_01)."""

    def __init__(self, num_classes: int = 6, lambda_cramer: float = 0.50, smoothing: float = 0.10):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(label_smoothing=smoothing)
        self.num_classes = num_classes
        self.lambda_cramer = lambda_cramer

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        loss_ce = self.ce(logits, targets)
        probs = torch.softmax(logits, dim=1)
        cdf_pred = torch.cumsum(probs, dim=1)[:, :-1]
        k_indices = torch.arange(self.num_classes - 1, device=logits.device).unsqueeze(0)
        cdf_target = (targets.unsqueeze(1) <= k_indices).float()
        loss_cramer = torch.mean(torch.sum((cdf_pred - cdf_target) ** 2, dim=1))
        return loss_ce + self.lambda_cramer * loss_cramer


class WeightedCramerLoss(nn.Module):
    """CE (suavizado 0.10) + Cramer ponderada por frontera (scripts/v47_01:96-120)."""

    def __init__(self, num_classes: int = 6, lambda_cramer: float = 0.50, smoothing: float = 0.10,
                 gamma: list[float] | None = None):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(label_smoothing=smoothing)
        self.num_classes = num_classes
        self.lambda_cramer = lambda_cramer
        if gamma is None:
            gamma = [1.2, 1.5, 1.3, 1.1, 1.0]
        self.register_buffer("gamma", torch.as_tensor(gamma, dtype=torch.float32))

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        loss_ce = self.ce(logits, targets)
        probs = torch.softmax(logits, dim=1)
        cdf_pred = torch.cumsum(probs, dim=1)[:, :-1]
        k_indices = torch.arange(self.num_classes - 1, device=logits.device).unsqueeze(0)
        cdf_target = (targets.unsqueeze(1) <= k_indices).float()
        diff_sq = (cdf_pred - cdf_target) ** 2
        weighted_sq = diff_sq * self.gamma.unsqueeze(0)
        loss_cramer = torch.mean(torch.sum(weighted_sq, dim=1))
        return loss_ce + self.lambda_cramer * loss_cramer


class ConsistentSmoothedCramerLoss(nn.Module):
    """CE (suavizado 0.10) + Cramer sobre CDF objetivo suavizada (scripts/v51_01:53-74; v53_01:145-163)."""

    def __init__(self, num_classes: int = 6, lambda_cramer: float = 0.50, smoothing: float = 0.10):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(label_smoothing=smoothing)
        self.num_classes = num_classes
        self.lambda_cramer = lambda_cramer
        self.smoothing = smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        loss_ce = self.ce(logits, targets)
        probs = torch.softmax(logits, dim=1)
        cdf_pred = torch.cumsum(probs, dim=1)[:, :-1]
        k_indices = torch.arange(self.num_classes - 1, device=logits.device).unsqueeze(0)
        cdf_step = (targets.unsqueeze(1) <= k_indices).float()
        cdf_target = ((1.0 - self.smoothing) * cdf_step
                      + self.smoothing * (k_indices + 1.0) / self.num_classes)
        loss_cramer = torch.mean(torch.sum((cdf_pred - cdf_target) ** 2, dim=1))
        return loss_ce + self.lambda_cramer * loss_cramer


def _construir_modelo(nombre: str, n_features: int, n_clases: int) -> nn.Module:
    """Crea la arquitectura exacta del componente."""
    tipo = CONFIGS[nombre]["modelo"]
    if tipo == "tabular":
        return TabularResNet(in_features=n_features, hidden=256, num_classes=n_clases,
                             drop=0.2)
    if tipo == "dense_se":
        return TabularDenseNet(in_features=n_features, init_dim=192, growth_rate=128,
                               num_classes=n_clases, drop=0.20)
    return SETabularResNet(in_features=n_features, hidden=256, num_classes=n_clases,
                           drop=0.2, reduction=4)


def _construir_loss(nombre: str, n_clases: int) -> nn.Module:
    """Crea la perdida exacta del componente."""
    tipo = CONFIGS[nombre]["loss"]
    if tipo == "ce":
        return nn.CrossEntropyLoss()
    if tipo == "ordinal_ce":
        return OrdinalCrossEntropyLoss(num_classes=n_clases, lambda_dist=0.2)
    if tipo == "ordinal_dist":
        return OrdinalDistanceLoss(num_classes=n_clases, lambda_dist=0.2)
    if tipo == "smooth_ordinal":
        return SmoothOrdinalDistanceLoss(num_classes=n_clases, lambda_dist=0.2, smoothing=0.10)
    if tipo == "cramer":
        return CramerDistanceLoss(num_classes=n_clases, lambda_cramer=0.50, smoothing=0.10)
    if tipo == "weighted_cramer":
        return WeightedCramerLoss(num_classes=n_clases, lambda_cramer=0.50, smoothing=0.10,
                                  gamma=[1.2, 1.5, 1.3, 1.1, 1.0])
    if tipo == "consistent_cramer":
        return ConsistentSmoothedCramerLoss(num_classes=n_clases, lambda_cramer=0.50,
                                            smoothing=0.10)
    raise ValueError(f"loss desconocida: {tipo}")


def _semilla(nombre: str, fold: int) -> int:
    """Semilla exacta por fold de cada componente."""
    return SEED + fold if CONFIGS[nombre]["seed"] == "42+fold" else SEED + fold * 100


def _entrenar_una(dispositivo, cfg: dict, semilla: int, X_tr: np.ndarray, y_tr: np.ndarray,
                   n_features: int, n_clases: int, nombre: str) -> nn.Module:
    """Entrena un modelo con la receta exacta del componente (con o sin SWA)."""
    torch.manual_seed(semilla)
    np.random.seed(semilla)
    base_model = _construir_modelo(nombre, n_features, n_clases).to(dispositivo)
    criterion = _construir_loss(nombre, n_clases).to(dispositivo)
    optimizer = optim.AdamW(base_model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    scheduler = None
    if cfg["coseno"]:
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=cfg["epocas"], eta_min=cfg["eta_min"])
    ds = TensorDataset(torch.from_numpy(X_tr).float(), torch.from_numpy(y_tr).long())
    loader = DataLoader(ds, batch_size=cfg["batch"], shuffle=True)
    swa_model = AveragedModel(base_model) if cfg["swa_start"] is not None else None
    for ep in range(cfg["epocas"]):
        base_model.train()
        for bx, by in loader:
            bx, by = bx.to(dispositivo), by.to(dispositivo)
            optimizer.zero_grad()
            out = base_model(bx)
            loss = criterion(out, by)
            loss.backward()
            optimizer.step()
        if scheduler is not None:
            scheduler.step()
        if swa_model is not None and ep >= cfg["swa_start"]:
            swa_model.update_parameters(base_model)
    if swa_model is not None:
        if cfg.get("bn_batch") is None:
            update_bn(loader, swa_model, device=dispositivo)
        else:
            loader_bn = DataLoader(ds, batch_size=cfg["bn_batch"], shuffle=False)
            update_bn(loader_bn, swa_model, device=dispositivo)
        modelo = swa_model
    else:
        modelo = base_model
    modelo.eval()
    return modelo


def _predecir(modelo: nn.Module, X: np.ndarray, dispositivo) -> np.ndarray:
    """Softmax en eval sobre una matriz (sin gradientes)."""
    with torch.no_grad():
        tensor = torch.from_numpy(X).float().to(dispositivo)
        return torch.softmax(modelo(tensor), dim=1).cpu().numpy()


def entrenar_componente(nombre: str, X_train: np.ndarray, X_test: np.ndarray, y: np.ndarray,
                        fold_idx: np.ndarray, dispositivo: str | None = None
                        ) -> tuple[np.ndarray, np.ndarray]:
    """Reentrena un componente V38-V53 y devuelve (oof, test) con las formas originales."""
    cfg = CONFIGS[nombre]
    if dispositivo is None:
        dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
    n, n_features = X_train.shape
    n_test = X_test.shape[0]
    n_clases = len(CLASSES)
    oof = np.zeros((n, n_clases), dtype=np.float32)
    test = np.zeros((n_test, n_clases), dtype=np.float32)
    for fold in range(5):
        tr = fold_idx != fold
        va = fold_idx == fold
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_train[tr])
        X_val = scaler.transform(X_train[va])
        X_te = scaler.transform(X_test)
        if cfg.get("clip") is not None:
            X_tr = np.clip(X_tr, -cfg["clip"], cfg["clip"])
            X_val = np.clip(X_val, -cfg["clip"], cfg["clip"])
            X_te = np.clip(X_te, -cfg["clip"], cfg["clip"])
        modelo = _entrenar_una(dispositivo, cfg, _semilla(nombre, fold), X_tr, y[tr],
                               n_features, n_clases, nombre)
        oof[va] = _predecir(modelo, X_val, dispositivo)
        if cfg["test"] == "media":
            test += _predecir(modelo, X_te, dispositivo) / 5.0
    if cfg["test"] == "full":
        scaler = StandardScaler()
        X_full = scaler.fit_transform(X_train)
        X_te = scaler.transform(X_test)
        modelo = _entrenar_una(dispositivo, cfg, SEED, X_full, y, n_features, n_clases,
                               nombre)
        test = _predecir(modelo, X_te, dispositivo)
    return oof, test


def hgb_componente(X_train: np.ndarray, X_test: np.ndarray, y: np.ndarray,
                   fold_idx: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """V39-HGB: HistGradientBoosting por fold (OOF) y full-train (test)."""
    n = len(y)
    oof = np.zeros((n, len(CLASSES)), dtype=np.float64)
    for fold in range(5):
        tr = fold_idx != fold
        va = fold_idx == fold
        clf = HistGradientBoostingClassifier(
            max_iter=200, max_leaf_nodes=31, learning_rate=0.05,
            min_samples_leaf=50, l2_regularization=1.0, random_state=SEED + fold)
        clf.fit(X_train[tr], y[tr])
        oof[va] = clf.predict_proba(X_train[va])
    clf = HistGradientBoostingClassifier(
        max_iter=200, max_leaf_nodes=31, learning_rate=0.05,
        min_samples_leaf=50, l2_regularization=1.0, random_state=SEED)
    clf.fit(X_train, y)
    test = clf.predict_proba(X_test)
    return oof, test

### Módulo `consenso_v2.py`

In [ ]:
"""Consensos Product-of-Experts de V53/V54/V55/V56 (porte fiel de los scripts de construccion).

Product-of-Experts bayesiano en el espacio de log-probabilidades:
    log P_geom(y = k | x) = sum_m w_m log P_m(y = k | x) - log Z(x)

CONSENSOS congela, por version, los miembros EN ORDEN de apilado y los pesos del script
ANTES de la normalizacion final (v53/v54 los derivan de W_V48; v55/v56 son literales):
- v53_02_build_submission.py:51-59,75-76 (7 miembros).
- v54_01_build_submission.py:52-72,89-93 (8 miembros, rama Cramer 0.80 V43 + 0.20 V47).
- v55_01_build_submission.py:53-56,73 (8 miembros, incluye CatBoost V49).
- v56_01_build_submission.py:53-56,74 (9 miembros, incluye V51 y V53).
Los pesos se normalizan al combinarlos (WEIGHTS_GEOM = WEIGHTS_GEOM / WEIGHTS_GEOM.sum()).
ASCII puro. Sin efectos en import.
"""

from __future__ import annotations

import numpy as np

#: Regla plug-in congelada por clase (v53_02_build_submission.py:51; age_umbrales_validacion.json).
PESOS_PLUG_IN = np.array([1.0000, 1.1749, 1.2721, 1.1630, 1.0010, 0.8128])

#: Pesos V48 de origen de las derivaciones de v53/v54 (v53_02:54-55, v54_01:55-56).
W_V48 = [0.5628, 0.0832, 0.1724, 0.0754, 0.0513, 0.0549]

#: Peso de Dense-SE-TabularNet en v53 (v53_02:56) y en v54 (v54_01:58).
W_DENSE_V53 = 0.0250
W_DENSE_V54 = 0.0200

#: Reparto de la rama Cramer de v54: 0.80 V43 + 0.20 V47 (v54_01:64-65).
REPARTO_CRAMER_V54 = (0.80, 0.20)

#: Epsilon del clip de probabilidades (v53_02:81, v54_01:98, v55_01:74, v56_01:75).
EPS = 1e-12

#: Especificaciones congeladas por version. Los valores float64 de v53/v54 son el resultado
#: EXACTO de la derivacion jerarquica (ver pesos_geometricos); v55/v56 son los literales.
CONSENSOS = {
    "v53": {
        "miembros": ["v30", "v43", "v42", "v40", "v39_ordinal", "v39_hgb", "v53_dense"],
        "pesos": [0.54873, 0.08112000000000001, 0.16809000000000002, 0.07351500000000001,
                  0.050017500000000006, 0.053527500000000006, 0.025],
        "fuente": "scripts/v53_02_build_submission.py:54-59,75-76",
    },
    "v54": {
        "miembros": ["v30", "v43", "v47", "v42", "v40", "v39_ordinal", "v39_hgb", "v53_dense"],
        "pesos": [0.551544, 0.06522880000000002, 0.016307200000000004, 0.16895200000000002,
                  0.07389200000000001, 0.050274000000000006, 0.053802, 0.02],
        "fuente": "scripts/v54_01_build_submission.py:55-72,89-93",
    },
    "v55": {
        "miembros": ["v30", "v43", "v47", "v42", "v40", "v39_ordinal", "v39_hgb", "v49_catboost"],
        "pesos": [0.52829, 0.07720, 0.02895, 0.17931, 0.07890, 0.05062, 0.04869, 0.00805],
        "fuente": "scripts/v55_01_build_submission.py:53-56,73",
    },
    "v56": {
        "miembros": ["v30", "v43", "v47", "v42", "v40", "v39_ordinal", "v39_hgb",
                     "v51_consistent", "v53_dense"],
        "pesos": [0.55939, 0.06023, 0.02089, 0.16873, 0.06080, 0.04376, 0.05204, 0.01035,
                  0.02381],
        "fuente": "scripts/v56_01_build_submission.py:53-56,74",
    },
}


def pesos_geometricos(version: str) -> np.ndarray:
    """Vector de pesos pre-normalizacion de una version, derivado como el script.

    v53: W_V48 normalizado * (1 - 0.0250) + [0.0250] (v53_02_build_submission.py:54-58).
    v54: W_V48 * (1 - 0.0200) con rama Cramer 0.80/0.20 y + [0.0200] (v54_01:55-71).
    v55/v56: literales del script (v55_01:55, v56_01:55).
    """
    if version == "v53":
        w48 = np.array(W_V48, dtype=np.float64)
        w48 = w48 / w48.sum()
        rem = 1.0 - W_DENSE_V53
        return np.array(list(w48 * rem) + [W_DENSE_V53])
    if version == "v54":
        w48 = np.array(W_V48, dtype=np.float64)
        w48 = w48 / w48.sum()
        base = w48 * (1.0 - W_DENSE_V54)
        reparto_v43, reparto_v47 = REPARTO_CRAMER_V54
        return np.array([base[0], base[1] * reparto_v43, base[1] * reparto_v47, base[2],
                         base[3], base[4], base[5], W_DENSE_V54])
    return np.array(CONSENSOS[version]["pesos"], dtype=np.float64)


def poe_ponderado(modelos: list[np.ndarray], pesos: list[float]) -> np.ndarray:
    """PoE ponderado en log-espacio con normalizacion final (v55_01:77-84, v56_01:77-85).

    Los pesos se normalizan (WEIGHTS_GEOM / WEIGHTS_GEOM.sum()); las probabilidades se
    recortan a [EPS, 1.0] antes del log; se desplaza por el maximo por fila y se
    renormalizan las filas. Entrada: modelos (n, 6) en orden de apilado.
    """
    w = np.asarray(pesos, dtype=np.float64)
    w = w / w.sum()
    log_geom = sum(wi * np.log(np.clip(m, EPS, 1.0)) for wi, m in zip(w, modelos))
    p_geom = np.exp(log_geom - np.max(log_geom, axis=1, keepdims=True))
    return p_geom / p_geom.sum(axis=1, keepdims=True)


def regla_plug_in(prob: np.ndarray) -> np.ndarray:
    """Decision plug-in por clase: argmax(prob * PESOS_PLUG_IN) (v53_02:88-89)."""
    return np.argmax(prob * PESOS_PLUG_IN, axis=1)

### Módulo `analisis_v57.py`

In [ ]:
"""Analisis matematico y estadistico de V57 (porte fiel de scripts/v57_analisis_matematico_estadistico.py).

Cuatro bloques sobre los OOF de 169.336 filas (miembros de V56 en orden y regla plug-in):
1. sensibilidad_simplex: perturbacion +-20% de cada coordenada de pesos (v57:66-94).
2. curva_power_mean: familia P_alpha con alpha en [-0.15, +0.15]; alpha=0 es el PoE (v57:101-119).
3. deciles_confiabilidad: deciles de confianza maxima vs exactitud (v57:125-149).
4. brecha_menciones: autocalibracion con/sin mencion (n_mention de dense_train.parquet; v57:151-152,165-178).

Los estados se escriben en ASCII (el original usa tildes: OPTIMO/SUB-OPTIMO). Sin efectos en import.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score


#: Grilla de alpha del bloque Power-Mean (v57_analisis_matematico_estadistico.py:104).
ALPHAS = [-0.15, -0.10, -0.05, -0.01, 0.0, 0.01, 0.05, 0.10, 0.15]

#: Epsilon del clip de probabilidades (v57:46).
EPS = 1e-12

#: Nombres de los 9 miembros de V56 en orden de apilado (v57:48-52).
NOMBRES_V56 = ["V30 LightGBM Optuna", "V43 Cramer-SWA", "V47 Asym-Cramer", "V42 Smooth-SWA",
               "V40 SE-TabOrd", "V39 TabOrd", "V39 HistGB", "V51 Consistent-Cramer",
               "V53 Dense-SE"]


def macro_f1(y_true: np.ndarray, pred: np.ndarray) -> float:
    """Macro-F1 canonico sobre indices 0-5 (v57:62)."""
    return float(f1_score(y_true, pred, average="macro"))


def f1_por_folds(y_true: np.ndarray, pred: np.ndarray, fold_idx: np.ndarray) -> list[float]:
    """Macro-F1 por fold 0-4 (v57:63)."""
    return [float(f1_score(y_true[fold_idx == f], pred[fold_idx == f], average="macro"))
            for f in range(5)]


def sensibilidad_simplex(modelos: list[np.ndarray], y_true: np.ndarray, pesos: list[float],
                         nombres: list[str] | None = None) -> list[dict]:
    """Bloque 1: sensibilidad coordenual +-20% en el simplex (v57:66-94)."""
    w_base = np.asarray(pesos, dtype=np.float64)
    w_base = w_base / w_base.sum()
    p_base = poe_ponderado(modelos, w_base)
    f1_base = macro_f1(y_true, regla_plug_in(p_base))
    if nombres is None:
        nombres = [f"miembro_{i}" for i in range(len(modelos))]
    registros = []
    for i, nombre in enumerate(nombres):
        w_low = w_base.copy()
        w_low[i] *= 0.80
        w_low /= w_low.sum()
        f1_low = macro_f1(y_true, regla_plug_in(poe_ponderado(modelos, w_low)))

        w_high = w_base.copy()
        w_high[i] *= 1.20
        w_high /= w_high.sum()
        f1_high = macro_f1(y_true, regla_plug_in(poe_ponderado(modelos, w_high)))

        registros.append({
            "modelo": nombre,
            "peso_base": float(w_base[i]),
            "f1_minus_20pct": f1_low,
            "delta_minus_20pct": f1_low - f1_base,
            "f1_plus_20pct": f1_high,
            "delta_plus_20pct": f1_high - f1_base,
            "estado": "ESTACIONARIO" if (f1_low < f1_base and f1_high < f1_base)
                      else "NO_ESTACIONARIO",
        })
    return registros


def curva_power_mean(modelos: list[np.ndarray], y_true: np.ndarray,
                     pesos: list[float]) -> list[dict]:
    """Bloque 2: Macro-F1(alpha) de la familia Power-Mean; alpha=0 es el PoE (v57:101-119)."""
    w_base = np.asarray(pesos, dtype=np.float64)
    w_base = w_base / w_base.sum()
    p_poe = poe_ponderado(modelos, w_base)
    f1_base = macro_f1(y_true, regla_plug_in(p_poe))
    registros = []
    for alpha in ALPHAS:
        if alpha == 0.0:
            p = p_poe
        else:
            p_alpha = sum(wi * (np.clip(m, EPS, 1.0) ** alpha)
                          for wi, m in zip(w_base, modelos))
            p = p_alpha ** (1.0 / alpha)
            p = p / p.sum(axis=1, keepdims=True)
        f1 = macro_f1(y_true, regla_plug_in(p))
        registros.append({
            "alpha": alpha,
            "macro_f1": f1,
            "delta_vs_poe": f1 - f1_base,
            "estado": "OPTIMO GLOBAL ESTACIONARIO (PoE)" if alpha == 0.0 else "SUB-OPTIMO",
        })
    return registros


def deciles_confiabilidad(prob: np.ndarray, y_true: np.ndarray) -> list[dict]:
    """Bloque 3: deciles de confianza maxima vs exactitud (v57:125-149)."""
    pred = regla_plug_in(prob)
    conf = np.max(prob, axis=1)
    correct = (pred == y_true)
    deciles = pd.qcut(conf, 10, labels=False)
    registros = []
    for d in range(10):
        mask = (deciles == d)
        registros.append({
            "decil": int(d),
            "confianza_min": float(conf[mask].min()),
            "confianza_max": float(conf[mask].max()),
            "confianza_media": float(conf[mask].mean()),
            "exactitud": float(correct[mask].mean()),
            "conteo": int(mask.sum()),
        })
    return registros


def brecha_menciones(prob: np.ndarray, y_true: np.ndarray, n_mention: np.ndarray) -> dict:
    """Bloque 4: brecha confianza-exactitud con/sin mencion (v57:151-152,165-178)."""
    pred = regla_plug_in(prob)
    conf = np.max(prob, axis=1)
    correct = (pred == y_true)
    has_men = (n_mention > 0)
    return {
        "tweets_con_mencion": {
            "proporcion": float(has_men.mean()),
            "exactitud": float(correct[has_men].mean()),
            "confianza_media": float(conf[has_men].mean()),
            "gap": float(conf[has_men].mean() - correct[has_men].mean()),
        },
        "tweets_sin_mencion": {
            "proporcion": float((~has_men).mean()),
            "exactitud": float(correct[~has_men].mean()),
            "confianza_media": float(conf[~has_men].mean()),
            "gap": float(conf[~has_men].mean() - correct[~has_men].mean()),
        },
    }


def analisis_completo(modelos: list[np.ndarray], y_true: np.ndarray, pesos: list[float],
                      n_mention: np.ndarray, nombres: list[str] | None = None,
                      fold_idx: np.ndarray | None = None) -> dict:
    """Los cuatro bloques de V57 en un resumen con el esquema del JSON original."""
    prob = poe_ponderado(modelos, pesos)
    pred = regla_plug_in(prob)
    return {
        "f1_base": macro_f1(y_true, pred),
        "folds_base": None if fold_idx is None else f1_por_folds(y_true, pred, fold_idx),
        "sensibilidad_simplex": sensibilidad_simplex(modelos, y_true, pesos, nombres),
        "curva_power_mean": curva_power_mean(modelos, y_true, pesos),
        "deciles_confiabilidad": deciles_confiabilidad(prob, y_true),
        "brecha_menciones": brecha_menciones(prob, y_true, n_mention),
    }

### Módulo `test_side.py`

In [ ]:
"""Lado TEST de la cadena del campeon: re-ajuste de la cadena sobre el train completo.

Reproduce el camino de test del pipeline v12/V30 (matriz 279) sin dependencias del proyecto
original (prohibido importar `src`/`scripts`; solo `cadena. y `cadena.nb_svm`):

    - representaciones dispersas: porte de ``src/pipeline.py:157-198``
      (``build_representations``), ajustadas solo con train y transformando test.
    - 13 bloques de scores de test: porte de ``src/pipeline.py:199-288``
      (``fit_base_models``) en el orden de ``OOF_FILES`` (``src/pipeline.py:59-65``), con la
      transformacion de ``MEMBER_FILES`` (``scripts/phase2_ponderacion.py:49-55``): ``log``
      si la fuente es probabilistica, crudo si no.
    - NB-SVM de palabras (ADV07): porte de ``scripts/phase25_submission_fresca.py:92-103``.
    - ensamblado (m, 279): porte de ``scripts/phase25_submission_fresca.py:108-111``.

Reglas del porte: ASCII puro, sin efectos en import, sin aleatoriedad anadida (semilla 42).
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge, SGDClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import LinearSVC


#: src/config.py:59 -- TEXT_COL = "text".
TEXT_COL = "text"

#: Componentes de la SVD del miembro E019b (src/pipeline.py:52).
SVD_COMPONENTS = 64


def adjacent_pair_margins(x_train, y_train: np.ndarray, x_valid) -> np.ndarray:
    """Margenes de 5 SVM entrenados **solo** con cada par de rangos contiguos (E-028a).

    Porte de ``src/pipeline.py:101-115``. Concentra la capacidad del modelo en las fronteras
    donde ocurre el error (la confusion del ensamble es casi toda entre rangos adyacentes).
    """
    margins = np.zeros((x_valid.shape[0], len(ADJACENT_PAIRS)), dtype=np.float32)
    for position, (lower, upper) in enumerate(ADJACENT_PAIRS):
        mask = (y_train == lower) | (y_train == upper)
        binary_target = (y_train[mask] == upper).astype(int)
        model = LinearSVC(C=0.2, max_iter=3000)
        model.fit(x_train[mask], binary_target)
        margins[:, position] = model.decision_function(x_valid)
    return margins


def ordinal_cumulative_probabilities(x_train, y_index: np.ndarray, x_valid) -> np.ndarray:
    """Modelo ordinal acumulativo (Frank & Hall 2001) sobre la representacion dispersa.

    Porte de ``src/pipeline.py:118-136``. Entrena K-1 clasificadores binarios ``P(y > k)`` y
    los combina en una distribucion: ``P(y = k) = P(y > k-1) - P(y > k)`` (E-021a: 0,5527).
    """
    thresholds = np.arange(1, len(CLASSES))
    cumulative = np.ones((x_valid.shape[0], len(thresholds) + 1), dtype=np.float64)
    for position, threshold in enumerate(thresholds):
        binary_target = (y_index >= threshold).astype(int)
        model = LogisticRegression(C=1.0, max_iter=1000, solver="liblinear",
                                   random_state=SEED)
        model.fit(x_train, binary_target)
        cumulative[:, position + 1] = model.predict_proba(x_valid)[:, 1]
    cumulative = np.clip(np.minimum.accumulate(cumulative, axis=1), 0.0, 1.0)
    probabilities = np.column_stack([-np.diff(cumulative, axis=1), cumulative[:, -1]])
    probabilities = np.clip(probabilities, 1e-12, None)
    return probabilities / probabilities.sum(axis=1, keepdims=True)


def rocchio_probabilities(x_train, y: np.ndarray, x_valid) -> np.ndarray:
    """Clasificador de centroides (Rocchio) con similitud coseno (E-021b: 0,4017).

    Porte de ``src/pipeline.py:139-154``. Miembro deliberadamente debil pero **diverso**.
    """
    centroids = []
    for label in CLASSES:
        mask = y == label
        centroid = np.asarray(x_train[mask].mean(axis=0)).ravel()
        centroids.append(centroid / (np.linalg.norm(centroid) + 1e-12))
    centroid_matrix = np.vstack(centroids)
    valid_norms = np.sqrt(np.asarray(x_valid.multiply(x_valid).sum(axis=1)).ravel()) + 1e-12
    similarity = np.asarray(x_valid @ centroid_matrix.T) / valid_norms[:, None]
    shifted = similarity - similarity.max(axis=1, keepdims=True)
    exponentials = np.exp(shifted * 8.0)
    return exponentials / exponentials.sum(axis=1, keepdims=True)


def representaciones_test(train: pd.DataFrame, test: pd.DataFrame) -> dict:
    """Ajusta los vectorizadores con train y transforma train y test.

    Porte de ``src/pipeline.py:157-198`` (``build_representations``). Configs congeladas
    ``CHAR_300K``/``CHAR_700K``/``MENTION``/``HASHTAG``/``WORD`` de ``cadena.`
    (= ``src/pipeline.py:42-51``). Los tokens de menciones y hashtags se extraen con
    ``tokens_of`` (``src/pipeline.py:75-78``) y patrones ``r"@\\w+"`` / ``r"#\\w+"``.

    Returns
    -------
    dict
        Matrices dispersas de cada canal mas el detalle de dimensiones (mismas claves que el
        original: x300/x700/xh/xw de train y test, y "dimensiones").
    """
    char300 = TfidfVectorizer(**CHAR_300K)
    mention300 = TfidfVectorizer(**MENTION)
    x300_train = hstack([char300.fit_transform(train[TEXT_COL]),
                         mention300.fit_transform(tokens_of(train[TEXT_COL], r"@\w+"))]).tocsr()
    x300_test = hstack([char300.transform(test[TEXT_COL]),
                        mention300.transform(tokens_of(test[TEXT_COL], r"@\w+"))]).tocsr()

    char700 = TfidfVectorizer(**CHAR_700K)
    mention700 = TfidfVectorizer(**MENTION)
    x700_train = hstack([char700.fit_transform(train[TEXT_COL]),
                         mention700.fit_transform(tokens_of(train[TEXT_COL], r"@\w+"))]).tocsr()
    x700_test = hstack([char700.transform(test[TEXT_COL]),
                        mention700.transform(tokens_of(test[TEXT_COL], r"@\w+"))]).tocsr()

    hashtag = TfidfVectorizer(**HASHTAG)
    xh_train = hstack([x300_train,
                       hashtag.fit_transform(tokens_of(train[TEXT_COL], r"#\w+"))]).tocsr()
    xh_test = hstack([x300_test,
                      hashtag.transform(tokens_of(test[TEXT_COL], r"#\w+"))]).tocsr()

    word = TfidfVectorizer(**WORD)
    xw_train = word.fit_transform(train[TEXT_COL])
    xw_test = word.transform(test[TEXT_COL])

    return {"x300_train": x300_train, "x300_test": x300_test,
            "x700_train": x700_train, "x700_test": x700_test,
            "xh_train": xh_train, "xh_test": xh_test,
            "xw_train": xw_train, "xw_test": xw_test,
            "dimensiones": {"char300+menciones": int(x300_train.shape[1]),
                            "char700+menciones": int(x700_train.shape[1]),
                            "char300+menciones+hashtags": int(xh_train.shape[1]),
                            "word12": int(xw_train.shape[1])}}


def base_models_test(reps: dict, y: np.ndarray, dense_train: np.ndarray, dense_test: np.ndarray,
                     textos_train: pd.Series, textos_test: pd.Series) -> list[np.ndarray]:
    """Ajusta los 13 modelos base sobre todo el train y devuelve sus scores de test.

    Porte de ``src/pipeline.py:199-288`` (``fit_base_models``). Los 13 bloques van en el orden
    de ``OOF_FILES`` (``src/pipeline.py:59-65``) con la transformacion ``log`` de
    ``MEMBER_FILES`` (``scripts/phase2_ponderacion.py:49-55``) cuando la fuente es
    probabilistica; el resto va crudo:

        1.  E013  LinearSVC(C=0.2, max_iter=3000) sobre x300  -> margenes (raw, 6)
        2.  E016  LinearSVC(C=0.2, max_iter=3000) sobre x700  -> margenes (raw, 6)
        3.  E-009 OneVsRest(LogReg C=4.0) sobre x300          -> proba log (6)
        4.  E-011 SGDClassifier(log_loss) sobre x300          -> proba log (6)
        5.  E-010 ComplementNB(alpha=0.3) sobre x300          -> proba log (6)
        6.  E014  LinearSVC(C=0.2, max_iter=3000) sobre xh    -> margenes (raw, 6)
        7.  E015_ordinal  Ridge(alpha=1.0, sparse_cg) sobre x300 -> continuo (raw, 1)
        8.  E015_softmax  log(softmax_ordinal(continuo))      -> log (6)
        9.  E017  OneVsRest(LogReg C=4.0) sobre xw            -> proba log (6)
        10. E019b MLP(256,128) sobre [35 densas + SVD64 de x300] -> proba log (6)
        11. E021a ordinal acumulativo sobre x300              -> log (6)
        12. E021b centroides Rocchio sobre x300               -> log (6)
        13. E028a margenes de 5 SVM de pares adyacentes       -> raw (5)

    Total 72 columnas (11 bloques de 6 + 1 de 1 + 1 de 5). ``textos_train``/``textos_test``
    forman parte de la interfaz congelada del porte; el cuerpo literal del original no los
    consume (toda la senal de texto entra por ``reps``). El ``detail`` con la varianza
    explicada por la SVD del original no forma parte de la interfaz (se retorna solo la lista).

    Parameters
    ----------
    reps : dict
        Salida de ``representaciones_test``.
    y : np.ndarray
        Etiquetas de train (strings de ``CLASSES``).
    dense_train, dense_test : np.ndarray
        35 features densas (``DENSE_FEATURES``, sin las 3 de emoji).
    textos_train, textos_test : pd.Series
        Textos crudos (no consumidos por el cuerpo portado).

    Returns
    -------
    list[np.ndarray]
        13 bloques de scores de test en el orden de ``OOF_FILES``.
    """
    scores: list[np.ndarray] = []

    model = LinearSVC(C=0.2, max_iter=3000).fit(reps["x300_train"], y)
    scores.append(to_scores(model.decision_function(reps["x300_test"]), model.classes_, False))
    model = LinearSVC(C=0.2, max_iter=3000).fit(reps["x700_train"], y)
    scores.append(to_scores(model.decision_function(reps["x700_test"]), model.classes_, False))

    model = OneVsRestClassifier(LogisticRegression(C=4.0, max_iter=1000, solver="liblinear",
                                                   random_state=SEED))
    model.fit(reps["x300_train"], y)
    scores.append(to_scores(model.predict_proba(reps["x300_test"]), model.classes_, True))

    model = SGDClassifier(loss="log_loss", alpha=1e-6, max_iter=30, tol=1e-4,
                          random_state=SEED).fit(reps["x300_train"], y)
    scores.append(to_scores(model.predict_proba(reps["x300_test"]), model.classes_, True))
    model = ComplementNB(alpha=0.3).fit(reps["x300_train"], y)
    scores.append(to_scores(model.predict_proba(reps["x300_test"]), model.classes_, True))

    model = LinearSVC(C=0.2, max_iter=3000).fit(reps["xh_train"], y)
    scores.append(to_scores(model.decision_function(reps["xh_test"]), model.classes_, False))

    ordinal_target = np.array([CLASS_MIDPOINTS[label] for label in y])
    ridge = Ridge(alpha=1.0, solver="sparse_cg", max_iter=2000, random_state=SEED)
    ridge.fit(reps["x300_train"], ordinal_target)
    continuous_test = ridge.predict(reps["x300_test"])
    scores.append(continuous_test.reshape(-1, 1))
    scores.append(np.log(np.clip(softmax_ordinal(continuous_test), 1e-12, None)))

    model = OneVsRestClassifier(LogisticRegression(C=4.0, max_iter=1000, solver="liblinear",
                                                   random_state=SEED))
    model.fit(reps["xw_train"], y)
    scores.append(to_scores(model.predict_proba(reps["xw_test"]), model.classes_, True))

    # MLP sobre densas + SVD del espacio de caracteres (miembro diverso, E019b).
    svd = TruncatedSVD(n_components=SVD_COMPONENTS, random_state=SEED)
    z_train = svd.fit_transform(reps["x300_train"])
    z_test = svd.transform(reps["x300_test"])
    scaler = StandardScaler().fit(np.hstack([dense_train, z_train]))
    encoder = LabelEncoder().fit(y)
    mlp = MLPClassifier(hidden_layer_sizes=(256, 128), alpha=1e-3, batch_size=512,
                        learning_rate_init=1e-3, max_iter=60, early_stopping=True,
                        n_iter_no_change=6, random_state=SEED)
    mlp.fit(scaler.transform(np.hstack([dense_train, z_train])), encoder.transform(y))
    probabilities = mlp.predict_proba(scaler.transform(np.hstack([dense_test, z_test])))
    scores.append(np.log(np.clip(probabilities, 1e-12, None)))

    # Miembro ordinal acumulativo (Frank & Hall): 5 clasificadores binarios sobre x300.
    y_index = np.array([CLASSES.index(label) for label in y])
    ordinal_probabilities = ordinal_cumulative_probabilities(reps["x300_train"], y_index,
                                                             reps["x300_test"])
    scores.append(np.log(np.clip(ordinal_probabilities, 1e-12, None)))

    # Miembro de centroides (Rocchio): debil pero diverso.
    centroids = rocchio_probabilities(reps["x300_train"], y, reps["x300_test"])
    scores.append(np.log(np.clip(centroids, 1e-12, None)))

    # Supervision ordinal fina: margenes entre pares de rangos contiguos.
    pair_margins = adjacent_pair_margins(reps["x300_train"], y, reps["x300_test"])
    scores.append(pair_margins)

    return scores


def nb_word_test(textos_train: pd.Series, y: np.ndarray,
                 textos_test: pd.Series) -> np.ndarray:
    """NB-SVM de palabras (ADV07) re-ajustado sobre el train completo.

    Porte de ``scripts/phase25_submission_fresca.py:92-103`` (identico en
    ``scripts/phase8_holdout_v12.py:153-164``). Mismo vectorizador (``WORD`` de 
    analyzer word, ngramas 1-2, min_df=3, sublinear_tf, 300k, lowercase) ajustado sobre
    **todo el train**; log-count ratio por clase sobre todo el train (``nb_log_count_ratio``,
    ``scripts/phase7_nbsvm_canal_meta.py:57-61``); ``LinearSVC(C=0.2, max_iter=3000)``
    uno-vs-rest por clase y ``decision_function`` sobre el test.

    Returns
    -------
    np.ndarray
        (m, 6) float32, margenes crudos en el orden de ``CLASSES`` (entran sin transformar
        como ultimas 6 columnas de la matriz 279).
    """
    word = TfidfVectorizer(**WORD)
    x_train_w = word.fit_transform(textos_train)
    x_test_w = word.transform(textos_test)
    margenes = np.zeros((len(textos_test), len(CLASSES)), dtype=np.float32)
    for posicion, etiqueta in enumerate(CLASSES):
        binario = (y == etiqueta).astype(int)
        ratio = nb_log_count_ratio(x_train_w, binario)
        modelo = LinearSVC(C=0.2, max_iter=3000)
        modelo.fit(x_train_w.multiply(ratio).tocsr(), binario)
        margenes[:, posicion] = modelo.decision_function(x_test_w.multiply(ratio).tocsr())
    return margenes


def matriz_test_fresca(bloques_72: list[np.ndarray], nb_test: np.ndarray,
                       densas_test: np.ndarray, estructuradas_test: np.ndarray,
                       supervisadas_test: np.ndarray, nb_word: np.ndarray) -> np.ndarray:
    """Ensambla el lado test (m, 279) en el orden EXACTO del campeon.

    Porte de ``scripts/phase25_submission_fresca.py:108-111``. Orden de columnas:

        [72 scores del pipeline (13 bloques)] + [nb_test 6] + [densas 35] +
        [estructuradas 87] + [supervisadas 73] + [nb_word 6]  = 279

    Parameters
    ----------
    bloques_72 : list[np.ndarray]
        13 bloques de ``base_models_test`` (6x11 + ordinal 1 + pares 5).
    nb_test : np.ndarray
        (m, 6) NB-SVM de char300+menciones (ADV05 lado test; ver ``cadena.nb_svm``).
    densas_test, estructuradas_test, supervisadas_test : np.ndarray
        (m, 35), (m, 87) y (m, 73) respectivamente.
    nb_word : np.ndarray
        (m, 6) salida de ``nb_word_test`` (ADV07 lado test).

    Returns
    -------
    np.ndarray
        (m, 279); falla con ``AssertionError`` si el ancho no es 279.
    """
    matriz = np.concatenate(bloques_72 + [nb_test, densas_test, estructuradas_test,
                                          supervisadas_test, nb_word], axis=1)
    assert matriz.shape[1] == 279, f"lado test inesperado: {matriz.shape}"
    return matriz

# FASE 1 — Comprensión del problema

## 1.1 La competición, la tarea y la métrica

La competencia plantea un problema de clasificación de texto supervisada: dado el texto de un tuit en
español, predecir a cuál de **seis rangos de edad ordenados** pertenece su autor (`13-17`, `18-24`, `25-34`,
`35-49`, `50-64`, `65-xx`). El corpus oficial tiene **169.336 tuits de entrenamiento** y **112.892 de
prueba**; el texto es corto (mediana de 93 caracteres) y está normalizado a ASCII (sin tildes ni eñes), lo
que elimina parte de la señal morfológica del español.

La métrica oficial del leaderboard es **F1 Score**, que en esta competencia coincide con la **macro-F1
sin ponderar sobre las seis clases**, la misma que el proyecto adoptó como métrica de trabajo: es la
lectura más exigente y estable para comparar modelos, porque no deja que las clases mayoritarias oculten
los errores en las minoritarias.
Todas las decisiones de adopción se tomaron sobre esta métrica en validación cruzada, y la calibración
posterior contra el leaderboard (sección 5.5) confirmó que es un estimador conservador y bien comportado
del puntaje real.

La competición puntúa sobre dos particiones del test: un **leaderboard público** (una fracción del test)
durante el torneo y un **leaderboard privado** (el resto) al cierre. El proyecto decidió desde el inicio
no usar el leaderboard como herramienta de búsqueda — ni para elegir modelos ni para calibrar umbrales —
porque eso sobreajusta la partición pública y castiga el resultado privado.

## 1.2 Restricciones inviolables

El brief del curso y las indicaciones del profesor fijaron restricciones que condicionaron cada decisión
técnica del proyecto:

| Restricción | Alcance en este proyecto |
|---|---|
| **Sin transformadores ni LLM** | Ningún modelo de atención (BERT, RoBERTa…) ni modelo generativo, tampoco como extractor de características. |
| **Sin embeddings** | Ni preentrenados (Word2Vec, GloVe, FastText) ni entrenados desde cero. Toda representación es estadística o de ingeniería léxica. |
| **Sin datos ni etiquetas externas** | Solo el corpus de la competición. Ningún recurso etiquetado de terceros. |
| **Sin metadatos** | El identificador nunca se usó como característica; no hay fuga temporal ni de autor. |
| **Test intocable** | El archivo de prueba jamás se usó para ajustar vectorizadores, modelos, selección de variables ni umbrales. |

Estas restricciones no son un detalle administrativo: explican por qué el proyecto invirtió tanto esfuerzo
en **representaciones clásicas** (TF-IDF (Salton y Buckley, «Term-Weighting Approaches in Automatic Text Retrieval», Information Processing & Management, 1988) de palabras y de caracteres (Sapkota, Bethard, Montes y Solorio, «Not All Character N-grams Are Created Equal: A Study in Authorship Attribution», NAACL-HLT 2015), rasgos estilométricos, canales
léxicos (Sap et al., «Developing Age and Gender Predictive Lexica over Social Media», EMNLP 2014)) y en **combinadores** de modelos lineales y de árboles, en lugar de recurrir al camino corto de
un modelo preentrenado (Liu, Singh y Mneimneh, «A Comparative Analysis of Classic and Deep Learning Models for Inferring Gender and Age of Twitter Users», DeLTA 2021).

## 1.3 Gobernanza experimental

El proyecto se condujo con disciplina de pre-registro: **cada experimento se definió y congeló antes de
ejecutarse** (pregunta, diseño, criterio de decisión y presupuesto), y solo después se corría. Un
experimento se adoptaba únicamente si superaba un **gate** estricto, que combinaba varias condiciones:

1. **Mejora en validación agrupada:** delta positivo sobre el campeón vigente, medido en la partición
   canónica de 5 folds agrupados por clústeres de near-duplicados (sección 2.4; Kaufman, Rosset, Perlich y Stitelman, «Leakage in Data Mining: Formulation, Detection, and Avoidance», ACM Transactions on Knowledge Discovery from Data, 2012).
2. **Consistencia entre folds:** la mejora debía aparecer en la mayoría de los folds, no en uno solo.
3. **Significancia estadística:** intervalo de confianza del 95 % por *bootstrap* (Efron, «Bootstrap Methods: Another Look at the Jackknife», The Annals of Statistics, 1979) pareado que excluyera
   el cero, y verificación en una **segunda partición** con semilla distinta.
4. **Control negativo:** cada línea de investigación incluía un gemelo de control (por ejemplo, el mismo
   modelo con las filas o las variables barajadas) para separar señal real de ruido de selección.

Cuando una línea no superaba el gate, se **cerraba con evidencia** y quedaba registrada — el proyecto
acumuló decenas de cierres documentados, que son parte central del valor del trabajo: los fracasos
medidos son lo que permitió no repetir caminos y concentrar el esfuerzo en lo que sí aportaba.

Cada adopción quedó registrada en un **acta de decisión** (con contexto, decisión, alternativas y
consecuencias) y cada corrida en un **registro experimental** con su configuración y su resultado. Este
notebook reconstruye esa cadena completa y verifica que los números reproducidos coinciden con los
documentados.

# FASE 2 — Comprensión de los datos

## 2.1 Auditoría de integridad del corpus

Antes de cualquier modelado, el proyecto auditó la integridad del corpus: formas, tipos, valores nulos,
duplicados exactos, el relleno de ancho fijo de la exportación (el campo de texto viene con espacios a la
derecha hasta 140 caracteres — un artefacto, no señal) y la normalización a ASCII. Esta auditoría fijó
decisiones que atraviesan todo el pipeline: limpiar el relleno con `strip` antes de vectorizar, no
depender de tildes ni eñes, y tratar los duplicados con cuidado (lo que motiva la sección 2.4).

In [ ]:
# Auditoria de integridad: formas, nulos, relleno, ASCII y duplicados exactos.
print("=== forma y columnas ===")
print("train:", train.shape, list(train.columns))
print("test :", test.shape, list(test.columns))
print("nulos en train:", int(train.isna().sum().sum()),
      "| nulos en test:", int(test.isna().sum().sum()))

anchos = train["text"].str.len()
print("\n=== longitudes (train) ===")
print("min:", int(anchos.min()), "| mediana:", int(anchos.median()), "| max:", int(anchos.max()))
print(f"ancho exacto de 140 (relleno de exportacion): {float((anchos == 140).mean()) * 100:.1f}%")

no_ascii = float(train["text"].map(lambda t: any(ord(c) > 127 for c in t)).mean())
print(f"filas con caracteres no ASCII: {no_ascii * 100:.3f}%")

print("\n=== duplicados e identificadores ===")
print("textos duplicados exactos en train:", int(train["text"].duplicated().sum()))
print("ids unicos (train/test):", bool(train["id"].is_unique), bool(test["id"].is_unique))
print("clases presentes:", sorted(train["age_range"].unique().tolist()))

In [ ]:
# Auditoria ampliada: vacios, ruido de etiqueta, id como proxy y signo de apertura.
vacios_train = int((train["text"].str.strip() == "").sum())
vacios_test = int((test["text"].str.strip() == "").sum())
print(f"textos vacios: train {vacios_train} | test {vacios_test}")

normalizado = (train["text"].fillna("").astype(str).str.lower()
               .str.replace(r"\s+", " ", regex=True).str.strip())
etiquetas_por_texto = train.assign(_norm=normalizado).groupby("_norm")["age_range"].nunique()
textos_contradictorios = etiquetas_por_texto[etiquetas_por_texto > 1].index
ruido = int(normalizado.isin(textos_contradictorios).sum())
print(f"ruido de etiqueta: {len(textos_contradictorios)} textos con {ruido} filas "
      f"({ruido / len(train) * 100:.2f}%) con etiquetas contradictorias")

apariciones_apertura = int(train["text"].str.count("\u00bf").sum() + test["text"].str.count("\u00bf").sum())
print(f"signo de apertura (corpus completo): {apariciones_apertura} apariciones (unico caracter no ASCII)")

deciles_id = pd.qcut(train["id"], 10, labels=False)
tabla_id = pd.crosstab(deciles_id, train["age_range"], normalize="index")
desviacion = float((tabla_id - 1 / len(CLASSES)).abs().max().max()) * 100
print(f"id como proxy de clase: desviacion maxima frente a 1/6 de {desviacion:.2f} pp")

## 2.2 Análisis exploratorio y textometría

El análisis exploratorio respondió tres preguntas que condicionaron la representación: ¿qué tan largos son
los textos?, ¿qué vocabulario los distingue entre rangos? y ¿qué tan informativa es la forma del texto
(emoticonos, mayúsculas, puntuación; Flekova y Gurevych, «Can We Hide in the Web? Large Scale Simultaneous Age and Gender Author Profiling in Social Media», PAN@CLEF 2013)? Las figuras muestran que la distribución de longitudes es muy
similar entre train y test (la representación puede confiar en la forma) y que cada rango de edad tiene
marcadores léxicos propios (Nguyen, Gravel, Trieschnigg y Meder, «How Old Do You Think I Am? A Study of Language and Age in Twitter», ICWSM 2013) — jerga juvenil en los extremos bajos, temas de actualidad y registro formal en
los altos — lo que justifica la representación por n-gramas de palabra y de carácter que se adopta en la
Fase 3.

In [ ]:
# Figura 2.1: distribucion de longitudes (caracteres y palabras).
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.6))
largos_train = train["text"].str.len()
largos_test = test["text"].str.len()
ejes[0].hist(largos_train, bins=60, color="#2b6cb0", alpha=0.85, label="train")
ejes[0].hist(largos_test, bins=60, color="#ed8936", alpha=0.55, label="test")
ejes[0].set_title("Longitud en caracteres (train vs test)")
ejes[0].set_xlabel("caracteres")
ejes[0].set_ylabel("tuits")
ejes[0].legend()
ejes[1].hist(train["text"].str.split().str.len(), bins=40, color="#2b6cb0", alpha=0.85)
ejes[1].set_title("Longitud en palabras (train)")
ejes[1].set_xlabel("palabras")
plt.tight_layout()
plt.show()

In [ ]:
# Figura 2.2: terminos mas frecuentes de cada rango de edad.
from collections import Counter

fig, ejes = plt.subplots(2, 3, figsize=(13, 6))
for posicion, etiqueta in enumerate(CLASSES):
    eje = ejes[posicion // 3][posicion % 3]
    palabras = Counter()
    for texto in train.loc[train["age_range"] == etiqueta, "text"]:
        palabras.update(token for token in texto.split() if len(token) >= 4)
    top = palabras.most_common(8)[::-1]
    eje.barh([t for t, _ in top], [c for _, c in top], color="#2b6cb0", alpha=0.85)
    eje.set_title(f"rango {etiqueta}", fontsize=10)
    eje.tick_params(labelsize=8)
plt.suptitle("Tokens frecuentes por rango (longitud >= 4)", y=1.0)
plt.tight_layout()
plt.show()

In [ ]:
# Figura 2.3: distribucion de clases y balance.
conteos = train["age_range"].value_counts().reindex(CLASSES)
fig, eje = plt.subplots(figsize=(8, 3.2))
eje.bar(conteos.index.astype(str), conteos.values, color="#2b6cb0")
for posicion, valor in enumerate(conteos.values):
    eje.text(posicion, valor + 150, f"{valor:,}", ha="center", fontsize=9)
eje.set_title("Tuits por rango de edad (train)")
eje.set_ylabel("tuits")
plt.tight_layout()
plt.show()
print(f"razon mayoritaria/minoritaria: {conteos.max() / conteos.min():.2f}x")
from sklearn.metrics import f1_score as _f1_suelo
suelo_mayoritaria = _f1_suelo(train["age_range"], [conteos.idxmax()] * len(train),
                              average="macro")
print(f"suelos de referencia: azar uniforme {1 / 6:.4f} | clase mayoritaria {suelo_mayoritaria:.5f}")

## 2.2b Textometría y contraste por edad

El corpus admite una caracterización cuantitativa más allá de las longitudes: riqueza léxica, densidad de
menciones y enlaces, mayúsculas y puntuación. La tabla contrasta diez de esas variables entre los seis
rangos con la prueba de Kruskal–Wallis (Kruskal y Wallis, «Use of Ranks in One-Criterion Variance
Analysis», Journal of the American Statistical Association, 1952) y reporta el tamaño del efecto
ε² = (H − k + 1)/(n − k); el barrido completo de 38 variables quedó documentado en
`reports/tables/eda_kruskal.csv`. La lectura coincide con la del informe: la longitud y la riqueza léxica
**crecen con la edad** (ε² ≈ 0.13 para `n_chars`), mientras que las marcas de oralidad juvenil
(emoticonos, alargamientos) se concentran en el extremo bajo. La mediana de longitud pasa de 73
caracteres en `13-17` a 111 en `65-xx`.

In [ ]:
# Figura 2.5: textometria por rango y contraste de Kruskal-Wallis.
from collections import Counter
from scipy.stats import kruskal

texto_train = train["text"]
palabras_train = texto_train.str.split()
n_palabras = palabras_train.str.len()
n_unicas = palabras_train.map(lambda lista: len(set(lista)))
cttr = n_unicas / np.sqrt(n_palabras.clip(lower=1))
entropia = palabras_train.map(
    lambda lista: float(-sum((cantidad / len(lista)) * np.log2(cantidad / len(lista))
                             for cantidad in Counter(lista).values())) if lista else 0.0)
variables_texto = {
    "n_chars": texto_train.str.len(),
    "n_words": n_palabras,
    "n_unique_words": n_unicas,
    "cttr": cttr,
    "lexical_entropy": entropia,
    "n_mention": texto_train.str.count(r"@\w+"),
    "n_url": texto_train.str.count(r"https?://\S+|www\.\S+"),
    "n_hashtag": texto_train.str.count(r"#\w+"),
    "all_caps_count": texto_train.str.count(r"\b[A-Z]{2,}\b"),
    "n_excl": texto_train.str.count(r"!"),
}
filas = []
for nombre, serie in variables_texto.items():
    grupos = [serie[train["age_range"] == clase].to_numpy() for clase in CLASSES]
    estadistico, p_valor = kruskal(*grupos)
    epsilon2 = (estadistico - len(CLASSES) + 1) / (len(train) - len(CLASSES))
    filas.append((nombre, estadistico, p_valor, epsilon2))
tabla_textometria = pd.DataFrame(filas, columns=["variable", "H", "p", "epsilon2"])
print(tabla_textometria.sort_values("H", ascending=False).to_string(index=False))
print(f"\nmediana de caracteres 13-17: {texto_train[train['age_range'] == '13-17'].str.len().median():.0f}"
      f" | 65-xx: {texto_train[train['age_range'] == '65-xx'].str.len().median():.0f}")

fig, eje = plt.subplots(figsize=(8, 3.2))
datos = [texto_train[train["age_range"] == clase].str.len().to_numpy() for clase in CLASSES]
eje.boxplot(datos, showfliers=False)
eje.set_xticks(range(1, len(CLASSES) + 1))
eje.set_xticklabels([str(clase) for clase in CLASSES])
eje.set_xlabel("rango de edad")
eje.set_ylabel("caracteres (sin relleno)")
eje.set_title("Longitud por rango: mediana y dispersion")
plt.tight_layout()
plt.show()

## 2.3 La estructura de los seis rangos

Los seis rangos son **ordinales**: `13-17 < 18-24 < 25-34 < 35-49 < 50-64 < 65-xx`. Esa estructura tiene
dos consecuencias prácticas que el proyecto explotó (y midió): primero, los errores graves (confundir
`13-17` con `65-xx`) son raros, mientras que las confusiones entre **rangos adyacentes** concentran la
mayor parte del error; segundo, la distancia entre clases admite formulaciones ordinales (regresión sobre
puntos medios, clasificadores acumulativos, pérdidas de transporte) (Nguyen, Smith y Rosé, «Author Age Prediction from Text using Linear Regression», LaTeCH@ACL-HLT 2011; Frank y Hall, «A Simple Approach to Ordinal Classification», ECML 2001; Frogner, Zhang, Mobahi, Araya y Poggio, «Learning with a Wasserstein Loss», NeurIPS 2015) que se exploran en la Fase 4. El
balance de clases es moderado (la razón mayoritaria/minoritaria ronda 1.06x), por lo que el desbalanceo no
fue el problema central — el problema fue la **frontera** entre rangos vecinos y el ruido intrínseco de la
tarea.

## 2.4 La fuga por near-duplicados y los folds canónicos

El corpus contiene **near-duplicados** (Kaufman, Rosset, Perlich y Stitelman, «Leakage in Data Mining: Formulation, Detection, and Avoidance», ACM Transactions on Knowledge Discovery from Data, 2012): textos idénticos o casi idénticos (retuits, copias con un emoji
distinto, el mismo chiste repetido) que aparecen tanto en entrenamiento como en prueba. Si la validación
cruzada se hace con folds aleatorios, un texto de validación puede tener su gemelo en entrenamiento: el
modelo lo memoriza y la validación **sobreestima** el desempeño real. Este fenómeno se midió y quedó
documentado como la decisión de protocolo más importante del proyecto: **la validación agrupada por
clústeres de near-duplicados**, donde un clúster entero cae en un solo fold. Así, ningún fold de
validación contiene un texto cuyo near-duplicado esté en el fold de entrenamiento.

Los clústeres se construyen con un procedimiento determinista (normalización del texto, comparación por
similitud de Jaccard (Jaccard, «The Distribution of the Flora in the Alpine Zone», New Phytologist, 1912) de tokens con *blocking* por tokens raros, unión de pares similares). La partición de
5 folds agrupados es el **protocolo congelado** del proyecto: todos los experimentos de este notebook se
evalúan con exactamente esos folds, que se **generan en la propia ejecución** con la regla documentada
(`StratifiedGroupKFold` con `shuffle=True` y semilla 42 sobre los clústeres) y se verifican por
propiedades: tamaños equilibrados, proporciones de clase estables y cero clústeres compartidos entre
folds. En versiones de scikit-learn distintas de la 1.8.0 la asignación puede variar y, con ella, los
valores absolutos (véase 6.2).

In [ ]:
# Figura 2.4: reconstruccion de los clusteres de near-duplicados (protocolo congelado).
t0 = time.time()
textos_todo = pd.concat([train["text"], test["text"]], ignore_index=True)
clusters = construir_clusters(textos_todo)
print(f"clusters: {clusters['cluster'].nunique():,} en {len(clusters):,} textos "
      f"({time.time() - t0:.1f}s)")

cluster_de = clusters["cluster"].to_numpy()
n_train = len(train)
pares = pd.Series(cluster_de[:n_train])
repetidos = pares.duplicated(keep=False)
exactos_test = float(test["text"].isin(set(train["text"])).mean())
compartidos_test = float(pd.Series(cluster_de[n_train:]).isin(
    set(cluster_de[:n_train])).mean())

fig, ejes = plt.subplots(1, 2, figsize=(11, 3.4))
tamanos = pd.Series(cluster_de).value_counts().value_counts().sort_index()
ejes[0].bar(tamanos.index.astype(str), tamanos.values, color="#2b6cb0")
ejes[0].set_title("Tamano de los clusters de near-duplicados")
ejes[0].set_xlabel("textos por cluster")
ejes[0].set_ylabel("numero de clusters")
ejes[0].tick_params(labelsize=8)
ejes[1].bar(["exactos test~train", "comparten cluster", "sin gemelo"],
            [exactos_test * 100, compartidos_test * 100, (1 - compartidos_test) * 100],
            color=["#c53030", "#ed8936", "#2b6cb0"])
ejes[1].set_title("Test con gemelo en train (%)")
ejes[1].set_ylabel("porcentaje")
plt.tight_layout()
plt.show()
print(f"train en clusters de tamano > 1: {float(repetidos.mean()) * 100:.1f}%")
print(f"test con duplicado exacto en train: {exactos_test * 100:.2f}%")
print(f"test que comparte cluster con train: {compartidos_test * 100:.2f}%")

In [ ]:
# Folds canonicos: se generan con la regla documentada (StratifiedGroupKFold con shuffle y semilla 42
# sobre los clusters de near-duplicados) y se verifican por propiedades, sin constantes opacas.
from sklearn.model_selection import StratifiedGroupKFold


def generar_folds(etiquetas, grupos, semilla=42, n_folds=5):
    """Asigna cada fila a un fold agrupado por cluster: ningun cluster cae en dos folds."""
    divisor = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=semilla)
    asignacion = np.zeros(len(etiquetas), dtype=np.int64)
    for numero, (_, validacion) in enumerate(
            divisor.split(np.zeros(len(etiquetas)), etiquetas, groups=grupos)):
        asignacion[validacion] = numero
    return asignacion


fold_idx = generar_folds(y, cluster_de[:n_train], semilla=42)
print("folds canonicos (semilla 42):", np.bincount(fold_idx).tolist())
for numero in range(5):
    en_fold = fold_idx == numero
    proporcion = np.bincount(clase[en_fold], minlength=len(CLASSES)) / en_fold.sum()
    print(f"  fold {numero}: {en_fold.sum():,} filas | clases "
          f"{np.round(proporcion * 100, 1).tolist()} %")

# Verificacion por propiedades: cobertura completa, tamano equilibrado y cero fuga entre folds.
assert sorted(np.unique(fold_idx).tolist()) == list(range(5))
tamanos = np.bincount(fold_idx)
diferencia = int(tamanos.max() - tamanos.min())
print(f"tamanos por fold: min {tamanos.min():,} | max {tamanos.max():,} | "
      f"diferencia {diferencia:,}")
assert diferencia <= 0.05 * len(fold_idx) / 5  # cota: 5% del fold medio
grupos = cluster_de[:n_train]
fugados = set()
for numero in range(5):
    fugados |= set(grupos[fold_idx == numero]) & set(grupos[fold_idx != numero])
print(f"clusters compartidos entre folds (debe ser 0): {len(fugados)}")
assert not fugados, "fuga de near-duplicados entre folds"
print("protocolo verificado: 5 folds agrupados sin fuga de near-duplicados.")

In [ ]:
# Suelo de memorizacion pura: particion ingenua frente a la particion agrupada congelada.
from sklearn.model_selection import StratifiedKFold

SENTINELA = "__no_cubierto__"
etiquetas = train["age_range"].to_numpy()
cluster_train = cluster_de[:n_train]


def oof_memorizacion(particiones):
    """Etiqueta mayoritaria del gemelo de cluster; las filas sin gemelo reciben un centinela."""
    prediccion = np.empty(len(etiquetas), dtype=object)
    cubiertas = 0
    for entrenamiento, validacion in particiones:
        etiquetas_por_cluster = {}
        for cluster, etiqueta in zip(cluster_train[entrenamiento], etiquetas[entrenamiento]):
            etiquetas_por_cluster.setdefault(cluster, []).append(etiqueta)
        for fila in validacion:
            gemelos = etiquetas_por_cluster.get(cluster_train[fila])
            if gemelos:
                prediccion[fila] = Counter(gemelos).most_common(1)[0][0]
                cubiertas += 1
            else:
                prediccion[fila] = SENTINELA
    return prediccion, cubiertas / len(etiquetas)


def macro_f1_etiquetas(etiquetas_reales, prediccion):
    """Macro-F1 sobre etiquetas de texto (no sombrea la version canonica por indices)."""
    return f1_score(etiquetas_reales, prediccion, average="macro", labels=CLASSES, zero_division=0)


ingenuas = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(cluster_train, etiquetas))
pred_ingenua, cobertura_ingenua = oof_memorizacion(ingenuas)
agrupadas = [(np.flatnonzero(fold_idx != k), np.flatnonzero(fold_idx == k)) for k in range(5)]
pred_agrupada, cobertura_agrupada = oof_memorizacion(agrupadas)

print("SUELO DE MEMORIZACION PURA (etiqueta del gemelo; sin gemelo = centinela)")
print(f"  particion estratificada ingenua : macro-F1 {macro_f1_etiquetas(etiquetas, pred_ingenua):.4f}"
      f" | cobertura {cobertura_ingenua * 100:.2f}%  (documentado: 0.0608 y 3.99%)")
print(f"  particion agrupada (usada)      : macro-F1 {macro_f1_etiquetas(etiquetas, pred_agrupada):.4f}"
      f" | cobertura {cobertura_agrupada * 100:.2f}%  (documentado: 0.0000 y 0.00%)")
print("  lectura: la validacion agrupada elimina por completo la ventaja de memorizar near-duplicados.")

## 2.5 Construcción del conjunto de datos de modelado

El corpus crudo no se usa directamente: el modelado consume una **matriz de 279 columnas** construida en
siete pasos, que se resumen aquí porque son el contrato que comparten todos los experimentos del notebook.

1. **Corpus y limpieza.** Se cargan `age_train.csv` (169.336 filas) y `age_test.csv` (112.892) con
   separador `;`; se recorta el relleno de ancho fijo (`strip`), se conserva la transliteración ASCII y se
   tratan los textos vacíos como cadenas vacías. La etiqueta es `age_range`, con seis clases.
2. **Particiones congeladas.** Los cinco folds agrupados por clúster de near-duplicados (sección 2.4) se
   embeben como constante: ninguna fila de validación comparte clúster con el entrenamiento.
3. **Representación.** TF-IDF de palabras (1-2) y de carácter `char_wb` (3-6), ajustado dentro de cada
   fold con el vocabulario, `min_df` y `max_features` de la receta documentada.
4. **Miembros (78 columnas).** Catorce clasificadores individuales (tabla de la sección 3.4) generan
   probabilidades o márgenes fuera de fold; se aplica `log` a las columnas indicadas.
5. **Features densas (35).** Textometría estilométrica por documento: longitudes, emoticonos,
   alargamientos, puntuación repetida, mayúsculas, dígitos, entropías y ratios de compresión.
6. **Features estructuradas (87).** Seis familias: distribución de longitudes, distribución de caracteres,
   riqueza léxica, forma de palabra, posición y entropías de puntuación.
7. **Features supervisadas (73) y ADV-07 (6).** Perfil de clase por entidad (menciones, hashtags, fuente
   de retuit), rareza léxica y léxico log-odds construidos **dentro de cada fold**, más seis márgenes
   NB-SVM de palabra.

La tabla fija el inventario de variables; la construcción efectiva y sus aserciones de forma se ejecutan en
la Fase 3 (secciones 3.3 y 3.4), donde la matriz se ensambla una sola vez
(`assert matriz.shape == (169336, 279)`) y todos los modelos posteriores la consumen.

| Bloque | Columnas | Contenido y ejemplos |
|---|---:|---|
| Miembros | 78 | Probabilidades o márgenes fuera de fold de 14 clasificadores (log si corresponde) |
| Densas | 35 | Longitudes, emoticonos, alargamientos, puntuación repetida, mayúsculas, dígitos, entropías, ratios de compresión |
| Estructuradas | 87 | Distribución de longitudes, de caracteres, riqueza léxica, forma de palabra, posición, entropías de puntuación |
| Supervisadas | 73 | Perfil de clase por entidad, rareza léxica y log-odds por clase (in-fold) |
| ADV-07 | 6 | Márgenes NB-SVM uno-vs-rest de palabra |
| **Total** | **279** | 273 del meta + 6 del canal de palabra |

Este resultado cierra la fase de comprensión de datos con el protocolo listo: corpus auditado, EDA hecho,
estructura ordinal entendida y partición honesta de validación verificada. Todo lo que sigue se evalúa con
estos folds.

## 2.5 Auditoría del historial V1-V37 [DOC]

*Sección documental: no re-ejecuta ningún script; cita cifras ya verificadas contra el registro experimental, las actas y los informes. Fuentes: `DECISIONS.md` (actas D-090 y D-091), `reports/auditoria_experimentos_v1_v37.md` (citado como `informe`), `experiments.csv` y los paquetes 06 y 07 de `plan/kaggle_notebook/paquetes/`.*

El cierre del ciclo V37 dejó pendiente verificar si las corridas anteriores se habían ejecutado correctamente y si sus decisiones estaban justificadas. Se auditó el historial completo V1-V37 con criterio de evidencia, no de opinión: cada afirmación debía citar su artefacto con ruta y línea. El objeto fue el registro completo de `experiments.csv`: 195 filas que no son 195 experimentos, sino 151 ejecuciones, 33 documentos (21 documentales y 12 pre-registros), 10 enmiendas y 1 corrida abortada (`V35-E1-INTENTO-1`, por `UnboundLocalError`); el inventario actualizado suma 200 filas al añadir las 5 filas `AUD-*` (`DECISIONS.md:3445-3446`; informe §3; paquete 07 §5). El peso del registro está en V1-V12: 113 de las 195 filas, donde los parámetros heredados acumulaban más riesgo (informe §2).

### 2.5.1 El protocolo de verificación

La auditoría usó seis revisiones independientes por especialidad (nlp, arquitectura, DL, kaggle, reviewer y QA) más la verificación propia del auditor; ningún hallazgo se aceptó sin comprobación contra la fuente primaria con ruta y línea (informe §2). Las restricciones fueron estrictas: no se re-ejecutó ningún script, no se abrió test, holdout ni leaderboard, no se generó ninguna submission y el campeón quedó intacto (D-090; informe §2). El protocolo de competencia se verificó por barrido del árbol sintáctico de 308 scripts: cero `fit` sobre test (los dos únicos `fit` con receptor de test calibran sobre OOF de train), vectorizadores ajustados solo sobre train, `id` nunca usado como característica, sin fuga de autor ni temporal (Kaufman, Rosset, Perlich y Stitelman, «Leakage in Data Mining: Formulation, Detection, and Avoidance», ACM Transactions on Knowledge Discovery from Data, 2012); el 4.2554% del test cae en un clúster de train y el esquema agrupado deja 0.00% de memorización frente al 3.99% del ingenuo (informe §6). La resolución de artefactos también se comprobó: de 262 citas, 230 existen en disco y las 32 restantes son globs, carpetas o referencias textuales; los 95 "huecos" iniciales eran problemas de formato de cita, no evidencia ausente (informe §3).

### 2.5.2 Los defectos confirmados

Se confirmaron 20 defectos (4 media-alta, 9 media, 7 baja); ninguno invalidó una adopción ni exigió cambiar el campeón (`experiments.csv:198`; paquete 06 §B.3). Los más relevantes:

| Defecto | Hallazgo | Fuente |
|---|---|---|
| K-01 (alta) | El "techo" de LB 0.70 queda refutado como prueba de imposibilidad (detalle en 2.5.3) | D-090; informe §4 |
| A-01 (alta) | La multiplicidad de las ~151 corridas no estaba cuantificada (detalle en 2.5.4) | D-090; informe §5 |
| K-02 (media) | El holdout de V12 está quemado: declara su criterio (Δ ≥ +0.003) sobre sí mismo con `confirmado: true`; es puerta de selección, no confirmación. Mitigado: el CV ya daba +0.0031 en 5/5 folds | informe §7; D-090 |
| N-01 (media) | El brazo NER `N1b` filtra spans con índices desalineados en ~55.8% de los documentos; sin impacto porque los seis brazos quedan bajo el gate | informe §7 |
| N-02 (media) | `es_core_news_lg` no estaba fijado en `requirements.txt`; corregido en D-091 con el wheel 3.8.0 (sha256 `7c6c2127…`) | D-091; informe §7 |
| A-03 (media) | `max_iter=40` en el MLP miembro del campeón (peso 0.25); su JSON de reoptimización no existe | informe §7 |
| A-06 (baja) | Falta `torch.manual_seed`: V18-P3/P4/P5 no son reproducibles | informe §7 |
| K-03 (media) | Registro de envíos obsoleto: contaba 15/12 cuando hay 16 puntuados | informe §7 |
| Registro | `V37-DIAG-POS` escrita con 21 de 22 campos; SHA256 del pre-registro V37 mal transcrito (`30d5d3cf…` -> `1d6d6573…`); ambos corregidos | informe §9; D-090 |

El propio artefacto de la auditoría pasó por revisión: su script de inventario tuvo tres defectos bloqueantes, detectados por `ml-reviewer` y corregidos antes de publicarse (informe §9).

### 2.5.3 El "techo" de LB 0.70 queda refutado (K-01)

El hallazgo más grave no fue de ejecución, sino de razonamiento. D-087 afirmó que el objetivo de LB 0.70 era inalcanzable porque el oráculo rendía +0.0568 frente a los +0.0626 exigidos; pero la rejilla del script era `TAUS = (0.30, 0.35, 0.40, 0.45, 0.50)` (`scripts/v35_00g_techo_analisis.py:69`) y el "techo" se tomó como su máximo, mientras la curva del oráculo es monótona creciente al bajar tau. La reproducción sobre `artifacts/p33_scores/todos/` (base 0.6159732; 65.683 errores) da:

| tau | filas corregibles | ganancia macro-F1 | ¿supera +0.0626? |
|---|---|---|---|
| 0.30 (máximo publicado) | 10.384 | +0.0593677 | no |
| 0.2930 (el punto que D-087 usa como objetivo) | 11.298 | +0.0646335 | sí |
| 0.28 | 13.199 | +0.0755813 | sí |
| 0.25 | 18.345 | +0.1054505 | sí |

En el punto exacto que D-087 define como objetivo el oráculo da +0.0646, un 3% por encima de lo exigido: el "no llegaría" era un artefacto del redondeo de la rejilla. La conclusión correcta es más fuerte y distinta: el objetivo queda exactamente en el borde de lo que un oráculo sobre la salida de V30 podría entregar (exigiría acertar ~16% de los errores con `p(y_true) >= 0.293`, conocimiento casi-oracular). LB 0.70 sigue siendo inalcanzable en la práctica; la prueba publicada no lo demuestra (D-090; informe §4). El IC95 [0.635, 0.653] de D-087 queda etiquetado como declarado, no estimado (acción `AUD-IC95-TECHO`; D-091).

### 2.5.4 Multiplicidad de las ~151 corridas (A-01)

El único defecto de marco analítico: la multiplicidad de las ~151 corridas nunca se cuantificó (el único control previo era el Bonferroni interno de V37, 18 pruebas por clase). Con `macro_f1_std` mediano 0.0042 en las 76 filas que lo declaran, el error estándar de la media de 5 folds es ~0.00188; el gate vigente acumulaba estos falsos positivos esperados:

| Gate | z aproximado | Falsos positivos esperados en 151 corridas |
|---|---|---|
| Δ ≥ +0.0015 | ~0.80 | ~32 |
| Δ ≥ +0.0030 | ~1.60 | ~8 |
| Bonferroni de familia 151 | — | exigiría Δ ≥ 0.0147 |

El argumento "ninguno pasó el gate" no está protegido contra multiplicidad; la afirmación correcta es "ninguno pasó el gate *y* su bootstrap pareado (Efron, «Bootstrap Methods: Another Look at the Jackknife», The Annals of Statistics, 1979) excluyendo 0" (informe §5; D-090).

### 2.5.5 El sesgo del inner split de Optuna (D-091)

El inner split de Optuna (Akiba et al., «Optuna: A Next-generation Hyperparameter Optimization Framework», KDD 2019) usa `scheme="stratified"` con `groups=np.arange(...)` en cinco scripts del camino al campeón (`v29_04`, `v29_06`, `v30_01`, `v30_02`, `v31_05`). La medición reconstruyó las dos particiones interiores exactamente como las construye el código real, sobre los 5 folds canónicos (169.336 filas, 164.018 clústeres):

| Nivel | solape de clúster en la validación interior |
|---|---|
| outer agrupado (canónico) | 0.000% |
| inner ingenuo (código real) | 3.907% |
| inner agrupado (propuesto) | 0.000% |

Con el criterio declarado antes de correr (diferencia > 0.02 = material), la medida fue 0.039071, casi el doble, y consistente en los 5 folds (3.787%-4.152%). La medición corrige la cifra previa de D-090 ("sin impacto medible": 0.963% vs 1.030%), errónea por dos motivos: medía solapamiento de texto exacto cuando los clústeres agrupan por Jaccard (Jaccard, «The Distribution of the Flora in the Alpine Zone», New Phytologist, 1912), y comparaba contra cero teórico en vez del inner agrupado. El leakage está en la *selección* de hiperparámetros, no en la estimación: la cifra externa del campeón usó el outer agrupado y no cambia; lo que no es reproducible bajo un esquema honesto es el proceso de selección. Re-optimizar quedó pendiente de decisión y no se ejecutó (D-091; paquete 06 §B.4). Nota de numeración: existen dos actas D-091 (inner split, 2026-10-01, y V38, 2026-10-16); al citar, indicar línea y contexto (paquete 06 §B.4).

### 2.5.6 Consecuencias: qué no invalida y por qué

La auditoría no invalidó ninguna adopción, y la razón es verificable: las cuatro adopciones reales del proyecto (V19-B1, V20, V30 y el plug-in) tienen IC95 pareado excluyendo 0 y confirmación externa. El bootstrap pareado corrige la varianza de estimación; no corrige la selección del máximo sobre 151 corridas, pero las adopciones que importan no dependen de esa corrección (informe §5). El campeón quedó verificado bit a bit en 0.6159732487657944 (SHA256 `29cbe6cc…`; 112.892 filas; 6 clases; 0 nulos) y el historial es auditable con `scripts/auditoria_v1v37_inventario.py` (6/6 checks) (D-090; paquete 07 §5). Quedaron reservas declaradas: el IC95 del techo es un rango declarado (ya etiquetado) y el `max_iter=40` del MLP miembro del campeón sigue sin revisar porque su JSON de reoptimización no existe (informe §10). Ninguna reserva exigió cambiar el campeón.

# FASE 3 — Preparación y representación (fase histórica V1–V17: V1–V12 re-ejecutada, V13–V17 documentadas)

Esta fase reconstruye la primera mitad del proyecto: cómo se pasó del texto crudo a una representación
que funciona, cómo se construyeron las features de segunda generación y cómo se llegó al meta-modelo de
14 miembros (V12), la base sobre la que después se construyó todo el resto. Cada experimento re-ejecutable
se vuelve a ejecutar aquí desde cero y su resultado se contrasta con el documentado; los que exigen
artefactos congelados o procesos largos quedan documentados.

## 3.1 Limpieza y normalización

La limpieza es deliberadamente mínima, y eso es una decisión medida: el corpus ya viene normalizado a
ASCII (sin tildes ni eñes) y sin URLs problemáticas, así que las transformaciones agresivas (lematizar,
quitar stopwords, minúsculas forzadas) fueron probadas en su momento y **perdían señal**. Lo único que se
hace siempre es eliminar el relleno de espacios del ancho fijo de la exportación (`strip`), que es un
artefacto y no contenido. Los vectorizadores se configuran con `lowercase=True` y `sublinear_tf=True`
(escala logarítmica de frecuencias), que resultaron estables en todos los barridos.

## 3.2 Representaciones comparadas (V1–V4)

La primera pregunta del proyecto fue cómo representar el texto **sin embeddings**: la apuesta fue la
combinación de n-gramas de palabra (capturan léxico y jerga: «crush», «examen», «hipoteca») con n-gramas
de carácter `char_wb` (capturan ortografía, sufijos y variantes: «jajaja», «xq», emojis de texto) (Sapkota, Bethard, Montes y Solorio, «Not All Character N-grams Are Created Equal: A Study in Authorship Attribution», NAACL-HLT 2015). El
baseline documentado (E-001) unió `word(1-2)` y `char_wb(3-5)` con un `LinearSVC` (Fan et al., «LIBLINEAR: A Library for Large Linear Classification», Journal of Machine Learning Research, 2008) y alcanzó
**macro-F1 0.5344 ± 0.0050** en validación agrupada (y 0.55235 en el leaderboard — el primer envío del
proyecto). Aquí se re-ejecuta la variante documentada (la unión) con el mismo
protocolo, junto con los dos suelos de referencia: el azar
uniforme (1/6) y el **suelo estricto** de predecir siempre la clase mayoritaria (0.04854 documentado).
Las variantes por separado quedan documentadas con sus cifras medidas en su momento (solo palabra
0.4971 ± 0.0034; solo carácter 0.5408 ± 0.0050): re-ejecutarlas costaba ~5 minutos de vectorización y
no cambiaba ninguna decisión.

In [ ]:
# Utilidad: validacion cruzada agrupada (5 folds canonicos) para una representacion + modelo.
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.svm import LinearSVC


def cv_representacion(vectorizadores, fabrica_modelo, textos, etiquetas, folds):
    """Macro-F1 medio por folds de una lista de vectorizadores + un modelo lineal."""
    puntajes, n_features = [], 0
    for fold in range(5):
        tr = folds != fold
        va = folds == fold
        bloques_tr, bloques_va = [], []
        for vectorizador in vectorizadores:
            bloques_tr.append(vectorizador.fit_transform(textos[tr]))
            bloques_va.append(vectorizador.transform(textos[va]))
        x_tr = hstack(bloques_tr).tocsr()
        x_va = hstack(bloques_va).tocsr()
        n_features = x_tr.shape[1]
        modelo = fabrica_modelo()
        modelo.fit(x_tr, etiquetas[tr])
        puntajes.append(float(f1_score(etiquetas[va], modelo.predict(x_va), average="macro")))
    return float(np.mean(puntajes)), float(np.std(puntajes)), n_features

In [ ]:
# Representaciones V1-V4: word, char_wb y la union documentada (E-001), con LinearSVC.
TEXTOS = train["text"].to_numpy()


def fabrica_word():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True,
                           max_features=300_000, lowercase=True)


def fabrica_char():
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=5,
                           sublinear_tf=True, max_features=300_000, lowercase=True)


def fabrica_svc():
    return LinearSVC(C=1.0, max_iter=3000)


variantes = {
    "union word+char (E-001)": [fabrica_word(), fabrica_char()],
}
resultados_representaciones = {}
for nombre, vectorizadores in variantes.items():
    t0 = time.time()
    media, desviacion, n_features = cv_representacion(vectorizadores, fabrica_svc, TEXTOS, y, fold_idx)
    resultados_representaciones[nombre] = media
    print(f"{nombre:26} macro-F1 {media:.4f} +- {desviacion:.4f} | "
          f"{n_features:,} features | {time.time() - t0:.0f}s")
print("\nreferencia documentada: E-001 (union + LinearSVC) = 0.5344 +- 0.0050")

In [ ]:
# Suelos de referencia y figura comparativa.
mayoritaria = pd.Series(y).value_counts().idxmax()
f1_mayoritaria = float(f1_score(y, np.full(len(y), mayoritaria), average="macro"))

fig, eje = plt.subplots(figsize=(9, 3.4))
nombres = list(resultados_representaciones) + ["suelo estricto", "azar uniforme"]
valores = list(resultados_representaciones.values()) + [f1_mayoritaria, 1 / 6]
colores = ["#2b6cb0", "#c53030", "#a0aec0"]
eje.bar(nombres, valores, color=colores)
eje.axhline(0.5344, color="#2f855a", linestyle="--", linewidth=1,
            label="E-001 documentado (0.5344)")
for posicion, valor in enumerate(valores):
    eje.text(posicion, valor + 0.004, f"{valor:.4f}", ha="center", fontsize=9)
eje.set_ylabel("macro-F1 (CV agrupada)")
eje.set_title("Representaciones y suelos (V1-V4 re-ejecutada)")
eje.legend(fontsize=8)
plt.xticks(rotation=12, fontsize=9)
plt.tight_layout()
plt.show()
print(f"suelo estricto (clase mayoritaria '{mayoritaria}'): {f1_mayoritaria:.5f} (documentado 0.04854)")

## 3.3 Features densas, estructuradas y supervisadas

Sobre la representación base, el proyecto construyó tres bloques de features que después alimentan al
meta-modelo (y que en el campeón ocupan 195 de las 279 columnas). Este bloque de trabajo se adoptó como
`AGE_ADVANCED_V01` tras un gate estricto: **+0.0031 de CV (positivo en 5/5 folds), +0.0052 en el holdout
congelado y +0.0043 en el leaderboard** — una de las pocas adopciones del proyecto, y de las más sólidas.

- **Densas (35 columnas):** textometría y estilometría del texto (Schler, Koppel, Argamon y Pennebaker, «Effects of Age and Gender on Blogging», AAAI Spring Symposium 2006; Argamon, Koppel, Pennebaker y Schler, «Mining the Blogosphere: Age, gender and the varieties of self-expression», First Monday 2007) — longitudes, riqueza léxica (TTR,
  hapax, entropía), proporción de stopwords, emoticonos, mayúsculas, puntuación múltiple. Son locales al
  documento (no dependen del corpus), totalmente deterministas y sin parámetros aprendidos.
- **Estructuradas (87 columnas):** seis familias de forma — distribución de longitudes de palabra,
  distribución de caracteres, riqueza léxica avanzada (MATTR, Yule, Maas), forma de las palabras,
  posición de entidades (menciones, hashtags, URLs; Nguyen, Gravel, Trieschnigg y Meder, «How Old Do You Think I Am? A Study of Language and Age in Twitter», ICWSM 2013) y entropía de estructura (puntuación, compresión).
- **Supervisadas (73 columnas):** rasgos que **usan las etiquetas del fold de entrenamiento** y por eso
  se calculan estrictamente in-fold (nunca ven la validación): probabilidad de cada rango dada una
  mención/hashtag/fuente de retuit (con suavizado sobre el prior), rareza léxica contra el vocabulario
  del fold (con centinela para palabras fuera de vocabulario) y puntuaciones léxicas por clase con
  log-odds de Monroe (Monroe, Colaresi y Quinn, «Fightin' Words: Lexical Feature Selection and Evaluation for Identifying the Content of Political Conflict», Political Analysis, 2008).

In [ ]:
# Construccion de los tres bloques de features (densas, estructuradas, supervisadas in-fold).
t0 = time.time()
dense = extraer_textometria(train)
dense_test = extraer_textometria(test)
print(f"densas: {dense.shape} | {time.time() - t0:.0f}s")

t0 = time.time()
estructuradas = extraer_estructuradas(train)
estructuradas_test = extraer_estructuradas(test)
print(f"estructuradas: {estructuradas.shape} | {time.time() - t0:.0f}s")

t0 = time.time()
supervisadas = supervisadas_infold(train["text"], y, fold_idx)
supervisadas_te = supervisadas_test(train["text"], y, test["text"])
print(f"supervisadas (in-fold): {supervisadas.shape} | {time.time() - t0:.0f}s")

print("\nreferencia: 35 densas + 87 estructuradas + 73 supervisadas = 195 columnas del meta")

In [ ]:
# Figura 3.2: tamano de los bloques y varianza explicada por familia (resumen visual).
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.4))
ejes[0].bar(["densas", "estructuradas", "supervisadas"], [35, 87, 73],
            color=["#2b6cb0", "#2f855a", "#ed8936"])
ejes[0].set_title("Tamano de cada bloque de features")
ejes[0].set_ylabel("columnas")
varianza = pd.DataFrame({"densas": dense.std(), "estructuradas": estructuradas.std(),
                         "supervisadas": supervisadas.std()})
nulos = float((varianza == 0).to_numpy().mean()) * 100
ejes[1].bar(["densas", "estructuradas", "supervisadas"],
            [float(dense.std().mean()), float(estructuradas.std().mean()),
             float(supervisadas.std().mean())], color=["#2b6cb0", "#2f855a", "#ed8936"])
ejes[1].set_title(f"Desviacion tipica media (columnas nulas: {nulos:.1f}%)")
plt.tight_layout()
plt.show()

## 3.4 Las fases PH y el meta-modelo (V5–V12)

Con las representaciones y features listas, la pregunta cambió de «¿cómo representar?» a «¿cómo
combinar?». El proyecto construyó **14 miembros diversos** — cada uno con una representación, un modelo o
una formulación distinta — y los combinó en un meta-modelo:

| # | Miembro | Representación | Modelo | CV documentada |
|---|---|---|---|---|
| 1 | E-013 | char_wb(3-6) 300k + menciones | LinearSVC C=0.2 (márgenes) | 0.5628 |
| 2 | E-016 | char_wb(3-6) 700k + menciones | LinearSVC C=0.2 (márgenes) | 0.5658 |
| 3 | E-009 | char300 + menciones | LogReg OvR C=4.0 | 0.5603 |
| 4 | E-011 | char300 + menciones | SGD log-loss | 0.5579 |
| 5 | E-010 | char300 + menciones | ComplementNB a=0.3 | 0.5370 |
| 6 | E-014 | char300 + menciones + hashtags | LinearSVC C=0.2 | 0.5644 |
| 7 | E-015o | char300 + menciones | Ridge ordinal (puntos medios) | 0.3973 |
| 8 | E-015s | ídem + softmax(-\|d\|/5) | distribución suave | — |
| 9 | E-017 | word(1-2) 300k | LogReg OvR C=4.0 | 0.5015 |
| 10 | E-019b | 35 densas + SVD(64) de char300 | MLP(256,128) | 0.4071 |
| 11 | E-021a | char300 + menciones | Frank & Hall (5 binarios) | 0.5527 |
| 12 | E-021b | char300 + menciones | Centroides Rocchio + softmax | 0.4017 |
| 13 | E-028a | char300 + menciones | 5 SVM por par adyacente | 0.7428 (fronteras) |
| 14 | ADV-05 | char300 + menciones | NB-SVM OvR | 0.5641 |

El meta-modelo final (V12) toma esas 14 salidas (78 columnas tras las transformaciones log), les añade
las 195 features de la sección 3.3 y las 6 columnas del NB-SVM de palabra (ADV-07; Wang y Manning, «Baselines and Bigrams: Simple, Good Sentiment and Topic Classification», ACL 2012; Mukherjee y Liu, «Improving Gender Classification of Blog Authors», EMNLP 2010), formando la **matriz
de 279 columnas**, y combina dos cabezas — una regresión logística y un MLP(64) — promediando sus
probabilidades. V12 alcanzó **CV 0.6005** (0.6077 en el holdout congelado) y **0.62182 en el
leaderboard**, el primer salto grande del proyecto.

In [ ]:
# Reconstruccion de los 14 miembros (OOF con los folds canonicos de la seccion 2.4).
t0 = time.time()
oofs_miembros = construir_miembros_oof(train["text"], y, fold_idx, dense)
print(f"14 miembros reconstruidos en {time.time() - t0:.0f}s")

documentados = {"e013": 0.5628, "e016": 0.5658, "e009": 0.5603, "e011": 0.5579, "e010": 0.5370,
                "e014": 0.5644, "e015_ordinal": 0.3973, "e015_softmax": None, "e017": 0.5015,
                "e019b": 0.4071, "e021a": 0.5527, "e021b": 0.4017, "e028a": None, "adv05": 0.5641}
print(f"\n{'miembro':14} {'CV reconstruida':>15} {'documentada':>12}")
for nombre, oof in oofs_miembros.items():
    if oof.ndim == 1:
        pred = np.argmax(softmax_ordinal(oof), axis=1)
    else:
        pred = np.argmax(oof, axis=1)
    cv_miembro = float(f1_score(clase, pred, average="macro"))
    referencia = documentados.get(nombre)
    texto_ref = f"{referencia:.4f}" if referencia else "(n/d)"
    print(f"{nombre:14} {cv_miembro:15.4f} {texto_ref:>12}")

In [ ]:
# ADV-07 (NB-SVM de palabra) y la matriz 279 del campeon.
t0 = time.time()
adv07 = np.zeros((len(y), 6), dtype=np.float32)
for fold in range(5):
    tr = np.flatnonzero(fold_idx != fold)
    va = np.flatnonzero(fold_idx == fold)
    adv07[va] = nb_svm_margins(train["text"], y, tr, va, WORD)
print(f"ADV-07 reconstruido en {time.time() - t0:.0f}s")

meta273 = build_features_273(oofs_miembros, densas35(dense),
                             estructuradas.to_numpy(float), supervisadas)
matriz = matriz_279(meta273, adv07)
print("meta 273:", meta273.shape, "| matriz 279:", matriz.shape, matriz.dtype)
assert matriz.shape == (169336, 279)

In [ ]:
# Mapa de aporte individual de los 14 miembros del meta (el meta-modelo V12 queda documentado en
# 0.6005; su reconstruccion costaba ~3.5 min y era una verificacion historica).
fig, eje = plt.subplots(figsize=(11, 3.6))
nombres = list(oofs_miembros)
valores = []
for nombre in nombres:
    oof = oofs_miembros[nombre]
    pred = np.argmax(softmax_ordinal(oof), axis=1) if oof.ndim == 1 else np.argmax(oof, axis=1)
    valores.append(float(f1_score(clase, pred, average="macro")))
eje.bar(nombres, valores, color="#2b6cb0", alpha=0.85, label="miembros (reconstruidos)")
eje.axhline(0.6005, color="#2f855a", linestyle="--", label="meta V12 (0.6005 documentado)")
eje.axhline(0.5344, color="#c53030", linestyle=":", label="baseline E-001 (0.5344)")
eje.set_ylabel("macro-F1 (CV agrupada)")
eje.set_title("Los 14 miembros del meta-modelo y su combinacion (V12)")
eje.legend(fontsize=8)
plt.xticks(rotation=45, fontsize=8, ha="right")
plt.tight_layout()
plt.show()

# FASE 4 — Modelado

Esta fase recorre el modelado desde V18 hasta la frontera de 2026. Primero se documenta la búsqueda de
frontera, ordinal y selección de features (V18–V29), que dejó lecciones y cerró caminos; después se
re-ejecuta el corazón del campeón: los cuatro cuartos de V20, el ensemble V30 y la regla plug-in por
clase (Lipton, Elkan y Narayanaswamy, «Thresholding Classifiers to Maximize F1 Score», arXiv, 2014); a continuación se documentan los fracasos seleccionados; luego se re-ejecutan los componentes
neuronales V38–V43 (la superación del techo lineal); se documentan las falsaciones V44–V52, que
delimitan con precisión qué técnicas no aportan y por qué; y se cierra con el arco 2026 (V53–V57):
los componentes nuevos, el consenso de nueve modelos que alcanzó el récord histórico de validación
local y la auditoría matemática del techo.

## 4.1 V18-V29: frontera, ordinal y selección [DOC]

Esta sección es documental: reconstruye, con cifras y fuentes, el tramo V18-V29, que va de la búsqueda de objetivos de frontera al giro ordinal, la selección supervisada de features y la búsqueda de capacidad con Optuna (Akiba et al., «Optuna: A Next-generation Hyperparameter Optimization Framework», KDD 2019). Las fuentes son `experiments.csv`, `DECISIONS.md`, `RUNLOG.md`, los paquetes de porte y los informes de `reports/`; nada se re-ejecuta aquí. Las cinco primeras versiones se conocen a través de la reconciliación documental de V24, porque nunca tuvieron eslabón propio en la cadena de generadores del notebook consolidado.

### 4.1.1 V18-V22: la frontera y el giro ordinal

V18-V22 no tuvieron eslabón propio en la cadena de cinco scripts que genera el notebook consolidado, de modo que su contenido solo quedó disponible al reconstruirse documentalmente en V24 (D-059). La fase V18 puso a prueba objetivos de entrenamiento para la frontera. Su premisa partía de un número derivado: V13 P3a publicó que el especialista binario sobre las 279 variables acertaba 0.4944 y 0.4942, y la auditoría P0 de V18 lo dio por bueno. La corrección D-048 midió al especialista sin la compuerta y obtuvo 0.7136 y 0.7493, por encima del 0.6864 y 0.7218 del TF-IDF (Salton y Buckley, «Term-Weighting Approaches in Automatic Text Retrieval», Information Processing & Management, 1988) disperso: la señal de frontera estaba en las variables de V12 todo el tiempo, y V12 extraía 0.4963 de los 0.7136 disponibles.

Con esa premisa corregida, V18 midió cuatro mecanismos y no adoptó ninguno: el re-peso de clases dejó 0.6001 (-0.0004, 3 de 5 folds) y en su nivel fuerte movió la frontera +0.082 a cambio de -0.0164 global; el margen de adyacencia movió una frontera +0.0392 y la otra -0.0552, acopladas por 25-34; el término multi-objetivo movió la frontera +0.0038 como máximo, con un mejor brazo de -0.0012; y la destilación (Hinton, Vinyals y Dean, «Distilling the Knowledge in a Neural Network», NIPS 2014 Deep Learning Workshop, 2015) transfirió señal real, pero con una mejora máxima de +0.0008 y 0 de 5 folds (`experiments.csv:104-107`; `DECISIONS.md:1187-1245`, D-049). La razón correcta la dio D-050: el bloqueo no es de magnitud, porque el gate de +0.003 se cruza con unas 478 correcciones netas de las 28.721 que V12 comete en la primera frontera, sino de identificabilidad, porque no existe un selector que diga cuáles corregir.

V19 y V20 aportaron después los dos mecanismos adoptados del arco: el árbol boosting (Ke et al., «LightGBM: A Highly Efficient Gradient Boosting Decision Tree», NeurIPS 2017) como tercer miembro (0.6079388; delta +0.0074; 5 de 5 folds; IC95 [+0.0064, +0.0085]; neto +1.311; LB 0.62578; D-051) y la cadena de cinco umbrales anidados (Frank y Hall, «A Simple Approach to Ordinal Classification», ECML 2001) como cuarto miembro (0.6097782; delta +0.0018; 5 de 5 folds; IC95 [+0.0012, +0.0025]; neto +294; LB 0.62689; D-053, D-054). V21 y V22 cerraron la familia ordinal por sus otros ejes: sustituir el discriminador B1 no mejora nada (D-056); el forense resolvió que V20 es una cadena de cinco binarios independientes (Caso A); y ni compartir representación (-0.000163) ni el coste ordinal (-0.0149 y -0.0706) mejoran el macro-F1 (D-057).

### 4.1.2 V23: CORAL / cumulative-link

V22 dejó abierta una pregunta con evidencia de poder importar: si las cinco fronteras ordinales compartieran su vector de pesos y sus umbrales estuvieran ordenados, la monotonía dejaría de ser una corrección posterior (V20 la viola en 18.927 de 169.336 filas, el 11.18 %) para ser una garantía de construcción. V23 implementó la formulación cumulative-link (Cao, Mirjalili y Raschka, «Rank consistent ordinal regression for neural networks with application to age estimation», Pattern Recognition Letters, 2020) en `src/ordinal_coral.py` con `numpy` y `scipy`, sin librería externa: un único vector $w$ y cinco umbrales $\theta_k = \gamma + \sum_{j<k} \mathrm{softplus}(\delta_j)$, con $P(Y \le k \mid x) = \sigma(\theta_k - w^\top x)$, verificados contra diferencias finitas (error relativo 3.15e-10) y contra la densidad de clase calculada por otra vía (diferencia 0.0). El diseño incluyó dos controles: el control A reproduce V20 exacto (0.6097782) y el control B son cinco binarios lineales independientes (0.4690055), que separan el efecto de compartir pesos del de perder no linealidad.

| Modelo | macro-F1 | Delta vs V20 | Folds + | Monotonía |
|---|---|---|---|---|
| V20 (control A) | 0.6097782 | - | 3/5 | 18.927 violaciones |
| Cinco binarios lineales (control B) | 0.4690055 | -0.140773 | 0/5 | no medido |
| CORAL (peso compartido) | 0.4837017 | -0.126077 | 0/5 | 0 violaciones |

La descomposición es el resultado útil: compartir pesos y ordenar umbrales vale +0.014696 frente al control B, luego la hipótesis era correcta en su plano lineal; pero la no linealidad de LightGBM vale -0.140773, un orden de magnitud mayor. CORAL resuelve la monotonía de verdad (0 violaciones frente a 18.927) y aun así pierde 0.1261: es la segunda confirmación independiente de que la consistencia ordinal no era el cuello de botella. La decisión fue ORDINAL_SHARED_WEIGHTS = NO_SIGNAL y CORAL = CLOSED, sin submission y sin bootstrap (Efron, «Bootstrap Methods: Another Look at the Jackknife», The Annals of Statistics, 1979), porque el pre-registro lo reservaba al caso de superar el gate. El mapa de errores se rehace entero: 36 de 36 celdas de la matriz de confusión cambian, con distancia L1 57.650 (`experiments.csv:129`; `DECISIONS.md:1683-1757`, D-058; `reports/v23_coral_analysis.md`).

### 4.1.3 V24: la tabla maestra

V24 fue la reconciliación documental que cerró la deuda descubierta en V23: el notebook consolidado llegaba a la sección 30 (V17) mientras el campeón era V20. La causa resultó estructural y no editorial: la fuente de verdad del notebook es una cadena de cinco scripts, uno por fase, y V18, V19, V20, V21 y V22 nunca tuvieron eslabón (`fase09_99_entrega.py`, `fase12_notebook_final.py`, `phase28_notebook_final_v2.py`, `v17_notebook.py` y `v23_02_auditoria.py`). Las 103 celdas eran correctas para lo que contenían, pero estaban incompletas; tras añadir el eslabón que faltaba (`v24_notebook_reconciliacion.py`) quedaron 121 celdas y 33 secciones, con las doce versiones V12 a V23 presentes.

V24 no añadió ningún experimento y produjo la tabla maestra (`reports/v24_tabla_maestra.csv`) y una figura de evolución construidas desde artefactos, no desde memoria. El registry arrastraba un defecto preexistente: tres líneas de cabecera en vez de una, con lo que declaraba 130 experimentos y tenía 128; se corrigió a una sola cabecera y el conteo real quedó en 128. También se documentaron sin sobrescribir dos discrepancias: V20 registrado como 0.609615, que era la salida del modelo compartido de D-057 (el canónico es 0.6097782), y V12 registrado como 0.6004 redondeado (el canónico pooled es 0.6005017). La auditoría de la fase fue PASS: 9 celdas ejecutadas con 0 errores y coherencia numérica 7 de 7. La decisión fue reconciliar sin experimentar, sin submissions y sin abrir hipótesis nuevas (`DECISIONS.md:1759-1814`, D-059; `reports/v24_reconciliacion.md`).

### 4.1.4 V25: selección supervisada de features

Con el frente de representaciones cerrado, V25 respondió una pregunta acotada: ¿contiene V20 un subconjunto significativo de features ruidosas que, al eliminarse con selección supervisada hecha dentro de cada fold, mejore el macro-F1 de forma reproducible? La comparación usó el mismo código de V20 y una sola variable: T1 conservó el top 50 % por `mutual_info_classif` (K = 139 de 279), estandarizado dentro de cada training fold, porque el estimador por vecino más cercano no es invariante a escala (las 73 features supervisadas llegan a magnitud 59.439 frente a las probabilidades de miembro, que viven en [0, 1]).

El control T0 reprodujo V20 exacto (0.6097782). T1 dio 0.6091492: delta -0.000629, 1 de 5 folds positivos, 1.042 rescates contra 1.143 daños, neto -101 y solo 3.545 filas cambiadas. El hallazgo útil es por bloque: sobreviven con frecuencia 1.000 las 78 probabilidades del ensamble y las 6 de NB-SVM, y nunca sobreviven 74 de las 87 estructuradas ni 30 de las 35 densas; 137 features fueron estables en los cinco folds. La selección no elimina ruido, elimina señal: lo que el mutual information ordena primero son las predicciones de los miembros que sostienen el modelo. La decisión fue SUPERVISED_FEATURE_SELECTION = CLOSED, sin probar otro K ni otro selector y sin bootstrap, porque el gate de delta no se cumplió (`experiments.csv:131`; `DECISIONS.md:1816-1882`, D-060; `reports/v25_feature_selection.md`).

### 4.1.5 V26: representación de la evidencia del ensemble

V26 partió de una distinción que gobierna el análisis: V20 ya es un meta-modelo. De sus 279 features, 84 son predicciones de modelos base (78 columnas de los 14 miembros del ensamble de V12 y 6 de NB-SVM) y el meta-modelo es LightGBM (Ke et al., 2017). La pregunta no era construir un meta-modelo, sino si su representación de la evidencia es explotable de otra forma. Un gap analysis de 15 familias, buscado por equivalencia funcional en el código, dejó cinco genuinamente abiertas (H02, H03, H11, H12 y H14) y ejecutó una sola, H02: la reescalada afín de las 84 columnas de evidencia, con media y desviación estimadas solo dentro del training fold.

Dos afirmaciones se midieron antes de decidir, y las dos son contrarias a la intuición: un árbol no es invariante a una transformación monótona en esta implementación de LightGBM (un z-score afín cambia la predicción, con diferencia máxima 0.219 sobre 135.468 filas de un fold), y un percentil no es monótono estricto (colapsa empates, con diferencia máxima 0.166). El experimento dio T1 = 0.6097770, delta -0.000001, 2 de 5 folds, 555 rescates contra 552 daños y neto +3 sobre 1.809 filas cambiadas (IC95 [-0.000375, +0.000397]). La transformación cambia el modelo y no cambia la métrica: lo que el meta-modelo necesita de las probabilidades de los miembros es el orden, y el orden ya se conserva. H02 quedó CLOSED, y H03, H11, H12 y H14 se dejaron documentadas sin ejecutar (`DECISIONS.md:1884-1946`, D-061; `reports/v26_final_research.md`).

### 4.1.6 V27: H12, H11, H14 y H03

V27 evaluó en orden las cuatro hipótesis que V26 dejó abiertas, con V20 congelado. H12 (fiabilidad por modelo y clase) se cerró por resultado y por equivalencia: la tabla se replica por fila, es constante, y una columna constante no puede generar splits; el delta fue exactamente 0.0000000 y 0 filas cambiaron. H11 (restricciones monótonas nativas de LightGBM, con cero usos en el proyecto) se cerró con -0.000355, 2 de 5 folds y neto -55, sobre 42 features con restricciones fijadas antes de entrenar. H14 (reescalar la salida del blend) se cerró por equivalencia funcional sin experimento: es H02, H03, H01 o un cambio de combinador, y los cuatro ya estaban cerrados. H03 (percentil) fue el único positivo: delta +0.000272, 3 de 5 folds, 564 rescates contra 513 daños, neto +51 y 1.764 filas cambiadas, con IC95 que incluye el cero; quedó MARGINAL, no finalista y sin submission.

El hallazgo estructural que explica los tres fracasos: el meta-modelo de V20 divide por 7 de sus 279 features, todas del bloque de evidencia de los miembros (0 de 35 densas, 0 de 87 estructuradas, 0 de 73 label-informed y 0 de 6 NB-SVM), y no es efecto del submuestreo. Eso explica de una vez por qué la selección de V25 no podía mejorar, por qué reescalar no movió la métrica y por qué las restricciones monótonas no podían actuar. Ninguna de las cuatro familias quedó abierta y V20 permaneció como baseline absoluto (`DECISIONS.md:1967-1984`, D-062; `reports/v27_compliance.md`).

### 4.1.7 V28: mecanismos estructuralmente distintos

V28 preguntó si existe una forma estructuralmente distinta de combinar, estabilizar o descomponer la evidencia que V20 no aproveche, y evaluó seis técnicas. El gap analysis cerró cuatro sin gastar experimento: B (geometría del consenso), porque `advanced_04_incertidumbre.py` implementa un superconjunto ya medido (+0.0001 al pasar de 267 a 318 features); D (fusión de evidencia), porque es un combinador y H04 con H06 están cerrados con número; E, por duplicación de B; y F (subespacio aleatorio), porque `HIERBA` ya usa `colsample_bytree = 0.5`, que es exactamente aleatorizar la exposición de cada árbol manteniendo todas las columnas.

Las dos ejecutadas fueron negativas. A, bagging (Breiman, «Bagging Predictors», Machine Learning, 1996) del meta-modelo con N = 5 miembros y bootstrap de Poisson(1), dio -0.0003120, 1 de 5 folds y neto -46, con un dato que hace informativo al resultado: los cinco miembros acordaban solo en el 91.89 % de las decisiones, luego la varianza existía y promediarla empeoró. C, cinco binarios disjuntos por par adyacente, dio -0.0675530, 0 de 5 folds y neto -10.996, con IC95 que excluye el cero; los pares reducen sus errores locales (de 39.556 a 32.826), pero un clasificador que solo ve dos clases no puede colocar una fila lejos de su par, y las clases no frontera se desploman. La decisión fue cerrar A y C y mantener V20: el cuello restante es la limitación de los datos (`DECISIONS.md:1987-2003`, D-063; `reports/v28_final_research.md`).

### 4.1.8 V29: Optuna y búsqueda de ensemble diverso

V29 atacó los dos ejes que quedaban: la capacidad de un miembro individual y la composición del ensemble, con la hipótesis de que V20 estaba limitado por falta de diversidad útil y no por capacidad. El paso más rentable no fue entrenar, sino leer: el historial contenía 51 matrices OOF reutilizables en 17 familias y ninguna se reentrenó. La diversidad histórica quedó cerrada con número: de los 47 modelos fuera del círculo de V20, 0 tienen rescate único mayor que daño único, y con el pool histórico los tres métodos de ensemble eran negativos (todas 2 de 5 folds).

La capacidad individual sí era un cuello: un LightGBM afinado con Optuna (Akiba et al., 2019) alcanzó 0.6121698 frente a los 0.6097782 de V20 (delta +0.002392), el primer modelo individual que supera a V20 en todo el proyecto, con la misma matriz de 279 features, los mismos folds y validación interior anidada. Y el ensemble mejoró con él y solo con él: el mejor de los tres métodos llegó a 0.6125960 (delta +0.002818, 5 de 5 folds, neto +480). No hubo finalista porque el gate exige +0.003 y el mejor resultado cubre el 94 % del gate; además, ese margen valdría +0.0014 en el leaderboard según la calibración, frente a una oscilación de 0.0034 entre modelos vecinos del propio proyecto. La hipótesis quedó contestada con matiz: el cuello era parcialmente la capacidad, no la diversidad, y el margen es real pero insuficiente (`experiments.csv:139`; `DECISIONS.md:2005-2025`, D-064; `reports/V29_FINAL_RESEARCH_REPORT.md`).

### 4.1.9 Tabla resumen

| Versión | Pieza | Resultado exacto | Decisión |
|---|---|---|---|
| V18 | Objetivos de frontera (P1/P3/P4/P5) | P1 -0.0004 (3/5); P3 +0.0392/-0.0552; P4 -0.0012; P5 +0.0008 (0/5) | Sin adopciones; V12 sigue campeón (D-049, D-050) |
| V19 | Boosting como tercer miembro | 0.6079388; +0.0074; 5/5; IC95 [+0.0064, +0.0085]; LB 0.62578 | ADOPTAR challenger (D-051) |
| V20 | Cadena de umbrales anidados como cuarto miembro | 0.6097782; +0.0018; 5/5; IC95 [+0.0012, +0.0025]; LB 0.62689 | CONFIRMED_CHALLENGER (D-053, D-054) |
| V21 | Sustitución del discriminador B1 | LinearSVC -0.0106; boosting 0.7606 < 0.7632 | B1_SPECIALIST = NO_SIGNAL (D-056) |
| V22 | Forense de V20 y ejes ordinales | Compartido -0.000163 (3/5); coste -0.0149/-0.0706 | WRONG_OBJECTIVE; familia cerrada (D-057) |
| V23 | CORAL cumulative-link | 0.4837017; -0.126077; 0/5; 0 violaciones de monotonía | ORDINAL_SHARED_WEIGHTS = NO_SIGNAL; CORAL = CLOSED (D-058) |
| V24 | Reconciliación documental y tabla maestra | 121 celdas y 33 secciones; registry de 130 a 128 | Sin experimento ni hipótesis nuevas (D-059) |
| V25 | Selección supervisada de features | -0.000629; 1/5; neto -101 | SUPERVISED_FEATURE_SELECTION = CLOSED (D-060) |
| V26 | H02: reescalada de la evidencia | -0.000001; 2/5; neto +3 | H02 CLOSED (D-061) |
| V27 | H12, H11, H14 y H03 | 0.0000000; -0.000355; equivalencia; +0.000272 | Tres cerradas; H03 MARGINAL (D-062) |
| V28 | Bagging (A) y pares disjuntos (C) | -0.0003120 (1/5); -0.0675530 (0/5) | A y C CLOSED; 4 por equivalencia (D-063) |
| V29 | Optuna individual y ensemble diverso | 0.6121698 individual; 0.6125960 ensemble (+0.002818; 5/5) | Sin finalista: 94 % del gate (D-064) |

## 4.2 V30: los cuatro cuartos, el ensemble y la regla plug-in

Con la matriz de 279 columnas fija, la pregunta pasó a ser cómo exprimirla. La respuesta llegó en cuatro
pasos (V19->V20, V29, V30 y la regla plug-in), cada uno con su acta y su verificación:

**V19 → V20 — «cuatro cuartos».** V19 añadió un árbol de boosting (LightGBM con configuración
pre-registrada; Ke et al., «LightGBM: A Highly Efficient Gradient Boosting Decision Tree», NeurIPS 2017) como tercer miembro del meta-modelo. V20 fue más allá y definió un candidato con cuatro
componentes de peso igual (0.25 cada uno): la regresión logística, el MLP(64), el árbol y una **cadena de
cinco umbrales acumulativos** (cinco clasificadores binarios «¿el rango es ≤ k?» que se corrigen para ser
monótonos y se convierten en distribución; Frank y Hall, «A Simple Approach to Ordinal Classification», ECML 2001). La cadena aporta la estructura ordinal de forma explícita.
V20 alcanzó **CV 0.6097782** y **0.62689** en el leaderboard.

**V29 — búsqueda con Optuna.** Una exploración sistemática de LightGBM (Ke et al., 2017) con Optuna (Akiba et al., «Optuna: A Next-generation Hyperparameter Optimization Framework», KDD 2019) encontró una
configuración distinta por fold (el mejor *trial* individual logró **0.6121698** de CV). Importante: las
configuraciones quedaron **congeladas** en el artefacto de la búsqueda — nada se re-optimiza después.

**V30 — el ensemble.** El campeón lineal del proyecto mezcla tres miembros — V20 (cuartos intactos), el
LightGBM de V29 y un LightGBM refinado por Optuna (Akiba et al., 2019) — con pesos que son la **media de los vectores óptimos
por fold, renormalizada** (`0.3548 / 0.3354 / 0.3098`, congelados). El resultado: **CV 0.6134471** y
**0.63302** en el leaderboard.

**La regla plug-in por clase.** La última pieza fue una regla de decisión (Lipton, Elkan y Narayanaswamy, «Thresholding Classifiers to Maximize F1 Score», arXiv, 2014) que reemplaza el `argmax`
plano por un `argmax` con **pesos por clase** (`[1.0000, 1.1749, 1.2721, 1.1630, 1.0010, 0.8128]`): los
pesos se ajustaron por ascenso por coordenadas en espacio logit, **dentro de cada fold** (usando los
otros cuatro como validación interna), y el vector final es la media de los cinco, congelada en un
pre-registro. La intuición es que la macro-F1 no trata igual a todas las clases: mover la frontera de una
clase mayoritaria puede costar más de lo que ayuda, y el plug-in re-calibra esa decisión sin tocar el
modelo. El sanity check del proyecto exige reproducir exactamente la mejora documentada
(**0.6134471 → 0.6159732487657944** en el OOF canónico, sin ningún cambio en filas de alta confianza).

In [ ]:
# V19/V20: los tres miembros de V19 y la cadena de cinco umbrales (OOF, folds canonicos).
t0 = time.time()
p19_logreg, p19_mlp, p19_arbol = v19_oofs(matriz, y, clase, fold_idx)
seis_oof = v20_cadena_oof(matriz, clase, fold_idx)
v20_oof = 0.25 * p19_logreg + 0.25 * p19_mlp + 0.25 * p19_arbol + 0.25 * seis_oof
print(f"reconstruccion V19/V20: {time.time() - t0:.0f}s\n")
for nombre, probabilidad in [("V19 logreg", p19_logreg), ("V19 MLP", p19_mlp),
                             ("V19 arbol", p19_arbol), ("cadena 5 umbrales", seis_oof),
                             ("V20 (cuartos)", v20_oof)]:
    print(f"  {nombre:20} macro-F1 {macro_f1(clase, np.argmax(probabilidad, axis=1)):.6f}")
print("\n  V20 documentado: CV 0.6097782")

In [ ]:
# V29/V30: los dos LightGBM con configuracion congelada por fold, la mezcla y la regla plug-in.
t0 = time.time()
v29_oof = lgbm_refit_oof(matriz, clase, fold_idx, V29_CONFIGS,
                         base={"random_state": SEED})
v30_oof = lgbm_refit_oof(matriz, clase, fold_idx, V30_CONFIGS)
oof_v30 = mezcla(v20_oof, v29_oof, v30_oof)
print(f"reconstruccion V29/V30: {time.time() - t0:.0f}s\n")
print(f"  V29 individual   : {macro_f1(clase, np.argmax(v29_oof, axis=1)):.7f} (doc 0.6121698)")
print(f"  V30 individual   : {macro_f1(clase, np.argmax(v30_oof, axis=1)):.7f} (doc 0.6122387)")
cv_v30_oof = macro_f1(clase, np.argmax(oof_v30, axis=1))
cv_plugin_oof = macro_f1(clase, regla_plug_in(oof_v30))
print(f"  V30 mezcla       : {cv_v30_oof:.7f} (doc 0.6134471)")
print(f"  V30 + plug-in    : {cv_plugin_oof:.10f} (doc 0.6159732487657944)")

In [ ]:
# Verificacion del orden de clases (diagonal) y figura de la progresion hasta el plug-in.
medias_condicionales = np.zeros((6, 6))
for fila in range(6):
    medias_condicionales[fila] = oof_v30[clase == fila].mean(axis=0)
diagonal = [int(np.argmax(medias_condicionales[fila])) for fila in range(6)]
assert diagonal == list(range(6)), f"orden de clases roto: {diagonal}"
print("diagonal de orden de clases verificada:", diagonal)

fig, ejes = plt.subplots(1, 2, figsize=(11.5, 3.8))
imagen = ejes[0].imshow(medias_condicionales, cmap="Blues")
ejes[0].set_title("E[P(clase) | rango real] sobre el OOF de V30")
ejes[0].set_xticks(range(6))
ejes[0].set_xticklabels(CLASSES, rotation=45, fontsize=8)
ejes[0].set_yticks(range(6))
ejes[0].set_yticklabels(CLASSES, fontsize=8)
plt.colorbar(imagen, ax=ejes[0], fraction=0.046)
hitos = ["V12", "V20", "V30", "V30\n+plug-in"]
valores = [0.6005, 0.6097782, cv_v30_oof, cv_plugin_oof]
ejes[1].bar(hitos, valores, color=["#a0aec0", "#90cdf4", "#4299e1", "#2b6cb0"])
ejes[1].set_ylim(0.59, 0.62)
ejes[1].set_ylabel("macro-F1 (CV agrupada)")
ejes[1].set_title("Progresion hasta la regla plug-in")
for posicion, valor in enumerate(valores):
    ejes[1].text(posicion, valor + 0.0004, f"{valor:.4f}", ha="center", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Lado TEST del campeon: la matriz de 279 de test (representaciones + 13 modelos + features + NB-SVM).
def adv05_test(textos_train, etiquetas, textos_test):
    """ADV-05 lado test: NB-SVM char300+menciones ajustado sobre todo el train."""
    textos_todo = pd.concat([textos_train, textos_test], ignore_index=True)
    etiquetas_todo = np.concatenate([etiquetas, np.array(["13-17"] * len(textos_test))])
    tr = np.arange(len(textos_train))
    va = np.arange(len(textos_train), len(textos_train) + len(textos_test))
    config = {**CHAR_300K, "mentions": {"config": MENTION, "pattern": r"@\w+"}}
    return nb_svm_margins(textos_todo, etiquetas_todo, tr, va, config)


t0 = time.time()
reps_test = representaciones_test(train, test)
bloques_72 = base_models_test(reps_test, y, densas35(dense), densas35(dense_test),
                              train["text"], test["text"])
nb_test = adv05_test(train["text"], y, test["text"])
nb_word_test_matrix = nb_word_test(train["text"], y, test["text"])
matriz_test = matriz_test_fresca(bloques_72, nb_test, densas35(dense_test),
                                 estructuradas_test.to_numpy(float), supervisadas_te, nb_word_test_matrix)
print(f"lado test completo: {time.time() - t0:.0f}s | matriz test {matriz_test.shape}")
assert matriz_test.shape == (112892, 279)

In [ ]:
# Probabilidades de test del candidato V30 (sin plug-in): cuartos + los dos LightGBM reajustados.
t0 = time.time()
p20_test = v20_test(matriz, matriz_test, y, clase)
p29_test = lgbm_refit_test(matriz, clase, matriz_test, V29_CONFIGS)
p30_test = lgbm_refit_test(matriz, clase, matriz_test, V30_CONFIGS)
prob_v30_test = mezcla(p20_test, p29_test, p30_test)
print(f"test V30: {time.time() - t0:.0f}s | forma {prob_v30_test.shape}")
print("decisiones V30+plug-in sobre test:")
pred_v30_test = regla_plug_in(prob_v30_test)
for posicion, etiqueta in enumerate(CLASSES):
    print(f"  {etiqueta}: {int((pred_v30_test == posicion).sum()):,}")

### La segunda partición del campeón (semilla 987654)

El proyecto verificó el linaje V30 también sobre una **segunda partición agrupada**: misma
agrupación de clústeres, semilla 987654, con solape parcial con la canónica — es una
re-ejecución en otra partición, no una partición independiente, y así quedó declarado. Esta
segunda evaluación cumple dos papeles: confirma el linaje fuera de la partición canónica
(0.6144120 de base y 0.6164700 con el plug-in, los valores documentados) y es, además, la que el
proyecto usa como miembro V30 en la **evaluación publicada del campeón V48 y de los consensos del
arco 2026 (secciones 5.2 y 4.6)**: su OOF es el que entró en los consensos geométricos de los
récords. La receta es idéntica (cuartos + dos LightGBM con configuraciones y pesos congelados); la
única diferencia es que los reajustes de esta partición usan el techo completo de árboles, sin early
stopping (Ke et al., 2017), tal como los ejecutó el script original de la confirmación. La partición se
genera aquí mismo con la misma función de la sección 2.4 y la semilla 987654, de modo que el código
queda legible y verificable en el punto donde se usa.

In [ ]:
# Segunda particion (semilla 987654): la misma regla sobre los mismos clusters, generada junto a su uso.
fold_idx2 = generar_folds(y, cluster_de[:n_train], semilla=987654)
print("segunda particion (semilla 987654):", np.bincount(fold_idx2).tolist())

In [ ]:
# La segunda particion del campeon (semilla 987654): validacion del linaje V30.
t0 = time.time()
v20_segunda = cuartos_v20_oof(matriz, y, clase, fold_idx2)
v29_segunda = lgbm_refit_oof_sin_es(matriz, clase, y, fold_idx2, V29_CONFIGS)
v30_segunda = lgbm_refit_oof_sin_es(matriz, clase, y, fold_idx2, V30_CONFIGS)
oof_v30_segunda = mezcla(v20_segunda, v29_segunda, v30_segunda)
print(f"segunda particion reconstruida: {time.time() - t0:.0f}s")
print(f"  V30 segunda        : {macro_f1(clase, np.argmax(oof_v30_segunda, axis=1)):.7f} "
      f"(doc 0.6144120)")
print(f"  V30 + plug-in segunda: {macro_f1(clase, regla_plug_in(oof_v30_segunda)):.7f} "
      f"(doc 0.6164700)")

## 4.3 Fracasos seleccionados [RE/DOC]

Esta sección documenta ocho fracasos seleccionados del proyecto: el MLP denso, CatBoost, FTRL-Proximal, CORAL, el modelado ordinal, los especialistas de frontera, los canales lingüísticos y la selección supervisada de features. Cada subsección declara **[RE]** si el experimento es re-ejecutable en minutos con insumos que el propio notebook reconstruye, o **[DOC]** si exige artefactos congelados (la matriz de 279 features, los OOF de V30) o procesos de horas. En todos los casos se documenta la hipótesis, el resultado exacto con su fuente y la razón del cierre: los fracasos se conservan como evidencia y no se esconden.

### 4.3.1 [RE] MLP denso (E-019/E-020)

Se entrenaron dos MLP en CPU, porque la GPU estaba ocupada por una carga ajena: un MLP(128.64) sobre las 35 features densas y un MLP(256.128) sobre densas más SVD(64) del espacio de caracteres; después se midió su aporte como miembro del ensamble. La hipótesis era que un modelo no lineal sobre densas podía competir con los lineales dispersos y que la SVD comprimiría sin perder señal.

El resultado fue doble. Como clasificadores quedaron muy por debajo: 0.3381 ± 0.0040 (peor que el LightGBM denso, 0.3567) y 0.4071 ± 0.0060, con una SVD que explica solo el 7.8 % de la varianza. Como miembro del ensamble, en cambio, el MLP llevó el meta-modelo de 0.5784 a 0.5851 (+0.0066), el mayor aporte individual desde el canal de menciones. La decisión D-015 fue precisa: el deep learning no se usa como modelo principal, pero sí como miembro por decorrelación de errores; en la era V19 el MLP(64) fue un tercio del blend (LogReg + MLP + árbol). La subsección se marca [RE] porque reconstruir el bloque de 35 densas y reentrenar cuesta unos 5-6 minutos de CPU; sin esa reconstrucción, el resultado es documentable (`experiments.csv:29-31`; `DECISIONS.md:265-281`, D-015; `RUNLOG.md:207-215`).

### 4.3.2 [DOC] CatBoost (V32-E2)

El ciclo 2 de la reapertura V32 probó CatBoost MultiClass (Prokhorenkova, Gusev, Vorobev, Dorogush y Gulin, «CatBoost: unbiased boosting with categorical features», NeurIPS 2018) sobre la matriz 279 congelada (cap de 3000 iteraciones, lr 0.03, depth 6, l2 3.0, early stopping 150), con un gemelo LightGBM afín (A2, control decisional) y etiquetas barajadas (A3, control negativo). La hipótesis era que el ordered boosting aportaría información distinta del LightGBM afilado del pool. El pre-registro exigió una enmienda previa: la métrica de evaluación se fijó como TotalF1:average=Macro, porque el default Weighted de catboost 1.2.10 no coincide con el macro-F1.

El resultado fue negativo en los tres brazos: A1 combinado 0.6148474, delta -0.0011258 (1 de 5 folds; IC95 [-0.0019378, -0.0003467]); A2 -0.0012871 y A3 -0.0012138. La comparación A1 contra A2 no fue favorable (+0.00017; IC95 contiene el cero; p+ = 0.76), y el control barajado rindió casi como A1 y por encima de A2: la señal atribuible al mecanismo es indistinguible del ruido. A1 rescató 9 errores de alta confianza frente a 9.780 del campeón. El cierre fue CLOSED por el árbol pre-registrado, sin segunda partición, y la reapertura V32 quedó agotada. Se marca [DOC] porque el entrenamiento consumió 1149 s de CPU y exige la matriz 279 congelada (`experiments.csv:173`; `DECISIONS.md:2823-2849`, D-082; `RUNLOG.md:2213`).

### 4.3.3 [DOC] FTRL-Proximal (V32-V35)

Cuatro ciclos pre-registrados probaron FTRL-Proximal (McMahan et al., 2013) sobre TF-IDF (Salton y Buckley, «Term-Weighting Approaches in Automatic Text Retrieval», Information Processing & Management, 1988) ajustada por fold, con la hipótesis de que el aprendizaje online proximal con L1-L2 explícito extraería otra región del espacio de texto. El primero (E3) entrenó OvR con 10 pasadas: A1 dio 0.6147759, delta -0.0011973 (1 de 5 folds), empatado bit a bit con su control barajado, y el brazo no aprendió (standalone 0.040169, por debajo del suelo trivial 0.048542). El segundo (V33) reformuló el diseño como softmax multiclase de una pasada con alpha derivado, y la puerta de convergencia G1-G6 lo cortó en el fold 0: ll_train 1.8048 frente al techo 1.6126, con colapso de frecuencias a una clase. El tercero (V34) dio la forma fuerte de la hipótesis, con alpha derivado en forma cerrada y 10 pasadas: la puerta cerró la línea por mecanismo, con G1 (ll_train de 10.09 a 19.17 frente al techo 1.7816) mientras el gemelo SGD (Bottou, «Large-Scale Machine Learning with Stochastic Gradient Descent», COMPSTAT 2010) sano descendía de 1.0622 a 0.4042; el presupuesto no era el límite, el mecanismo sí. El cuarto (V35) cubrió lambda1 > 0 con un anclaje derivado (0.23993793862904017) y la norma L2 se duplicó (razón máxima/mínima 2.1304), lo que refutó la hipótesis de acotación y cerró la familia de forma definitiva.

El cierre es por MECANISMO, no por gates de delta: la integración y el bootstrap (Efron, «Bootstrap Methods: Another Look at the Jackknife», The Annals of Statistics, 1979) no se ejecutan por pre-registro, y reabrir la familia exige un acta nueva con hipótesis mecánica nueva. La evidencia previa ya lo anunciaba: 27 celdas de alpha y 0 de 27 por debajo de H(prior) = 1.7914537. Se marca [DOC] porque exige el núcleo FTRL propio y la TF-IDF por fold, con fail-fast en el fold 0 (`experiments.csv:175, 178, 182, 190`; `DECISIONS.md:2930`, D-083; `:2852`, D-084; `:2980`, D-085; `:3072`, D-086; `RUNLOG.md:2232, 2272, 2315, 2404`).

### 4.3.4 [DOC] CORAL / cumulative-link (V23)

V23 implementó la formulación cumulative-link (Cao, Mirjalili y Raschka, «Rank consistent ordinal regression for neural networks with application to age estimation», Pattern Recognition Letters, 2020) con un único vector de pesos y cinco umbrales ordenados por construcción (véase la sección 4.1), para responder si compartir pesos entre las cinco fronteras ordinales aporta señal que los cinco binarios independientes de V20 no capturan, y si garantizar la monotonía por construcción compra métrica. El diseño incluyó dos controles: A reproduce V20 exacto (0.6097782) y B son cinco binarios lineales independientes (0.4690055), que separan el efecto de compartir pesos del de perder no linealidad.

| Modelo | macro-F1 | Delta vs V20 | Folds + | Monotonía |
|---|---|---|---|---|
| V20 (control A) | 0.6097782 | - | 3/5 | 18.927 violaciones |
| Cinco binarios lineales (control B) | 0.4690055 | -0.140773 | 0/5 | no medido |
| CORAL (peso compartido) | 0.4837017 | -0.126077 | 0/5 | 0 violaciones |

La descomposición es el resultado útil: compartir pesos y ordenar umbrales vale +0.014696 frente al control B, luego la hipótesis era correcta en su plano lineal; pero la no linealidad de LightGBM vale -0.140773, un orden de magnitud mayor. CORAL resuelve la monotonía de verdad (0 violaciones frente a 18.927) y aun así pierde 0.1261, y el mapa de errores se rehace entero (36 de 36 celdas de la matriz de confusión cambian; distancia L1 57.650; aciertos en la diagonal de 103.331 a 80.712). El cierre fue ORDINAL_SHARED_WEIGHTS = NO_SIGNAL y CORAL = CLOSED, sin submission y sin bootstrap por pre-registro: la consistencia ordinal no era el cuello de botella. Se marca [DOC] porque consume 2470 s de CPU y depende del módulo `src/ordinal_coral.py` y de la matriz 279 (`experiments.csv:129`; `DECISIONS.md:1683-1757`, D-058; `reports/v23_coral_analysis.md`; `reports/v23_coral_audit.md`).

### 4.3.5 [DOC] Modelado ordinal (V22)

V22 probó dos ideas ordinales sobre la matriz 279 de V20, sin reentrenar ningún discriminador en la segunda. La primera fue compartir representación: un único LightGBM multiclase del que se derivan las cinco CDF por suma, de modo que la monotonía queda garantizada por construcción. La segunda fue un coste de error proporcional a la distancia ordinal, que alinea la pérdida con la naturaleza ordenada de las etiquetas.

El modelo compartido dio 0.6096150, delta -0.000163 (3 de 5 folds; IC95 [-0.00078, +0.00045]; neto -13; 4.451 cambios), con 0 violaciones de monotonía frente a las 18.927 de V20: resolver el problema gratis no compró métrica. El coste ordinal fue peor: con alfa 1 el macro-F1 cayó a 0.5949198 (-0.0149; neto -3.995) y con alfa 2 a 0.5392168 (-0.0706; neto -13.838), porque los errores del modelo no están concentrados a grandes distancias y forzar su evitación degrada la frontera y la global a la vez. La decisión D-057 fue WRONG_OBJECTIVE, y con ella quedó cerrada la familia ordinal en cuatro subfamilias, junto con D-054, D-055 y D-056. Se marca [DOC] porque las dos corridas duran 33 s y 32 s, pero operan sobre la matriz 279 congelada (`experiments.csv:124, 127`; `DECISIONS.md:1633-1682`, D-057).

### 4.3.6 [DOC] Especialistas de frontera (V13-P3a y V22-B1)

Dos generaciones de especialistas binarios de frontera se probaron y cerraron. P3a entrenó un clasificador binario por par de fronteras sobre las mismas 279 variables de V12, con la hipótesis de que un modelo dedicado al par vencería a la predicción de seis clases restringida al par. El resultado fue negativo en las dos fronteras centrales: 0.4963 a 0.4944 (LogReg) y 0.4946 (MLP) en 18-24|25-34, con deltas de par de -0.0017 y -0.0029 en la variante MLP; 0.4974 a 0.4942 y 0.4945 en 25-34|35-49; y la global bajó de 0.6005 a 0.5993 con 6.944 filas modificadas. La pista se cerró: el especialista dedicado era peor que V12 restringida al par. La corrección D-048 precisó después el diagnóstico: sin la compuerta, el especialista sobre las 279 rinde 0.7136 y 0.7493 (frente a 0.6864 y 0.7218 del TF-IDF disperso), luego la señal existe y el problema era de identificabilidad, no de información.

La segunda generación (V22-B1) sustituyó solo el primer discriminador, con las otras cuatro fronteras congeladas: LinearSVC C 0.2 dio 0.5992098 (-0.0106; B1 0.6782), C 1.0 dio 0.5992058 (B1 0.6784) y un boosting especializado dio 0.6097377 con B1 0.7606, por debajo del 0.7632 de V20. La decisión fue B1_SPECIALIST = NO_SIGNAL, con razón estructural: las cinco CDF comparten features y modelo, y el primer umbral del multiclase ya está a su nivel. Se marca [DOC] porque ambos bloques exigen la matriz 279 congelada (P3a: 154 s; B1: 33 s) (`experiments.csv:86, 103, 125-126`; `DECISIONS.md:1142`, D-048; `reports/v3_especialistas_franjas.csv`).

### 4.3.7 [DOC] Canales lingüísticos (V31)

V31 construyó un banco de representaciones con modelo, folds, semilla y vectorizador fijos (LinearSVC C = 1.0; Fan et al., «LIBLINEAR: A Library for Large Linear Classification», Journal of Machine Learning Research, 2008; folds canónicos de V20) para separar "falta señal" de "falta capacidad", y probó dos vías de integración sobre V30. La hipótesis era que la edad se manifiesta en canales que el TF-IDF de palabras no expresa: URL, ortografía, composición, discurso, posición y forma de palabra. El banco (base `raw` = 0.4979941) dio la tabla siguiente.

| Banco | Representación | macro-F1 | Delta vs raw | Folds + |
|---|---|---|---|---|
| v31_02 URL | url_canal | 0.4987321 | +0.000738 | 3/5 |
| v31_02 URL | url_struct | 0.4926034 | -0.005391 | 0/5 |
| v31_02 URL | url_mask | 0.4910249 | -0.006969 | 0/5 |
| v31_02 URL | solo_url | 0.1563043 | -0.341690 | 0/5 |
| v31_03 canales | ortografía | 0.4999868 | +0.0019927 | 5/5 |
| v31_03 canales | composición | 0.5016523 | +0.0036582 | 5/5 |
| v31_03 canales | discurso | 0.5030002 | +0.0050061 | 5/5 |
| v31_03 canales | ortografía+composición+discurso | 0.5048142 | +0.0068201 | 5/5 |
| v31_03 canales | url (como canal) | 0.4998877 | +0.0018936 | 5/5 |
| v31_03 canales | posición | 0.4956907 | -0.0023034 | 0/5 |
| v31_03 canales | forma_palabra | 0.4950099 | -0.0029842 | 0/5 |
| v31_03 canales | todos | 0.4961184 | -0.0018757 | 1/5 |

Las tres familias buenas (ortografía, composición y discurso) son señal real y consistente: su combinación bate al texto solo en +0.0068201 con 5 de 5 folds, y el canal URL explícito aporta +0.0018936. Posición y forma de palabra salieron negativas y se cerraron. En la integración, el canal como cuarto miembro (N1) dio +0.0000642 sobre el control reoptimizado: la mayor parte del aparente +0.0004119 frente a V30 congelado era el efecto de reoptimizar pesos, no el canal. Como 48 features del meta-espacio (N2) dio +0.0002948 sobre el control barajado. Ambos quedan muy por debajo del gate de +0.0015. La decisión D-067 fue: URL CLOSED; posición y forma_palabra CLOSED; ortografía, composición y discurso MARGINAL (señal real sin sitio donde V30 no la exprese ya); 0 finalistas y 0 submissions. Se marca [DOC] porque el banco consumió 3802 s y la integración exige los OOF de V30 (`reports/v31_representation_benchmark.csv`; `experiments.csv:143-148`; `DECISIONS.md:2086-2152`, D-067).

### 4.3.8 [DOC] Selección supervisada de features (V25)

V25 respondió una pregunta acotada: ¿contiene V20 features ruidosas que, al eliminarse con selección supervisada dentro de cada fold, mejoren el macro-F1? La comparación usó el mismo código y los mismos folds de V20, con una sola variable: T1 conservó el top 50 % por `mutual_info_classif` (K = 139 de 279), estandarizado dentro del training fold. El control T0 reprodujo V20 exacto (0.6097782) y T1 dio 0.6091492: delta -0.000629, 1 de 5 folds positivos, 1.042 rescates contra 1.143 daños, neto -101 y 3.545 filas cambiadas.

El hallazgo por bloque explica el fracaso: sobreviven con frecuencia 1.000 las 78 probabilidades del ensamble y las 6 de NB-SVM, y nunca sobreviven 74 de las 87 estructuradas ni 30 de las 35 densas; 137 features fueron estables en los cinco folds. La selección no elimina ruido, elimina señal: lo que el mutual information ordena son las predicciones de los miembros que sostienen el modelo. La decisión fue SUPERVISED_FEATURE_SELECTION = CLOSED, sin probar otro K ni otro selector y sin bootstrap por pre-registro. Se marca [DOC] porque la corrida consume 953 s y exige la matriz 279 congelada (`experiments.csv:131`; `DECISIONS.md:1816-1882`, D-060; `reports/v25_feature_selection.md`).

## 4.5 V44-V52: falsaciones documentadas [DOC]

Tras la adopción de V43, el proyecto ejecutó nueve versiones más (V44-V52) para intentar superar el récord o para delimitar su techo. Todas comparten el protocolo canónico: validación de 5 folds `StratifiedGroupKFold` agrupada por `dup_clusters.parquet` (169.336 filas OOF, cero fuga in-fold) y evaluación con la regla plug-in congelada. La compuerta de adopción exigía consistencia por folds e intervalo de confianza pareado (bootstrap B=1.000; Efron, «Bootstrap Methods: Another Look at the Jackknife», The Annals of Statistics, 1979) que excluyera el cero (p < 0.05); ningún envío se hacía sin superarla. La única adopción del tramo es V48; las demás quedaron como falsaciones documentadas.

| Versión | Falsación (mecanismo) | Resultado exacto | Folds / p | Decisión |
|---|---|---|---|---|
| V44 | Boundary-EMD L1 + ablaciones (POS 305, Multi-Sample Dropout, Mixup, stacking logístico, XGBoost) | OOF 0.6183063; delta -0.0001720 vs V43 | 1/5; p=0.8550 | Cerrada |
| V45 | Batería de 21 hipótesis (discretización 1D, compuertas, Nystroem, meta-árbitro, reoptimización anidada del plug-in) | Ninguna supera a V43; peor: discretización 1D -0.04167; reoptimización del plug-in -0.0003374 | n/d (gate por CV) | Cerrada (el techo del oráculo se analiza en 5.4) |
| V46 | RankGauss (QuantileTransformer normal) | Standalone 0.6055514; ensamble 0.6184456, delta -0.0000327 vs V43 | 2/5; p no reportado | Cerrada |
| V47 | Weighted-Cramer + recorte [-6, +6]; enlace acumulativo | Nominal +0.0000743 vs V43; enlace acumulativo 0.5810757 (-0.033) | 3/5; p=0.2620; IC95 [-0.0001503, +0.0002838] | Cerrada |
| V48 | PoE geométrico (media geométrica en log-probabilidades) | CV 0.6186451; delta +0.0002199 vs V43 recomputado; LB 0.63724 | 4/5; p=0.0250; IC95 [+0.0000007, +0.0004095] | ADOPTADO (campeón) |
| V49 | CatBoost oblivious trees | Standalone 0.6066473; PoE de 7 modelos 0.6185635, delta -0.0000816 vs V48 | 2/5; p=0.7480; IC95 [-0.0003022, +0.0001361] | Cerrada |
| V50 | TabTransformer MHSA; matriz W 6x6 desacoplada | MHSA fold 0: 0.6073432 (blend -0.0001058; sustitución -0.0005204); W nominal +0.0000474 | 4/5 (W); p=0.1940; IC95 [-0.0000633, +0.0001577] | Cerrada |
| V51 | Consistent Smoothed Cramér-von Mises | Standalone 0.6128371; ensamble 0.6186149, delta -0.0000302 vs V48 | 3/5; p=0.6900; IC95 [-0.0001337, +0.0000721] | Cerrada |
| V52 | Power-Mean; Dual-Transport 7-model PoE; SNN/SELU; paradoja de Simpson | Dual-Transport 0.6185841, delta -0.0000610 vs V48; SNN fold 0 -0.00913; regla por segmento global -0.0013763 | 3/5 (Dual); p=0.7040; IC95 [-0.0002945, +0.0001803] | Cerrada |

Nota de referencias: V43 tiene dos cifras documentadas (0.6184783 publicada y 0.6184252 recomputada); los deltas "vs V43" se citan tal como están registrados y el delta de V48 se declara con ambas referencias en su apartado.

### 4.5.1 V44: la distancia L1 en las fronteras

Hipótesis. El 68.3% del error de V43 se concentraba en las fronteras intermedias contiguas (10.929 errores directos entre 18-24 y 25-34, y 7.965 entre 25-34 y 35-49). Penalizar de forma asimétrica esas fronteras con la distancia de Wasserstein-1 (L1 sobre la CDF, con gamma = [1.0, 1.4, 1.4, 1.1, 1.0]; Frogner, Zhang, Mobahi, Araya y Poggio, «Learning with a Wasserstein Loss», NeurIPS 2015) debía destrabar el cuello de botella; se añadieron cinco ablaciones (26 features POS, Multi-Sample Dropout (Inoue, «Multi-Sample Dropout for Accelerated Training and Better Generalization», arXiv, 2019), Mixup tabular (Zhang, Cissé, Dauphin y Lopez-Paz, «mixup: Beyond Empirical Risk Minimization», ICLR 2018), stacking logístico y XGBoost (Chen y Guestrin, «XGBoost: A Scalable Tree Boosting System», KDD 2016)). Resultado. A pesar de un beneficio aparente en fold 0 (+0.000438), el OOF de 5 folds fue 0.6183063: delta -0.0001720 frente a V43, solo 1/5 folds positivos y p=0.8550. Ninguna ablación mejoró en fold 0: POS -0.000776, Mixup -0.000500, stacking -0.001288 y XGBoost -0.000403. Lección. La norma L1 produce gradientes de paso discontinuos que oscilan alrededor del umbral óptimo; Cramér-von Mises (L2) (Cramér, «On the composition of elementary errors: First paper: Mathematical deductions», Scandinavian Actuarial Journal, 1928; von Mises, «Wahrscheinlichkeitsrechnung und ihre Anwendung in der Statistik und theoretischen Physik», Franz Deuticke, 1931) queda confirmada como estrictamente superior. (D-096; `experiments.csv:208`; `RUNLOG.md:2664-2679`).

### 4.5.2 V46: RankGauss

Hipótesis. La matriz de 279 variables es muy asimétrica: 124 variables (44.4%) con |skew| > 3, 26 con |skew| > 10 y 71 con más del 50% de ceros. Normalizar por cuantiles (RankGauss, $z = \Phi^{-1}(F(x))$) eliminaría los picos de gradiente de los outliers y estabilizaría las compuertas de atención SE. Resultado. El modelo standalone cayó a 0.6055514 frente a 0.6128402 de V43 con StandardScaler (-0.00729), el peor de la familia neuronal; en ensamble promedió 0.6184456 (delta -0.0000327 vs V43, solo 2/5 folds). La reoptimización libre del símplex convergió a [0.556, 0.084, 0.175, 0.080, 0.050, 0.055] con 0.6183841 (-0.0000942). Lección. Forzar distribuciones cero-infladas hacia una normal destruye la señal discreta del indicador de ausencia; el pico de asimetría no era el problema. (D-098; `experiments.csv:210`; `RUNLOG.md:2703-2716`).

### 4.5.3 V47: Weighted-Cramer con recorte

Hipótesis. La auditoría de la matriz encontró 109 variables con |z| > 10σ y 16 con |z| > 50σ (máximo 120.94σ), observaciones que distorsionan la varianza de lote en BatchNorm (Ioffe y Szegedy, «Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift», ICML 2015). Recortar in-fold a [-6, +6] y ponderar la Cramér en los cuellos de botella (gamma = [1.2, 1.5, 1.3, 1.1, 1.0], sin normalizar) debía estabilizar el entrenamiento; en paralelo se probó el enlace acumulativo (proportional odds; McCullagh, 1980). Resultado. El enlace acumulativo colapsó a 0.5810757 (-0.033) y quedó falsado. V47 rindió 0.6110824 standalone y 0.6184996 como miembro de reemplazo (delta +0.0000743 vs V43), pero solo 3/5 folds positivos; bootstrap p=0.2620 con IC95 [-0.0001503, +0.0002838]. Lección y decisión. Un delta nominal positivo no basta: con el IC95 conteniendo el cero no hay evidencia para tocar la submission (compuerta anti-shakeup). (D-099; `experiments.csv:211`; `RUNLOG.md:2718-2732`).

### 4.5.4 V48: el PoE geométrico (adoptado)

Hipótesis. La media aritmética del ensamble (modelo de mezcla, tipo OR) es vulnerable a la confianza espuria de un solo miembro; el producto de expertos bayesiano (media geométrica en log-probabilidades, $\log P_{\text{geom}} = \sum_m w_m \log P_m - \log Z$; Hinton, «Training Products of Experts by Minimizing Contrastive Divergence», Neural Computation, 2002) actúa como consenso estricto (tipo AND) y debía mejorar la certidumbre conjunta en las fronteras ordinales. Resultado. Los pesos óptimos por nested CV convergieron a w* = [0.5628, 0.0832, 0.1724, 0.0754, 0.0513, 0.0549]; el CV subió a 0.6186451 (delta +0.0002199 frente a V43 recomputado 0.6184252; +0.0001668 frente al publicado 0.6184783; +0.0026719 frente al plug-in congelado). Fueron 4/5 folds positivos (el quinto, -0.000008), bootstrap con media +0.0002151 e IC95 [+0.0000007, +0.0004095], p=0.0250; 290 filas cambiaron respecto a V43 (0.26%). En Kaggle marcó 0.63724 (ref. 57283487). Decisión. ADOPTADO como campeón: única adopción del tramo. (D-101; `experiments.csv:212`; `RUNLOG.md:2734-2762`; `CHAMPION.md`).

### 4.5.5 V49: CatBoost oblivious

Hipótesis. El campeón rinde 0.6760801 en tweets con mención (63.9% del corpus: 108.186 en train y 72.085 en test) pero solo 0.5043177 sin mención (36.1%: 61.150 y 40.807), porque el bloque supervisado de 73 variables colapsa al prior. Un CatBoost de árboles oblivious (mismo corte en todo el nivel; Prokhorenkova, Gusev, Vorobev, Dorogush y Gulin, «CatBoost: unbiased boosting with categorical features», NeurIPS 2018) debía capturar patrones globales en el segmento sin mención. Resultado. Standalone 0.6066473; en el PoE de 7 modelos pesó 0.0372 en media y el global quedó en 0.6185635 (delta -0.0000816 vs V48); 2/5 folds positivos; p=0.7480 con IC95 [-0.0003022, +0.0001361]. Lección. Falsación #26: los árboles simétricos no rescatan la señal del segmento sin menciones y no aportan al consenso. (D-102; `experiments.csv:213`; `RUNLOG.md:2763-2777`).

### 4.5.6 V50: MHSA y la matriz W desacoplada

Hipótesis. Dos vías de frontera: (a) un TabTransformer (Huang, Khetan, Cvitkovic y Karnin, «TabTransformer: Tabular Data Modeling Using Contextual Embeddings», arXiv, 2020) con self-attention multi-cabeza (8 tokens de dimensión 64, 2 capas, 4 cabezas; Vaswani et al., «Attention Is All You Need», NIPS 2017) como rama neuronal, y (b) desacoplar el PoE por clase con una matriz W de 6x6, para que cada rango optimice su propia mezcla. Resultado. (a) MHSA rindió 0.6073432 en fold 0 (frente a 0.61284 de SE-TabNet; Arik y Pfister, «TabNet: Attentive Interpretable Tabular Learning», AAAI 2021); añadirlo degradó el blend de 0.6227569 a 0.6226511 (-0.0001058) y sustituir la rama SE lo hundió a -0.0005204. (b) La matriz W dio 0.6186925 nominal (+0.0000474, 4/5 folds), pero el IC95 [-0.0000633, +0.0001577] contiene el cero y p=0.1940; los pesos óptimos resultaron casi invariantes entre clases (V30: 0.5516 a 0.5701). Lección. La atención sobre tokens sintéticos carece del sesgo inductivo localizado de Squeeze-and-Excitation (Hu, Shen y Sun, «Squeeze-and-Excitation Networks», CVPR 2018), y estimar 36 parámetros en lugar de 6 añade varianza sin separación defendible. (D-103; `experiments.csv:214`; `RUNLOG.md:2778-2787`).

### 4.5.7 V51: Consistent Smoothed Cramér

Hipótesis. En V43 la entropía cruzada con suavizado (Szegedy, Vanhoucke, Ioffe, Shlens y Wojna, «Rethinking the Inception Architecture for Computer Vision», CVPR 2016; Müller, Kornblith y Hinton, «When Does Label Smoothing Help?», NeurIPS 2019) empujaba los logits hacia una distribución dispersa mientras la CDF dura del término de Cramér (Cramér, 1928) los atraía hacia un delta de Dirac: una contradicción de gradientes. Unificar la CDF objetivo con el suavizado consistente, $\tilde{F}_y(k) = (1-\epsilon)\,\mathbb{I}(y \le k) + \epsilon\,\frac{k+1}{K}$, debía eliminar esa tensión y mejorar la calibración de frontera. Resultado. Standalone 0.6128371, indistinguible del baseline (0.6128402; delta -0.0000031); en ensamble 0.6186149 (delta -0.0000302 vs V48), 3/5 folds positivos, p=0.6900 con IC95 [-0.0001337, +0.0000721]. Como séptimo miembro tampoco mejoró (delta entre -0.00020 y +0.00002). Lección. Falsación #29: la contradicción de gradientes no era el cuello de botella; la calibración de V43 ya estaba en su punto. (D-104; `experiments.csv:215`; `RUNLOG.md:2788-2803`).

### 4.5.8 V52: Power-Mean, SNN y la paradoja de Simpson

Hipótesis. Cuatro líneas: (a) el Power-Mean $P_\alpha(k) \propto (\sum_m w_m P_m(k)^\alpha)^{1/\alpha}$ como puente continuo entre la media aritmética (α=1) y el PoE (α→0); (b) un ensamble Dual-Transport de 7 modelos que acopla la EMD L1 de V44 con la Cramér L2 de V43; (c) redes autonormalizantes SNN (SELU, inicialización LeCun normal, AlphaDropout; Klambauer, Unterthiner, Mayr y Hochreiter, «Self-Normalizing Neural Networks», NeurIPS 2017) para eliminar el ruido de minilote de BatchNorm; (d) reglas de decisión desacopladas por segmento. Resultado. (a) El óptimo bajo la compuerta es α→0: α=-0.30 dio 0.618721 nominal pero solo 3/5 folds; α=0.50 (-0.000242) y α=1.00 (-0.000208) fallaron con 0/5 folds. (b) Dual-Transport 0.6185841 (delta -0.0000610 vs V48; 3/5 folds; p=0.7040; IC95 [-0.0002945, +0.0001803]). (c) SNN en fold 0: 0.608477, -0.00913 frente a SE-TabNet. (d) Subir el F1 del segmento sin menciones (+0.000185) hundió el global en -0.0013763 (0.6172689 vs 0.6186451). Lección. Falsaciones #30 y #31: el PoE geométrico es el óptimo de la familia de operadores, y la métrica armónica no admite reglas por segmento sin romper la calibración marginal, porque los FP y FN de ambos segmentos se suman en el denominador (Simpson, «The Interpretation of Interaction in Contingency Tables», Journal of the Royal Statistical Society: Series B (Methodological), 1951). (D-105; `experiments.csv:216`; `RUNLOG.md:2804-2817`).

### 4.5.9 El patrón: el PoE geométrico como óptimo de la familia de operadores

La familia de operadores de fusión quedó caracterizada. La media aritmética (α=1) degrada -0.000208 con 0/5 folds; el rank-averaging colapsa a 0.5958594 (degradación -0.02257) porque destruye la densidad continua que la regla plug-in necesita; el Power-Mean confirma que la dirección correcta es α→0; y la media geométrica pasó de +0.0001509 con 4/5 folds y p=0.0930 (con los pesos fijos de V43) a la adopción definitiva cuando los pesos se optimizaron en nested CV (V48, p=0.0250). Bajo la compuerta pre-registrada, el PoE geométrico es el único punto adoptable de la familia. Las ocho falsaciones restantes (V44, V45, V46, V47, V49, V50, V51 y V52) comparten una lectura ya medida en el cierre de canales: una representación puede tener señal real en aislamiento y no aportar nada al integrarse, porque el modelo existente la expresa mejor; las variantes de pérdida, arquitectura u operador solo añaden varianza. (D-100; D-105; `reports/INFORME_EXPERIMENTOS_Y_LEADERBOARD.md:128`).

Fuentes: `experiments.csv:208-216`; `DECISIONS.md` D-096 a D-105 (`DECISIONS.md:3696-3885`); `RUNLOG.md:2664-2817`; `CHAMPION.md`; paquetes `06_fracasos_y_analisis.md` (B.5) y `04_neuronales.md` (contexto V44/V46/V47); `reports/INFORME_EXPERIMENTOS_Y_LEADERBOARD.md`.

## 4.6 V53-V57: la frontera 2026, el récord de CV y el techo estacionario [RE/DOC]

*Sección mixta: re-ejecuta los componentes y consensos del arco 2026 (V47/V51/V53, los PoE de V53/V54/V56 y la auditoría V57) y documenta lo que exige artefactos congelados o entrenamientos largos (V55 con CatBoost). Fuentes: `RUNLOG.md` (entradas V53-V57), `DECISIONS.md` (D-106 a D-110), `experiments.csv` y los JSON `experiments/v53_*.json` a `v57_*.json`. La procedencia de cada mecanismo en el estado del arte está en `reports/state_of_art.csv` (filas C13-C17), `reports/age_literature_methods_registry.csv` y `references.bib`.*

Tras el récord de V48 (CV 0.6186451; LB 0.63724), el proyecto siguió empujando la frontera con tres piezas nuevas —una arquitectura densa, dos variantes del transporte ordinal de Cramér y un árbol simétrico— más una auditoría matemática del consenso. La pregunta era si el espacio logarítmico admitía miembros aún más diversos sin degradar las cohortes minoritarias; la respuesta quedó medida: el techo de validación local subió a **0.6189929** (V56, 5/5 folds positivos, bootstrap pareado p = 0.0010 frente a V48) y la **selección dual V48/V56** quedó ratificada para Kaggle (V48 conserva el mejor LB, 0.63724; V56 aporta la mejor validación local, LB 0.63717).

### 4.6.1 V53 — Dense-SE-TabularNet y el consenso de siete modelos

La red densa conecta cada bloque con todos los siguientes (192 -> 320 -> 448 -> 576) para preservar las representaciones tempranas sin dilución aditiva (Huang, Liu, van der Maaten y Weinberger, «Densely Connected Convolutional Networks», CVPR 2017), con compuertas Squeeze-and-Excitation (Hu, Shen y Sun, «Squeeze-and-Excitation Networks», CVPR 2018) y la pérdida de Cramér consistente-suavizada (Cramér, 1928; Székely y Rizzo, 2013; véanse las secciones 4.4 y 4.5). Como clasificador individual rinde 0.6111343; integrada al consenso geométrico de siete modelos (V30, V43, V42, V40, V39-ordinal, V39-HGB y la red densa) con la regla plug-in congelada alcanza **CV 0.6188259** (bootstrap p = 0.0280 vs V48) y **LB 0.63706**.

### 4.6.2 V54 — el consenso jerárquico multi-pérdida

La rama de transporte ordinal se unificó en un ensamble jerárquico: 0.80 de Cramér simétrico (V43) más 0.20 de Cramér asimétrico ponderado con recorte a [-6, +6] (V47, documentado en la sección 4.5) más la red densa y los demás miembros, todo en espacio logarítmico (Hinton, «Training Products of Experts by Minimizing Contrastive Divergence», Neural Computation, 2002; Heskes, «Selecting Weighting Factors in Logarithmic Opinion Pools», NeurIPS, 1998). El diagnóstico de V53 —una red densa sin amortiguar contraía la cohorte 13-17— se corrigió con el peso jerárquico. Resultado: **CV 0.6188887** (p = 0.0050 vs V48; 180 filas modificadas vs V48, 0.16%) y **LB 0.63712**.

### 4.6.3 V55 — el octa-modelo limpio y el blindaje del 30% privado

Al constatar que las redes densas rinden por debajo de las residuales en aislamiento (0.6111 vs ~0.6128), V55 excluyó formalmente las arquitecturas subóptimas e inyectó un árbol simétrico CatBoost con peso 0.00805 (Prokhorenkova, Gusev, Vorobev, Dorogush y Gulin, «CatBoost: unbiased boosting with categorical features», NeurIPS 2018) en un consenso PoE de ocho modelos de máxima calidad. Resultado: CV 0.6182436 (4/5 folds positivos vs V48) y **LB 0.63707**, adoptado como protocolo de blindaje para la evaluación privada.

### 4.6.4 V56 — el consenso no lineal de nueve modelos (récord histórico de CV)

La integración armónica definitiva: nueve modelos de sesgos ortogonales en espacio logarítmico —V30 (LightGBM), V43 (Cramér-SWA), V47 (Cramér asimétrico winsorizado), V42 (suavizado-SWA), V40 (SE ordinal), V39-ordinal, V39-HGB, V51 (Cramér consistente) y V53 (red densa)— con la regla plug-in (Lipton, Elkan y Narayanaswamy, 2014). Resultado: **CV 0.6189929 — máximo histórico absoluto del proyecto** (delta +0.0003478 vs V48), con 5/5 folds positivos frente a V54, bootstrap pareado p = 0.0010 vs V48, 194 filas modificadas vs V48 (0.17%) y **LB 0.63717** confirmado en Kaggle (puesto #1).

### 4.6.5 V57 — geometría del símplex y concavidad Power-Mean

La auditoría matemática de cierre caracterizó la cuenca de generalización del consenso: sensibilidad coordenual en el símplex de 9 pesos (Delta^8), concavidad de la familia continua Power-Mean para alpha en [-0.15, +0.15], desalineamiento de pérdidas sustitutas continuas en validación anidada y confiabilidad decil a decil sobre las 169.336 predicciones OOF. La conclusión formal: V56 habita un **óptimo estacionario absoluto** inmune al sobreajuste del leaderboard público, lo que ratificó la selección dual V48/V56 y el blindaje del 30% privado (D-110).

| Versión | Mecanismo | CV local | Kaggle LB | Cierre |
|---|---|---|---|---|
| V53 | Dense-SE-TabularNet + PoE de 7 modelos | 0.6188259 | 0.63706 | Ex-récord CV (p = 0.0280 vs V48) |
| V54 | PoE jerárquico de 8 (Cramér simétrico + asimétrico) | 0.6188887 | 0.63712 | Ex-récord CV (p = 0.0050) |
| V55 | Octa-modelo limpio + CatBoost simétrico | 0.6182436 | 0.63707 | Protocolo de blindaje (4/5 vs V48) |
| V56 | PoE no lineal de 9 modelos + plug-in | **0.6189929** | 0.63717 | **Récord histórico de CV** (5/5; p = 0.0010) |
| V57 | Auditoría matemática del símplex 9D | 0.6189929 | 0.63717 | Techo estacionario confirmado |

El estado final del proyecto queda así: **V48 es el campeón oficial de leaderboard** (0.63724, re-ejecutado en la sección 5.2 de este notebook) y **V56 el récord de validación local** (0.6189929), con la selección dual ratificada por la auditoría de V57. Las celdas siguientes re-ejecutan el arco completo —los tres componentes nuevos, los consensos publicados y la auditoría matemática—; V55 queda documentado (su miembro CatBoost cuesta ~19 minutos de entrenamiento y no es récord).

# FASE 5 — Evaluación

## 5.1 El protocolo de validación y su reproducibilidad

Toda la evaluación del proyecto se apoya en el mismo protocolo, definido en la Fase 2 y usado sin
excepciones: **validación cruzada de 5 folds agrupados por clústeres de near-duplicados** (folds
generados con la regla documentada de la sección 2.4; Kaufman, Rosset, Perlich y Stitelman, «Leakage in Data Mining: Formulation, Detection, and Avoidance», ACM Transactions on Knowledge Discovery from Data, 2012), con la métrica **macro-F1 sin ponderar** calculada de dos maneras
complementarias — *pooled* (todas las predicciones juntas) y media por folds — más pruebas de
significancia cuando una decisión estaba en juego (bootstrap (Efron, «Bootstrap Methods: Another Look at the Jackknife», The Annals of Statistics, 1979) pareado con re-muestreo de clústeres, B=1000,
intervalo de confianza al 95 % y fracción de folds positivos). Las decisiones importantes se confirmaron
además en una **segunda partición** con semilla distinta.

Este diseño tiene una propiedad que conviene explicitar: el CV agrupado es **conservador** respecto del
leaderboard (sección 5.5). Al eliminar los near-duplicados entre folds, mide la capacidad de generalizar
a textos nuevos; el leaderboard, en cambio, contiene near-duplicados del train y los «cobra», de modo que
el puntaje real queda sistemáticamente por encima del CV local. El proyecto verificó esa brecha en cada
envío y la usó como calibración, nunca como objetivo.

## 5.4 Análisis matemáticos y estadísticos [DOC]

Cuatro análisis delimitan el desempeño del campeón: la cota superior de un selector perfecto (el techo oracle), la refutación de la prueba que declaraba inalcanzable el LB 0.70, la multiplicidad estadística de las ~151 corridas y el sesgo del inner split de Optuna (Akiba et al., «Optuna: A Next-generation Hyperparameter Optimization Framework», KDD 2019). Se presentan en lenguaje llano, con su receta y sus fuentes.

### 5.4.1 El techo oracle del ensamble: 0.693547

El cálculo parte de las 169.336 predicciones OOF de los seis miembros del ensamble (V30, V43, V42, V40, V39-ordinal y V39-HGB). Con la mezcla fija p_blend = 0.555·V30 + 0.085·V43 + 0.175·V42 + 0.080·V40 + 0.050·V39_ord + 0.055·V39_hgb y la regla plug-in congelada (Lipton, Elkan y Narayanaswamy, «Thresholding Classifiers to Maximize F1 Score», arXiv, 2014) w = [1.0000, 1.1749, 1.2721, 1.1630, 1.0010, 0.8128], el blend comete 65.271 errores (38.55%). El oráculo se construye por sustitución por fila: para cada error, si algún miembro acierta la etiqueta verdadera bajo su propia regla plug-in, la predicción se reemplaza por y_true. El macro-F1 del vector corregido es 0.693547. La descomposición ordinal muestra dónde está el error: 61.94% a distancia 1 (40.429 filas), 23.59% a distancia 2 (15.395) y 14.47% a distancia 3 o más (9.447); el 85.5% está a distancia ≤ 2. Advertencia de lectura obligatoria: es un oráculo de selección por fila sobre miembros ya entrenados, la cota superior de un selector perfecto con la misma evidencia, no un modelo. El intento de realizarlo (árbitro V17-P9) falló con 3 rescates y 22 daños; no debe presentarse como objetivo alcanzable. (D-097, `DECISIONS.md:3720-3759`; `notebooks/99_entrega_consolidado.ipynb` sección 44.1; `RUNLOG.md:2681-2701`).

### 5.4.2 La refutación del techo de LB 0.70 (K-01)

El acta D-087 declaraba inalcanzable un LB de 0.70 apoyándose en un oráculo de umbrales. La auditoría D-090 revisó la prueba y la refutó: la rejilla del script solo exploraba tau ≥ 0.30 (TAUS = (0.30, 0.35, 0.40, 0.45, 0.50), en `scripts/v35_00g_techo_analisis.py:69`) y el techo se tomó como su máximo, pero la curva del oráculo es monótona creciente al bajar tau: un umbral más permisivo corrige más filas. La reproducción sobre `artifacts/p33_scores/todos/` (base 0.6159732, 65.683 errores) lo cuantifica:

| tau | filas corregibles | ganancia macro-F1 | ¿supera +0.0626? |
|---|---|---|---|
| 0.30 (máximo publicado) | 10.384 | +0.0593677 | no |
| 0.2930 | 11.298 | +0.0646335 | sí |
| 0.28 | 13.199 | +0.0755813 | sí |
| 0.25 | 18.345 | +0.1054505 | sí |

La conclusión correcta es más fuerte y distinta: el objetivo queda exactamente en el borde de lo que un oráculo sobre la salida de V30 podría entregar, pues exigiría acertar ~16% de los errores con p(y_true) ≥ 0.293, es decir, conocimiento casi-oracular. El LB 0.70 sigue siendo inalcanzable en la práctica, pero la prueba publicada no lo demuestra. (D-090, `DECISIONS.md:3462-3478`; `experiments.csv:198-200`; `plan/auditoria/02c_protocolo_y_qa.md`; `PROJECT_STATE.md:20-27`).

### 5.4.3 La multiplicidad de las corridas (A-01)

El registro acumuló ~151 corridas y cada adopción se decidió con un gate de delta sobre el CV. Con un macro_f1_std mediano de 0.0042 (76 filas que lo declaran), el error estándar de la media de 5 folds es SE ≈ 0.00188. En esas condiciones, un gate de Delta ≥ +0.0015 equivale a z ≈ 0.80 y, sobre una familia de 151 pruebas, se esperarían del orden de 32 falsos positivos; con Delta ≥ +0.0030 (z ≈ 1.60), unos 8. Una corrección de Bonferroni de familia 151 exigiría Delta ≥ 0.0147, umbral que ninguna adopción del proyecto declara por sí sola. La auditoría lo registró como defecto de marco (A-01), no de ejecución. Lo que sostiene las conclusiones: las cuatro adopciones reales (V19-B1, V20, V30 y el plug-in por clase) tienen IC95 pareado que excluye el cero y confirmación externa en el leaderboard, y el único control de multiplicidad previo era el Bonferroni interno de V37 (18 pruebas por clase). (D-090, `DECISIONS.md:3484-3494`; `experiments.csv:198`; `plan/auditoria/05_sintesis_y_cierre.md:13-56`).

### 5.4.4 El sesgo del inner split de Optuna

Los cinco scripts del camino al campeón (`v29_04`, `v29_06`, `v30_01`, `v30_02`, `v31_05`) seleccionan hiperparámetros con un inner split estratificado sin grupos; como el outer split está agrupado por clústeres de near-duplicados, un mismo clúster puede caer a ambos lados del inner. La medición reconstruye las dos particiones interiores exactamente como las construye el código real, sobre los 5 folds canónicos (169.336 filas, 164.018 clústeres):

| Nivel | solape de clúster en la validación interior |
|---|---|
| outer agrupado (canónico) | 0.000% |
| inner ingenuo (código real) | 3.907% |
| inner agrupado (propuesto) | 0.000% |

El criterio se declaró antes de correr (diferencia > 0.02 = material); lo medido fue 0.039071, casi el doble, y consistente en los 5 folds (3.787%-4.152%). Esto corrige la medición previa registrada en D-090 ("0.963% vs 1.030%"), que medía solapamiento de texto exacto cuando los clústeres agrupan por Jaccard (Jaccard, «The Distribution of the Flora in the Alpine Zone», New Phytologist, 1912) y comparaba contra cero teórico en vez del inner agrupado. Alcance: el leakage está en la SELECCIÓN de hiperparámetros, no en la estimación externa, que usó el outer agrupado; la cifra publicada del campeón no cambia. La re-optimización quedó pendiente de decisión y no se ejecutó. Nota de citación: hay dos actas numeradas D-091 (inner split, 2026-10-01, y V38, 2026-10-16); al citar, usar el título. (D-091 inner split, `DECISIONS.md:3534-3606`; `experiments.csv:201`; `plan/auditoria/06_accion_inner_split.md`; `scripts/auditoria_medir_inner_split_optuna.py`).

Fuentes: `plan/kaggle_notebook/paquetes/06_fracasos_y_analisis.md` (B.1-B.4); `DECISIONS.md` (D-090, D-091 inner split, D-097); `experiments.csv:198-201`; `plan/auditoria/05_sintesis_y_cierre.md` y `06_accion_inner_split.md`; `reports/auditoria_experimentos_v1_v37.md`.

## 5.5 Calibración CV <-> leaderboard [DOC]

La relación entre la validación agrupada y el leaderboard de Kaggle se midió con los envíos oficiales del proyecto. Esta sección fija la ley lineal, la brecha media y su lectura, y deja la progresión completa de pares CV/LB como contexto.

### 5.5.1 La ley lineal entre CV y LB

Con los cuatro pares canónicos cuyos extremos son verificables en disco (v01 0.5344444/0.55235; v07 0.586/0.60372; v08 0.5866/0.60376; v12 0.6005017473/0.62182), la regresión del leaderboard sobre el CV es $\text{LB} = 0.00223 + 1.02825 \times \text{CV}$. El informe publica un R2 de 0.998; recomputado sobre esos cuatro pares, el ajuste da 0.996776 (≈0.9968, `experiments/age_cierre_auditoria.json`), de modo que la cifra 0.998 no es reconstruible de los pares documentados: la discrepancia se declara aquí (ver `07_checklist_verificacion.md`, sección 0.4). La desviación residual es 0.00147. La primera regresión, que incluía el puntaje degenerado de v08 (0.39563, subida con formato roto, incidente D-021), dio pendiente 0.231 y R2 0.0043; al excluir ese punto se obtiene la línea final. Lectura práctica: la brecha es un offset aditivo estable; un gate de +0.003 en CV vale +0.00308 en LB, luego los gates locales ni engañan ni regalan puntos. (D-050, `DECISIONS.md:1247-1298`; `reports/INFORME_EXPERIMENTOS_Y_LEADERBOARD.md:62-66`; `reports/age_lb_calibracion.csv`; `notebooks/99_entrega_consolidado.ipynb` sección 38.1).

### 5.5.2 La brecha media y su lectura

El informe cita la brecha media CV → LB como +0.0177 ± 0.0023 sobre diez mediciones. La recomputación del checklist sobre once envíos históricos da media +0.01768 y sd 0.00204 (≈0.0020), y sobre los doce modernos ≈0.0010; el ±0.0023 es, por tanto, una cifra de prosa. Sobre los cuatro pares canónicos de la regresión, la brecha media es 0.01853 con sd 0.00164 y rango 0.01716-0.02132 (`reports/age_lb_calibracion.csv`). La lectura: la validación agrupada es conservadora. El test contiene near-duplicados (Kaufman, Rosset, Perlich y Stitelman, «Leakage in Data Mining: Formulation, Detection, and Avoidance», ACM Transactions on Knowledge Discovery from Data, 2012) del train que el CV agrupado elimina por diseño (los clústeres agrupan por similitud Jaccard; Jaccard, «The Distribution of the Flora in the Alpine Zone», New Phytologist, 1912) y el leaderboard los cobra; por eso el LB supera sistemáticamente al CV con un offset estable. No es un artefacto de la métrica ni un error del protocolo: es una propiedad medida del corpus. (D-050; `reports/INFORME_EXPERIMENTOS_Y_LEADERBOARD.md:68`; `07_checklist_verificacion.md` sección 6; `notebooks/99_entrega_consolidado.ipynb:1120.1286-1287`).

### 5.5.3 Progresión de envíos (v01 → V48)

La tabla recoge los 22 envíos puntuados del arco; los CV son locales (recomputables) y los LB son puntajes oficiales de Kaggle citados como documentados, no recomputables en el notebook.

| Versión | CV local | LB Kaggle | Ganancia LB |
|---|---|---|---|
| v01 | 0.5344 | 0.55235 | - |
| v02 | 0.5557 | 0.57018 | +0.01783 |
| v03 | 0.5656 | 0.58081 | +0.01063 |
| v04 | 0.5748 | 0.59178 | +0.01097 |
| v05 | 0.5784 | 0.59518 | +0.00340 |
| v06 | 0.5851 | 0.60273 | +0.00755 |
| v07 | 0.5860 | 0.60372 | +0.00099 |
| v08 | 0.5866 | 0.60376 | +0.00004 |
| v09 | 0.5897 | 0.60807 | +0.00431 |
| v11 | 0.5966 | 0.61744 | +0.00937 |
| v12 | 0.6005 | 0.62182 | +0.00438 |
| V19 | 0.6079388 | 0.62578 | +0.00396 |
| V20 | 0.6097782 | 0.62689 | +0.00111 |
| V30 | 0.6134471 | 0.63302 | +0.00613 |
| V30 + plug-in | 0.6159732 | 0.63575 | +0.00273 |
| V38 | 0.6166579 | 0.63590 | +0.00015 |
| V39 | 0.6168017 | 0.63588 | -0.00002 |
| V40 | 0.6175912 | 0.63685 | +0.00097 |
| V41 | 0.6181119 | 0.63722 | +0.00037 |
| V42 | 0.6184162 | 0.63716 | -0.00006 |
| V43 | 0.6184783 | 0.63722 | +0.00006 |
| V48 | 0.6186451 | 0.63724 | +0.00002 |

Notas. El registro conserva 22 versiones puntuadas y 23 puntajes: v08 tiene un segundo puntaje de 0.39563 procedente de una subida con formato roto (incidente D-021). El candidato v10 se construyó y validó (CV 0.5904) pero no se envió. La ganancia acumulada del arco es +0.08489 (de 0.55235 a 0.63724). La ley de calibración se ajustó solo con los cuatro pares canónicos (v01, v07, v08, v12); el resto de la tabla ilustra la consistencia del offset. (`reports/INFORME_EXPERIMENTOS_Y_LEADERBOARD.md:34-68`; `CHAMPION.md`; `07_checklist_verificacion.md` secciones 2 y 6).

Fuentes: `reports/INFORME_EXPERIMENTOS_Y_LEADERBOARD.md`; `reports/age_lb_calibracion.csv`; `experiments/age_cierre_auditoria.json`; `DECISIONS.md` (D-050, D-021); `CHAMPION.md`; `plan/kaggle_notebook/paquetes/07_checklist_verificacion.md` (secciones 0.4-0.5 y 6).

[DOC]

## Anexo A. Inventario completo de experimentos

**Fuentes:** `experiments.csv` (registro completo, 220 filas), `experiments/auditoria_v1v37_inventario.json` (clasificación por tipo de las 200 filas V1-V37, auditoría del 2026-10-01), `CHAMPION.md`, `DECISIONS.md` (actas D-090, D-091, D-101 y D-106 a D-110) y `plan/kaggle_notebook/paquetes/07_checklist_verificacion.md`.

**Alcance.** La tabla de la sección A.4 reproduce, en el orden del registro, las 220 filas de `experiments.csv`: toda ejecución, pre-registro, enmienda, documento y corrida bloqueada del proyecto. Las celdas de texto se recortan a ~80 caracteres; el texto completo de cada campo vive en `experiments.csv` y en el artefacto citado. Verificación de la nota recibida: `experiments.csv` ya incluye las 15 corridas del arco V38-V52 (una por bloque, de 38.0 a 52.0); no fue necesario agregarlas a mano. El arco final V53-V57 (frontera 2026: red densa, consenso jerárquico, récord de CV y auditoría matemática) se incorporó al registro y a este anexo el 2026-10-18.

### A.1 Resumen por tipo de registro

| Tipo | Inventario V1-V37 (2026-10-01) | Acta D-090 (195 filas) | Arco V38-V52 | Arco V53-V57 | Total `experiments.csv` |
|---|---:|---:|---:|---:|---:|
| EJECUTADO | 151 | 151 | 15 | 5 | 171 |
| DOCUMENTAL | 26 | 21 | 0 | 0 | 26 |
| PREREGISTRO | 12 | 12 | 0 | 0 | 12 |
| ENMIENDA | 10 | 10 | 0 | 0 | 10 |
| BLOQUEADO | 1 | 1 | 0 | 0 | 1 |
| **Total** | **200** | **195** | **15** | **5** | **220** |

### A.2 Resumen por familia

Vista solicitada sobre las 220 filas (los 7 registros de auditoría transversal se agrupan en su propia familia; el inventario original los repartía dentro de V18-V29 y V30-V37, como se anota abajo):

| Familia | Filas | Detalle |
|---|---:|---|
| V1-V12 (fundacional y fases PH) | 113 | 37 de V1-V4 (`E-*`, `B*`, `ADV-*`) + 76 de V5-V12 (fases `PH*`) |
| V13-V17 (búsqueda residual) | 13 | 10 ejecutadas + 3 documentales |
| V18-V29 (frontera, ordinal y selección) | 25 | 12 de V18-V23 + 15 de V24-V29 − 2 registros de auditoría reasignados |
| V30-V37 (campeón, reapertura FTRL y diagnósticos) | 42 | 14 + 10 + 16 + 7 − 5 registros de auditoría reasignados |
| V38-V52 (arco neuronal y falsaciones) | 15 | una fila por bloque 38.0-52.0, todas ejecutadas |
| V53-V57 (frontera 2026: DenseNet, PoE jerárquico y techo estacionario) | 5 | una fila por bloque 53.0-57.0, todas ejecutadas |
| Auditoría transversal (`AUD-*`, `AUDIT-*`, `G14-AUDIT`) | 7 | 7 documentales |
| **Total** | **220** | 171 ejecutadas + 26 documentales + 12 pre-registros + 10 enmiendas + 1 bloqueada |

Familias originales del inventario V1-V37 (200 filas, para trazabilidad): V1-V4 37; V5-V12 76; V13-V17 13; V18-V23 12; V24-V29 15; V30-V31 14; V32-V33 10; V34-V36 16; V37 7.

### A.3 Notas de lectura

1. La columna `tipo` proviene de la auditoría V1-V37 (2026-10-01). Las 15 filas V38-V52 son posteriores a esa auditoría y se marcan como EJECUTADO porque reportan mediciones con artefacto citado; V45, V50 y V52 no tienen script dedicado y quedan documentadas en `notebooks/99_entrega_consolidado.ipynb`, `DECISIONS.md` y `RUNLOG.md`.
2. Doble conteo del acta D-090: registró 195 filas (151 + 21 + 12 + 10 + 1); el inventario actualizado añade las 5 filas `AUD-*` y suma 200 (151 + 26 + 12 + 10 + 1). Fuente: `plan/contexto.md:294` e `reports/INFORME_EXPERIMENTOS_Y_LEADERBOARD.md:12`.
3. La auditoría V1-V37 confirmó 20 defectos (4 media-alta, 9 media, 7 baja) y 0 rechazos del campeón; el campeón V30 + plug-in quedó intacto bit a bit (D-090/D-091).
4. Referencias Kaggle del arco final: V48 57283487; V43 57277515; V41 57275353; V40 57274929; V39 57274330; V38 57273347 (`07_checklist_verificacion.md` §2).
5. La fila `V25-00-INVENTARIO` no tiene campo `resultado` en `experiments.csv` (celda vacía en el origen).
6. Existe una colisión de numeración de actas: hay dos `D-091` (inner split, 2026-10-01, y V38 TabularNet, 2026-10-16); citar por título además del número.
7. Las 5 filas V53-V57 (2026-10-18) son posteriores al arco V38-V52 y se marcan EJECUTADO; sus actas son D-106 a D-110, sus submissions viven en `submissions/` y su procedencia SOTA está en `reports/state_of_art.csv` (filas C13-C17).

### A.4 Tabla completa (`experiments.csv`, 220 filas)

| id | tipo | bloque y etapa | descripcion breve | resultado | decision (resumen) |
|---|---|---|---|---|---|
| E-001 | EJECUTADO | 2.0 — B - baseline | Que desempeno da la union TF-IDF word(1-2)+char_wb(3-5) con LinearSVC? | macro-F1 0.5344 +- 0.0050 | Conservar como baseline de referencia; comparar contra E-002..E-007 (ablacion... |
| E-002a | EJECUTADO | 2.0 — B - representacion | ¿Qué desempeño da la representación 'word 1-gram'? | macro-F1 0.5025 ± 0.0040 (39.409 features) | pendiente |
| E-002b | EJECUTADO | 2.0 — B - representacion | ¿Qué desempeño da la representación 'word 1-2 gram'? | macro-F1 0.4971 ± 0.0034 (111.865 features) | pendiente |
| E-002c | EJECUTADO | 2.0 — B - representacion | ¿Qué desempeño da la representación 'word 1-3 gram'? | macro-F1 0.4941 ± 0.0039 (151.431 features) | pendiente |
| E-003a | EJECUTADO | 2.0 — B - representacion | ¿Qué desempeño da la representación 'char_wb 2-4'? | macro-F1 0.5261 ± 0.0048 (121.285 features) | pendiente |
| E-003b | EJECUTADO | 2.0 — B - representacion | ¿Qué desempeño da la representación 'char_wb 3-5'? | macro-F1 0.5408 ± 0.0050 (224.628 features) | pendiente |
| E-003c | EJECUTADO | 2.0 — B - representacion | ¿Qué desempeño da la representación 'char_wb 3-6'? | macro-F1 0.5413 ± 0.0049 (300.000 features) | pendiente |
| E-005a | EJECUTADO | 2.0 — B - representacion | ¿Qué desempeño da la representación 'union word 1-2 + char 3-5 (normalizado)'? | macro-F1 0.3902 ± 0.0035 (183.263 features) | pendiente |
| E-007a | EJECUTADO | 2.0 — B - representacion | ¿Qué desempeño da la representación 'union sin IDF (TF puro)'? | macro-F1 0.5339 ± 0.0046 (336.493 features) | pendiente |
| E-005b | EJECUTADO | 2.0 — B - representacion | ¿Qué desempeño da la representación 'marcadores <url>/<user> + union'? | macro-F1 0.4107 ± 0.0024 (218.709 features) | pendiente |
| E-005c | EJECUTADO | 2.0 — B - representacion | ¿Qué desempeño da la representación 'solo quitar URLs + union'? | macro-F1 0.5277 ± 0.0038 (292.170 features) | pendiente |
| E-005d | EJECUTADO | 2.0 — B - representacion | ¿Qué desempeño da la representación 'solo quitar menciones + union'? | macro-F1 0.4094 ± 0.0026 (260.025 features) | pendiente |
| E-005e | EJECUTADO | 2.0 — B - representacion | ¿Qué desempeño da la representación 'solo quitar puntuación + union'? | macro-F1 0.5264 ± 0.0043 (292.313 features) | pendiente |
| E-004b | EJECUTADO | 2.0 — B - representacion | ¿Qué desempeño da la representación 'char_wb 3-6 + word 1-gram'? | macro-F1 0.5354 ± 0.0037 (339.409 features) | pendiente |
| E-008 | EJECUTADO | 2.0 — B - representacion | ¿Qué desempeño da la representación 'char_wb 3-6 + canal de menciones'? | macro-F1 0.5557 ± 0.0052 (314.420 features) | pendiente |
| B0 | EJECUTADO | 2.0 — B - baseline | ¿Cuál es el suelo estadístico (clase mayoritaria/prior)? | macro-F1 0.0485 ± 0.0000 | pendiente |
| B1 | EJECUTADO | 2.0 — B - baseline | ¿Un modelo probabilístico lineal supera al SVM sobre word n-grams? | macro-F1 0.5015 ± 0.0049 | pendiente |
| B5 | EJECUTADO | 2.0 — B - baseline | ¿La señal densa (textometría+estilo) compite con la dispersa? | macro-F1 0.3567 ± 0.0036 | pendiente |
| E-009 | EJECUTADO | 3.0 — D - modelos clasicos | ¿Cómo rinde LogisticRegression (OvR, liblinear, C=4) sobre la representación... | macro-F1 0.5603 ± 0.0033 | pendiente |
| E-010 | EJECUTADO | 3.0 — D - modelos clasicos | ¿Cómo rinde ComplementNB (alpha=0.3) sobre la representación ganadora? | macro-F1 0.5370 ± 0.0046 | pendiente |
| E-011 | EJECUTADO | 3.0 — D - modelos clasicos | ¿Cómo rinde SGDClassifier (log_loss, alpha=1e-6) sobre la representación gana... | macro-F1 0.5579 ± 0.0045 | pendiente |
| E-012 | EJECUTADO | 3.0 — D - modelos clasicos | ¿Cómo rinde LinearSVC (C=0.3) sobre la representación ganadora? | macro-F1 0.5624 ± 0.0036 | pendiente |
| E-014 | EJECUTADO | 3.0 — D-F | ¿Aporta un canal dedicado de hashtags? | macro-F1 0.5644 | documentada en DECISIONS.md |
| E-015 | EJECUTADO | 3.0 — D-F | ¿Aprovecha la estructura ordinal de las clases? | macro-F1 0.3973 | documentada en DECISIONS.md |
| E-016 | EJECUTADO | 3.0 — D-F | ¿Está el tope de 300k truncando vocabulario de caracteres? | macro-F1 0.5658 | documentada en DECISIONS.md |
| E-017 | EJECUTADO | 3.0 — D-F | ¿Aporta un modelo de palabra como miembro diverso del ensamble? | macro-F1 0.5015 | documentada en DECISIONS.md |
| E-018 | EJECUTADO | 3.0 — D-F | ¿Aprovecha el meta-modelo la longitud y el estilo del tweet? | macro-F1 0.5784 | documentada en DECISIONS.md |
| E-019a | EJECUTADO | 3.0 — D-F | ¿Un MLP sobre features densas compite con los lineales? | macro-F1 0.3381 | documentada en DECISIONS.md |
| E-019b | EJECUTADO | 3.0 — D-F | ¿Ayuda reducir el espacio de caracteres con SVD para un modelo no lineal? | macro-F1 0.4071 | documentada en DECISIONS.md |
| E-020 | EJECUTADO | 3.0 — D-F | ¿Aporta el MLP al ensamble pese a su baja exactitud? | macro-F1 0.5851 | documentada en DECISIONS.md |
| ADV-01 | EJECUTADO | 4.0 — Fase avanzada A | ¿Qué señal no está representada en el sistema actual? | 3 informes: gap analysis (107 líneas), signal coverage (18 fenómenos), backlo... | Construir features de segunda generación en esas cinco líneas. |
| ADV-02 | EJECUTADO | 4.0 — Fase avanzada B | ¿Aportan las features estructuradas y supervisadas al meta-modelo? | macro-F1 0.5897 vs baseline 0.5866 (delta +0.0031) | Adoptar como AGE_ADVANCED_V01. |
| ADV-03 | EJECUTADO | 4.0 — Fase avanzada C | ¿Se confirma el candidato fuera del CV de desarrollo? | holdout: base 0.5914 vs candidato 0.5966 (delta +0.0052) | Generar submission v09. |
| ADV-04 | EJECUTADO | 4.0 — Fase avanzada D | ¿Aportan las features de incertidumbre, acuerdo e interacción al meta-modelo? | +0.0002 frente al candidato v09 (0.5897 -> 0.5899) | Línea CERRADA: no se incorporan al candidato. |
| ADV-05 | EJECUTADO | 4.0 — Fase avanzada E | ¿Supera NB-SVM a TF-IDF+SVM y ayuda la selección de features del meta-modelo? | NB-SVM 0.5641 (vs 0.5658 del SVM TF-IDF); selección top-150 = 0.5896 vs 0.589... | NB-SVM se incorpora como miembro; la selección de features se CIERRA. |
| ADV-06 | EJECUTADO | 4.0 — Fase avanzada F | ¿Cuál es el mejor candidato final de la fase avanzada? | CV 0.5904 (v09: 0.5897; holdout 0.5966; LB v09 0.60807) | Generado como submission v10 y NO enviado: se reserva el último envío puntuado. |
| ADV-07 | EJECUTADO | 4.0 — Fase avanzada G | ¿Se puede mejorar la frontera 18-24/25-34 con especialistas y enrutamiento? | especialista con compuerta −0.0036; enrutamiento por margen +0.0000 (3.571 ac... | Línea CERRADA (D-019). No se gasta envío. |
| PH1-L1 | EJECUTADO | 5.0 — Fase avanzada v2 | ¿Cuántos tweets tienen algún modelo base acertando mientras el ensamble falla? | 25.94% de filas con rescate posible; rescatadores: softmax ordinal 36%, centr... | Reorientar la fase hacia el combinador (L6). |
| PH2-L6 | EJECUTADO | 5.0 — Fase avanzada v2 | ¿Mejora el ensamble si el combinador condiciona su confianza al tipo de tweet? | MLP 0.5967 vs LogReg 0.5906 (+0.0061, 5/5 folds); segmentado por tipo -0.0007... | Adoptar el meta-modelo MLP. |
| PH3-V11 | EJECUTADO | 5.0 — Fase avanzada v2 | ¿Se confirma el candidato v11 fuera del CV de desarrollo? | holdout: LogReg 0.5981 vs MLP 0.6024 (+0.0043); v09 previo 0.5966 | Generar submission v11 con el último envío disponible. |
| PH5-L2 | EJECUTADO | 5.0 — L2 residuales | ¿Aporta la residualización por longitud al combinador? | MLP con residuos 0.5948 vs sin residuos 0.5967 (-0.0019) | Línea CERRADA. |
| PH6-L3 | EJECUTADO | 5.0 — L3 contenido vs estilo | ¿Conserva el estilo señal demográfica que el contenido no captura? | estilo solo 0.2793 · contenido solo 0.5537 · aporte al ensamble +0.0008 (3/5) | Línea CERRADA; hallazgo interpretativo registrado. |
| PH7-L7 | EJECUTADO | 5.0 — L7 y combinador | ¿Aportan un NB-SVM de palabras y el promedio de combinadores? | NB-SVM palabra 0.5254 (+0.0009 al meta) · promedio de combinadores 0.6005 vs... | Adoptar el promedio de combinadores (candidato v12). |
| PH8-V12 | EJECUTADO | 5.0 — Candidato v12 | ¿Se confirma el candidato v12 en el holdout congelado? | holdout: LogReg 0.5983 · MLP 0.6034 · promedio 0.6077 (+0.0043 vs MLP) | Generar submission v12. |
| PH9-COMB | EJECUTADO | 5.0 — Fase 9 combinadores | ¿Mejora el promedio añadiendo un tercer combinador diverso? | MLP+LogReg 0.6005 = óptimo; +Ridge 0.5999; +SGD 0.5676; cuatro 0.5702. Indivi... | Línea CERRADA: el promedio de dos (v12) es el modelo final. |
| PH10-A | EJECUTADO | 5.0 — Fase 10 vía A: retrieval | ¿Aporta el retrieval train->test (inferencia) al modelo? | cobertura de clúster in-fold 0.00% (el CV agrupado la elimina por diseño); du... | Línea CERRADA con evidencia: permitida por las reglas, pero los datos dicen q... |
| PH10-B | EJECUTADO | 5.0 — Fase 10 vía B: POS spaCy | ¿Aportan las features de POS (spaCy es_core_news_md) al combinador? | MLP + POS 0.5944 vs base 0.5976 (-0.0032); ambas vías -0.0013 | Línea CERRADA. |
| PH11-RUIDO | EJECUTADO | 5.0 — Fase 11 ruido de etiqueta | ¿Mejora el combinador si se excluyen del entrenamiento las filas con etiqueta... | contradictorias (643 filas): +0.0001 (3/5) · duplicadas (2.123 filas): +0.001... | Mejora real pero por debajo del umbral de envío (+0.003). Se documenta como l... |
| PH12-AUD | EJECUTADO | 5.0 — Fase residual A | ¿El sustrato experimental es el que dicen los artefactos? | Reconstrucción exacta de la matriz (diff 0.0); LOFO 0.6004; acuerdo 100.0000%... | Base validada para la búsqueda de señal residual; D-004 (límite de envíos) qu... |
| PH13-ERR | EJECUTADO | 5.0 — Fase residual B | ¿Dónde falla el v12 y qué modelos rescatan esos errores? | 13.87% todos fallan; 26.70% rescatable; consenso Q1 error 60.2% vs Q5 18.1%;... | Insumo de hipótesis para C, E y G; ningún especialista duro se reabre. |
| PH14-CONS | EJECUTADO | 5.0 — Fase residual C | ¿Añaden el consenso por familia, las distancias de desacuerdo y el gating con... | +C1 0.5986 (-0.0018, 1/5); +C1+C2 0.6002 (-0.0002, 3/5); +C1+C2+C3 0.5998 (-0... | Familia CERRADA (D-028). No se adopta ninguna variante. |
| PH15-CALIB | EJECUTADO | 5.0 — Fase residual D | ¿Mejora la calibración de los miembros o el blending en logits al combinador? | D1a (T=1): promedio 0.6005 (+0.0001, 3/5) y logits 0.6007 (+0.0003, 3/5). D1b... | CERRADA (D-029). |
| PH15B-SUBSETS | EJECUTADO | 5.0 — Fase residual D5 | ¿Aporta diversidad un combinador entrenado en un subconjunto distinto de feat... | Producción 0.6004 idéntico (control de determinismo bajo concurrencia); MLP p... | CERRADA (D-029). |
| PH18-LIMPIEZA | EJECUTADO | 5.0 — Fase residual F | ¿Mejora el combinador si se limpia de forma selectiva el lado de entrenamiento? | F2 (un representante por clúster) 0.6003 (-0.0001, 2/5); F3 (clúster selectiv... | CERRADA sin adopción (D-031). |
| PH16-ORD-GREEDY | EJECUTADO | 5.0 — Fase residual G | ¿Aportan la edad implícita/consistencia ordinal y la selección greedy de miem... | G1 0.6001 (-0.0003, 1/5); G2 0.5770 (-0.0234). | CERRADA (D-030). |
| PH21-VISTA | EJECUTADO | 5.0 — Fase residual §13 multivista | ¿Aporta una vista sin contenido (estructura de tipos de token) diversidad de... | estructura: F1 individual 0.2285, rescate 18.17% de los errores del v12, +0.0... | Documentada como marginal NO adoptada (D-032); archivo candidato local sin en... |
| PH22-VISTA2 | EJECUTADO | 5.0 — Fase residual §13 refuerzo | ¿Mejora la vista estructural con un alfabeto más rico o combinada con los ext... | V1 ampliada 0.6009 (+0.0005, 4/5); V2 estructura+extremos 0.6001 (-0.0003); V... | CERRADA la vía de enriquecer la vista (D-032). |
| PH19-HOLDOUT-VISTA | EJECUTADO | 5.0 — Fase residual H | ¿Se confirma la vista estructural en el holdout congelado? | base 0.6077 (reproducido exacto) frente a vista 0.6108 -> +0.0031 en el holdo... | No adoptada por el gate pre-registrado de CV; documentada y con archivo candi... |
| PH24-DIAG | EJECUTADO | 5.0 — Fase residual I (integridad) | ¿Por qué la reconstrucción del v12 desde la caché da 93.88% y no 100%? | Todos los bloques coinciden: fuentes max/Δ/ 1.0e-05, NB 3.1e-05, densas 0.0,... | Hipótesis descartada con evidencia; continúa el diagnóstico. |
| PH25-FRESCA | EJECUTADO | 5.0 — Fase residual I/J | ¿Reproduce la matriz fresca la submission v12 enviada? ¿Es válido el candidat... | fresca vs v12 subido 93.8800%; caché vs v12 subido 93.8800%; fresca vs caché... | Se regenera el candidato con la matriz fresca (guardada en features/) y se do... |
| PH26-ESTABILIDAD | EJECUTADO | 5.0 — Fase residual I (integridad) | ¿Es reproducible el ajuste final del combinador? | LogReg 100.0000% (0 filas distintas, 418 iteraciones en ambas), MLP 100.0000%... | Se documenta como hallazgo de reproducibilidad del artefacto subido (D-034):... |
| PH27-V12REPRO | DOCUMENTAL | 5.0 — Fase residual I (cierre) | ¿Se puede dejar un artefacto v12 reconstruible desde el repositorio? | 112.892 filas, 8/8 checks; acuerdo con age_submission_v12.csv 93.8800%; distr... | Se guarda como age_submission_v12_reproducible.csv; no se envía. |
| PH30-COBERTURA | EJECUTADO | 5.0 — Capa de decisión (preparación) | ¿Qué formulaciones de decisión ordinal ya se evaluaron? | 9 líneas genuinamente abiertas (B, C, D, E, F, G, H, I y la decodificación de... | Matriz de cobertura publicada (reports/age_hypothesis_coverage.md). |
| PH31-DECISION | EJECUTADO | 5.0 — Capa de decisión A/E/G/F | ¿La decodificación ordinal, la proyección, el desempate por E[Y] o el coste o... | 14 capas; mejor G (tau 0.05) 0.6015 (+0.0011, 4/5, neto -117); A1 umbrales 0,... | CERRADA (D-037). |
| PH32-CORRECCION | EJECUTADO | 5.0 — Capa de decisión C/D/H/B | ¿La corrección suave de fronteras, la decodificación por confusión, la calibr... | 14 capas; mejor H 5 bandas 0.6006 (+0.0002, 3/5); C4 neto 0; D empeora monóto... | CERRADA (D-037). |
| PH33-CORRECTOR | EJECUTADO | 5.0 — Capa de decisión I | ¿Un corrector mínimo sobre P(V12) y contexto captura lo que el argmax no conv... | 5 variantes: I1 0.5999 (-0.0005), I2 0.5992, I3 0.6000, I4 0.5572, I5 0.5997.... | CERRADA (D-037). |
| PH34-INFORME-DEC | EJECUTADO | 5.0 — Capa de decisión (cierre) | ¿Dónde vive el error residual y la línea puede cerrarse? | Techo de información 23.485 filas (34.7% del error); agregación 45.213 filas... | Línea cerrada (D-037); V12 congelado; sin submission. |
| PH36-COBERTURA-LEX | DOCUMENTAL | 6.0 — P0 auditoria de cobertura | ¿Que hipotesis lexicas son genuinamente nuevas? | 0 experimentos y 0 lineas de codigo previas mencionan un canal lexico (nivel... | Cobertura publicada; 3 hipotesis abiertas (H-LEX-A/B/C). |
| PH37-CATALOGO-LEX | EJECUTADO | 6.0 — P1 catalogo de lexicones | ¿Que recursos lexicos son admisibles? | 1.588 declaradas -> 1.369 efectivas; cobertura registro 5.72% / vida 0.62% /... | Catalogo publicado con SHA256 por fichero. |
| PH38-BLOQUES-LEX | EJECUTADO | 6.0 — P2-P5 bloques, control, redundancia y ablacion | ¿Existe la señal lexica y es redundante con lo que V12 ya tiene? | Control lexicon-only: temas 0.2033 / registro 0.1520 / vida 0.0731 (azar 0.16... | Continuar a P6 con las tres familias y su control de permutacion. |
| PH39-FUSION-LEX | EJECUTADO | 6.0 — P6 fusion con V12 | ¿El bloque lexico mejora a V12 por encima de su control de permutacion? | V12 + Registro: +0.0001 (4/5) control 0.6005; V12 + Vida: +0.0000 (2/5) contr... | CERRAR la linea (D-039). |
| PH40-ANALISIS-ERROR-LEX | EJECUTADO | 6.0 — P7 analisis de error | ¿El lexico rescata errores reales de V12? | __registro: rescate 5541 / daño 5489 / neto +52; __registro___vida___temas: r... | Sin cambio en el veredicto. |
| V13-P0A-IDIOMA | DOCUMENTAL | 7.0 — P0a auditoria de idioma | ¿El codigo, la documentacion y los datos estan en espanol y sin caracteres de... | 6 ideogramas CJK y 260 emoji/flechas encontrados y corregidos; 0 errores y 0... | Regla V13: todo el codigo y la documentacion en espanol; auditoria automatica... |
| V13-P0B-COLINEALIDAD | EJECUTADO | 7.0 — P0b prueba de colinealidad | ¿La matriz completa de 279 variables de V12 tiene redundancia lineal que este... | 401 pares con /rho/>=0.80; 229 redundantes; 15 duplicados exactos; 54 cluster... | Criba de redundancia obligatoria antes de fusionar cualquier bloque nuevo; ni... |
| V13-P1C-IDENTIDADES | EJECUTADO | 7.0 — P1c identidad frente a evidencia lexica | ¿La senal de P(clase/token) es evidencia lexica de la edad o una tabla de con... | completo 0.3997 / sin cuentas 0.3410 / sin cuentas ni hashtags 0.2992; suelo... | P1 se mantiene como pista; el informe de P1b debe declarar que hay dos compon... |
| V13-P1A-TOKENS | EJECUTADO | 7.0 — P1a evidencia por token in-fold | ¿Hay señal en P(clase/token) para tokens simples, y es redundante con lo que... | LogReg 0.3997 y LinearSVC 0.3917 frente al suelo de azar 0.1667; cribado 1 de... | Procede P1b (fusion con control de permutado y gate). |
| V13-P17A-PRONOMBRES | EJECUTADO | 7.0 — P17a pronombres de persona (V14-V18 del articulo) | ¿Los pronombres de persona, la variable mas predictiva del articulo de refere... | LogReg 0.1918 y LinearSVC 0.1881 frente al suelo 0.1667; 0 de 26 variables re... | Procede P17b con puerta de rescate antes de pagar la fusion. |
| V13-P17B-PRONOMBRES | EJECUTADO | 7.0 — P17b fusion de pronombres (pre-chequeo y cierre) | ¿El bloque de pronombres, que no toca ninguna variable de V12, mejora la macr... | bloque solo 0.1922 / rescate 12.043 / dano 76.118 / neto -64.075 / fronteras... | CERRAR la pista P17 sin fusionar. El bloque se conserva por si otra combinaci... |
| V13-P20-MODALIDAD | EJECUTADO | 7.0 — P20 incertidumbre y modalidad | ¿Los marcadores de incertidumbre y modalidad llevan señal de edad en este cor... | LogReg 0.1270 y LinearSVC 0.1259, ambos POR DEBAJO del suelo de azar 0.1667;... | CERRAR la pista P20 sin fusionar. |
| V13-P1B-FUSION | EJECUTADO | 7.0 — P1b fusion con V12, control de permutado y gate | ¿La evidencia por token, que por si sola da 0.3997, mejora a V12 por encima d... | delta -0.0002 con 3/5 folds positivos / controles permutados 0.6000/0.6003/0,... | NO ADOPTAR. Cerrar la pista P1 y abrir P16 (por que falla toda fusion). |
| V13-P16-SUFIENCIA | EJECUTADO | 7.0 — P16 ¿V12 ya tenia la informacion de los bloques nuevos? | ¿El fracaso de cuatro fusiones consecutivas se explica por que V12 ya contien... | R2 mediano reconstruido: P1 -0.219 / P17 -0.004 / P20 -0.003 / control recons... | No concluir nada; repetir el experimento con complejidad controlada (P16c). |
| V13-P16B-ACTIVACION | EJECUTADO | 7.0 — P16b ¿donde es util el bloque? | ¿La senal del bloque esta concentrada en las filas donde el tiene actividad? | 1.65 tokens conocidos de media; 122.870 filas (72.6%) con al menos uno / el b... | Medir si existe un gatillo realizable (P16d). |
| V13-P16C-RECONSTRUCCION | EJECUTADO | 7.0 — P16c reconstruccion estable por complejidad | ¿Cuanta informacion de los bloques contiene V12 de verdad? | mejor R2 mediano: P1 +0.522 (max +0.959, 21 de 36 columnas por encima de 0.5)... | Concluir que la redundancia NO explica el fracaso; el problema es de combinac... |
| V13-P16D-PUERTA | EJECUTADO | 7.0 — P16d ¿existe una puerta realizable? | ¿Hay un gatillo, sin oraculo, que aproveche el bloque y llegue al gate? | AUC de la confianza del bloque 0.7173 / techo con puerta perfecta 0.6770 / el... | Cerrar P1 de forma definitiva y no abrir mas pistas de variables del mismo tipo. |
| V13-P3A-ESPECIALISTAS | EJECUTADO | 7.0 — P3a especialistas binarios de frontera | ¿Un modelo entrenado especificamente para cada frontera mejora la decision de... | 18-24/25-34: exactitud en el par 0.4963 -> 0.4944 (LogReg) y 0.4946 (MLP), ne... | CERRAR la pista P3 sin pagar la etapa 2 (control de permutado y gate completo). |
| V13-P19-CONJUNCIONES | EJECUTADO | 7.0 — P19 arboles de conjunciones por funcion (sin POS) | ¿La funcion gramatical de las conjunciones varia con la edad y un arbol la ca... | arboles 0.1489/0.1512/0.1539 y LogReg 0.1539 y LinearSVC 0.1522, TODOS por de... | CERRAR la pista P19 sin pagar la fusion. Ultima pista pre-registrada del plan. |
| V13-CIERRE | DOCUMENTAL | 7.0 — cierre y agregacion de entregables | - | 16 experimentos en la tabla maestra, 8 hipotesis pre-registradas registradas,... | Cerrar V13 con V12 congelado; la continuacion exige una decision de alcance,... |
| V14A-LINGUISTICO | EJECUTADO | 8.0 — V14a variables linguisticas del analizador espanol (D-043) | ¿Aportan signal las variables morfologicas, sintacticas y de entidades que da... | LogReg 0.2409 y LinearSVC 0.227 frente al suelo 0.1667 / cribado 4 de 37 redu... | Se fusiona de todas formas bajo el gate D-042, dejando escrito que el pre-che... |
| V14B-FUSION-LINGUISTICO | EJECUTADO | 8.0 — V14b fusion con V12 bajo el gate revisado D-042 | ¿Las variables del analizador espanol externo mejoran a V12 por encima de su... | delta -0.0015 con 1/5 folds positivos / controles permutados 0.5962/0.5975/0,... | NO ADOPTAR. Cerrar V14 y, con el, la via de relajar §3. |
| V15A-CORRELACION | EJECUTADO | 8.0 — V15a correlacion entrada-entrada y asociacion con la salida, in-fold | ¿Que variables de entrada estan correlacionadas entre si, y cuales se asocian... | 2 columnas constantes / pares con /rho/>=0.80: 402 / >=0.90: 245 / duplicados... | La particion in-fold habilita V15b; se declaran 112 variables casi informativas. |
| V15B-PODA | EJECUTADO | 8.0 — V15b evaluacion de la poda de variables redundantes | Si se eliminan las variables de entrada correlacionadas, ¿el modelo mejora, e... | macro-F1 0.5965 frente a 0.6005 de V12 / delta -0.004 con 0/5 folds positivos... | NO ADOPTAR la poda. V12 se queda con sus 279 variables. |
| V16-ARQUITECTURA | EJECUTADO | 9.0 — V16 arquitectura del meta-modelo: peso aprendido y combinador logistico | ¿El promedio fijo 0.5 LogReg + 0.5 MLP es la forma correcta de combinar las d... | A1 peso aprendido 0.6002 (delta -0.0003, 0/5 folds; pesos elegidos 0.1 0.5 0,... | NO ADOPTAR. Se conservan el promedio fijo 0.5 + 0.5 y el submission v01. La l... |
| V17-P0-AUDITORIA | DOCUMENTAL | 10.0 — V17 P0 auditoria de brechas previa a cualquier entrenamiento | Que pistas de V17 son genuinamente nuevas y cuales son sinonimos de lo ya cer... | 42 familias inventariadas; de las 10 pistas, 3 cerradas, 5 parcialmente proba... | La referencia se canoniza y las pistas se reordenan por lo que la auditoria r... |
| V17-P0-REPRODUCCION | EJECUTADO | 10.0 — V17 P0' reconciliacion de la referencia y verificacion inerte de A2 de... | Cual es el valor canonico de V12 y se reproduce el +0.0006 de A2 con sus fron... | Pooled 0.6005017473 frente a media de folds 0.6004103057: diferencia -9.14416... | Canonico 0.6005 pooled. A2 se reproduce: no hay STOP. |
| V17-P1-FRONTERAS | EJECUTADO | 10.0 — V17 P1 y P1.1 matriz de error por miembro en las fronteras y complemen... | Existe informacion complementaria en las dos fronteras centrales, y la tienen... | 18-24/25-34: V12 0.4963 frente a especialista 0.6864; rescata 14.356, rompe 3... | Se pasa a P1b para medir cuanto de ese techo es cobrable. P2 y P3 siguen abie... |
| V17-P1B-CURVA | EJECUTADO | 10.0 — V17 P1b curva de compromiso de la compuerta de ambiguedad | Cuanto de la mejora local se convierte en macro-F1 global con una compuerta s... | Mejor delta global +0.0002 en 18-24/25-34 y +0.0001 en 25-34/35-49; a partir... | Cerrar P3 en su formulacion de senal de V12. La via sigue abierta con otras s... |
| V17-P8-CONFIANZA | EJECUTADO | 10.0 — V17 P8 perfil de confianza de los errores de frontera | Existe alguna senal de confianza que distinga las filas que el especialista A... | Acuerdo entre miembros: 0.7070 y 0.7300. Margen del especialista: 0.7294 y 0,... | P9 queda como pista principal. P2 se considera redundante con E-028a y se dec... |
| V17-P1C-CONJUNTA | EJECUTADO | 10.0 — V17 P1c regla conjunta de las dos senales de P8 | Las dos senales que P8 separo, leidas a la vez y en sentido contrario, convie... | Mejor univariada +0.0000 y mejor conjunta +0.0000: ninguna llega al gate expl... | P9 NO esta indicado con estas entradas. Cerrada. La tercera frontera queda di... |
| V17-P9-ARBITRO | EJECUTADO | 10.0 — V17 P9 experto selectivo con arbitro aprendido, protocolo anidado | Puede un arbitro entrenado sobre las columnas derivadas que ya existen locali... | macro-F1 0.6004 frente a 0.6005 de V12 / delta -0.0001 con 0/5 folds positivo... | NO ADOPTAR. Cerrada. V12 sigue congelado en 0.6005 pooled. |
| V17-P5-COMBINADOR | EJECUTADO | 10.0 — V17 P5 meta-combinador local por frontera | Combinar la evidencia dentro de la frontera rinde mas que seleccionar entre V... | macro-F1 0.6003 frente a 0.6005 / delta -0.0002 con 1/5 folds / IC95 [-0.0004... | NO ADOPTAR. Cerrada. |
| V18-P0-AUDITORIA | DOCUMENTAL | 11.0 — V18 P0 auditoria de brechas previa a cualquier entrenamiento | Que pistas de V18 son genuinamente nuevas y cuales repiten algo ya cerrado, y... | 28 familias inventariadas; de las 8 pistas: 0 cerradas, 3 parcialmente probad... | Se pide medir la rejilla de dos representaciones por dos modelos antes de dec... |
| V18-P0-REPRODUCCION | EJECUTADO | 11.0 — V18 P0' reproduccion de la evidencia clave y cierre del confound de la... | Se reproducen V12, A2 de V16 y los especialistas de V17, y de donde viene la... | V12 0.6005 y A2 0.6011 reproducidos; especialistas {'18-24/25-34': 0.6864, '2... | Se corrige el diagnostico de P3a (D-048). La premisa de V18 se invierte: la s... |
| V18-P1-PESO-CLASES | EJECUTADO | 11.0 — V18 P1 peso de clases consciente de la frontera, tres niveles pre-regi... | Subir el peso de las tres clases centrales en el entrenamiento convierte la s... | nivel 0 base 0.6005 (delta +0.0000, fidelidad 0.000000: reproduce V12 exactam... | NO ADOPTAR. P1 se cierra con resultado estable negativo y con un diagnostico... |
| V18-P3-MARGEN | EJECUTADO | 11.0 — V18 P3 objetivo de margen de adyacencia en el entrenamiento, cinco brazos | Exigir que la clase verdadera venza a su vecina por un margen, durante el ent... | A control 0.5983 (delta -0.0022) / B1 generico 0.5985 (-0.0020) / B2 generico... | NO ADOPTAR. P3 se cierra. El hallazgo estructural es que las dos fronteras ce... |
| V18-P4-MULTIOBJETIVO | EJECUTADO | 11.0 — V18 P4 entrenamiento multi-objetivo: arquitectura frente a objetivo bo... | Un termino auxiliar aditivo sobre las dos fronteras, sin anadir parametros, c... | A control 0.5982 (delta -0.0023) / B arquitectura 0.5993 (-0.0012, 558 parame... | NO ADOPTAR. P4 se cierra. Un termino aditivo no mueve la frontera, y anadir u... |
| V18-P5-DESTILACION | EJECUTADO | 11.0 — V18 P5 destilacion del conocimiento de frontera, cross-fitted | Se puede transferir al student el conocimiento de un especialista de frontera... | control sin destilacion 0.5983 (delta -0.0022, 0/5) / mejor destilado lambda... | NO ADOPTAR. P5 se cierra. La destilacion es el mecanismo correcto, se midio c... |
| V18-CIERRE | DOCUMENTAL | 11.0 — V18 cierre de fase, documental | Que queda establecido, con medicion, sobre si un modelo global de seis clases... | 8 de 14 experimentos previstos, 0 adopciones, 4 vias cerradas, 4 pistas no ej... | V18 se cierra sin adopciones. La accion que este trabajo recomienda no es un... |
| AGE-CIERRE-AUDITORIA | DOCUMENTAL | 12.0 — AGE auditoria de cierre estrategico, sin entrenamiento | Existe informacion real en los artefactos que no este en las 279 columnas de... | masa de 18-24 mas 25-34 = 33.67 % de las filas / elasticidad +0.0061 y +0.006... | D-050. Se corrige la razon de D-049 y se documenta que el gap CV-LB no da pun... |
| AGE-DECISION-EJECUTIVA | DOCUMENTAL | 12.0 — AGE informe de decision ejecutiva, documental | Se ejecuta un experimento final o se cierra AGE con V12 como champion? | 3 de las 8 condiciones del brief fallan: hipotesis no competitiva, informacio... | CLOSE AGE - V12 REMAINS CHAMPION / V12 intacto en 0.6005017473 pooled, envio... |
| V19-B1-BOOSTING | EJECUTADO | 12.0 — V19 B1 boosting como tercer miembro de V12, con control de ruido | La informacion de las 279 variables se combina de forma no lineal y el meta l... | control filas barajadas 0.5909687 (delta -0.0095, 0 de 5 folds) / arbol aisla... | ADOPTAR como challenger, sujeto a las tres comprobaciones de la bateria de fa... |
| V19-B1-VERIFICACION | EJECUTADO | 12.0 — V19 B1 bateria de falsacion del +0.0074 | El efecto es real o el arbol esta exprimiendo el bloque de 73 columnas superv... | las 279 completas 0.6072312 / sin las 73 supervisadas 0.586956 / solo las 78... | El hallazgo se reformula: no es un arbol mejor, es que el COMBINADOR de V12 d... |
| V19-B1-CONFIRMACION | DOCUMENTAL | 12.0 — V19 B1 segunda particion, control de alineacion y auditoria de fuga | El efecto aparece en una particion que no se uso para elegir la combinacion,... | 2a particion 4 folds 0.6090431 (delta +0.0085, folds [0.6077, 0.606, 0.6102,... | CHALLENGER VIABLE: los cuatro criterios pasan. |
| V19-B1-CHALLENGER | DOCUMENTAL | 12.0 — V19 B1 construccion y validacion de la submission del challenger | Como cambia el candidato respecto a V12, y que leaderboard se espera? | OOF 5 folds 0.6079388 y 4 folds 0.6090431 (delta +0.0074) / 11.392 filas camb... | CHALLENGER ENVIADO Y PUNTUADO 0.62578. La subida es manual en este proyecto y... |
| V19-P0-AUDITORIA | DOCUMENTAL | 13.0 — V19 P0-P1-P2 auditoria exacta del champion, sin entrenar nada nuevo | Que es exactamente V19, donde esta su senal y cuanto margen hay antes de que... | configuracion identica al artefacto: True / atribucion: las 279 dan 0.6072312... | Auditoria cerrada. Registra que el hueco train a OOF es +0.0078, que es la ma... |
| V19-X1-EXPLOTACION | EJECUTADO | 13.0 — V19 X1 las dos unicas configuraciones autorizadas por el especialista | Cual es la micro-mejora mas plausible sobre V19 sin cambiar su mecanismo fund... | puerta de saturacion: ocupacion 97.5% con 1519 de 1800 arboles al techo, lueg... | NO ADOPTAR ninguna. Cerrar la explotacion de V19. V19 permanece champion con... |
| V20-AUDIT | DOCUMENTAL | 14.0 — V20 auditoria del espacio explorado antes de proponer nada | Que se ha probado ya, y que familias del brief son genuinamente nuevas? | nunca probados: RandomForest/ExtraTrees, HistGradientBoosting, CatBoost y la... | Seleccionadas tres familias: F1 informacion nueva, F2 clase de funcion distin... |
| V20-F1-CLASECOND | EJECUTADO | 14.0 — V20 F1 vista de clase condicional de las 122 features numericas | Aporta informacion nueva transformar las 122 features densas y estructuradas... | comparacion emparejada con el mismo aprendiz: arbol sobre las 279 = 0.6072312... | NO_SIGNAL. La familia se cierra con numero. |

| id | tipo | bloque y etapa | descripcion breve | resultado | decision (resumen) |
|---|---|---|---|---|---|

*Tabla (continua).*

| V20-F2-CLASEFUNC | EJECUTADO | 14.0 — V20 F2 clase de funcion alternativa sobre las mismas 279 | El efecto de V19 es de cualquier combinador no lineal, o es especifico del bo... | ExtraTrees en solitario sobre las 279: 0.5836218 (delta -0.0243, 0 de 5 folds... | NO_SIGNAL como candidato, y la familia se cierra por completo. |
| V20-F3-UMBRALES | EJECUTADO | 14.0 — V20 F3 cinco clasificadores de umbral anidado encadenados | Aporta algo encadenar cinco discriminadores de umbral anidado, cada uno con e... | exactitud balanceada por umbral: <= 13-17 vs >= 18-24 0.7632 (V12 0.9408), <=... | WEAK_SIGNAL pero el unico que sigue vivo. No llega al gate de +0.003, y su me... |
| V20-F3C-CANDIDATO | DOCUMENTAL | 14.0 — V20 F3c construccion y validacion de la submission del candidato | Como cambia el candidato respecto al envio de V19, y que LB se espera? | cambia 3.546 de 112.892 filas (3.14%) frente al envio de V19 / OOF 0.6097782... | CANDIDATO PREPARADO, NO ENVIADO. El envio es decision de Carlos; el cierre de... |
| V20-P2-PESOS | EJECUTADO | 15.0 — V20 P2 pesos ordinales: un solo parametro cambiado | Subir el peso del lado minoritario del primer umbral mejora su exactitud bala... | primer umbral 0.7632 -> 0.843 (delta +0.0797) / blend 0.593202 (delta -0.0166... | ORDINAL_WEIGHTS cerrada con numero. Sin submission: no mejora el OOF de V20.... |
| V21-B1-ESPECIALISTA | EJECUTADO | 16.0 — V21 mejorar unicamente el primer discriminador ordinal | Puede un clasificador diferente para 13-17 vs >=18, con las otras cuatro fron... | V20 reconstruido 0.6097782 (reproduce el control, SHA256 69b574f7) / LinearSV... | B1_SPECIALIST = NO_SIGNAL. Con el se cierran las tres subfamilias de la ordin... |
| V22-ORDINAL-COMPARTIDO | EJECUTADO | 17.0 — V22 modelo ordinal compartido: un unico multiclase con cinco CDF por suma | Compartir representacion entre las cinco fronteras produce una mejora real de... | V20 reconstruido 0.6097782 / compartido 0.6096150 / delta -0.000163 / 3/5 fol... | WRONG_OBJECTIVE. Sin submission. |
| V22-B1-SPECIALIST | EJECUTADO | 17.0 — V22 sustituir unicamente el primer discriminador ordinal | Puede un clasificador diferente para 13-17 vs >=18, con las otras cuatro fron... | LinearSVC C 0.2 0.5992098 (-0.0106, B1 0.6782) / LinearSVC C 1.0 0.5992058 (-... | B1_SPECIALIST = NO_SIGNAL. Cierra la sustitucion de B1. |
| V22-00-FRONTIER | EJECUTADO | 17.0 — V22 frontier audit del espacio restante antes de proponer nada | Existe alguna fuente de senal realmente nueva capaz de superar V20 = 0.62689? | 14 familias cerradas, 1 parcialmente explorada y 5 sin explorar; el id cerrad... | familia ordinal cerrada en sus cuatro subfamilias. Sin submission. |
| V22-01-COSTE-ORDINAL | EJECUTADO | 17.0 — V22 coste de error proporcional a la distancia ordinal en la decision | Penalizar los errores segun la distancia ordinal mejora el macro-F1 de seis c... | control V20 0.6097782 (reproduce exacto) / alfa 1 0.5949198 (-0.0149, neto -3... | NO_SIGNAL. Sin submission. |
| V23-00-TESTS-CORAL | EJECUTADO | 18.0 — V23 tests unitarios obligatorios de la implementacion CORAL, antes de... | es correcta la implementacion cumulative-link y la monotonia esta garantizada... | los cuatro tests Pasan / error relativo del gradiente 3.15e-10 / perdida iden... | los cuatro tests pasan; CORAL puede ejecutarse sobre AGE |
| V23-01-CORAL | EJECUTADO | 18.0 — V23 CORAL / cumulative-link con pesos compartidos frente a V20 | compartir pesos entre las cinco fronteras ordinales aporta senal util que cin... | control A reproduce 0.6097782 exacto / CORAL 0.4837017 (delta -0.126077, 0/5... | CORAL = CLOSED. Sin submission. No se abre ninguna familia nueva |
| V25-00-INVENTARIO | DOCUMENTAL | 19.0 — V25 auditoria de inventario y procedencia de las 279 features, antes d... | las 73 features label-informed son legales o se construyeron con labels de un... |  | procedencia verificada; se puede seleccionar sobre las 279 sin riesgo de leakage |
| V25-01-SELECCION-MI | EJECUTADO | 19.0 — V25 seleccion supervisada de features por informacion mutua dentro de... | V20 mejora eliminando features ruidosas mediante seleccion supervisada hecha... | T0 0.6097782 reproduce exacto / T1 0.6091492 / delta -0.000629 / 1/5 folds po... | SUPERVISED_FEATURE_SELECTION = CLOSED. Sin submission. No se prueba otro K ni... |
| V26-00-GAP-ANALYSIS | EJECUTADO | 20.0 — V26 gap analysis del espacio experimental antes de implementar ningun... | que familias estan cerradas, cuales parcialmente probadas y cuales son genuin... | 15 familias, 5 genuinamente abiertas: H02, H03, H11, H12, H14 / V20 YA es un... | V26_SELECTED_TRACK = H02, por novedad metodologica, evidencia de que V20 no l... |
| V26-01-REPRESENTACION | EJECUTADO | 20.0 — V26 primer experimento minimo: representacion afin de la evidencia de... | las probabilidades de los modelos base no son comparables como magnitudes abs... | control 0.6097782 reproduce exacto / T1 0.609777 / delta -1e-06 / 2/5 folds /... | H02 CLOSED. Sin submission. Las familias H03, H11, H12 y H14 siguen abiertas... |
| V27-01 | EJECUTADO | 21.0 — V27 H12 fiabilidad por modelo y clase entregada al meta-modelo | si la fiabilidad de cada miembro del ensemble depende de la clase de edad, pu... | control 0.6097782 reproduce True / T1 0.6097782 / delta +0.0 / 0/5 folds / re... | H12 CLOSED |
| V27-02 | EJECUTADO | 21.0 — V27 H11 restricciones monotonas nativas de LightGBM sobre la evidencia... | ciertas relaciones entre evidencia y clases pueden presentar estructura monot... | control 0.6097782 reproduce True / T1 0.6094234 / delta -0.000355 / 2/5 folds... | H11 CLOSED |
| V27-05 | EJECUTADO | 21.0 — V27 H03 transformacion a percentil de la evidencia de los modelos base | la evidencia de los modelos base tiene una estructura de rango util que una t... | control 0.6097782 reproduce True / T1 0.6100505 / delta +0.000272 / 3/5 folds... | H03 MARGINAL |
| V28-02 | EJECUTADO | 22.0 — V28 tecnica A: bagging del meta-modelo sobre las mismas 279 features | la estimacion de la frontera tiene varianza residual que un ensemble de ajust... | control 0.6097782 reproduce True / T1 0.6094663 / delta -0.000312 / 1/5 folds... | A CLOSED |
| V28-03 | EJECUTADO | 22.0 — V28 tecnica C: descomposicion pairwise en 5 pares adyacentes disjuntos | un clasificador binario disjoint por par adyacente puede aprender fronteras q... | control 0.6097782 reproduce True / T1 0.5422249 / delta -0.067553 / 0/5 folds... | C CLOSED |
| V29-01 | EJECUTADO | 23.0 — P4 · Optuna individual LightGBM | existe capacidad de modelo individual adicional dentro de la MISMA evidencia... | CV 0.6121698 / delta +0.002392 / 5 folds / 150 trials / mejor-trial outliers 0 | Pista individual: no supera el gate de +0.003, que no se exige para entrar po... |
| V30-01 | EJECUTADO | 24.0 — V30 refinamiento local del LightGBM de V29 | puede una busqueda mas estrecha y sembrada con el mejor trial de V29 consolid... | CV 0.6122387 / delta V20 +0.002461 / delta V29-LGBM +6.9e-05 / 3/5 folds fren... | EQUIVALE_A_V29 |
| V31-00 | EJECUTADO | 24.0 — P0: reproduccion y auditoria de V30 | V30 reproduce desde sus artefactos y la cifra publicada es la correcta? | recomputado 0.6134471 / publicado 0.6130051 / desviacion 0.000442 / LB predic... | BASELINE CORREGIDO: el operativo de V31+ es 0.6134471 |
| V31-01 | EJECUTADO | 24.0 — Track A: taxonomia de URL sobre train | la prevalencia de URL se distribuye de forma monotona por edad? | prevalencia global 40.20% / 13-17 33.4% -> 65-xx 50.0% / twitter status 0 / U... | MONOTONA, pero la taxonomia de dominios es casi vacia en este dataset |
| V31-02 | EJECUTADO | 24.0 — Track A2/A4: representacion de URL | cambiar la representacion de la URL mejora sobre el texto crudo? | url_canal 0.4987321 (+0.000738, 3/5) / raw 0.4979941 / url_struct 0.4926034 (... | MARGINAL: el canal URL explicito no llega al gate de +0.0015 |
| V31-03 | EJECUTADO | 24.0 — Tracks B/D/E: canales ortografia, composicion y discurso | la composicion linguistica aporta senal que el TF-IDF de palabras no expresa? | ortografia+composicion+discurso 0.5048142 (+0.006820, 5/5) / discurso 0.50300... | PROMISING en el banco; posicion y forma_palabra CERRADAS |
| V31-04 | EJECUTADO | 24.0 — Track I2: el canal es complementario de V30? | el canal aporta informacion que V30 no tiene, o es redundante? | acuerdo 0.75228 / rescate 11.761 / danio 30.187 / neto -18.426 / error compar... | MIEMBRO DEBIL PERO DIVERSO: pasa a la integracion |
| V31-ENS-N1 | EJECUTADO | 24.0 — Track N1: el canal como cuarto miembro de V30 | el canal aporta al meta-espacio de V30 como miembro de la mezcla ponderada? | V30 congelado 0.6134471 / V30 reoptimizado 0.6137948 (+0.0003477) / V30+canal... | APORTE MARGINAL: muy por debajo del gate de +0.0015. No se abre grid |
| V31-ENS-N2 | EJECUTADO | 24.0 — Track N2: el canal como features del meta-espacio | el canal aporta como columnas en el meta-espacio de 279, donde el arbol puede... | solo279 0.6116801 / +canal 0.6116856 (+0.0000055) / +canal barajado 0.6113908... | APORTE MARGINAL: +0.0002948 sobre el control barajado, bajo el gate |
| V31-J | EJECUTADO | 24.0 — Track J: mineria guiada por errores de V30 | los errores de alta confianza de V30 apuntan a una falta de representacion? | errores 65.403 (38.62%) / alta confianza 9.780 (15.0%) / baja confianza 55.62... | DIAGNOSTICO: la familia que ataca esto, los ordinales, ya esta cerrada (H10,... |
| G14-AUDIT | DOCUMENTAL | 25.0 — G14 auditoria de informacion de entrada | existe algun campo de entrada legal, presente en prediccion y que ningun mode... | esquema: `id`, `text`, `age_range` en train y `id`, `text` en test. El ZIP co... | NOT_ACTIONABLE. No se rescata el `id` con modulo, hash, posicion ni patrones... |
| AUDIT-COVERAGE | DOCUMENTAL | 26.0 — auditoria de cobertura metodologica | queda algun mecanismo de aprendizaje genuinamente nuevo que pueda convertir i... | 39 mecanismos auditados: 23 USED, 9 CLOSED, 1 INVALIDATED, 1 NOT_ACTIONABLE,... | AGE_RESEARCH_CLOSED: no quedan mecanismos metodologicamente defendibles y no... |
| AGE-CLOSURE | DOCUMENTAL | 27.0 — acta de cierre formal de la investigacion | queda alguna hipotesis nueva con justificacion suficiente para reabrir la inv... | V30 congelado en 0.6134471 (particion alterna 0.6143347, leaderboard 0.63302)... | AGE_RESEARCH_CLOSED. El mapa queda auditado dos veces: contra el codigo del p... |
| AGE-PLUGIN-CANON | EJECUTADO | 28.0 — plug-in por clase en la particion canonica | mejora macro-F1 una regla de decision por clase en vez de argmax? | delta +0.0020172 / 5/5 folds / bootstrap IC95 [+0.0010143, +0.0029480] sin ce... | PROMISING en la canonica; pasa a la particion alternativa |
| AGE-PLUGIN-ALT | EJECUTADO | 28.0 — plug-in por clase en la particion alternativa (confirmatorio, congelado) | se mantiene la mejora cuando cambia el reparto de entrenamiento/validacion? | V30 0.6144120 -> plug-in 0.6164700 / delta +0.0020580 / 5/5 folds / bootstrap... | PROMISING_CONFIRMED. No FINALIST: el delta esta bajo +0.003 y la particion al... |
| AGE-PLUGIN-SUBMISSION | DOCUMENTAL | 29.0 — submission experimental del plug-in por clase | se transfiere al test la mejora de decision confirmada internamente? | submission PREPARED: 112892 filas, SHA256 29cbe6cca4312a65... / 7408 filas ca... | PREPARED. No se sube automaticamente y no se proyecta score. Si se obtiene, s... |
| AGE-PLUGIN-ADOPTED | DOCUMENTAL | 30.0 — transferencia externa y adopcion del plug-in | se transfiere al test la mejora de decision confirmada internamente? | LB 0.63575 frente a 0.63302 de V30 / delta +0.00273 / POSITIVE_EXTERNAL_TRANS... | ADOPTED. El leaderboard entra como evidencia final y no como funcion objetivo... |
| AGE-LING-P0-GAP | EJECUTADO | 30.1 — P0 - gap analysis linguistico por operacion | existe informacion linguistica estructural (secuencias tipadas) no explotada... | secuencias tipadas (POS, lemma_POS, morfologia, dependencias, entidades, text... | P2 condicionado a la resolucion de P1 |
| AGE-LING-P1-LEGALITY | DOCUMENTAL | 30.1 — P1 - auditoria de legalidad de herramientas NLP | se pueden usar anotadores preentrenados de POS/NER/sintaxis en AGE? | es_core_news_md (3.8.0, GPL-3.0, SHA256 889a2fd0...) = AMBIGUOUS: D-001 lo pe... | P2 gateado; opciones A/B/C elevadas a Carlos |
| AGE-LING-RESOLUTION | EJECUTADO | 30.1 — resolucion de legalidad del anotador (D-076) y registro de modelos | se autoriza el anotador y con que condiciones? | anotador ALLOWED (condicional): etiquetas si, vectores no; 6 condiciones; md... | contrastar POS-secuencia (md y lg) contra el campeon adoptado en P2 |
| AGE-LING-P2-PREREG | PREREGISTRO | 30.2 — P2 pre-registro congelado del canal POS-secuencia | pos_seq aporta sobre el campeon y mejora al integrarse por promedio neutral? | pre-registro escrito y congelado; sin resultados aun | ejecutar P2 tal cual |
| AGE-LING-P2-POS-SEQ | EJECUTADO | 30.2 — P2 ejecucion: canal POS-secuencia contra el campeon adoptado | pos_seq aporta sobre el campeon y mejora al integrarse por promedio neutral? | md: delta -0.0086357 (0/5) NEGATIVO / lg: delta -0.0084168 (0/5) NEGATIVO | aplicado el arbol pre-registrado; sin submission |
| AGE-LING-P6-PREREG | PREREGISTRO | 30.3 — P6 pre-registro congelado del canal NER estructural | la estructura de tipos de entidad aporta al campeon bajo un combinador anidad... | pre-registro escrito y congelado; sin resultados aun | ejecutar P6 tal cual |
| AGE-LING-P6-NER | EJECUTADO | 30.3 — P6 ejecucion: canal NER estructural (tipos de entidad) contra el campeon | la estructura de tipos de entidad aporta al campeon bajo un combinador anidad... | N1/lg: delta -0.0010228 (2/5) no pasa / N1/md: delta -0.0009257 (1/5) no pasa... | PROMISING -> segunda particion; si no, cierre documentado de NER |
| AGE-LING-P7-PREREG | PREREGISTRO | 30.4 — P7 pre-registro congelado del texto tipado (texto original + lemma_POS) | una representacion lexica tipada conserva el contenido TF-IDF y anade funcion... | pre-registro escrito y congelado; sin resultados aun | ejecutar P7 tal cual |
| AGE-LING-P7-TYPED | EJECUTADO | 30.4 — P7 ejecucion: texto tipado (texto original + lemma_POS) contra el campeon | una representacion lexica tipada anade informacion complementaria al campeon? | CLOSED / lg: delta -0.0007606 (1/5) / md: delta -0.0006894 (2/5) / T solo md... | cierre documentado; P5 se evaluara una sola vez |
| AGE-LING-REMAINING-PREREG | PREREGISTRO | 30.5 — pre-registro congelado de las familias restantes (P3/P4/P5) | los canales restantes aportan al campeon bajo el mecanismo evaluado? | pre-registro escrito y congelado; sin resultados aun | ejecutar el ciclo tal cual |
| AGE-LING-P3-LEMMA-POS | EJECUTADO | 30.5 — ciclo final: P3-LEMMA-POS contra el campeon | el canal aporta al campeon bajo el mecanismo evaluado? | md: delta -0.0009172 (2/5) / lg: delta -0.0009230 (1/5) | cierre documentado de la familia |
| AGE-LING-P4-MORFOLOGIA | EJECUTADO | 30.5 — ciclo final: P4-MORFOLOGIA contra el campeon | el canal aporta al campeon bajo el mecanismo evaluado? | md: delta -0.0011668 (1/5) / lg: delta -0.0010333 (2/5) | cierre documentado de la familia |
| AGE-LING-P5-DEPENDENCIAS | EJECUTADO | 30.5 — ciclo final: P5-DEPENDENCIAS contra el campeon | el canal aporta al campeon bajo el mecanismo evaluado? | md: delta -0.0009132 (1/5) / lg: delta -0.0005440 (2/5) | cierre documentado de la familia |
| V32-E1-PREREG | PREREGISTRO | 32.0 — V32-E1 pre-registro congelado del challenger con perdida surrogate de... | una perdida que optimiza macro-F1 directamente produce un challenger compleme... | pre-registro escrito y congelado; sin resultados aun | ejecutar V32-E1 tal cual |
| V32-E1 | EJECUTADO | 32.1 — V32-E1 ejecucion: challenger MLP con perdida surrogate de macro-F1 (3... | una perdida que optimiza macro-F1 directamente produce un challenger compleme... | A1 comb 0.6125567 (delta -0.0034165, 0/5 folds, IC95 [-0.0044246, -0.0023612]... | no puede pasar a segunda particion (control negativo barajado) |
| V32-E2-PREREG | PREREGISTRO | 32.2 — V32-E2 (P2) pre-registro congelado del challenger CatBoost + gemelo LGBM | CatBoost sobre matriz_279 aporta complementariedad al campeon mas alla de la... | pre-registro escrito y congelado; sin resultados aun | ejecutar V32-E2 tal cual |
| V32-E2-PREREG-ENMIENDA | ENMIENDA | 32.2 — Enmienda pre-ejecucion del prereg V32-E2: eval_metric TotalF1 -> Total... | corregir la metrica de parada a la media no ponderada (macro-F1) antes de eje... | enmienda registrada; sin resultados del experimento | ejecutar V32-E2 con la enmienda |
| V32-E2 | EJECUTADO | 32.2 — V32-E2 ejecucion: challenger CatBoost MultiClass 279 (A1 decisional) +... | CatBoost sobre matriz_279 aporta complementariedad al campeon mas alla de la... | A1 CatBoost comb 0.6148474 (delta -0.0011258, 1/5 folds, IC95 [-0.0019378, -0... | delta <= 0 -> se cierra P2; seria el ultimo candidato priorizado (P3 FTRL sol... |
| V32-E3-PREREG | PREREGISTRO | 32.3 — V32-E3 (P3) pre-registro congelado del challenger FTRL-Proximal + geme... | FTRL-Proximal sobre TF-IDF aporta complementariedad al campeon mas alla del S... | pre-registro escrito y congelado; sin resultados aun | ejecutar V32-E3 tal cual |
| V32-E3 | EJECUTADO | 32.3 — V32-E3 ejecucion: challenger FTRL-Proximal sobre TF-IDF por fold (A1 d... | FTRL-Proximal sobre TF-IDF aporta complementariedad al campeon mas alla del S... | A1 FTRL comb 0.6147759 (delta -0.0011973, 1/5 folds, IC95 [-0.0019881, -0.000... | si delta_int(A3) >= delta_int(A1) (precedencia) |
| V33-E1-PREREG | PREREGISTRO | 33.0 — V33-E1 pre-registro congelado del challenger FTRL-Proximal softmax mul... | el online proximal con la semantica correcta aporta complementariedad al camp... | pre-registro escrito y congelado; sin resultados aun | ejecutar V33-E1 tal cual |
| V33-E1-PREREG-ENMIENDA | ENMIENDA | 33.0 — Enmienda pre-ejecucion del prereg V33-E1: banda G1 y shuffle del control | la banda G1 congelada es alcanzable por un modelo sano? | enmienda registrada; sin resultados del experimento | ejecutar V33-E1 con la enmienda |
| V33-E1 | EJECUTADO | 33.0 — V33-E1 ejecucion: challenger FTRL-Proximal softmax multiclase de una p... | el online proximal con la semantica correcta (una pasada, softmax multiclase,... | puerta G1-G6 cerrada en el fold 0 (pasa = False); sin integracion, sin bootst... | cierre documentado NO_ATRIBUIBLE_BY_GATE; revisar la puerta con el arquitecto |
| V34-E1-PREREG | PREREGISTRO | 34.0 — V34-E1 pre-registro congelado: falsacion de la linea FTRL-Proximal con... | existe algun alpha con el que FTRL-Proximal supere el predictor de prior sobr... | pre-registro escrito y congelado; sin resultados aun | ejecutar V34-E1 tal cual |
| V34-E1-PREREG-ENMIENDA-2 | ENMIENDA | 34.0 — Enmienda pre-ejecucion 2: presupuesto del control A2 con partial_fit | fit(max_iter=1, warm_start=True) x 10 es un presupuesto de 10 pasadas? | enmienda registrada; sin resultados del experimento | ejecutar V34-E1 con la enmienda |
| V34-E1-PREREG-ENMIENDA-3 | ENMIENDA | 34.0 — Enmienda pre-ejecucion 3: token_pattern del vectorizador a la receta c... | el vectorizador congelado de V34 es el canonico del proyecto? | enmienda registrada; sin resultados del experimento | ejecutar V34-E1 con la enmienda |
| V34-E1 | EJECUTADO | 34.0 — V34-E1 ejecucion: falsacion de la linea FTRL-Proximal (A1 decisional m... | la linea FTRL-Proximal softmax falla por MECANISMO aunque se le den 10 pasada... | veredicto LINEA_ONLINE_CERRADA_POR_MECANISMO; puerta G0-G6 del fold 0 con pas... | LINEA_ONLINE_CERRADA_POR_MECANISMO: cerrar la familia por mecanismo; reabrirl... |
| V35-E1-PREREG | PREREGISTRO | 35.0 — V35-E1 (via B) pre-registro congelado: lambda1 > 0 sobre el nucleo de V34 | el umbral L1 acota la norma de FTRL y elimina la divergencia? | pre-registro escrito y congelado; sin resultados aun | ejecutar V35-E1 tal cual |
| V35-E1-PREREG-ENMIENDA-1 | ENMIENDA | 35.0 — Enmienda pre-ejecucion 1: verificacion del anclaje de lambda1 con el n... | el z de A1 puede reproducir el umbral que lo define? | enmienda registrada; sin resultados del experimento | ejecutar V35-E1 con la enmienda |
| V35-E1-PREREG-ENMIENDA-2 | ENMIENDA | 35.0 — Enmienda pre-ejecucion 2: las sondas (k)-(m) son de firma en el mini;... | pueden las sondas (k)-(m) reproducir identicamente los tres casos del gemelo... | enmienda registrada; sin resultados del experimento | ejecutar V35-E1 con las enmiendas 1 y 2 |
| V35-E1-PREREG-ENMIENDA-3 | ENMIENDA | 35.0 — Enmienda pre-ejecucion 3: token_pattern del vectorizador corregido al... | el token_pattern congelado es utilizable? | enmienda registrada; sin resultados del experimento | ejecutar V35-E1 con las enmiendas 1, 2 y 3 |
| V35-E1-PREREG-ENMIENDA-4 | ENMIENDA | 35.0 — Enmienda pre-ejecucion 4: restitucion de las 6 secciones de contexto d... | el pre-registro de V35 tiene todo lo que el runner lee? | enmienda registrada; sin resultados del experimento | ejecutar V35-E1 con las enmiendas 1, 2, 3 y 4 |
| V35-E1-PREREG-ENMIENDA-5 | ENMIENDA | 35.0 — Enmienda pre-ejecucion 5: una sola declaracion de asimetria A1 vs A2 | el pre-registro declara una o dos no-equivalencias estructurales? | enmienda registrada; sin resultados del experimento | ejecutar V35-E1 con las enmiendas 1 a 5 |
| V35-E1-INTENTO-1 | BLOQUEADO | 35.1 — Corrida real bloqueada por bug de ambito (UnboundLocalError en A2) | el reinicio one-shot con lambda1 > 0 acota la norma de FTRL? | DEFECTO_DE_IMPLEMENTACION: exit 1, 42.82 s; A1 fold 0 completo, A2 inalcanzab... | re-emision unica por DEFECTO_DE_IMPLEMENTACION, desde cero, previa correccion... |
| V35-E1 | EJECUTADO | 34.0 — V35-E1 ejecucion: falsacion de la linea FTRL-Proximal (A1 decisional m... | la linea FTRL-Proximal softmax falla por MECANISMO aunque se le den 10 pasada... | veredicto LINEA_ONLINE_CERRADA_POR_MECANISMO; puerta G0-G6 del fold 0 con pas... | LINEA_ONLINE_CERRADA_POR_MECANISMO: cerrar la familia por mecanismo; reabrirl... |
| D-087 | EJECUTADO | 35.2 — Hito: reevaluacion del objetivo LB 0.70 y fijacion del techo | es alcanzable LB 0.70 bajo las restricciones vigentes? | NO ALCANZABLE: techo de la curva +0.0567955 < objetivo +0.0626; techo defendi... | LB 0.70 se retira como objetivo; techo fijado en 0.642; C1 pasa a D-088 |
| V36-E1-PREREG | PREREGISTRO | 36.0 — V36-E1 (via C1-A) pre-registro congelado: transiciones de clase (L4) | las transiciones de clase de caracter aportan senal ortogonal que el word y c... | pre-registro escrito y congelado; sin resultados aun | ejecutar V36-E1 tal cual; C1-A es one-shot en cualquier caso |
| V36-E1-PREREG-ENMIENDA-1 | ENMIENDA | 36.0 — Enmienda pre-ejecucion 1: 3 defectos del pre-registro y 2 controles de... | los controles de especificidad y de barajado pueden fallar? | enmienda registrada; sin resultados del experimento | ejecutar V36-E1 con la enmienda 1 |
| V36-E1 | EJECUTADO | 36.1 — V36-E1 via C1-A: bloque de transiciones de clase de caracter (L4) | el perfil de tipo de transicion (letra->digito, espacio->puntuacion) aporta s... | nodo SENAL_EXISTE_INTEGRACION_NEGATIVA; banco +0.0032273 (5/5, IC95 [+0.00223... | se cierra L4 por el nodo del arbol; C1-A es ONE-SHOT: no se reintenta con otr... |
| V37-PREREG | PREREGISTRO | 37.0 — V37 pre-registro congelado: diagnostico P(POS/edad), fases A-E | existe un patron estable y no redundante en la distribucion condicional P(POS... | pre-registro escrito y congelado; 9/9 aserciones de integridad; sonda E9 decl... | ejecutar el diagnostico V37 tal cual, sin tocar el campeon |
| V37-DIAG-POS | EJECUTADO | 37.1 — V37 diagnostico P(POS/edad): perfil, estabilidad, cota de potencia y t... | P(POS/edad) tiene potency suficiente para un canal predictivo, o solo ruido? | A1(md)=0.1720532 frente a azar uniforme 0.1666667 (+0.0054) y suelo estricto... | DIAGNOSTICO_CERRADO; campeon V30+plug-in conservado intacto; el canal de tran... |
| AUD-INV | DOCUMENTAL | 99.0 — Auditoria V1-V37 fase 1: inventario verificable de las 195 filas | Cuantas filas del registro son ejecuciones reales y cuales documentos, y exis... | 151 EJECUTADO + 21 DOCUMENTAL + 12 PREREGISTRO + 10 ENMIENDA + 1 BLOQUEADO =... | FASE 1 CERRADA; inventario congelado y reproducible; alimenta las fases 2 y 4 |
| AUD-REV | DOCUMENTAL | 99.1 — Auditoria V1-V37 fases 2a-2c: revision independiente por especialidad | Las ~151 ejecuciones usaron la particion, el codigo, las metricas, los contro... | 20 defectos confirmados (4 media-alta, 9 media, 7 baja) y 0 rechazos del campeon | ACEPTADO con correcciones: acta D-090; multiplicidad cuantificada y registrad... |
| AUD-ENV-LG | DOCUMENTAL | 99.2 — Accion 1 de la auditoria: fijar es_core_news_lg en el entorno declarado | Es el anotador grande instalable desde el entorno declarado, como ya lo estab... | sha256 fijado 7c6c212715a12f31aacde3361754436945ff7376fb24cde57d0c277c9c9b050... | ADOPTADO; cierra el hallazgo N-02 de la auditoria D-090, que era el mas accio... |
| AUD-IC95-TECHO | DOCUMENTAL | 99.3 — Accion 2 de la auditoria: etiquetar el IC95 del techo como declarado,... | El IC95 [0.635, 0.653] del techo de LB 0.642 de D-087, esta medido o declarado? | IC95 etiquetado como rango declarado, no estimado; y la prueba del oraculo ma... | ADOPTADO; cierra el hallazgo A-05 de la auditoria D-090 |
| AUD-INNER-SPLIT | DOCUMENTAL | 99.4 — Accion 3 de la auditoria: medir el sesgo del inner split que elige los... | El inner split de Optuna deja pasar near-duplicates entre train interior y va... | inner ingenuo 3.907 % de solape de cluster frente a inner agrupado 0.000 % (d... | SESGO MATERIAL confirmado. El campeon es VALIDO porque su estimacion externa... |
| V38-TABULAR-BLEND | EJECUTADO | 38.0 — Ensamble de representaciones no lineales profundas (TabularResNet en G... | Puede una red residual tabular profunda sobre las 279 meta-features complemen... | CV 0.6166579 (delta +0.0006847, 5/5 folds positivos); Bootstrap IC95 [+0.0002... | ADOPTADO como CHAMPION OFICIAL (D-091). Archivo age_submission_v38_tabular_bl... |
| V39-TRIO-BLEND | EJECUTADO | 39.0 — Ensamble trimodal (V30 + TabNet Ordinal + HistGB) con pérdida de dista... | ¿Puede una pérdida con regularización de distancia ordinal esperada guiar a l... | CV 0.6168017 (delta +0.0008285 vs V30, 5/5 folds positivos); Kaggle LB 0.6358... | BASELINE SUPERADO. Archivo age_submission_v39_trio_blend.csv subido y puntuad... |
| V40-SE-TABULAR-BLEND | EJECUTADO | 40.0 — Ensamble pénta-modal (V30 + SE-TabNet + TabOrd + TabV38 + HistGB) con... | ¿Puede la atención condicional por muestra (Squeeze-and-Excitation) mitigar l... | CV 0.6175912 (delta +0.0016180 vs V30, 5/5 folds positivos); Bootstrap paread... | ADOPTADO COMO NUEVO CHAMPION OFICIAL (D-092). Archivo age_submission_v40_se_t... |
| V41-SWA-SE-BLEND | EJECUTADO | 41.0 — Ensamble quinteto con SWA-SE-TabularResNet (Stochastic Weight Averagin... | ¿Puede el promediado estocástico de pesos (SWA) suavizar la superficie de pér... | CV 0.6181119 (delta +0.0021387 vs V30, delta +0.0005207 vs V40, 5/5 folds pos... | ADOPTADO COMO NUEVO CHAMPION OFICIAL (D-093). Archivo age_submission_v41_swa_... |
| V42-SMOOTH-SWA-BLEND | EJECUTADO | 42.0 — Ensamble con Smooth-SWA-SE-TabularResNet (Label Smoothing 0.10 + Cosin... | ¿Puede el suavizado de etiquetas regularizar las normas de logits y evitar el... | CV 0.6184162 (delta +0.0024430 vs V30, delta +0.0003043 vs V41, 5/5 folds pos... | REGISTRADO como exploración de regularización por entropía y cuenca plana. V4... |
| V43-CRAMER-SMOOTH-BLEND | EJECUTADO | 43.0 — Ensamble con Cramer-SWA-SE-TabularResNet (Distancia de Cramér-von Mise... | ¿Puede la regularización de transporte óptimo 1D (distancia de Cramér sobre l... | CV 0.6184783 (delta +0.0025050 vs V30, delta +0.0000621 vs V42, 5/5 folds pos... | ADOPTADO COMO CO-CHAMPION OFICIAL (D-095). Archivo age_submission_v43_cramer_... |
| V44-BOUNDARY-EMD-ABLATIONS | EJECUTADO | 44.0 — Diagnóstico estadístico de fronteras y falsificación rigurosa de Wasse... | ¿Puede la regularización Wasserstein-1 (EMD) ponderada en fronteras y el enri... | CV 0.6183063 (delta -0.0001720 vs V43, 1/5 folds positivos, p=0.8550); POS de... | CERRADO documentado (D-096). V43 Cramer-Smooth Blend preservado como CHAMPION... |
| V45-ORACLE-FALSIFICATIONS | EJECUTADO | 45.0 — Diagnóstico del Techo Teórico Oracle (0.6935) y falsación empírica de... | ¿Existe margen en el espacio de representaciones de V43 y pueden arbitrarse l... | Techo Oracle medido en 0.693547; Discretización 1D degrada a 0.580048; Focal-... | RATIFICADO COMO CHAMPION INVICTO (D-097). V43 Cramer-Smooth Blend se mantiene... |
| V46-RANKGAUSS-SWA-TABULARNET | EJECUTADO | 46.0 — Diagnóstico de Asimetría Tabular y Falsación de RankGauss (QuantileTra... | ¿Puede la transformación normal cuantílica RankGauss mitigar el sesgo de 124... | Standalone RankGauss 0.6055514 (delta -0.0104219 vs V30); Blend 5-fold CV 0.6... | CERRADO documentado (D-098). V43 Cramer-Smooth Blend ratificado como CHAMPION... |
| V47-WEIGHTED-CRAMER-CLIPPED-TABNET | EJECUTADO | 47.0 — Diagnóstico de Outliers en Matriz Tabular, Recorte Robusto [-6, +6], W... | ¿Puede la pérdida cuadrática de Cramér ponderada en fronteras combinada con r... | Standalone V47 0.6110824; Blend reemplazo 0.6184996 (delta +0.0000743 vs V43,... | CERRADO documentado (D-099, D-100). V43 Cramer-Smooth Blend ratificado como C... |
| V48-GEOMETRIC-POE-BLEND | EJECUTADO | 48.0 — Ensamble Bayesiano Product-of-Experts en Espacio Logarítmico (Hinton 2... | ¿Puede el consenso bayesiano Product-of-Experts en log-probabilidades (operad... | CV 0.6186451 (delta +0.0002199 vs V43, 4/5 folds positivos); Bootstrap paread... | ADOPTADO COMO NUEVO CHAMPION OFICIAL INDISCUTIDO (D-101). Establece récord ab... |
| V49-CATBOOST-OBLIVIOUS-TREES | EJECUTADO | 49.0 — Diagnóstico Estadístico de Menciones (0.6761 vs 0.5043) y Falsación de... | ¿Puede CatBoost con cortes simétricos u obliviosos capturar patrones globales... | Standalone CatBoost 0.6066473; Ensamble 7-modal PoE 0.6185635 (delta -0.00008... | CERRADO documentado (D-102). V48 Geometric PoE Blend ratificado como CHAMPION... |
| V50-TRANSFORMER-DECOUPLED-POE | EJECUTADO | 50.0 — Falsación #27 (Tabular Transformer MHSA) y Falsación #28 (Class-Decoup... | ¿Pueden la auto-atención multi-cabeza (MHSA) sobre tokens conceptuales latent... | TabTransformer standalone Fold 0 0.6073432 (blend delta -0.0001058); Matriz W... | CERRADO documentado (D-103). V48 Geometric PoE Blend ratificado como CHAMPION... |
| V51-CONSISTENT-SMOOTHED-CRAMER | EJECUTADO | 51.0 — Unificación Teórica de Ground Truth Acumulado y Falsación de Consisten... | ¿Puede unificar la CDF objetivo suavizada consistente (1-eps)*I(y<=k) + eps*(... | Standalone 0.6128371 (delta -0.0000031 vs V43); Blend reemplazo 0.6186149 (de... | CERRADO documentado (D-104). V48 Geometric PoE Blend ratificado como CHAMPION... |
| V52-POWER-MEAN-DUAL-TRANSPORT-SNN | EJECUTADO | 52.0 — Operadores Power-Mean Falsación #30 Dual-Transport 7-Model PoE y Falsa... | ¿Pueden operadores continuos Power-Mean o transporte dual L1/L2 o redes SNN c... | PoE alpha->0 confirmado óptimo global; Dual-Transport 7-Model PoE rinde 0.618... | CERRADO documentado (D-105). V48 Geometric PoE Blend ratificado como CHAMPION... |
| V53-DENSE-SE-TABULARNET-POE | EJECUTADO | 53.0 — Dense-SE-TabularNet (Huang et al. CVPR 2017) con Squeeze-and-Excitation... | ¿Puede la concatenación multi-escala de características densas (DenseNet) pres... | Standalone 0.6111343; PoE 7-Model Blend CV 0.6188259 (delta +0.0001807 vs V48,... | ADOPTADO COMO NUEVO RÉCORD ABSOLUTO DE VALIDACIÓN CRUZADA LOCAL (D-106). Submi... |
| V54-HIERARCHICAL-POE-CONSENSUS | EJECUTADO | 54.0 — Consenso Bayesiano Jerárquico Multi-Pérdida (8-Model PoE) y Récord Abso... | ¿Puede la unificación de transporte Cramér simétrico y asimétrico ponderado co... | CV 0.6188887 (NUEVO RÉCORD HISTÓRICO ABSOLUTO DEFINITIVO, delta +0.0002436 vs ... | ADOPTADO COMO NUEVO RÉCORD HISTÓRICO ABSOLUTO DEFINITIVO DE VALIDACIÓN CRUZADA... |
| V55-CLEAN-OCTA-CONSENSUS | EJECUTADO | 55.0 — Consenso Bayesiano Octa-Model Limpio y Protocolo de Blindaje Definitivo... | ¿Puede la exclusión de redes subóptimas (DenseNet standalone 0.6111) y la adic... | CV 0.6182436 (delta +0.0002974 vs V48 canónico, 4/5 folds positivos); Bootstra... | ADOPTADO PROTOCOLO DE BLINDAJE DEFINITIVO (D-108). Submission oficial age_subm... |
| V56-NONLINEAR-POE-CONSENSUS | EJECUTADO | 56.0 — Consenso Bayesiano No Lineal Multi-Modal (9-Model PoE) y Récord Absolut... | ¿Puede la integración no lineal de 9 modelos de sesgo complementario en espaci... | CV 0.6189929 (NUEVO RÉCORD HISTÓRICO ABSOLUTO EN CV, delta +0.0003478 vs V48, ... | ADOPTADO COMO NUEVO RÉCORD HISTÓRICO ABSOLUTO DEFINITIVO DE VALIDACIÓN CRUZADA... |
| V57-MATHEMATICAL-SIMPLEX-CONCAVITY | EJECUTADO | 57.0 — Análisis Matemáticos y Estadísticos Avanzados: Geometría del Símplex y ... | ¿Demuestra el análisis analítico y numérico que el consenso bayesiano de 9 mod... | CV 0.6189929 (TECHO ESTACIONARIO CONFIRMADO); 9/9 modelos estacionarios ante v... | ADOPTADA DEMOSTRACIÓN DE TECHO ESTACIONARIO Y BLINDAJE ANTI-SHAKEUP (D-110). R... |

[DOC]

## Anexo B. Referencias del estado del arte

**Criterio de inclusión.** Solo se listan referencias que aparecen en el repositorio: la sección 54 del notebook `99_entrega_consolidado.ipynb` (bibliografía SOTA formal del proyecto), la bibliografía verificada `references.bib` (52 fichas: la corriente de dominio 2006-2021 y las 22 fichas de métodos SOTA incorporadas en el cierre de 2026, volcadas en `reports/state_of_art.csv`), las citas inline de `DECISIONS.md`, `RUNLOG.md`, `FINAL_REPORT.md`, `CHAMPION.md` y docstrings de `scripts/`, y los informes de `reports/` (incluidas la adenda de cierre SOTA de octubre de 2026 en `age_literature_review_2026.md` y el registro de métodos `age_literature_methods_registry.csv`). El DOI/URL se incluye solo cuando está en el repositorio; si no está, se marca `[VERIFICAR]`.

### B.1 Perfilado de autor y edad en texto (corriente de dominio)

Referencias que guiaron el diseño (según `FINAL_REPORT.md` §4 y `references.bib`), con su aporte concreto:

1. **Schler, J., Koppel, M., Argamon, S. y Pennebaker, J. W. (2006).** Effects of Age and Gender on Blogging. Proceedings of the AAAI Spring Symposium on Computational Approaches to Analyzing Weblogs (SS-06-03). URL: http://u.cs.biu.ac.il/~koppel/papers/springsymp-blogs-07.10.05-final.pdf — Uso: esquema de 6 rangos idéntico al de la competencia y bloque de features baratas (pronombres, palabras funcionales).
2. **Argamon, S., Koppel, M., Pennebaker, J. W. y Schler, J. (2007).** Mining the Blogosphere: Age, gender and the varieties of self-expression. First Monday, 12(9). DOI: 10.5210/fm.v12i9.2003 — Uso: discurso "outer-directed" vs "inner-directed" como marco del bloque de estilo.
3. **Mukherjee, A. y Liu, B. (2010).** Improving Gender Classification of Blog Authors. EMNLP 2010. URL: https://aclanthology.org/D10-1021/ — Uso: normalización weighted log-count ratio, base del NB-SVM del proyecto.
4. **Nguyen, D., Smith, N. A. y Rosé, C. P. (2011).** Author Age Prediction from Text using Linear Regression. LaTeCH@ACL-HLT 2011. URL: https://dongnguyen.nl/publications/nguyen-latech2011.pdf — Uso: regresión ordinal auxiliar sobre el punto medio del rango (E-015).
5. **Nguyen, D., Gravel, R., Trieschnigg, D. y Meder, T. (2013).** How Old Do You Think I Am? A Study of Language and Age in Twitter. ICWSM 2013. URL: https://dongnguyen.nl/publications/nguyen-icwsm2013.pdf — Uso: catálogo de features de Twitter (RT, URL, hashtag, alargamiento) y prioridad de los n-gramas de carácter.
6. **Nguyen, D., Trieschnigg, D., Doğruöz, A. S., Gravel, R., Theune, M., Meder, T. y de Jong, F. (2014).** Why Gender and Age Prediction from Tweets is Hard: Lessons from a Crowdsourcing Experiment. COLING 2014. URL: https://dongnguyen.nl/publications/nguyen-coling2014.pdf — Uso: cota de realismo del problema (los anotadores humanos no superan al clasificador).
7. **Sap, M., Park, G., Eichstaedt, J., Kern, M., Stillwell, D., Kosinski, M., Ungar, L. y Schwartz, H. A. (2014).** Developing Age and Gender Predictive Lexica over Social Media. EMNLP 2014. DOI: 10.3115/v1/D14-1121 — Uso: inspiración de los lexicones internos por regresión L2 sobre n-gramas.
8. **Flekova, L. y Gurevych, I. (2013).** Can We Hide in the Web? Large Scale Simultaneous Age and Gender Author Profiling in Social Media. PAN@CLEF 2013, CEUR-WS Vol-1179. URL: https://ceur-ws.org/Vol-1179/ — Uso: bloque explícito de superficie/legibilidad combinado con n-gramas.
9. **Hovy, D. y Fornaciari, T. (2018).** Increasing In-Class Similarity by Retrofitting Embeddings with Demographic Information. EMNLP 2018. DOI: 10.18653/v1/D18-1070 — Uso: idea de aumentar la similitud intra-clase con SVD/LSA sin embeddings (representación original prohibida por D-001).
10. **Morgan-Lopez, A. A., Kim, A. E., Chew, R. F. y Ruddle, P. (2017).** Predicting age groups of Twitter users based on language and metadata features. PLOS ONE, 12(8), e0183537. DOI: 10.1371/journal.pone.0183537 — Uso: fija el baseline solo-texto como cota inferior al no haber metadatos.
11. **Liu, Y., Singh, L. y Mneimneh, Z. (2021).** A Comparative Analysis of Classic and Deep Learning Models for Inferring Gender and Age of Twitter Users. DeLTA 2021. DOI: 10.5220/0010559500480058 — Uso: justifica la apuesta clásica (TF-IDF + modelos lineales) frente a deep learning.
12. **Rangel, F., Rosso, P., Koppel, M., Stamatatos, E. e Inches, G. (2013).** Overview of the Author Profiling Task at PAN 2013. CEUR-WS Vol-1179. URL: https://ceur-ws.org/Vol-1179/CLEF2013wn-PAN-RangelEt2013.pdf — Uso: calibración de techos con 3 clases de edad.
13. **Rangel, F., Rosso, P., Chugur, I., Potthast, M., Trenkmann, M., Stein, B., Verhoeven, B. y Daelemans, W. (2014).** Overview of the 2nd Author Profiling Task at PAN 2014. CEUR-WS Vol-1180. URL: https://ceur-ws.org/Vol-1180/CLEF2014wn-Pan-RangelEt2014.pdf — Uso: calibración con 5 clases de edad.
14. **Rangel, F., Celli, F., Rosso, P., Potthast, M., Stein, B. y Daelemans, W. (2015).** Overview of the 3rd Author Profiling Task at PAN 2015. CEUR-WS Vol-1391. URL: https://ceur-ws.org/Vol-1391/inv-pap12-CR.pdf — Uso: referencia de techos según el ancho de las clases.
15. **Rangel, F., Rosso, P., Verhoeven, B., Daelemans, W., Potthast, M. y Stein, B. (2016).** Overview of the 4th Author Profiling Task at PAN 2016: Cross-Genre Evaluations. CEUR-WS Vol-1609. URL: https://ceur-ws.org/Vol-1609/16090750.pdf — Uso: anticipa el riesgo de sobreajuste al género textual (aquí, retuits).
16. **Rangel, F., Rosso, P., Potthast, M. y Stein, B. (2017).** Overview of the 5th Author Profiling Task at PAN 2017: Gender and Language Variety Identification in Twitter. CEUR-WS Vol-1866. URL: https://ceur-ws.org/Vol-1866/invited_paper_11.pdf — Uso: rasgos dialectales para el análisis lingüístico.
17. **Rangel, F., Rosso, P., Montes-y-Gómez, M., Potthast, M. y Stein, B. (2018).** Overview of the 6th Author Profiling Task at PAN 2018: Multimodal Gender Identification in Twitter. CEUR-WS Vol-2125. URL: https://ceur-ws.org/Vol-2125/invited_paper_15.pdf — Uso: confirma que la imagen aporta poco frente al texto.
18. **Rangel, F. y Rosso, P. (2019).** Overview of the 7th Author Profiling Task at PAN 2019: Bots and Gender Profiling. CEUR-WS Vol-2380. URL: https://ceur-ws.org/Vol-2380/ — Uso: en PAN 2019 (ES) los char n-grams superaron a los embeddings y los RT no deben limpiarse.
19. **Sapkota, U., Bethard, S., Montes, M. y Solorio, T. (2015).** Not All Character N-grams Are Created Equal: A Study in Authorship Attribution. NAACL-HLT 2015. DOI: 10.3115/v1/N15-1010. URL: https://aclanthology.org/N15-1010/ — Uso: justifica char n-grams 2-5 con marcado de espacios (char_wb).
20. **Sapkota, U., Solorio, T., Montes-y-Gómez, M. y Ramírez-de-la-Rosa, G. (2013).** Author Profiling for English and Spanish Text. PAN@CLEF 2013, CEUR-WS Vol-1179. URL: https://ceur-ws.org/Vol-1179/ — Uso: n-gramas de carácter y palabra + SVM en inglés y español.
21. **López-Monroy, A. P., Montes-y-Gómez, M., Escalante, H. J., Villaseñor-Pineda, L. y Villatoro-Tello, E. (2013).** INAOE's Participation at PAN'13: Author Profiling Task. CEUR-WS Vol-1179. URL: https://ceur-ws.org/Vol-1179/ — Uso: atributos de segundo orden como antecedente.
22. **Verhoeven, B., Daelemans, W. y Plank, B. (2016).** TwiSty: A Multilingual Twitter Stylometry Corpus for Gender and Personality Profiling. LREC 2016. DOI: 10.63317/3yne5b7ve4p2 — Uso: referencia metodológica (no entrenable por D-001: sin datos externos).
23. **Verhoeven, B. y Daelemans, W. (2014).** CLiPS Stylometry Investigation (CSI) corpus: A Dutch corpus for the detection of age, gender, personality, sentiment and deception in text. LREC 2014. DOI: 10.63317/5ihmqb5uqzzs — Uso: referencia metodológica de etiquetas de edad.

Nota: la ficha de Rosenthal y McKeown (2011) aparece en la revisión, pero `references.bib` la marca "NO VERIFICADO... No citar en la entrega final sin confirmar" (ver B.9).

### B.2 Aprendizaje estadístico y arquitecturas neuronales

Núcleo metodológico de la sección 54 del notebook, complementado con citas inline de `DECISIONS.md` y `RUNLOG.md`:

1. **Ke, G. et al. (2017).** LightGBM: A Highly Efficient Gradient Boosting Decision Tree. NeurIPS 30, 3146-3154. URL: https://proceedings.neurips.cc/paper/2017/hash/6449f44a102fde848669bdd9eb6b76fa-Abstract.html — la ficha bibliográfica está en `references.bib`; el uso de la librería sí es central (V19-B1, V29, V30 y el árbol `HIERBA` de V20).
2. **Prokhorenkova, L., Gusev, G., Vorobev, A., Dorogush, A. V. y Gulin, A. (2018).** CatBoost: unbiased boosting with categorical features. NeurIPS 31, 6638-6648. URL: https://proceedings.neurips.cc/paper/2018/hash/14491b756b3a51daac41c24863285549-Abstract.html — Uso: challenger P2 (V32-E2) y falsación de árboles obliviosos (V49).
3. **Gorishniy, Y., Rubachev, I., Khrulkov, V. y Babenko, A. (2021).** Revisiting Deep Learning Models for Tabular Data. NeurIPS 34, 18932-18943. URL: https://proceedings.neurips.cc/paper/2021/hash/9d86d83f925f2149e9edb0ac3b49229c-Abstract.html — Uso: fundamento del TabularResNet de los componentes neuronales V38-V47.
4. **Hu, J., Shen, L. y Sun, G. (2018).** Squeeze-and-Excitation Networks. CVPR 2018, 7132-7141. DOI: 10.1109/CVPR.2018.00745. URL: https://openaccess.thecvf.com/content_cvpr_2018/html/Hu_Squeeze-and-Excitation_Networks_CVPR_2018_paper.html — Uso: compuertas SE en los bloques residuales (V40-V47).
5. **Izmailov, P., Podoprikhin, D., Garipov, T., Vetrov, D. y Wilson, A. G. (2018).** Averaging Weights Leads to Wider Optima and Better Generalization. UAI 2018, 876-885. URL: https://arxiv.org/abs/1803.05407 — Uso: SWA en V41-V47 (D-094).
6. **Szegedy, C., Vanhoucke, V., Ioffe, S., Shlens, J. y Wojna, Z. (2016).** Rethinking the Inception Architecture for Computer Vision. CVPR 2016, 2818-2826. DOI: 10.1109/CVPR.2016.308 — Uso: label smoothing 0.10 (V42-V47).
7. **Müller, R., Kornblith, S. y Hinton, G. E. (2019).** When does label smoothing help? NeurIPS 32, 4694-4703. URL: https://proceedings.neurips.cc/paper/2019/hash/f1748d6b0fd9d439f71450117eba2725-Abstract.html — Uso: cita complementaria del suavizado de etiquetas.
8. **Loshchilov, I. y Hutter, F. (2017).** SGDR: Stochastic Gradient Descent with Warm Restarts. ICLR 2017. URL: https://openreview.net/forum?id=Skq89Scxx — Uso: enfriamiento cosenoidal `CosineAnnealingLR` en V38-V47.
9. **Klambauer, G., Unterthiner, T., Mayr, A. y Hochreiter, S. (2017).** Self-Normalizing Neural Networks. NeurIPS 30, 971-980. URL: https://proceedings.neurips.cc/paper/2017/hash/5d44ee6f2c3f71b73125876103c8f6c4-Abstract.html — Uso: falsación SNN con SELU y AlphaDropout (V52-B, D-105).
10. **Hinton, G. E. (2002).** Training Products of Experts by Minimizing Contrastive Divergence. Neural Computation, 14(8), 1771-1800. DOI: 10.1162/089976602760128018 — Uso: consenso Product-of-Experts del campeón V48 (D-101).
11. **Heskes, T. (1998).** Selecting weighting factors in logarithmic opinion pools. NeurIPS 11, 266-272. URL: https://proceedings.neurips.cc/paper/1998/hash/9a96876e2f8f3dc4f3cf45f02c61c0c1-Abstract.html — Uso: antecedente del pooling logarítmico ponderado.
12. **Huang, G., Liu, Z., van der Maaten, L. y Weinberger, K. Q. (2017).** Densely Connected Convolutional Networks. CVPR 2017, 4700-4708. DOI: 10.1109/CVPR.2017.243 — Uso: red densa multi-escala de V53 (Dense-SE-TabularNet) y componente del consenso no lineal V56.

### B.3 Ordinalidad y transporte óptimo

1. **Frank, E. y Hall, M. (2001).** A simple approach to ordinal classification. ECML 2001, 145-156. Springer. DOI: 10.1007/3-540-44795-4_13 — Uso: miembro E-021a del ensamble (5 binarios `P(y>k)`; 0.5527) y descomposición pairwise (V28-03).
2. **Cao, W., Mirjalili, V. y Raschka, S. (2020).** Rank consistent ordinal regression for neural networks with application to age estimation. Pattern Recognition Letters, 140, 325-331. DOI: 10.1016/j.patrec.2020.11.008 — Uso: CORAL en V22/V23 (cerrado en V23-01).
3. **McCullagh, P. (1980).** Regression Models for Ordinal Data. Journal of the Royal Statistical Society: Series B (Methodological), 42(2), 109-127. DOI: 10.1111/j.2517-6161.1980.tb01109.x — citado como "McCullagh 1980" en `DECISIONS.md` (D-099) y `RUNLOG.md` (V47); la hipótesis fue falsada (0.5810757).
4. **Cramér, H. (1928).** On the composition of elementary errors. Scandinavian Actuarial Journal, 1928(1), 13-74. DOI: 10.1080/03461238.1928.10416862 — Uso: base de la distancia de Cramér-von Mises empleada como pérdida en V43/V46/V47/V51.
5. **von Mises, R. (1931).** Wahrscheinlichkeitsrechnung und ihre Anwendung in der Statistik und theoretischen Physik. F. Deuticke, Leipzig. — Uso: segunda mitad del nombre de la distancia.
6. **Rizzo, M. L. y Székely, G. J. (2016).** Energy distance. Wiley Interdisciplinary Reviews: Computational Statistics, 8(1), 27-38. DOI: 10.1002/wics.1375; y Székely & Rizzo (2013), JSPI 143(8), 1249-1272, DOI: 10.1016/j.jspi.2013.03.018 — Uso: conexión transporte óptimo / estadísticos de energía citada junto a la pérdida de Cramér.
7. **Frogner, C., Zhang, C., Mobahi, H., Araya, M. y Poggio, T. A. (2015).** Learning with a Wasserstein Loss. NeurIPS 28, 2053-2061. URL: https://proceedings.neurips.cc/paper/2015/hash/70efdf2ec9b086079795c442636b55fb-Abstract.html — Uso: marco general del transporte óptimo ordinal (pérdidas V43-V47, V52-A).

### B.4 Selección de términos, NB-SVM y optimización de F-medida

1. **Wang, S. y Manning, C. D. (2012).** Baselines and Bigrams: Simple, Good Sentiment and Topic Classification. ACL 2012, 90-94. URL: https://aclanthology.org/P12-2018/ — Uso: NB-SVM (log-count ratio + LinearSVC por clase) implementado como miembros del ensamble ADV05 (char+menciones, CV 0.5641) y ADV07 (palabra, CV 0.5254); ver nota de verificación en B.9.
2. **Monroe, B. L., Colaresi, M. P. y Quinn, K. M. (2008).** Fightin' Words: Lexical Feature Selection and Evaluation for Identifying the Content of Political Conflict. Political Analysis, 16(4), 372-403. DOI: 10.1093/pan/mpn018 — Uso: log-odds con prior de Dirichlet para términos distintivos por clase (`class_lexicon_scores`, análisis contrastivo de V45).
3. **Lipton, Z. C., Elkan, C. y Narayanaswamy, B. (2014).** Thresholding Classifiers to Maximize F1 Score. arXiv:1402.1892. URL: https://arxiv.org/abs/1402.1892 — Uso: base teórica del ajuste de umbral de la cadena plug-in.
4. **Puthiya Parambath, S. A., Usunier, N. y Grandvalet, Y. (2014).** Optimizing F-Measures by Cost-Sensitive Classification. NIPS 2014. URL: https://arxiv.org/abs/1505.00199 — Uso: reducción de F-medida a clasificación cost-sensitive (evaluada y descartada como clasificador; ver `FINAL_REPORT.md` §4).
5. **Waegeman, W., Dembczyński, K., Jachnik, A., Cheng, W. y Hüllermeier, E. (2014).** On the Bayes-optimality of F-measure maximizers. Journal of Machine Learning Research, 15, 3333-3388. URL: https://arxiv.org/abs/1310.4849 — Uso: análisis del optimizador de F1.
6. **Narasimhan, H., Kar, P. y Jain, P. (2015).** Optimizing Non-decomposable Performance Measures: A Tale of Two Classes. ICML 2015. URL: https://arxiv.org/abs/1505.06812 — Uso: esquemas estocásticos para F-medida.
7. **Eban, E., Schain, M., Mackey, A., Gordon, A., Saurous, R. A. y Elidan, G. (2017).** Scalable Learning of Non-Decomposable Objectives. AISTATS 2017. URL: https://arxiv.org/abs/1608.04802 — Uso: cotas sobre la matriz de confusión para optimizar F.

### B.5 Integridad experimental y fenómenos estadísticos

1. **Kaufman, S., Rosset, S., Perlich, C. y Stitelman, O. (2012).** Leakage in data mining: Formulation, detection, and avoidance. ACM Transactions on Knowledge Discovery from Data (TKDD), 6(4), 1-21. DOI: 10.1145/2382577.2382579 — Uso: marco de la auditoría de fuga por near-duplicados y de los folds agrupados (D-003).
2. **Simpson, E. H. (1951).** The Interpretation of Interaction in Contingency Tables. Journal of the Royal Statistical Society: Series B (Methodological), 13(2), 238-241. DOI: 10.1111/j.2517-6161.1951.tb00088.x — Uso: fundamento de la paradoja de Simpson demostrada en V52 con métricas armónicas (D-105).

### B.6 Aprendizaje online: la línea FTRL

1. **McMahan, H. B., Holt, G., Sculley, D. et al. (2013).** Ad Click Prediction: a View from the Trenches. Proceedings of the 19th ACM SIGKDD International Conference on Knowledge Discovery and Data Mining (KDD '13), 1222-1230. DOI: 10.1145/2487575.2488215 — citado como "McMahan 2013" en `DECISIONS.md` (D-083), `PROJECT_STATE.md` y `FINAL_REPORT.md`; implementado como candidato P3 (V32-E3, V33-V35) y cerrado por mecanismo (degeneró a 0.040169, bajo el suelo trivial 0.048542).

### B.7 Comparación con el techo publicado (acta D-087)

Referencias usadas para fijar el techo alcanzable y comparar con lo publicado (según `DECISIONS.md` D-087):

1. **Sadeghi, Akbari y Jaziriyan (2024).** ExaAUAC. Journal of Big Data. DOI: 10.1007/s44163-024-00145-0 — Twitter árabe, 181k perfiles, los mismos 6 bins: macro-F1 0.44 con BERT fine-tuned + metadatos (0.58 colapsando a 3 grupos). El campeón (0.616) queda +0.176 por encima.
2. **Cimino, A., Dell'Orletta, F. y Nissim, M. (2020).** Overview of the EVALITA 2020 Topic, Age, and Gender Attribute Identification from Tweets (TAG-it) Task. CEUR-WS Vol-2765, paper165. URL: http://ceur-ws.org/Vol-2765/paper165.pdf — edad macro-F1 0.416 (UmBERT) frente a 0.382 (SVM).
3. **Lupo, Bose, Habibi, Hovy y Schwarz (2024).** DADIT. LREC-COLING 2024. arXiv: 2403.05700 — hasta +53% de F1 en edad con XLM.
4. **O'Connor et al. (2024).** Revisión de métodos de edad y género desde Twitter. JMIR. DOI: 10.2196/72521 — rango publicado de F1 0.31-0.94; el extremo alto usa edad autodeclarada verificada, protocolo incompatible.
5. **Morgan-Lopez, A. A., Kim, A. E., Bibeau, C. y Dubin, M. (2017).** Predicting age groups of Twitter users: Comparison of survey vs Twitter metadata. American Journal of Public Health, 107(8). DOI: 10.2105/AJPH.2017.303831 — 74% de F1 con lenguaje + metadatos frente a 57% solo metadatos.
6. **Álvarez-Carmona, M. Á. et al. (2015).** INAOE's participation at PAN'15: Author Profiling task. CEUR-WS Vol-1391, paper 116. URL: https://ceur-ws.org/Vol-1391/116-CR.pdf — publica accuracy media combinada, no macro-F1 desglosado por edad.
7. **Rangel, F. et al. (2020).** Overview de PAN 2020. PMC7148014 — mismo problema de métrica agregada; la cifra desglosada de edad de PAN 2015 no pudo verificarse.

### B.8 Mecanismos revisados en 2026 y no adoptados (evidencia de cierre)

Citados en `reports/age_literature_review_2026.md`, `reports/age_literature_methods_registry.csv` y `reports/age_literature_gap_registry.csv` como evidencia para cerrar o mantener cerradas las familias C1-C5. Ninguno fue adoptado.

| Referencia (como aparece en el repo) | Papel en la decisión |
|---|---|
| Díaz y Marathe (CVPR 2019), SORD | Revisado en profundidad en C1 (2026): inspiró la combinación de etiquetas suaves con distancia ordinal adoptada en V42/V43; el candidato puro se cerró por desajuste de métrica nominal |
| Vargas et al. (Pattern Recognition 2022; Information Fusion 2023; Information Sciences 2023; ESWA 2024) | Familias de kernel ordinales; la CE categorica rinde mejor en métricas nominales |
| Castagnos et al. (COLING 2022), OLL | Pérdida ordinal de la familia C1 |
| Kasai et al. (arXiv 2405.11775) | Comparativa explícita: trade-off ordinal vs nominal en texto |
| Garg y Manwani (ACML 2020), Ord-NLL | Ruido ordinal explícito (C2); colapsa a target suave en expectativa |
| Yuan et al. (PRICAI 2023) y Takezaki et al. (IEEE Access 2026) | Evidencia adicional de Ord-NLL (C2) |
| ORCU (arXiv 2410.15658v4); stick-breaking (Neurocomputing 2020); Beckham y Pal (ICML 2017) | Regularización de forma unimodal (C3); ganancia de calibración, no de F1 |
| Dou et al. (COLING 2022), DePro; Ravfogel et al. (ACL 2020), INLP; Chew et al. (EACL Findings 2024), NFL; Seroussi et al. (CoNLL 2011) | Separación estilo/contenido (C4); la ganancia es estrictamente OOD |
| Wang et al. (AAAI 2023), CPL | Geometría ordinal de clases (C5); dependiente de un extractor prohibido |
| Natarajan et al. (NeurIPS 2013) | Desplazamiento de umbral por ruido; el intercepto óptimo ya estaba medido en 0 (P33) |

Detalles bibliográficos completos (título, venue, DOI/URL) de estas referencias: `[VERIFICAR]` en su mayoría; el repositorio las cita por autor, año y venue abreviado en los informes indicados.

### B.9 Referencias descartadas o marcadas por verificación

1. **Rosenthal, S. y McKeown, K. (2011).** Age Prediction in Blogs: A Study of Online Social Networks, Style and Personality. Informe técnico, Columbia University. DOI/URL: `[VERIFICAR]` — `references.bib` la marca "NO VERIFICADO: no se localizó DOI ni URL confirmada... No citar en la entrega final sin confirmar".
2. **Wang y Manning (2012)** (ficha completa en B.4): `reports/tables/dataset_provenance.md` registra que su ficha no se localizó en Crossref ni en el índice de ACL Anthology con las consultas realizadas; DOI/URL: https://aclanthology.org/P12-2018/.
3. **DOI 10.1007/s44163-026-01695-1**: citado como no verificable en D-087 (acceso bloqueado); no se incluye como referencia utilizable.

### B.10 Herramientas estándar citadas inline

Nota: fichas de la publicación original; no provienen del `references.bib` del repositorio. Se incorporaron en la segunda pasada de citas inline (2026-10-17) a las secciones del notebook.

1. **Salton, G. y Buckley, C. (1988).** «Term-Weighting Approaches in Automatic Text Retrieval». Information Processing & Management, 24(5), 513-523. — Uso: TF-IDF (00_portada, 01_fase1, fase3_5_residual_v13v17, fase4_1_v18_v29, fase4_3_fracasos, anexo_c_glosario).
2. **Fan, R.-E., Chang, K.-W., Hsieh, C.-J., Wang, X.-R. y Lin, C.-J. (2008).** «LIBLINEAR: A Library for Large Linear Classification». Journal of Machine Learning Research, 9, 1871-1874. — Uso: LinearSVC (03_fase3, fase3_5_residual_v13v17, fase4_3_fracasos, anexo_c_glosario).
3. **Bottou, L. (2010).** «Large-Scale Machine Learning with Stochastic Gradient Descent». COMPSTAT 2010, 177-186. — Uso: SGD (fase4_3_fracasos, anexo_c_glosario).
4. **Rennie, J. D. M., Shih, L., Teevan, J. y Karger, D. R. (2003).** «Tackling the Poor Assumptions of Naive Bayes Text Classifiers». ICML 2003, 616-623. — Uso: ComplementNB (anexo_c_glosario).
5. **Hoerl, A. E. y Kennard, R. W. (1970).** «Ridge Regression: Biased Estimation for Nonorthogonal Problems». Technometrics, 12(1), 55-67. — Uso: Ridge (anexo_c_glosario).
6. **Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M. y Duchesnay, É. (2011).** «Scikit-learn: Machine Learning in Python». Journal of Machine Learning Research, 12, 2825-2830. — Uso: scikit-learn (06_fase6).
7. **Akiba, T., Sano, S., Yanase, T., Ohta, T. y Koyama, M. (2019).** «Optuna: A Next-generation Hyperparameter Optimization Framework». KDD 2019, 2623-2631. — Uso: Optuna (04_fase4, fase4_1_v18_v29, fase2_5_auditoria_historial, fase5_4_matematicos, anexo_c_glosario).
8. **Efron, B. (1979).** «Bootstrap Methods: Another Look at the Jackknife». The Annals of Statistics, 7(1), 1-26. — Uso: bootstrap (01_fase1, 05_fase5, fase2_5_auditoria_historial, fase4_1_v18_v29, fase4_3_fracasos, fase4_5_v44_v52, anexo_c_glosario).
9. **Chen, T. y Guestrin, C. (2016).** «XGBoost: A Scalable Tree Boosting System». KDD 2016, 785-794. — Uso: XGBoost (ablación de V44, fase4_5_v44_v52).
10. **Zhang, H., Cissé, M., Dauphin, Y. N. y Lopez-Paz, D. (2018).** «mixup: Beyond Empirical Risk Minimization». ICLR 2018. — Uso: mixup (ablación de V44, fase4_5_v44_v52).
11. **Inoue, H. (2019).** «Multi-Sample Dropout for Accelerated Training and Better Generalization». arXiv:1905.09788. — Uso: Multi-Sample Dropout (ablación de V44, fase4_5_v44_v52).
12. **Arik, S. Ö. y Pfister, T. (2021).** «TabNet: Attentive Interpretable Tabular Learning». AAAI 2021, 6679-6687. — Uso: TabNet (mención «SE-TabNet» en fase4_5_v44_v52).
13. **Huang, X., Khetan, A., Cvitkovic, M. y Karnin, Z. (2020).** «TabTransformer: Tabular Data Modeling Using Contextual Embeddings». arXiv:2012.06678. — Uso: TabTransformer (V50, fase4_5_v44_v52).
14. **Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, Ł. y Polosukhin, I. (2017).** «Attention Is All You Need». Advances in Neural Information Processing Systems 30 (NIPS 2017), 5998-6008. — Uso: self-attention/MHSA (V50, fase4_5_v44_v52).
15. **Ioffe, S. y Szegedy, C. (2015).** «Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift». ICML 2015, 448-456. — Uso: BatchNorm (04b_fase4_nn, fase4_5_v44_v52, anexo_c_glosario).
16. **Jaccard, P. (1912).** «The Distribution of the Flora in the Alpine Zone». New Phytologist, 11(2), 37-50. — Uso: índice de Jaccard (02_fase2, fase2_5_auditoria_historial, fase5_4_matematicos, fase5_5_calibracion, anexo_c_glosario).
17. **Breiman, L. (1996).** «Bagging Predictors». Machine Learning, 24(2), 123-140. — Uso: bagging (V28-A, fase4_1_v18_v29).
18. **Hinton, G., Vinyals, O. y Dean, J. (2015).** «Distilling the Knowledge in a Neural Network». NIPS 2014 Deep Learning Workshop (arXiv:1503.02531). — Uso: destilación (V18-P5, fase4_1_v18_v29).
19. **Studer, S., Bui, T. B., Drescher, C., Hanuschkin, A., Winkler, L., Peters, S. y Müller, K.-R. (2021).** «Towards CRISP-ML(Q): A Machine Learning Process Model with Quality Assurance Methodology». Machine Learning and Knowledge Extraction, 3(2), 392-413. — Uso: metodología CRISP-ML(Q) (00_portada).
20. **Rocchio, J. J. (1971).** «Relevance Feedback in Information Retrieval». The SMART Retrieval System: Experiments in Automatic Document Processing, Prentice-Hall, 313-323. — Uso: centroides Rocchio (anexo_c_glosario).

[DOC]

## Anexo C. Glosario

Definiciones breves de los términos tal como los usa este proyecto. Cuando una definición cita una cifra, se indica su fuente.

- **macro-F1.** Media aritmética del F1 de las seis clases ($\text{macro-F1} = \frac{1}{6}\sum_{c} F1_c$); es la métrica de la competencia. Anclas: azar uniforme 0.1666667, suelo estricto de clase mayoritaria B0 0.0485421, primer baseline E-001 0.53444 y campeón V48 0.6186451 en CV / 0.63724 en LB (`experiments.csv`; `CHAMPION.md`).
- **rangos.** Las seis clases de edad `["13-17","18-24","25-34","35-49","50-64","65-xx"]` en el orden canónico de `src/config.py`, casi balanceadas por diseño (16.04%-17.04%, razón 1.062). Sus puntos medios (15/21/29.5/42/57/70) alimentan las cabezas ordinales (Nguyen, Smith y Rosé, «Author Age Prediction from Text using Linear Regression», LaTeCH@ACL-HLT 2011) (`07_checklist_verificacion.md`; `paquetes/01_miembros.md`).
- **OOF** (*out-of-fold*). Predicción de cada fila hecha por el modelo del fold en el que esa fila quedó fuera. Toda la evaluación del proyecto se apoya en el OOF; la convención canónica es pooled (0.6005017473 en V12) y se reporta también la media de folds (0.6004103057, efecto Jensen) (`experiments.csv` V17-P0-REPRODUCCION).
- **LOFO** (*leave-one-fold-out*). Protocolo estricto de ajuste y evaluación: para cada fold, el modelo se entrena con el train de los otros cuatro folds y predice el fold retenido; así se generan los scores del meta-modelo sin fuga (`paquetes/01_miembros.md` §1.4).
- **folds agrupados.** Partición canónica `StratifiedGroupKFold` de 5 folds (semilla 42) agrupada por clúster de near-duplicados (Kaufman, Rosset, Perlich y Stitelman, «Leakage in Data Mining: Formulation, Detection, and Avoidance», ACM Transactions on Knowledge Discovery from Data, 2012) (`dup_clusters.parquet`, 164.018 clústeres); tamaños 33868/33868/33867/33866/33867. Garantiza que ningún near-duplicado del train caiga en validación (`07_checklist_verificacion.md`; `paquetes/04_neuronales.md`).
- **near-duplicado / Jaccard.** Dos textos son near-duplicados si su índice de Jaccard (Jaccard, «The Distribution of the Flora in the Alpine Zone», New Phytologist, 1912) $J(A,B)=|A\cap B|/|A\cup B| \ge 0.8$ sobre tokens normalizados, con *blocking* por tokens raros. El 4.26% de las filas de test comparte clúster con train y el 92.8% de los clústeres tiene etiqueta única; un baseline de memorización da 0.0608 con split ingenuo y 0.0000 con el agrupado (`FINAL_REPORT.md` §3).
- **meta-modelo.** Segundo nivel que combina las salidas OOF de los miembros con features densas, estructuradas y supervisadas. En la cadena V11/V12 es `StandardScaler + LogisticRegression` sobre 90 features (49 scores + 6 del MLP + 35 densas) (`FINAL_REPORT.md` §7; `paquetes/01_miembros.md`).
- **miembros.** Los 14 modelos base de la cadena V12 (LinearSVC 300k/700k (Fan et al., «LIBLINEAR: A Library for Large Linear Classification», Journal of Machine Learning Research, 2008), LogReg, SGD (Bottou, «Large-Scale Machine Learning with Stochastic Gradient Descent», COMPSTAT 2010), ComplementNB (Rennie, Shih, Teevan y Karger, «Tackling the Poor Assumptions of Naive Bayes Text Classifiers», ICML 2003), SVM de hashtags, Ridge ordinal (Hoerl y Kennard, «Ridge Regression: Biased Estimation for Nonorthogonal Problems», Technometrics, 1970; Nguyen et al., 2011), softmax ordinal, LogReg de palabra, MLP densas+SVD, Frank & Hall (Frank y Hall, «A Simple Approach to Ordinal Classification», ECML 2001), Rocchio (Rocchio, «Relevance Feedback in Information Retrieval», The SMART Retrieval System, 1971), cinco SVM de pares y NB-SVM (Wang y Manning, «Baselines and Bigrams: Simple, Good Sentiment and Topic Classification», ACL 2012)), cuyas 78 columnas de scores abren la matriz de 279. En el arco final, los "miembros" del campeón son los 6 expertos del PoE de V48 (`paquetes/01_miembros.md`; `CHAMPION.md`).
- **cuartos.** Los cuatro componentes de V20, cada uno con peso 0.25: LogReg + MLP(64) + árbol LightGBM `HIERBA` (Ke et al., «LightGBM: A Highly Efficient Gradient Boosting Decision Tree», NeurIPS 2017) + cadena de cinco clasificadores binarios acumulativos (Frank y Hall, 2001). El OOF de V20 es el miembro de mayor peso dentro de V30 (0.3548) (`paquetes/03_v30_plugin.md` §4).
- **cadena de umbrales.** Cinco discriminadores binarios encadenados que ajustan $P(y \le k)$ frente a $P(y > k)$ sobre las 279 variables; la CDF resultante es monótona por construcción (máximo acumulado) y las probabilidades por clase salen de sus diferencias (V20-F3; `scripts/v20_f3_umbrales.py`).
- **plug-in.** Regla de decisión por clase $\hat{y} = \arg\max_c (w_c \cdot p_c)$ con el vector congelado `w = [1.0000, 1.1749, 1.2721, 1.1630, 1.0010, 0.8128]`, calibrado para macro-F1 en validación y congelado en pre-registro (Lipton, Elkan y Narayanaswamy, «Thresholding Classifiers to Maximize F1 Score», arXiv, 2014). Aporta +0.0025261 global (CV 0.6159732487657944) y +0.00273 en LB; cambia 7.408 filas de test (6.56%) sin tocar ninguna de alta confianza (`paquetes/03_v30_plugin.md`; `CHAMPION.md`).
- **PoE (product-of-experts).** Consenso multiplicativo en espacio logarítmico: $\log P(y|x) \propto \sum_m w_m \log P_m(y|x) - \log Z(x)$ (Hinton, «Training Products of Experts by Minimizing Contrastive Divergence», Neural Computation, 2002; Heskes, «Selecting Weighting Factors in Logarithmic Opinion Pools», NeurIPS 1998). A diferencia de la mezcla aritmética (tipo OR), el producto exige concordancia entre expertos (tipo AND). V48 usa 6 expertos con `w = [0.5628, 0.0832, 0.1724, 0.0754, 0.0513, 0.0549]`, logra +0.0002199 de CV (bootstrap p=0.0250; Efron, «Bootstrap Methods: Another Look at the Jackknife», The Annals of Statistics, 1979) y LB 0.63724 (`CHAMPION.md`; D-101).
- **SWA** (*stochastic weight averaging*). Promedio de los pesos visitados en las últimas K épocas de la trayectoria de optimización, seguido de recalibración de BatchNorm (Ioffe y Szegedy, «Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift», ICML 2015) (`update_bn`), para encontrar mínimos más planos y robustos. V41 promedia 8 checkpoints; V42-V47, 10; la variante SWA-SE es miembro del campeón (`paquetes/04_neuronales.md`; Izmailov, Podoprikhin, Garipov, Vetrov y Wilson, «Averaging Weights Leads to Wider Optima and Better Generalization», UAI 2018).
- **label smoothing.** Suavizado de las etiquetas objetivo en la entropía cruzada (`label_smoothing=0.10` en V42-V47) para evitar el colapso de margen en fronteras adyacentes; convive con la pérdida ordinal en la misma función de pérdida (`paquetes/04_neuronales.md`; Szegedy, Vanhoucke, Ioffe, Shlens y Wojna, «Rethinking the Inception Architecture for Computer Vision», CVPR 2016; Müller, Kornblith y Hinton, «When Does Label Smoothing Help?», NeurIPS 2019); su formulación ordinal (SORD; Díaz y Marathe, «Soft Labels for Ordinal Regression», CVPR 2019) se revisó en la fase de falsación de 2026 (sección 4.5).
- **distancia de Cramér.** Distancia de Cramér-von Mises (transporte óptimo $L_2$ 1D sobre la CDF): $\mathcal{L}_{\text{Cramer}} = \sum_{k=0}^{K-2}(F_p(k) - \mathbb{I}(y \le k))^2$. Penaliza cuadráticamente el desplazamiento acumulado de la masa de probabilidad y favorece distribuciones predictivas unimodales y centradas; es la pérdida del miembro Cramer-SWA del campeón (lambda 0.50) (Cramér, «On the composition of elementary errors: First paper: Mathematical deductions», Scandinavian Actuarial Journal, 1928; von Mises, «Wahrscheinlichkeitsrechnung und ihre Anwendung in der Statistik und theoretischen Physik», Franz Deuticke, 1931) (`paquetes/04_neuronales.md` §V43; D-095).
- **ordinal.** Todo modelo o pérdida que explota el orden de los rangos. El proyecto probó: Ridge sobre puntos medios (Hoerl y Kennard, 1970; E-015; Nguyen et al., 2011), softmax ordinal, Frank & Hall (E-021a, 0.5527; Frank y Hall, 2001), la cadena de umbrales (V20, miembro del campeón), pérdidas de distancia esperada (V39-V40; Frogner, Zhang, Mobahi, Araya y Poggio, «Learning with a Wasserstein Loss», NeurIPS 2015), Cramér (V43; Cramér, 1928), EMD (V44, falsada) y enlace acumulativo (V47, falsado con 0.5810757; McCullagh, 1980) (`experiments.csv`; `RUNLOG.md`).
- **CORAL.** Regresión ordinal rank-consistent con pesos compartidos y K-1 salidas sigmoidales (Cao, Mirjalili y Raschka, «Rank consistent ordinal regression for neural networks with application to age estimation», Pattern Recognition Letters, 2020). Implementada y testeada en V23-00 y falsada en V23-01 (0.4837017, delta -0.126077, 0/5 folds); familia cerrada (`experiments.csv`; `reports/age_literature_review_2026.md`).
- **FTRL.** Aprendizaje lineal online Follow-The-Regularized-Leader proximal (McMahan et al., 2013), tercera vía de la reapertura V32-V35 sobre TF-IDF (Salton y Buckley, «Term-Weighting Approaches in Automatic Text Retrieval», Information Processing & Management, 1988). La variante OvR degeneró (0.040169, bajo el suelo trivial 0.048542) y la familia se cerró por mecanismo en V34-V35 sin integrar nada (`DECISIONS.md` D-083/D-086; `experiments.csv`).
- **NB-SVM.** Modelo lineal sobre features de Naive Bayes: se multiplica la matriz TF-IDF por el log-count ratio de cada clase y se ajusta un LinearSVC por clase (Fan et al., 2008; Wang y Manning, 2012). Entra como miembro del ensamble (ADV05 char+menciones, CV 0.5641; ADV07 de palabra, 0.5254), no como modelo principal (`paquetes/01_miembros.md`; `FINAL_REPORT.md` §4).
- **TF-IDF (word / char_wb).** Ponderación frecuencia de término por frecuencia inversa de documento (Salton y Buckley, 1988). `word` usa n-gramas de palabras (1-2, `min_df=3`, 300k features); `char_wb` usa n-gramas de carácter con frontera de palabra (3-6, 300k; Sapkota, Bethard, Montes y Solorio, «Not All Character N-grams Are Created Equal: A Study in Authorship Attribution», NAACL-HLT 2015), la representación ganadora junto al canal de menciones (E-008, 0.5557) (`experiments.csv`; `FINAL_REPORT.md` §6).
- **SVD.** Descomposición en valores singulares truncada (`TruncatedSVD(n_components=64)`, ajustada in-fold) para comprimir la representación dispersa antes del MLP denso E-019b (0.4071 individual; aporta al ensamble vía E-020, 0.5851) (Hovy y Fornaciari, «Increasing In-Class Similarity by Retrofitting Embeddings with Demographic Information», EMNLP 2018) (`paquetes/01_miembros.md`).
- **Optuna.** Optimizador bayesiano de hiperparámetros (TPE) (Akiba et al., «Optuna: A Next-generation Hyperparameter Optimization Framework», KDD 2019) usado en V29 (150 trials) y V30 (75 trials) sobre LightGBM, con holdout interior por fold. La auditoría D-091 midió que el inner split ingenuo dejaba 3.907% de solape de clúster frente a 0.000% del agrupado (`experiments.csv`; `DECISIONS.md` D-091).
- **bootstrap pareado.** Remuestreo con reemplazo de las diferencias por fila entre dos modelos (n=1000) para estimar el IC95 y el p-valor de un delta sin supuestos paramétricos (Efron, 1979). Ejemplo: V48 vs V43, IC95 [+0.0000007, +0.0004095], p=0.0250 (`CHAMPION.md`; D-101).
- **IC95.** Intervalo de confianza al 95% de un delta (percentiles del bootstrap pareado). Ejemplo: V40 vs V30, [+0.0008660, +0.0023367], p<1e-6 (`experiments.csv`; D-092).
- **pre-registro.** Documento congelado antes de ejecutar un experimento, con hipótesis, brazos, métrica, gate y reglas de parada; todo cambio posterior se registra como enmienda pre-ejecución. El inventario V1-V37 registra 12 pre-registros y 10 enmiendas (`experiments.csv`; D-083).
- **gate de adopción.** Umbral pre-declarado para adoptar un cambio. En las fases finales: delta de CV $\ge$ +0.0003 con IC95 excluyendo cero (V38-V43); en fases previas, +0.0015 o +0.0030 según el pre-registro. La auditoría D-090 cuantificó el riesgo de multiplicidad: con SE ~0.00188, el gate de +0.0015 produce ~32 falsos positivos esperados y el de +0.0030 ~8; Bonferroni para 151 pruebas exigiría +0.0147 (`DECISIONS.md` D-090; `07_checklist_verificacion.md` §5).
- **submission.** CSV de 112.892 filas con columnas exactas `[id, age_range]`, mismo orden de ids que el test, sin NaN y con las seis clases presentes; se valida con listas de checks (V48: 8/8; pipeline del plug-in: 10/10) y se sella con SHA256 (V48: `bf2554571ca6676195af66a053e28411fc1cfc9e3946fd95bac1cae0ce71df62`) (`paquetes/03_v30_plugin.md`; `CHAMPION.md`).
- **leaderboard público / privado.** El público muestra el puntaje parcial durante la competición; el privado retiene ~30% de las filas de test y decide la clasificación final (defensa ante shakeup) (`scripts/v47_01_weighted_cramer_clipped_swa_tabularnet_5folds.py`; `CHAMPION.md`).
- **CV corregida.** Métrica de validación recomputada contra los artefactos cuando una cifra publicada no era reconstruible. Caso canónico (D-066): V30 publicado 0.6130051 (pesos por fold no persistidos) frente a 0.6134471 (pesos medios, lo realmente enviado), que es el baseline operativo de V31+; el campeón con plug-in congelado global es 0.6159732487657944 (`DECISIONS.md` D-066; `reports/v37_diagnostico_pos.md`).
- **Dense-SE-TabularNet.** Red densa tabular de V53: cada bloque se concatena con todos los siguientes (192 -> 320 -> 448 -> 576), con compuertas SE y la pérdida de Cramér consistente-suavizada; standalone 0.6111343 (Huang et al., CVPR 2017; Hu et al., CVPR 2018; sección 4.6).
- **Power-Mean.** Familia continua de operadores de agregación entre la media geométrica (alpha -> 0) y la aritmética (alpha = 1); la auditoría V57 demostró que el consenso V56 habita el óptimo de concavidad para alpha en [-0.15, +0.15] (sección 4.6).
- **selección dual (V48/V56).** Protocolo de envío ratificado por V57: V48 (mejor leaderboard, 0.63724) y V56 (mejor validación local, 0.6189929) conviven como candidatos oficiales; la submission de este notebook corresponde a V48.